# ML-03 · V4.03 FINAL FULL RESEARCH
## Экономика, инфраструктура, демографическая и миграционная динамика муниципалитетов

**Единый исследовательский Google Colab-ноутбук, четыре этапа, один запуск («Выполнить все»).**

Источник: приватный GitHub `AntonDozhdikov/Demograf_sovetnik`, фиксированный commit Stage I, авторизация через Colab Secrets `TRIAD-RU-Colab`. Вызов GPU не требуется. Исследование не реконструирует отсутствующие наблюдения и не выдаёт диагностические ассоциации за причинный эффект.

Промежуточные операции, ошибки источников и проверки фиксируются в журнале и результирующих таблицах; в интерфейсе поддерживается **ровно один сквозной прогресс-бар**. Выгрузки: полный технический Evidence/модели в GitHub и компактный научный ZIP со сводными Excel/CSV, графиками, причинным протоколом, JSON QA и копией ноутбука.

**Порядок запуска:** все ячейки строго сверху вниз. Для самостоятельной оценки ATT указываются пути к реальным верифицированным реестрам воздействий и стабильных границ муниципалитетов. При отсутствии таких источников причинная процедура выдаёт проверяемый статус `NOT_IDENTIFIED`, не создаёт искусственных данных и не останавливает полный прогнозный эксперимент.

### Этап I / IV. Источники, инвентаризация и муниципальная панель

Проверка 52 актов Stage I по SHA-256, дополнительных источников БДМО, кодов показателей, территориальной и временной привязки. Формирование воспроизводимой панели с паспортами единиц измерения, исключений и неопределённости.

In [ ]:
# 01 · ML03 V4.03 · FULL по умолчанию, строгие межъячеечные контракты
CFG = {
    'MODE':'FULL',                    # 'SMOKE' для быстрой проверки / 'FULL' для исследования
    'PUBLISH_FULL_PASS':False,       # строгая научная публикация запрещена без прохождения FULL QA
    'REPO':'AntonDozhdikov/Demograf_sovetnik',
    'SOURCE_REF':'58473f995440daac295ca32532959f101bfa14ea',
    'HANDOFF':'handoff/07_v2',
    'SEGMENT':'stage2/ml03_factors',
    'VERSION':'stage2-ml3-v4.0.3-final',
    'WORKDIR':'/content/ml03_factors',
    'SEED':1729,
    'DOWNLOAD_ALL_STAGE1':True,     # все 52 SHA-verified Stage I в локальный cache; JOIN только подходящих единиц
    'MAX_SINGLE_ASSET_GB':8.0,
    'MIN_DISK_FREE_GB':6.0,
    'MAX_EVIDENCE_GB':1.9,
    'HORIZONS':[1,3],
    'TARGETS':['migration_per_1000','population_change_pct'],
    'FEATURE_PUBLICATION_LAG':1,
    'DEV_YEARS':2,
    'TEST_YEARS':2,
    'MIN_TRAIN_YEARS':3,
    'MIN_TRAIN_ROWS':120,
    'MIN_TEST_ROWS':40,
    'MIN_BOOTSTRAP_REGIONS':8,
    'MAX_FEATURES_PER_GROUP':12,
    'MAX_FEATURES_FROM_ONE_ASSET':5,
    'MAX_CANDIDATE_CODES_PER_ASSET':20,
    'MAX_CATALOG_CODES_PER_ASSET':None,  # не скрывать нерассмотренные коды
    'MAX_CODE_SLICES_PER_ASSET_DOMAIN':None,
    'MIN_ECON_DOMAINS':2,
    'MIN_SOCIAL_DOMAINS':2,
    'MIN_CANONICAL_TEST_YEARS':2,
    'MIN_SPATIAL_MATCHES':30,
    'SPATIAL_PUBLICATION_YEAR':None,
    'SPATIAL_PUBLICATION_EVIDENCE':'',
    'SPATIAL_BOUNDARY_CROSSWALK_VERIFIED':False,
    'SPATIAL_BOUNDARY_CROSSWALK_EVIDENCE':'',
    'MIN_SPATIAL_CANONICAL_TEST_ROWS':40,
    'MAX_FUTURE_MUTATION_ROWS':30,
    'MIN_BOUNDARY_SENSITIVITY_REGIONS':8,
    'MIN_FE_ENTITY_OBSERVATIONS':3,
    'MIN_INDICATOR_MUNICIPALITIES':25,
    'MIN_INDICATOR_OBS_TRAIN':100,
    'MIN_INDICATOR_YEARS_TRAIN':2,
    'MIN_RECENT_MUNICIPALITY_YEARS':70,
    'CATALOG_RECENT_WINDOW':5,
    'MAX_DIMENSIONAL_SCANS':1500,  # если превышено — NOT_EXHAUSTIVE, не научный PASS
    'MAX_DIMENSIONAL_SLICES_PER_CODE':2000,  # размерные комбинации проверяются SQL-фильтром до лимита,
    'MIN_MUNICIPAL_YEAR_SUPPORT':60,
    'TARGET_LABEL_AVAILABILITY_LAG':1,
    'MAX_NDFL_SLICE_VALUES':400,  # ограничение числа тестируемых допустимых разрезов, не отказ от всего кода
    'FAIL_FAST_SQL':True,
    'SQL_PREFLIGHT_REAL_CODES':True,
    'EXPLORE_SECONDARY_MIGRATION_WINDOW':True,
    'PRIMARY_TEST_WINDOWS':{'migration_per_1000':[2021,2022],'population_change_pct':[2022,2023]},
    'MAX_FEATURE_STALENESS_YEARS':3,
    'ENABLE_REGIONAL_ECONOMIC_CONTEXT':True,
    'REGIONAL_ECONOMIC_MAX_FEATURES':5,
    'REGIONAL_ECONOMIC_MIN_REGIONS':10,
    'REGIONAL_CONTEXT_IS_NOT_MUNICIPAL_G1':True,
    'MIN_DISTINCT_EVALUATION_YEARS':2,
    'MAX_CONFLICT_SHARE':0.002,
    'CORRELATED_PROXY_CUTOFF':0.95,
    'MODEL_FAMILIES':['RIDGE','HGB','CATBOOST','LIGHTGBM','ELASTICNET'],
    'SEEDS_FINAL':[1729,1733,1747,1753,1777,1783,1799,1801,1811,1823],
    'SEED_REPETITION_REPORT':True,
    'ENABLE_CHECKPOINTS':True,
    'BOOTSTRAPS':499,
    'MAX_IMPORTANCE_ROWS':1200,
    'MAX_IMPORTANCE_FEATURES':22,
    'MAX_SHAP_ROWS':260,
    'SMOKE_MUNICIPALITIES':500,
    'DUCKDB_MEMORY':'6GB',
    'DUCKDB_THREADS':4,
    'EXCEL_MAX_ROWS_PER_SHEET':110000,
    # Source enrichment: additive official BDMO sections outside the immutable Stage I 52.
    'GITHUB_COLAB_SECRET':'TRIAD-RU-Colab',  # имя секрета, не значение; резерв — GITHUB_TOKEN
    'RUN_CAUSAL_EXPERIMENT':True,
    'VERIFIED_BOUNDARY_HISTORY_INPUT':'', # optional path, source/years must be independently certified
    'VERIFIED_PUBLICATION_VINTAGES_INPUT':'', # optional indicator/year public-release receipts
    'VERIFIED_SPATIAL_VINTAGE_INPUT':'', # optional historical geocodes and geometry crosswalk  # исторический источник, без реконструкции
    'G5_RETROSPECTIVE_EXPLORATORY':True,  # без подмены прогноза прошлым знанием
    'H5_CALENDAR_AUDIT':True,  # все невозможные пятилетние задачи документируются
    'SOURCE_PUBLICATION_AUDIT':True,  # даты публикации из документированных источников, без домыслов
    'ENABLE_EXTERNAL_BDMO':True,
    'BDMO_EXTERNAL_MAX_SOURCE_GB':12.0,
    'BDMO_EXTERNAL_MAX_UNPACKED_GB':20.0,
    'BDMO_EXTERNAL_MAX_MEMBERS_PER_SECTION':150,
    # Research-grade explainable ensemble: selected solely by chronological DEV.
    'ADVANCED_ENABLE':True,
    'ADVANCED_REQUIRE_ALL_FAMILIES':True,
    'ADVANCED_N_JOBS':4,
    'ADVANCED_MAX_CANDIDATE_FAMILIES':6,
    'ADVANCED_ENSEMBLE_K':3,
    'ADVANCED_EXPORT_MEMBERS':True,
    'ADVANCED_WEIGHT_RIDGE':0.02,
    'ADVANCED_EXPLAIN_ENABLE':True,
    'ADVANCED_SHAP_ROWS':180,
    'ADVANCED_BACKGROUND_ROWS':120,
    'ADVANCED_SHAP_TOLERANCE':3e-4,
    'ADVANCED_EXPLAIN_BOOTSTRAPS':299,
    'ADVANCED_PERM_ROWS':700,
    'ADVANCED_PERM_REPEATS':3,
    'ADVANCED_PDP_TRAIN_ROWS':450,
    'ADVANCED_PDP_TOP':4,
    'ADVANCED_PDP_BINS':13,
    'CAUSAL_TREATMENT_REGISTER':'',       # CSV с подтвержденными данными о вмешательствах; без синтетики
    'CAUSAL_BOUNDARY_REGISTER':'',        # CSV с документально подтвержденными границами
    'ARCHIVES_INCLUDE_RAW_SOURCE_CACHE':False, # raw Stage I cache can be very large
    'AUTO_COLAB_DOWNLOAD':True,  # только компактный научный архив; тяжёлый не скачивается
    'GITHUB_TECHNICAL_PUBLISH':True, # тяжелый Evidence + модели только в приватный Release репозитория
    'RESEARCH_ARCHIVE_MAX_MB':45,   # лимит научного ZIP для быстрой загрузки
    'RESEARCH_LOCAL_SHAP_SAMPLE':1500,
    'GITHUB_VERIFY_ASSET_SHA_IF_AVAILABLE':True,
    'SCIENTIFIC_RELEASE_ALLOWED':True,  # решает строгий gate после реального FULL; GitHub запись отдельно выключена
}
assert CFG['MODE'] in ('SMOKE','FULL')
assert all(h>0 and isinstance(h,int) for h in CFG['HORIZONS'])
assert CFG['FEATURE_PUBLICATION_LAG']>=1
# Версия и режим отображаются одним общим индикатором в следующей ячейке.

#### Проверка вычислительной среды, зависимости и единый виджет выполнения.

In [ ]:
# 02 · ENVIRONMENT, MINIMAL COLAB DEPENDENCIES, AUDIT LOG
import sys, os, shutil, platform, subprocess, importlib.util, json, time, hashlib, re, math, itertools, warnings, zipfile, io, base64, traceback, statistics, datetime, collections, urllib.parse, ast, zlib, pprint
from pathlib import Path
WORK=Path(CFG['WORKDIR']);WORK.mkdir(parents=True,exist_ok=True)
for d in ('cache','evidence','charts','models'): (WORK/d).mkdir(exist_ok=True)
usage=shutil.disk_usage(str(WORK))
# Среда и свободное место пишутся в технический журнал, а не в вывод ноутбука.
ensure_disk=usage.free/2**30>=CFG['MIN_DISK_FREE_GB']
if not ensure_disk:raise RuntimeError('INSUFFICIENT_COLAB_DISK_CAPACITY')
NEEDED={'duckdb':'duckdb>=1.1,<2','pyarrow':'pyarrow>=17,<24','pandas':'pandas>=2.1,<3',
        'numpy':'numpy>=1.26,<3','scipy':'scipy>=1.11,<2','sklearn':'scikit-learn>=1.4,<2',
        'matplotlib':'matplotlib>=3.8,<4','openpyxl':'openpyxl>=3.1,<4',
        'nbformat':'nbformat>=5.9,<6','requests':'requests>=2.31,<3',
        'ipywidgets':'ipywidgets>=8,<9','joblib':'joblib>=1.3,<2'}
OPTIONAL={'catboost':'catboost>=1.2,<2','lightgbm':'lightgbm>=4,<5',
          'linearmodels':'linearmodels>=6,<9','shap':'shap>=0.46,<1','xgboost':'xgboost>=2.0,<4'}
missing=[v for k,v in NEEDED.items() if importlib.util.find_spec(k) is None]
if missing:
    result=subprocess.run([sys.executable,'-m','pip','-q','install',*missing],capture_output=True,text=True)
    if result.returncode:raise RuntimeError('MANDATORY_PIP_DEPENDENCY_FAILURE')
optional_missing=[v for k,v in OPTIONAL.items() if importlib.util.find_spec(k) is None]
if optional_missing:
    result=subprocess.run([sys.executable,'-m','pip','-q','install',*optional_missing],capture_output=True,text=True)
    if result.returncode:
        if CFG['MODE']=='FULL' and CFG['ADVANCED_ENABLE'] and CFG['ADVANCED_REQUIRE_ALL_FAMILIES']:
            raise RuntimeError('ADVANCED_FULL_DEPENDENCY_INSTALL_FAILED: '+result.stderr[-1000:])
        pass  # Уточнение о недоступности — в статусе экспериментов, без засорения интерфейса.
import numpy as np, pandas as pd, duckdb, requests, pyarrow as pa, pyarrow.parquet as pq
import matplotlib;matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge,ElasticNet
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
from openpyxl import Workbook,load_workbook
from openpyxl.styles import Font,PatternFill,Alignment
from openpyxl.utils import get_column_letter
import nbformat,joblib
pd.options.mode.copy_on_write=True
STATUS={};WARNINGS=[];TRACE=WORK/'ml03_technical.log';START=time.monotonic()
if TRACE.exists():TRACE.unlink()
def log(msg,severity='INFO'):
    text=str(msg)
    text=re.sub(r'(?i)(?:gh'+'p_|github'+'_pat_)[a-z0-9_]+','[REDACTED]',text)
    with open(TRACE,'a',encoding='utf-8') as f:
        f.write(datetime.datetime.now(datetime.timezone.utc).isoformat()+f' [{severity}] '+text+'\n')
def mark(name,status,reason=''):
    STATUS[name]={'status':status,'reason':str(reason)[:480]}
    log(name+': '+status+': '+str(reason)[:480])
    if status not in ('PASS','NOT_FEASIBLE','CAUSAL_NOT_IDENTIFIED'):
        WARNINGS.append({'step':name,'status':status,'reason':str(reason)[:480]})
def ensure(ok,msg):
    if not ok:raise AssertionError(msg)
def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
log('Environment OK: Python='+sys.version.split()[0]+' CPU='+str(os.cpu_count())+' free_disk_gib='+str(round(usage.free/2**30,1)))
# Fail before expensive work if the required runtime is not available.
ensure(pd.__version__.split('.')[0] in ('2',), 'PANDAS_MAJOR_VERSION_NOT_TESTED') if 'ensure' in globals() else None



# ML-03 V4.03: РОВНО ОДНО визуальное окно прогресса (ipywidgets.HTML).
# Виджет отображается один раз в cell 02. Межъячеечные обновления меняют его value,
# не создавая новых FloatProgress, tqdm, display() или многократных выводов Colab.
from IPython.display import display as _single_display
try:
    import ipywidgets as _widgets
except ImportError:
    _widgets=None
_PHASE_BOUNDS={'ENV': (0, 26), 'AUTH': (26, 47), 'INPUT': (47, 187), 'BDMO': (187, 210), 'INVENTORY': (210, 238), 'SCAN': (238, 301), 'FACTORS': (301, 310), 'PANEL': (310, 357), 'TASKS': (357, 383), 'DEV': (383, 472), 'TRAIN': (472, 582), 'ADVANCED': (582, 684), 'STATS': (684, 723), 'SHAP': (723, 786), 'SENSITIVITY': (786, 820), 'EXCEL': (820, 831), 'EVIDENCE': (831, 847), 'FINAL': (847, 850), 'CAUSAL_LOAD': (850, 870), 'CAUSAL_REG': (870, 884), 'CAUSAL_PANEL': (884, 898), 'CAUSAL_EST': (898, 927), 'CAUSAL_EXPORT': (927, 940), 'ARCHIVE_RESEARCH': (940, 962), 'ARCHIVE_TECH': (962, 995), 'DELIVERY': (995, 1000)}
_PHASE_TITLES={'ENV': 'I/IV · Среда', 'AUTH': 'I/IV · GitHub и каталог', 'INPUT': 'I/IV · 52 источника', 'BDMO': 'I/IV · Дополнительные БДМО', 'INVENTORY': 'I/IV · Инвентаризация', 'SCAN': 'I/IV · Предметный отбор', 'FACTORS': 'I/IV · Готовность факторов', 'PANEL': 'I/IV · Панель', 'TASKS': 'II/IV · План эксперимента', 'DEV': 'II/IV · Подбор моделей DEV', 'TRAIN': 'II/IV · Обучение и тест', 'ADVANCED': 'II/IV · Ансамбли', 'STATS': 'III/IV · Статистика и абляции', 'SHAP': 'III/IV · Объяснения SHAP', 'SENSITIVITY': 'III/IV · Утечки и устойчивость', 'EXCEL': 'III/IV · Книга Excel', 'EVIDENCE': 'IV/IV · Evidence и контроль', 'FINAL': 'IV/IV · Проверка основного контура', 'CAUSAL_LOAD': 'IV/IV · Причинный анализ: вход', 'CAUSAL_REG': 'IV/IV · Реестр воздействий', 'CAUSAL_PANEL': 'IV/IV · Причинный дизайн', 'CAUSAL_EST': 'IV/IV · Причинные оценки', 'CAUSAL_EXPORT': 'IV/IV · Причинный протокол', 'ARCHIVE_RESEARCH': 'IV/IV · Научные таблицы и графики', 'ARCHIVE_TECH': 'IV/IV · GitHub Evidence', 'DELIVERY': 'IV/IV · Итоговая QA и архив'}
class _OneProgress:
    def __init__(self):
        self.n=0;self.total=1000;self.desc='Инициализация';self.postfix='';self._last=0.;self._last_percent=-1
        self.widget=_widgets.HTML(value=self._html()) if _widgets else None
        if self.widget is not None:_single_display(self.widget)
    def _html(self):
        import html
        pct=int(round(100*self.n/self.total));name=html.escape(self.desc)
        sub=html.escape(self.postfix)
        return ('<div style="max-width:920px;padding:15px 17px;border:1px solid #d4dce5;'
          'border-radius:13px;background:#0b1c31;color:#f5f9ff;font:14px Arial,sans-serif">'
          '<div style="display:flex;justify-content:space-between;gap:14px;margin-bottom:10px">'
          '<b>ML-03 V4.03 · Объединённый запуск · '+name+'</b><b>'+str(pct)+'%</b></div>'
          '<div role="progressbar" aria-valuenow="'+str(pct)+'" aria-valuemin="0" aria-valuemax="100"'
          ' style="height:15px;border-radius:9px;background:#35445b;overflow:hidden">'
          '<div style="width:'+str(pct)+'%;height:100%;background:linear-gradient(90deg,#21a5af,#61d8b0);'
          'border-radius:9px"></div></div><div style="margin-top:8px;color:#a8c5df;font-size:12px">'
          +sub+'</div></div>')
    def update(self,increment):
        self.n=min(self.total,max(0,self.n+int(increment)));self.refresh()
    def set_description_str(self,s,refresh=False):self.desc=str(s);self.refresh(force=refresh)
    def set_postfix_str(self,s,refresh=False):self.postfix=str(s);self.refresh(force=refresh)
    def refresh(self,force=False):
        now=time.monotonic();pct=int(100*self.n/self.total)
        if not force and now-self._last<.3 and pct==self._last_percent:return
        self._last=now;self._last_percent=pct
        if self.widget is not None:self.widget.value=self._html()
        elif force or pct%10==0:log(f'PROGRESS {pct}% {self.desc} {self.postfix}')
    def close(self):self.refresh(force=True)
_PROGRESS_BAR=_OneProgress();_PROGRESS_LAST=None
PROGRESS_LEDGER=[]
def progress(phase,done=1,total=1,detail=''):
    global _PROGRESS_LAST
    if phase not in _PHASE_BOUNDS:raise KeyError('PROGRESS_UNKNOWN_PHASE_'+str(phase))
    start,end=_PHASE_BOUNDS[phase];total=max(1,int(total));done=max(0,min(int(done),total))
    goal=start+int((end-start)*done/total)
    goal=max(goal,_PROGRESS_BAR.n)  # монотонная шкала, определяется фактическим выполнением этапов
    detail=str(detail).replace('\n',' ')[:85]
    state=(_PHASE_TITLES[phase],done,total,detail)
    if state!=_PROGRESS_LAST:
        _PROGRESS_BAR.desc=state[0];_PROGRESS_BAR.postfix=f'{done}/{total} · {detail}'
        _PROGRESS_LAST=state
    if goal>_PROGRESS_BAR.n:_PROGRESS_BAR.update(goal-_PROGRESS_BAR.n)
    else:_PROGRESS_BAR.refresh()
    if done==total:PROGRESS_LEDGER.append({'phase':phase,'completed':done,'total':total,'bar_value':_PROGRESS_BAR.n})
    return None

def quiet_print(*args,**kwargs):
    # Отладочные сообщения доступны в ml03_technical.log, но не создают сотни строк в Colab.
    try:log(' '.join(str(a) for a in args))
    except Exception:pass

# Every warning goes to the technical log, not the live notebook output. Exceptions remain visible.
def _ml03_warning_to_log(message, category, filename, lineno, file=None, line=None):
    log(f'{category.__name__}: {message} [{Path(filename).name}:{lineno}]','WARN')
warnings.showwarning=_ml03_warning_to_log

progress('ENV',1,1,'Среда и зависимости подготовлены')

# Не маскируем научные ошибки: подавляется только многословный pandas PerformanceWarning.
warnings.filterwarnings('ignore', message='DataFrame is highly fragmented.*', category=pd.errors.PerformanceWarning)

#### Безопасная загрузка только по SHA-256: токен читается из Secrets и не включается в артефакты.

In [ ]:
# 03 · SECURE AUTH / READ-ONLY PREFLIGHT
from getpass import getpass
# Colab secret name is TRIAD-RU-Colab; never write its VALUE to logs, ZIP or notebook.
_TOKEN_KEY=str(CFG.get('GITHUB_COLAB_SECRET','TRIAD-RU-Colab'))
TOKEN=''
try:
    from google.colab import userdata
    for _secret in (_TOKEN_KEY,'GITHUB_TOKEN'):
        try:TOKEN=str(userdata.get(_secret) or '').strip()
        except (KeyError,ValueError,Exception):TOKEN=''
        if TOKEN:break
except ImportError:pass
if not TOKEN:
    TOKEN=(os.environ.get('TRIAD_RU_COLAB') or os.environ.get('GITHUB_TOKEN') or '').strip()
if not TOKEN:TOKEN=getpass("GitHub token (скрытый ввод, если секрет Colab не найден): ").strip()
if not TOKEN: raise RuntimeError("BLOCKED_AUTH: требуется токен приватного GitHub")
API=f"https://api.github.com/repos/{CFG['REPO']}"
SESSION=requests.Session()
SESSION.headers.update({"Authorization":"Bearer "+TOKEN,"Accept":"application/vnd.github+json",
                        "X-GitHub-Api-Version":"2022-11-28","User-Agent":"Demograf-sovetnik-ML03"})
# В трассировки не передаются объекты HTTP-ошибок с заголовками/URL авторизации.
def api(method,endpoint,*,expected=(200,),**kwargs):
    url=endpoint if endpoint.startswith('https://') else API+endpoint
    r=SESSION.request(method,url,timeout=kwargs.pop('timeout',(20,90)),**kwargs)
    if r.status_code not in expected:
        code="ACCESS_DENIED" if r.status_code in (401,403) else "NOT_FOUND_OR_NO_RIGHTS" if r.status_code==404 else "HTTP_FAILURE"
        raise RuntimeError(f"{code}: GitHub HTTP {r.status_code}; resource={endpoint.split('?')[0][:130]}")
    return r
repo_obj=api('GET','').json()
ensure(repo_obj.get('private') is True,"Ожидался приватный репозиторий; настройки не меняем")
source_sha=api('GET',f"/commits/{CFG['SOURCE_REF']}").json()['sha']
SOURCE_COMMIT_SHA=source_sha  # CFG не изменяется после ячейки 01: воспроизводимый источник

def repo_bytes(path):
    obj=api('GET',f"/contents/{urllib.parse.quote(path,safe='/')}?ref={source_sha}").json()
    if obj.get('content'):
        return base64.b64decode(obj['content'])
    # Для крупных git blobs получаем Git blob, не доверяя download_url с временными параметрами.
    blob=api('GET','/git/blobs/'+obj['sha']).json()
    if blob.get('encoding') != 'base64': raise RuntimeError("UNKNOWN_GIT_BLOB_ENCODING")
    return base64.b64decode(blob['content'])
RECEIPT=json.loads(repo_bytes(CFG['HANDOFF']+'/publication_receipt.json'))
raw_index=repo_bytes(CFG['HANDOFF']+'/dataset_index.json')
receipt_index=next((x for x in RECEIPT['files'] if x['path'].endswith('/dataset_index.json')),None)
ensure(receipt_index is not None and hashlib.sha256(raw_index).hexdigest()==receipt_index['sha256'],
       'HANDOFF_INDEX_SHA_MISMATCH')
INDEX=json.loads(raw_index); ASSETS=INDEX['assets']
ensure(len(ASSETS)==52 and len({a['asset_id'] for a in ASSETS})==len(ASSETS),
       'Нарушен контракт индекса: ожидаются 52 уникальных массива')
ensure(all(a['sha256_verified'] and a['locator']['type'] in ('git','release') for a in ASSETS),
       'Индекс содержит неверифицированные/неизвестные источники')
mark('PREFLIGHT_CATALOG','PASS',f"52 актива; immutable source SHA {source_sha[:12]}")
quiet_print(f'GitHub READ: OK; handoff={INDEX["version"]}; sources={len(ASSETS)}; SHA={source_sha[:12]}')
progress('AUTH',1,1,'Каталог и commit подтверждены')

#### Загрузка первичных 52 активов и независимая проверка физических Parquet.

In [ ]:
# 04 · V3.00 · VERIFIED 52 SOURCE INVENTORY, MUNICIPAL/REGIONAL/ORGANIZATION GRAIN
import tempfile
source_receipts=[]; DOWNLOAD_QA=[]; LOCAL_ASSETS={}
def safe_remote_download(asset):
    """SHA-verified cache; GitHub git/release; no untrusted URL in logs."""
    name=hashlib.sha256(asset['asset_id'].encode()).hexdigest()[:22]+'.parquet'
    dst=WORK/'cache'/name
    if dst.exists() and digest(dst)==asset['sha256']:
        if asset['asset_id'] not in {x['asset_id'] for x in source_receipts}:
            source_receipts.append({'asset_id':asset['asset_id'],'sha256':asset['sha256'],'local_bytes':dst.stat().st_size,'source_ref':source_sha,'cache_hit':True})
        LOCAL_ASSETS[asset['asset_id']]=str(dst)
        return dst
    dst.unlink(missing_ok=True); tmp=dst.with_suffix('.part');tmp.unlink(missing_ok=True)
    loc=asset['locator']; total=0
    try:
        if loc['type']=='git':
            path=urllib.parse.quote(loc['path'],safe='/')
            obj=api('GET',f'/contents/{path}?ref={source_sha}').json()
            blob=api('GET','/git/blobs/'+obj['sha'],timeout=(30,240)).json()
            ensure(blob.get('encoding')=='base64','GIT_BLOB_ENCODING_UNSUPPORTED')
            buf=base64.b64decode(blob['content'])
            with open(tmp,'wb') as f:f.write(buf)
            total=len(buf)
        else:
            u=API+'/releases/assets/'+str(int(loc['asset_id']))
            r=SESSION.get(u,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,1200))
            if r.status_code!=200:
                r.close();raise RuntimeError('RELEASE_ACCESS_HTTP_'+str(r.status_code))
            if 'json' in r.headers.get('Content-Type','').lower():
                j=r.json();r.close();download=j.get('browser_download_url','')
                host=urllib.parse.urlparse(download)
                ensure(host.scheme=='https' and host.hostname in ('github.com','api.github.com'), 'INVALID_RELEASE_HOST')
                r=SESSION.get(download,stream=True,timeout=(30,1200))
                if r.status_code!=200 or 'json' in r.headers.get('Content-Type','').lower():
                    code=r.status_code;r.close();raise RuntimeError('RELEASE_BINARY_HTTP_'+str(code))
            try:
                with open(tmp,'wb') as f:
                    for block in r.iter_content(4*1024*1024):
                        if block:
                            total+=len(block)
                            ensure(total<=int(CFG['MAX_SINGLE_ASSET_GB']*2**30),'ASSET_LIMIT_EXCEEDED')
                            f.write(block)
            finally:r.close()
        ensure(digest(tmp)==asset['sha256'],'INPUT_SHA_MISMATCH_'+asset['asset_id'])
        tmp.replace(dst);LOCAL_ASSETS[asset['asset_id']]=str(dst)
        source_receipts.append({'asset_id':asset['asset_id'],'sha256':asset['sha256'],'local_bytes':dst.stat().st_size,'source_ref':source_sha,'cache_hit':False})
        return dst
    except Exception:tmp.unlink(missing_ok=True);raise

def get_asset(aid):
    a=next((v for v in ASSETS if v['asset_id']==aid),None)
    if a is None:raise KeyError('UNKNOWN_HANDOFF_ASSET '+aid)
    return a

SOURCE_CLASSES={
 'G1':['02::labor','02::socio1','02::socio2','02::socio3','02::5ndfl','02::7ndfl'],
 'G2':['02::housing','02::construction','06::BDMO6::BDMO6_member00::full'],
 'G3':['02::education','02::social'],
 'G4':['02::budget'],
 'G5':['06::TRANSPORT','06::D04::D04_member00::spatial_reference::derived']}
def source_group(a):
    return next((g for g,pp in SOURCE_CLASSES.items() if any(a['asset_id'].startswith(p) for p in pp)),'CONTEXT_ONLY')
BASE_ASSET='01::0'
# Дополнительный обход всего реестра Stage I: длинные муниципальные наборы независимо от
# папки/источника; запрещено превращать региональные и организационные записи в ОКТМО.
SOURCE_ADAPTER_AUDIT=[]
ECONOMY_MANIFEST=json.loads(repo_bytes('manifests/02_economy_full_v1.json'))
ECONOMY_SOURCE_MANIFEST_QA=pd.DataFrame([{'source_key':k,'source':v.get('source'),
 'rows_in_manifest':v.get('rows'),'column_count':len(v.get('columns') or []),
 'has_object_level':('object_level' in (v.get('columns') or [])),
 'has_municipal_key':bool({'oktmo','object_oktmo'}&set(v.get('columns') or [])),
 'report_grain':'REQUIRES_CONTENT_VALIDATION' if 'object_level' in (v.get('columns') or []) else 'MUNICIPAL_LONG_CANDIDATE' if 'oktmo' in (v.get('columns') or []) else 'UNKNOWN'}
 for k,v in ECONOMY_MANIFEST.get('files',{}).items()])

def eligible_municipal_long_schema(a):
    cc=set(a.get('columns_manifest') or [])
    ok=('oktmo' in cc or 'object_oktmo' in cc) and {'indicator_code','indicator_name','indicator_value','year'}.issubset(cc)
    return bool(ok)

def classify_asset_for_scan(a):
    declared=source_group(a)
    if declared!='CONTEXT_ONLY':return declared
    if eligible_municipal_long_schema(a):return 'DISCOVERY_MULTIDOMAIN'
    return 'CONTEXT_ONLY'

FACTOR_ASSETS=[a for a in ASSETS if a['asset_id']!='01::0' and
               classify_asset_for_scan(a)!='CONTEXT_ONLY']
# An education organization's OKTMO is a LOCATION OF ORGANIZATION, not proof of
# municipal total or municipal population coverage. Keep a full exclusion receipt.
ORGANIZATION_GRAIN_PREFIXES=('04::E02::','05::E04::','05::E05::')
ORGANIZATION_GRAIN_EXCLUSIONS=[{'asset_id':a['asset_id'],
    'reason':'ORGANIZATION_LEVEL_NOT_MUNICIPAL_AGGREGATE; NO_VALIDATED_ORGANIZATION_TO_MUNICIPAL_CONCORDANCE',
    'status':'EXPLICITLY_EXCLUDED_FROM_MUNICIPAL_MODEL'} for a in ASSETS
    if a['asset_id'].startswith(ORGANIZATION_GRAIN_PREFIXES)]
UNADAPTED_MUNICIPAL_WIDE=[a['asset_id'] for a in ASSETS
 if {'oktmo','year'}.issubset(set(a.get('columns_manifest') or [])) and
 not eligible_municipal_long_schema(a) and a['asset_id']!='01::0'
 and not a['asset_id'].startswith(ORGANIZATION_GRAIN_PREFIXES)]
SOURCE_ADAPTER_AUDIT=[{'asset_id':a['asset_id'],'declared_group':source_group(a),
 'scan_group':classify_asset_for_scan(a),'schema_type':('LONG_MUNICIPAL' if eligible_municipal_long_schema(a)
 else 'NON_MUNICIPAL_OR_WIDE_UNVERIFIED'),
 'has_oktmo':int(bool({'oktmo','object_oktmo'}&set(a.get('columns_manifest') or []))),
 'decision':('SCAN' if a in FACTOR_ASSETS else 'INDEX_ONLY_NO_VALIDATED_MUNICIPAL_GRAIN')}
 for a in ASSETS]

DOWNLOAD_ASSETS=(ASSETS if (CFG['MODE']=='FULL' and CFG['DOWNLOAD_ALL_STAGE1']) else
    [get_asset(BASE_ASSET)]+FACTOR_ASSETS)
if CFG['MODE']=='SMOKE':
    allow={'01::0','02::5ndfl::00::00','02::7ndfl::00::00',
           '02::housing::00::00','02::education::00::00','02::budget::00::00',
           '02::labor::00::00','02::social::00::00','02::construction::00::00',
           '02::socio1::00::00','02::socio2::00::00','02::socio3::00::00',
           '06::D04::D04_member00::spatial_reference::derived'}
    DOWNLOAD_ASSETS=[a for a in DOWNLOAD_ASSETS if a['asset_id'] in allow]
progress('INPUT',0,len(DOWNLOAD_ASSETS),'Сверка файлов с SHA-256')
input_map=pd.DataFrame([{'asset_id':a['asset_id'],'source':a['source_id'],
 'rows_manifest':a['rows_actual'],'group':source_group(a),
 'storage':a['locator']['type'],'status':'INDEXED','sha256':a['sha256']} for a in ASSETS])

# All 52 assets are indexed, including 03-05 (not misinterpreted as municipal observations).
# FULL downloads every asset by default, while only validated relevant sources enter ML-03 JOINs.
for ai,a in enumerate(DOWNLOAD_ASSETS,1):
    try:
        p=safe_remote_download(a)
        DOWNLOAD_QA.append({'asset_id':a['asset_id'],'source_group':source_group(a),'status':'SHA_PASS',
                             'bytes':p.stat().st_size,'error':''})
    except Exception as ex:
        DOWNLOAD_QA.append({'asset_id':a['asset_id'],'source_group':source_group(a),'status':'ERROR',
                             'bytes':0,'error':type(ex).__name__+': '+str(ex)[:120]})
        log('download '+a['asset_id']+': '+type(ex).__name__,'WARN')
    progress('INPUT',ai,len(DOWNLOAD_ASSETS),f'SHA: {sum(v["status"]=="SHA_PASS" for v in DOWNLOAD_QA)} из {len(DOWNLOAD_ASSETS)}')
DOWNLOAD_QA=pd.DataFrame(DOWNLOAD_QA)
required=['01::0']+[a['asset_id'] for a in FACTOR_ASSETS if a in DOWNLOAD_ASSETS]
missing=[aid for aid in required if aid not in LOCAL_ASSETS]
ensure('01::0' in LOCAL_ASSETS,'BLOCKED_INPUT_D01_NOT_AVAILABLE')
mark('PASS_INPUT','PASS' if not missing else 'FAIL',
     f'stage1 available={len(LOCAL_ASSETS)}/{len(DOWNLOAD_ASSETS)}; required missing={len(missing)}')
mark('ALL_STAGE1_DOWNLOAD','PASS' if CFG['MODE']=='FULL' and CFG['DOWNLOAD_ALL_STAGE1'] and len(LOCAL_ASSETS)==len(ASSETS) else 'PARTIAL',
     f'SHA downloaded={len(LOCAL_ASSETS)}/{len(ASSETS)}; includes independent education sources without ML05 JOIN')
if CFG['MODE']=='FULL' and CFG['DOWNLOAD_ALL_STAGE1']:
    ensure(len(LOCAL_ASSETS)==len(ASSETS),
           'FULL_STAGE1_INCOMPLETE_DOWNLOADS; review DOWNLOAD_QA and auth/release assets')

input_map['status']=input_map['asset_id'].map({r['asset_id']:'SHA_VERIFIED' for r in source_receipts}).fillna('INDEX_ONLY_OR_UNAVAILABLE')
STAGE1_PHYSICAL_AUDIT=[]
for a in ASSETS:
    aid=a['asset_id'];p=LOCAL_ASSETS.get(aid)
    if not p:
        STAGE1_PHYSICAL_AUDIT.append({'asset_id':aid,'status':'INDEXED_NOT_DOWNLOADED',
          'expected_rows':a.get('rows_actual'),'actual_rows':None,'metadata_columns':None,'sha_verified':False})
        continue
    try:
        pf=pq.ParquetFile(p);nr=pf.metadata.num_rows;nc=len(pf.schema.names)
        schema_columns=set(pf.schema.names);manifest_columns=set(a.get('columns_manifest') or [])
        ok=nr==a.get('rows_actual') and (not manifest_columns or manifest_columns==schema_columns)
        STAGE1_PHYSICAL_AUDIT.append({'asset_id':aid,'status':'PASS' if ok else 'SCHEMA_OR_ROWS_MISMATCH',
          'expected_rows':a.get('rows_actual'),'actual_rows':nr,'metadata_columns':nc,'sha_verified':True,
          'schema_columns_missing':','.join(sorted(manifest_columns-schema_columns))})
    except Exception as exc:
        STAGE1_PHYSICAL_AUDIT.append({'asset_id':aid,'status':'PARQUET_METADATA_ERROR',
          'expected_rows':a.get('rows_actual'),'actual_rows':None,'metadata_columns':None,
          'sha_verified':True,'exception_type':type(exc).__name__})
STAGE1_PHYSICAL_AUDIT=pd.DataFrame(STAGE1_PHYSICAL_AUDIT)
physical_errors=STAGE1_PHYSICAL_AUDIT.status.isin(['SCHEMA_OR_ROWS_MISMATCH','PARQUET_METADATA_ERROR']).sum()
mark('STAGE1_PHYSICAL_AUDIT','PASS' if physical_errors==0 else 'FAIL',
  f'{len(STAGE1_PHYSICAL_AUDIT)} registered assets; {len(LOCAL_ASSETS)} physical reads; metadata mismatches={int(physical_errors)}')
if physical_errors and CFG['MODE']=='FULL':raise RuntimeError('STAGE1_PHYSICAL_AUDIT_FAILED')

con=duckdb.connect(str(WORK/'ml03.duckdb'))
con.execute('SET enable_progress_bar = false')  # единый индикатор — только _PROGRESS_BAR
con.execute(f"SET memory_limit='{CFG['DUCKDB_MEMORY']}'");con.execute(f"SET threads={int(CFG['DUCKDB_THREADS'])}")
base=pd.read_parquet(LOCAL_ASSETS['01::0'],columns=['oktmo','region','mun_type','municipality','year',
    'population','average_population','migration','migration_rate','births','deaths'])
for c in ('oktmo','region','mun_type','municipality'):
    base[c]=base[c].astype('string').str.strip()
base=base.rename(columns={'oktmo':'entity_id','region':'region_name','year':'observation_year'})
for c in ('population','average_population','migration','migration_rate','births','deaths'):
    base[c]=pd.to_numeric(base[c],errors='coerce')
base['observation_year']=pd.to_numeric(base['observation_year'],errors='coerce')
base=base[base.entity_id.notna()&base.entity_id.ne('')&base.observation_year.between(1990,2026)].copy()
base['observation_year']=base.observation_year.astype(int)
base['entity_id']=base.entity_id.astype(str)
repeats=base[base.duplicated(['entity_id','observation_year'],keep=False)].copy()
if len(repeats):
    valuecols=['population','average_population','migration','migration_rate','births','deaths','region_name','mun_type']
    diversity=repeats.groupby(['entity_id','observation_year'])[valuecols].nunique(dropna=False)
    ensure(diversity.le(1).all().all(),'D01_CONFLICTING_ENTITY_YEAR')
    base=base.drop_duplicates(['entity_id','observation_year'])
base.loc[~base.population.gt(0),'population']=np.nan
base=base.sort_values(['entity_id','observation_year']).reset_index(drop=True)
YEARS=sorted(base.observation_year.unique().tolist())

# Independent target audit: no forced equivalence between reported migration_rate and net-migration counts.
MIGRATION_AUDIT=[];record=base.migration_rate
for denom in ['average_population','population']:
    for numerator_scale in [1.,10.,100.,1000.]:
        candidate=base.migration/base[denom]*numerator_scale
        good=record.notna() & candidate.notna() & base[denom].gt(0)
        if good.sum()<100:continue
        delta=(record[good]-candidate[good]).abs()
        consistent=(delta<=np.maximum(1e-5,0.05*record[good].abs())).mean()
        MIGRATION_AUDIT.append({'denominator':denom,'numerator_scale':numerator_scale,
           'n_pairs':int(good.sum()),'agreement_5pct':float(consistent),'median_abs_error':float(delta.median()),
           'median_observed_rate':float(record[good].median())})
MIGRATION_QA=pd.DataFrame(MIGRATION_AUDIT).sort_values(['agreement_5pct','n_pairs','median_abs_error'],ascending=[False,False,True])
if len(MIGRATION_QA):
    best=MIGRATION_QA.iloc[0];valid_rate=bool(best.agreement_5pct>=0.90 and best.n_pairs>=300)
else:valid_rate=False
if valid_rate:
    denom=best['denominator'];scale=best['numerator_scale']
    base['migration_per_1000']=1000.0*base.migration/base[denom]  # persons per 1000 residents; source migration_rate is a fraction when scale=1
    mark('MIGRATION_TARGET_SEMANTICS','PASS',f'{denom}, source_ratio_scale={scale}, converted_to_per_1000=True, n={best.n_pairs}, agreement={best.agreement_5pct:.3f}')
else:
    base['migration_per_1000']=np.nan
    mark('MIGRATION_TARGET_SEMANTICS','NOT_FEASIBLE','Reported migration rate inconsistent with all inspected denominators/scales; source-definition audit required')
base['boundary_uncertainty']='UNVERIFIED_CROSS_YEAR_BOUNDARIES'
base['source_vintage']=source_sha
base_keys=base[['entity_id','observation_year','region_name']].drop_duplicates()
con.register('base_keys',base_keys)
quiet_print('Демография D01:',len(base),'наблюдений;',base.entity_id.nunique(),'ОКТМО;',min(YEARS),'-',max(YEARS))

# Test windows are declared in CONFIG; D01-only presence checks prevent selecting
# a future year with no migration labels. Selection never examines outcome magnitudes.
TARGET_TEST_WINDOWS={t:list(map(int,CFG['PRIMARY_TEST_WINDOWS'][t])) for t in CFG['TARGETS']}
TARGET_CALENDAR_PREFLIGHT=[]
# D01 does not contain population_change_pct: construct availability from real
# population at target year and at origin=target-h, without evaluating magnitude.
_POP_SOURCE=base.loc[base.population.gt(0),['entity_id','observation_year','population']].copy()
_POP_SOURCE= _POP_SOURCE.drop_duplicates(['entity_id','observation_year'])
for target,years in TARGET_TEST_WINDOWS.items():
    ensure(target in ('migration_per_1000','population_change_pct'), 'UNKNOWN_D01_TARGET_'+str(target))
    for year in years:
        by_h={}
        for h in CFG['HORIZONS']:
            if target=='migration_per_1000':
                observed=base.loc[base.observation_year.eq(year),'migration_per_1000'].notna()
                n=int(observed.sum())
            else:
                current=_POP_SOURCE.loc[_POP_SOURCE.observation_year.eq(year),['entity_id']]
                origin=_POP_SOURCE.loc[_POP_SOURCE.observation_year.eq(year-int(h)),['entity_id']]
                n=int(current.merge(origin,on='entity_id',how='inner',validate='one_to_one').shape[0])
            by_h[int(h)]=n
        # Record each horizon separately; no silent reuse of h1 counts for h3/h5.
        for h,n in by_h.items():
            TARGET_CALENDAR_PREFLIGHT.append({'target':target,'target_year':int(year),'horizon':int(h),
              'available_labels':int(n),'role':'PREREGISTERED_TWO_YEAR_PRIMARY',
              'status':'OBSERVED' if n>0 else 'MISSING',
              'availability_derived_from':'D01_MIGRATION_PER_1000' if target=='migration_per_1000' else 'D01_POPULATION_ORIGIN_AND_TARGET'})
TARGET_CALENDAR_PREFLIGHT=pd.DataFrame(TARGET_CALENDAR_PREFLIGHT)
if TARGET_CALENDAR_PREFLIGHT.loc[TARGET_CALENDAR_PREFLIGHT.horizon.eq(1),'status'].eq('MISSING').any():
    mark('PRIMARY_TARGET_D01_GATE','FAIL','Preregistered target years are unobserved; revise protocol explicitly, never silently shift')
    if CFG['MODE']=='FULL':raise RuntimeError('PRIMARY_TARGET_D01_GATE_FAILURE: '+TARGET_CALENDAR_PREFLIGHT.to_json(orient='records'))
else:mark('PRIMARY_TARGET_D01_GATE','PASS',f'Observed outcome labels in primary h=1 windows: {TARGET_TEST_WINDOWS}')

#### Сканирование БДМО и отбор фактических экономических, жилищных, социальных и бюджетных индикаторов.

In [ ]:
# 05 · V4.00 · SUBJECT-VALIDATED FACTOR INVENTORY · SQL/LEVEL CHECKED
# Дополнительные муниципальные источники БДМО: только проверяемые фактические данные.
# Разделы основного каталога Stage I (52) остаются неизменными и отдельно аудируются.
import html as _html
from html.parser import HTMLParser as _HTMLParser
BDMO_EXTERNAL_QA_ROWS=[];BDMO_EXTERNAL_ASSETS=[];BDMO_EXTERNAL_RECEIPTS=[]
BDMO_OFFICIAL_CATALOGUE='https://tochno.st/datasets/bdmo'
BDMO_EXTRA_SECTIONS={1:'G1',2:'G1',3:'G3',6:'G5',8:'G2',9:'G1',19:'G3',32:'G1',33:'G3',39:'G1',42:'G4',70:'G5'}
BDMO_EXTERNAL_DESIGN=pd.DataFrame([{'section':s,'assigned_group':g,'source':'BDMO/Rosstat via tochno.st',
    'data_grain':'municipality_year_long, validated at query time',
    'selection':'all codes catalogued, only preregistered semantic domains retained'}
    for s,g in BDMO_EXTRA_SECTIONS.items()])

class _LinkCollector(_HTMLParser):
    def __init__(self):super().__init__();self.hrefs=[]
    def handle_starttag(self,tag,attrs):
        if tag=='a':
            d=dict(attrs)
            if d.get('href'):self.hrefs.append(_html.unescape(d['href']))

def _supplementary_urls():
    r=SESSION.get(BDMO_OFFICIAL_CATALOGUE,timeout=(15,60));r.raise_for_status()
    parser=_LinkCollector();parser.feed(r.text)
    u=set(parser.hrefs)|set(re.findall(r'https?://[^\s\"<>]+?(?:\.zip)',_html.unescape(r.text)))
    result={}
    for sec in BDMO_EXTRA_SECTIONS:
        pattern=rf'/by_(?:section|indicator)/data_section{sec}_.*?\.zip'
        matches=[v for v in u if re.search(pattern,v) and urllib.parse.urlparse(v).hostname=='storage.yandexcloud.net']
        # Fallback is the same dated catalogue release already audited in Stage I,
        # not a claim that the remote file exists: HTTP response must validate it.
        if not matches:
            base_url=next((x.get('source_url') for x in ASSETS if x.get('source_id')=='BDMO' and '/by_section/' in str(x.get('source_url'))),None)
            if base_url and sec!=32:
                mm=re.search(r'data_section\d+_',base_url)
                if mm:matches=[base_url[:mm.start()]+'data_section'+str(sec)+'_'+base_url[mm.end():].replace('data_section13_',f'data_section{sec}_')]
        if matches:result[sec]=max(matches,key=lambda s:re.search(r'v(20\d{6})',s).group(1) if re.search(r'v(20\d{6})',s) else '')
    return result

def _supplement_one_section(sec,url,group):
    pu=urllib.parse.urlparse(url)
    ensure(pu.scheme=='https' and pu.hostname=='storage.yandexcloud.net' and '/tochno-st-catalog/Rosstat/' in pu.path,'BDMO_EXTERNAL_URL_UNTRUSTED')
    zpath=WORK/'cache'/f'bdmo_external_{sec}.zip';part=zpath.with_suffix('.part')
    side=zpath.with_suffix('.source_url.txt')
    if zpath.exists() and (not side.exists() or side.read_text(encoding='utf-8')!=url):
        zpath.unlink()
    if not zpath.exists():
        part.unlink(missing_ok=True);nr=0;h=hashlib.sha256()
        try:
            rr=SESSION.get(url,timeout=(30,180),stream=True);rr.raise_for_status()
            with open(part,'wb') as fd:
                for buf in rr.iter_content(4*1024*1024):
                    if not buf:continue
                    nr+=len(buf)
                    ensure(nr<=CFG['BDMO_EXTERNAL_MAX_SOURCE_GB']*2**30,'BDMO_EXTERNAL_ZIP_SIZE_LIMIT')
                    if nr%(512*1024*1024)<len(buf):
                        ensure(shutil.disk_usage(WORK).free>CFG['MIN_DISK_FREE_GB']*2**30,
                               'BDMO_EXTERNAL_DISK_SAFETY_STOP')
                    h.update(buf);fd.write(buf)
            rr.close();ensure(zipfile.is_zipfile(part),'BDMO_EXTERNAL_NOT_ZIP')
            part.replace(zpath);side.write_text(url,encoding='utf-8')
        except Exception:part.unlink(missing_ok=True);raise
    ensure(zipfile.is_zipfile(zpath),'BDMO_EXTERNAL_CACHE_NOT_ZIP')
    source_sha=digest(zpath);indexed=0
    with zipfile.ZipFile(zpath) as z:
        ensure(z.testzip() is None,'BDMO_EXTERNAL_ZIP_CRC')
        infos=[i for i in z.infolist() if not i.is_dir() and i.filename.lower().endswith('.parquet')]
        infos.sort(key=lambda x:x.filename)
        if not infos:
            BDMO_EXTERNAL_QA_ROWS.append({'section':sec,'status':'SOURCE_FORMAT_NOT_PARQUET',
                'reason':'The validated archive contains no direct Parquet member; ' +
                         'nested/CSV sources require separate schema and grain validation',
                'members_sample':json.dumps([i.filename for i in z.infolist()[:20]],ensure_ascii=False)})
            return 0
        ensure(len(infos)<=CFG['BDMO_EXTERNAL_MAX_MEMBERS_PER_SECTION'],'BDMO_EXTERNAL_TOO_MANY_MEMBERS')
        for idx,member in enumerate(infos):
            ensure(member.file_size<CFG['BDMO_EXTERNAL_MAX_UNPACKED_GB']*2**30,'BDMO_EXTERNAL_UNPACKED_MEMBER_LIMIT')
            dest=WORK/'cache'/f'bdmo_external_{sec}_{idx:03d}.parquet'
            if not dest.exists() or dest.stat().st_size!=member.file_size:
                tmp=dest.with_suffix('.part')
                try:
                    with z.open(member) as src,open(tmp,'wb') as dst:shutil.copyfileobj(src,dst,length=4*1024*1024)
                    ensure(tmp.stat().st_size==member.file_size,'BDMO_EXTERNAL_MEMBER_BYTES')
                    tmp.replace(dest)
                except Exception:tmp.unlink(missing_ok=True);raise
            info=pq.ParquetFile(dest);cols=info.schema.names
            needed={'year','indicator_code','indicator_name','indicator_value','oktmo'}
            if not needed.issubset(cols):
                BDMO_EXTERNAL_QA_ROWS.append({'section':sec,'member':member.filename,'status':'SCHEMA_UNSUPPORTED','columns':','.join(cols[:35])});continue
            aid=f'SUPP::BDMO::{sec}::{idx:03d}'
            entry={'asset_id':aid,'source_id':'BDMO','role':'municipal_long_supplement',
                'segment':'BDMO_EXTERNAL','columns_manifest':cols,'rows_actual':info.metadata.num_rows,
                'source_url':url,'source_sha256':source_sha,'sha256':digest(dest),
                'locator':{'type':'verified_external_cache','path':str(dest)}}
            BDMO_EXTERNAL_ASSETS.append(entry);LOCAL_ASSETS[aid]=str(dest)
            SOURCE_CLASSES[group].append(aid)
            if entry not in FACTOR_ASSETS:FACTOR_ASSETS.append(entry)
            BDMO_EXTERNAL_RECEIPTS.append({'asset_id':aid,'group':group,'source_url':url,'source_sha256':source_sha,
                'parquet_sha256':entry['sha256'],'source_member':member.filename,'rows':entry['rows_actual'],
                'columns':len(cols),'status':'SHA_VERIFIED_AND_SCHEMA_CHECKED'})
            BDMO_EXTERNAL_QA_ROWS.append({'section':sec,'member':member.filename,'status':'ADAPTED',
                'asset_id':aid,'rows':entry['rows_actual'],'group':group})
            indexed+=1
    return indexed

if CFG['ENABLE_EXTERNAL_BDMO'] and CFG['MODE']=='FULL':
    try:
        urls=_supplementary_urls();BDMO_EXTERNAL_QA_ROWS.append({'section':0,'status':'CATALOGUE_OPENED','n_urls_found':len(urls)})
        for _bi,(section,group) in enumerate(BDMO_EXTRA_SECTIONS.items(),1):
            progress('BDMO',_bi-1,len(BDMO_EXTRA_SECTIONS),f'Раздел {section}')
            if section not in urls:
                BDMO_EXTERNAL_QA_ROWS.append({'section':section,'group':group,'status':'URL_NOT_DISCOVERED'});continue
            try:
                n=_supplement_one_section(section,urls[section],group)
                if n==0:log(f'BDMO section {section}: no verified Parquet member; review format audit','WARN')
                quiet_print(f'Дополнительный БДМО раздел {section}: {n} Parquet; SHA проверен')
            except Exception as ex:
                BDMO_EXTERNAL_QA_ROWS.append({'section':section,'group':group,'status':'SOURCE_LOAD_FAILED',
                    'reason':type(ex).__name__+': '+str(ex)[:240]})
                log(f'External BDMO section {section} rejected: {type(ex).__name__}: {str(ex)[:180]}','WARN')
    except Exception as ex:
        BDMO_EXTERNAL_QA_ROWS.append({'section':0,'status':'SOURCE_CATALOGUE_UNAVAILABLE',
            'reason':type(ex).__name__+': '+str(ex)[:240]})
        log('BDMO external catalogue unavailable: '+type(ex).__name__,'WARN')
progress('BDMO',1,1,f'Проверено разделов: {len(BDMO_EXTRA_SECTIONS)}')
BDMO_EXTERNAL_RECOVERY=pd.DataFrame([{
    'section':int(s),'group':g,'status':next((x.get('status') for x in reversed(BDMO_EXTERNAL_QA_ROWS)
        if x.get('section')==s),'NOT_ATTEMPTED'),
    'source_url':urls.get(s,'') if 'urls' in locals() else '',
    'must_not_claim_loaded_without_sha':True}
    for s,g in BDMO_EXTRA_SECTIONS.items()])
BDMO_EXTERNAL_QA=pd.DataFrame(BDMO_EXTERNAL_QA_ROWS)
mark('EXTERNAL_BDMO_SOURCE_AUDIT','PASS' if len(BDMO_EXTERNAL_ASSETS)>=3 else 'PARTIAL',
     f'External verified Parquet={len(BDMO_EXTERNAL_ASSETS)}, sections={sorted(set(int(s.split("::")[2]) for s in LOCAL_ASSETS if s.startswith("SUPP::BDMO::")))}; not part of original 52')
_ORIGINAL_GET_ASSET=get_asset
def get_asset(aid):
    for _a in BDMO_EXTERNAL_ASSETS:
        if _a['asset_id']==aid:return _a
    return _ORIGINAL_GET_ASSET(aid)

PATTERNS = {
 'G1':'зарплат|заработ|оплат.{0,12}труда|доход|инвестиц|занятост|занятых|безработ|предпринимател|предприяти|оборот|производств|налог|уровен.{0,8}жизни|средн.{0,15}душ',
 'G2':'жил|жилищ|квартир|строительств|коммунал|водоснабж|водоотвед|теплоснабж|газификац|электроснабж|канализац|благоустрой',
 'G3':'образован|школ|дошколь|детск.{0,8}сад|медицин|врач|фельдшер|больниц|поликлиник|амбулатор|социал|койко|здравоохран|спорт|культур',
 'G4':'бюджет|расход|дотаци|субсид|субвенци|межбюджет|налогов.{0,10}доход|казн',
 'G5':'расстояни|транспорт|доступност|пространств|территори|площад.{0,18}(?:земл|муницип)|автомобил.{0,12}дорог|покрыти.{0,12}дорог|связь|телефон|интернет'}
SEMANTIC_DOMAINS = {
 'G1':[('wages','заработн.{0,10}плат|средн.{0,12}зарплат|оплат.{0,7}труда|фонд.{0,10}оплат'),('employment','численност.{0,10}(?:занят|работающ|работник)|число.{0,10}(?:занят|работающ|работник)|уровень.{0,10}безработ|число.{0,12}безработ|рабоч.{0,8}мест|занят.{0,10}насел'),('business','мал.{0,12}предприят|средн.{0,12}предприят|предпринимател|субъект.{0,12}мсп|оборот.{0,12}организац|зарегистрированн.{0,15}организац'),('investment','инвестиц|капитальн.{0,10}вложен'),('retail','розничн.{0,14}торгов|оборот.{0,15}торгов|общественн.{0,15}питан'),('paid_services','платн.{0,12}услуг|услуг.{0,12}населен'),('income','денежн.{0,12}доход|сумм.{0,12}доход|совокупн.{0,10}доход|налогооблагаем.{0,10}доход|среднедушев.{0,12}доход|доход.{0,12}населени')],
 'G2':[('housing_stock','жил.{0,8}фонд|жил.{0,9}площад|обеспеченн.{0,12}жиль|площад.{0,12}жил'),('housing_construction','ввод.{0,30}(?:жил|жиль|квартир)|введен.{0,20}(?:жил|квартир)|строительств.{0,14}жил|построенн.{0,12}квартир'),('housing_access','улучшивш.{0,12}жилищ|нуждающ.{0,12}жилищ|семе.{0,12}жилищ'),('utilities','водоснабж|водоотвед|теплоснабж|газификац|канализац|коммунал|благоустроенн.{0,12}жил|электроснабж')],
 'G3':[('health','врач|фельдшер|поликлиник|больниц|больнич|здравоохран|медицин|амбулатор|койко|медработ'),('education','образован|школ|дошколь|детск.{0,8}сад|обучающ|учащ|педагог'),('social','социальн.{0,14}(учрежден|обслужив|защит|услуг)|дом.{0,12}культур|спортивн.{0,10}сооруж')],
 'G5':[('territorial_area','площад.{0,18}(?:земл|муницип)|площад.{0,8}территори'),('roads','автомобил.{0,12}дорог|протяж[её]нност.{0,12}дорог|дорог.{0,12}покрыти'),('connectivity','телефонн|интернет|связ[ьи]|доступност.{0,12}сети')],
 'G4':[('municipal_spending','расход|содержание.{0,12}работник|затрат'),('municipal_revenue','доход.{0,12}бюджет|налогов.{0,12}доход|собственн.{0,10}доход'),('intergovernmental','дотаци|субвенци|субсид|межбюджет|трансферт'),('budget_balance','дефицит|профицит|долг|сбалансированност|бюджет')]}
EXCLUDED_INCOME='ценн.{0,8}бумаг|дивиденд|вычет|ставк.{0,12}налог|исчисленн.{0,12}налог|сумм.{0,12}налог|полученн.{0,8}сведен|количеств.{0,12}сведен|числ.{0,9}сведен|операц.{0,14}бумаг|код.{0,8}доход'
REJECT_NONMEASUREMENT='методик|справочн|код.{0,7}показател|инструкция|наименование|описание|классификатор'

def classify_indicator(group,asset_id,name,unit):
    title=re.sub(r'\s+',' ',str(name).casefold()).strip();un=str(unit).casefold()
    if not title or re.search(REJECT_NONMEASUREMENT,title):return None,'NON_MEASUREMENT'
    if group not in SEMANTIC_DOMAINS:return None,'NO_REGISTERED_DOMAIN'
    if group=='G1' and re.search(EXCLUDED_INCOME,title):return None,'NARROW_TAX_OR_DEDUCTION_PROXY'
    if group=='G1' and 'ndfl' in asset_id.lower():
        if not re.search(r'денежн.{0,12}доход|сумм.{0,12}доход|совокупн.{0,10}доход|налогооблагаем.{0,10}доход',title):return None,'NDFL_NOT_INCOME_AMOUNT'
        if re.search(r'штук|единиц|количеств|сведен|запис',un):return None,'NDFL_COUNT_NOT_INCOME'
    # 'Продано товаров (предприятия со штатом >15)' measures SALES, not workers.
    if group=='G1' and re.search(r'продано товар|продаж.{0,25}товар|отгружено товар|отгрузк.{0,20}товар',title):
        return 'business','SALES_VOLUME_NOT_EMPLOYMENT'
    # A count of medical social workers is not automatically hospital infrastructure.
    for dom,pattern in SEMANTIC_DOMAINS[group]:
        if re.search(pattern,title):return dom,''
    return None,'NO_SUBSTANTIVE_MATCH'

TOTAL_DIM_VALUES={'всего','итого','все','все категории','total','all','общий','общая','общие'}
NDFL_RISK_DIMENSIONS=('individual_type','income','deduction','tax_rate','report_type','index_form','tax_authority')

def eligible_slice(asset_id,dims):
    if 'ndfl' not in asset_id.lower():return True,'CATEGORY_SPECIFIED_NO_COLLAPSE'
    for k in NDFL_RISK_DIMENSIONS:
        if k in dims and str(dims[k]).casefold().strip() not in TOTAL_DIM_VALUES:
            return False,'NON_VERIFIED_TOTAL_NDFL_DIMENSION_'+k
    return True,'NDFL_ALL_RISK_DIMENSIONS_EXPLICIT_TOTAL_OR_ABSENT'

ensure(classify_indicator('G1','02::5ndfl::00::00','Количество сведений о доходах','Штука')[0] is None,'SEMANTIC_CANARY_TAX_COUNTS')
ensure(classify_indicator('G1','02::labor','Средняя начисленная заработная плата','рублей')[0]=='wages','SEMANTIC_CANARY_WAGES')
ensure(classify_indicator('G3','02::social','Количество врачей','человек')[0]=='health','SEMANTIC_CANARY_HEALTH')
ensure(classify_indicator('G1','SUPP::BDMO','Продано товаров несобственного производства (без субъектов малого предпринимательства), средняя численность работников которых превышает 15 человек','рублей')[0]=='business','SEMANTIC_CANARY_SALES_NOT_EMPLOYMENT')
ensure(not eligible_slice('02::5ndfl::00::00',{'income':'2204','tax_rate':'9'})[0],'SEMANTIC_CANARY_NDFL_SLICES')
mark('SEMANTIC_CANARY','PASS','Tax counts, wage concept, medicine and narrow NDFL slice guards')

PRIMARY_TARGET_YEARS=sorted(set(sum(TARGET_TEST_WINDOWS.values(),[])))
FACTOR_CUTOFF_BY_H={int(h):int(min(min(yy) for yy in TARGET_TEST_WINDOWS.values())-h-CFG['FEATURE_PUBLICATION_LAG']) for h in CFG['HORIZONS']}
TRAIN_CATALOG_MAX_YEAR=max(FACTOR_CUTOFF_BY_H.values());TRAIN_CATALOG_MIN_YEAR=TRAIN_CATALOG_MAX_YEAR-CFG['CATALOG_RECENT_WINDOW']
CATALOG_ROWS=[];FEATURE_PARTS=[];conflict_rows=[];SOURCE_QA=[];FEATURE_CANDIDATES=[];SELECTED=[];CANDIDATE_AUDIT=[];G5_DIAGNOSTICS=[];SCAN_AUDIT=[];FACTOR_ELIGIBILITY=[];DIMENSION_AUDIT=[]
LAST_SLICE_REASON='NOT_STARTED'

def qstr(v):return "'"+str(v).replace("'","''")+"'"
def qident(s):
    v=str(s);ensure(bool(v) and '\x00' not in v,'INVALID_SOURCE_COLUMN')
    return '"'+v.replace('"','""')+'"'

SQL_FAILURES=[];SQL_EXECUTION_AUDIT=[];REAL_SQL_PREFLIGHT=[]

def sql_run(sql,params=None,*,asset_id='',code='',stage='',preview=False):
    args=list(params or []);t0=time.monotonic()
    try:
        result=con.execute(sql,args);data=result.fetchone() if preview else result.df()
        SQL_EXECUTION_AUDIT.append({'asset_id':asset_id,'code':str(code),'stage':stage,'status':'PASS','elapsed_ms':int((time.monotonic()-t0)*1000),'sql_sha256':hashlib.sha256(sql.encode()).hexdigest()})
        return data
    except Exception as exc:
        safe_sql=re.sub(r'(?i)(ghp_|github_pat_)[a-z0-9_]+','[REDACTED]',sql)
        detail={'asset_id':asset_id,'code':str(code),'stage':stage,'status':'SCAN_ERROR','exception_type':type(exc).__name__,'message':str(exc)[:2000],'sql':safe_sql[:12000],'sql_sha256':hashlib.sha256(sql.encode()).hexdigest(),'parameters_types':[type(v).__name__ for v in args],'elapsed_ms':int((time.monotonic()-t0)*1000)}
        SQL_FAILURES.append(detail);log('SQL_SCAN_ERROR '+json.dumps({k:v for k,v in detail.items() if k!='sql'},ensure_ascii=False),'WARN')
        raise

def column_key(a):
    cc=a.get('columns_manifest') or []
    return 'oktmo' if 'oktmo' in cc else 'object_oktmo' if 'object_oktmo' in cc else None

def dimensions_for(a):
    cc=a.get('columns_manifest') or [];aid=a['asset_id']
    if 'ndfl' in aid.lower():return [k for k in NDFL_RISK_DIMENSIONS if k in cc]
    omit={'indicator_section_code','indicator_section','indicator_code','indicator_name','indicator_unit','indicator_period','indicator_value','year','region_id','region_name','region','oktmo','object_oktmo','oktmo_stable','oktmo_history','oktmo_year_from','oktmo_year_to','mun_level','mun_district','municipality','mun_type','mun_type_oktmo','object_name','object_level','object_okato','source','comment','version_date','report_date','report_region_id','report_region_name','null_value_reason'}
    return [c for c in cc if c not in omit]

def val_sql(alias='a'):
    return f"TRY_CAST(REPLACE(REPLACE(REPLACE(TRIM(CAST({alias}.indicator_value AS VARCHAR)), ' ', ''), chr(160), ''), ',', '.') AS DOUBLE)"

def _grp_eligible_from_source(a):
    g=classify_asset_for_scan(a)
    return ['G1','G2','G3','G4'] if g=='DISCOVERY_MULTIDOMAIN' else [g] if g in SEMANTIC_DOMAINS else []

def inventory_asset(a,path,group):
    key=column_key(a);cc=set(a.get('columns_manifest') or [])
    if key is None or not {'indicator_code','indicator_name','indicator_value','year'}.issubset(cc):return pd.DataFrame()
    unit=f"COALESCE(TRIM(CAST(a.{qident('indicator_unit')} AS VARCHAR)), '')" if 'indicator_unit' in cc else "''"
    k=qident(key);yr='TRY_CAST(a.'+qident('year')+' AS INTEGER)'
    level_predicate=("regexp_matches(lower(coalesce(cast(a."+qident('object_level')+" AS VARCHAR),'')), 'муниципал|городск.{0,8}округ')" if 'object_level' in cc else 'TRUE')
    sql=f"""SELECT TRIM(CAST(a.{qident('indicator_code')} AS VARCHAR)) AS indicator_code,
      TRIM(CAST(a.{qident('indicator_name')} AS VARCHAR)) AS indicator_name,{unit} AS indicator_unit,
      COUNT(DISTINCT TRIM(CAST(a.{k} AS VARCHAR))) AS n_municipalities,
      COUNT(DISTINCT {yr}) AS n_years,
      COUNT(DISTINCT CASE WHEN {yr} BETWEEN {TRAIN_CATALOG_MIN_YEAR} AND {TRAIN_CATALOG_MAX_YEAR}
      THEN TRIM(CAST(a.{k} AS VARCHAR))||':'||CAST({yr} AS VARCHAR) END) AS recent_keys,
      MIN({yr}) AS min_year,MAX({yr}) AS max_year
      FROM read_parquet(?) a JOIN base_keys b ON TRIM(CAST(a.{k} AS VARCHAR))=b.entity_id
         AND {yr}=b.observation_year
      WHERE {yr}<={TRAIN_CATALOG_MAX_YEAR} AND {level_predicate}
       AND regexp_matches(lower(coalesce(cast(a.{qident('indicator_name')} AS VARCHAR),'')),?)
       AND a.{qident('indicator_code')} IS NOT NULL
      GROUP BY 1,2,3 HAVING COUNT(DISTINCT a.{k})>={CFG['MIN_INDICATOR_MUNICIPALITIES']}
      ORDER BY indicator_code,indicator_name,indicator_unit"""
    return sql_run(sql,[str(path),PATTERNS[group]],asset_id=a['asset_id'],stage='INVENTORY')

def _slice_dimensions_supported(a,dims):
    if 'ndfl' not in a['asset_id'].lower():return True,'NON_NDFL_DIMENSIONAL_SLICE'
    ok,reason=eligible_slice(a['asset_id'],dims)
    if not ok:return False,reason
    if any(k in dims for k in ('income','deduction','tax_rate','individual_type','report_type','index_form')):
        return True,'NDFL_EXPLICIT_TOTAL_VALUES_NO_COLLAPSE'
    return True,'NDFL_UNSTRATIFIED_NO_AGGREGATION'

def factor_raw_sql(a):
    key=column_key(a);cc=set(a.get('columns_manifest') or []);dims=dimensions_for(a)
    ensure(key is not None and {'indicator_code','indicator_name','indicator_value','year'}.issubset(cc),'INVALID_FACTOR_SCHEMA')
    d_expr=','.join(f"COALESCE(NULLIF(TRIM(CAST(a.{qident(d)} AS VARCHAR)),''),'∅') AS {qident('dim_'+str(i).zfill(2))}" for i,d in enumerate(dims))
    d_expr=','+d_expr if d_expr else ''
    unitfilter=f" AND COALESCE(TRIM(CAST(a.{qident('indicator_unit')} AS VARCHAR)),'')=?" if 'indicator_unit' in cc else ''
    level_filter=(" AND regexp_matches(lower(coalesce(cast(a."+qident('object_level')+" AS VARCHAR),'')), 'муниципал|городск.{0,8}округ')" if 'object_level' in cc else '')
    raw=f"""SELECT TRIM(CAST(a.{qident(key)} AS VARCHAR)) AS entity_id,
       TRY_CAST(a.{qident('year')} AS INTEGER) AS observation_year,{val_sql()} AS val{d_expr}
       FROM read_parquet(?) a JOIN base_keys b
         ON TRIM(CAST(a.{qident(key)} AS VARCHAR))=b.entity_id
         AND TRY_CAST(a.{qident('year')} AS INTEGER)=b.observation_year
       WHERE TRIM(CAST(a.{qident('indicator_code')} AS VARCHAR))=?
         AND TRIM(CAST(a.{qident('indicator_name')} AS VARCHAR))=? {unitfilter} {level_filter}"""
    return raw,dims,bool(unitfilter)

def select_dimensional_slice(a,path,code,name,unit,group):
    global LAST_SLICE_REASON
    LAST_SLICE_REASON='NO_VALID_CANDIDATE'
    raw,dims,with_unit=factor_raw_sql(a)
    baseparams=[str(path),str(code),str(name)]+([str(unit)] if with_unit else [])
    d_alias=[qident('dim_'+str(i).zfill(2)) for i in range(len(dims))]
    d_select=','.join(d_alias)+',' if dims else ''
    pos=' GROUP BY '+','.join(map(str,range(1,len(dims)+1))) if dims else ''
    summary_sql=f"""WITH v AS ({raw}) SELECT {d_select}
       COUNT(DISTINCT CASE WHEN observation_year<={TRAIN_CATALOG_MAX_YEAR} AND val IS NOT NULL
           THEN entity_id||':'||CAST(observation_year AS VARCHAR) END) AS n_train_keys,
       COUNT(DISTINCT CASE WHEN observation_year BETWEEN {TRAIN_CATALOG_MIN_YEAR} AND {TRAIN_CATALOG_MAX_YEAR}
         AND val IS NOT NULL THEN entity_id||':'||CAST(observation_year AS VARCHAR) END) AS n_recent_keys,
       COUNT(DISTINCT CASE WHEN observation_year<={TRAIN_CATALOG_MAX_YEAR} AND val IS NOT NULL
         THEN observation_year END) AS n_train_years FROM v {pos}"""
    if 'ndfl' in a['asset_id'].lower():
        filter_terms=[f'lower(trim({d_alias[i]})) IN ('+','.join(qstr(s) for s in sorted(TOTAL_DIM_VALUES))+')' for i,d in enumerate(dims) if d in NDFL_RISK_DIMENSIONS]
        predicate=' AND '.join(filter_terms) if filter_terms else 'TRUE'
        budget=max(1,int(CFG['MAX_NDFL_SLICE_VALUES']))
    else:
        predicate='TRUE';budget=max(1,int(CFG['MAX_DIMENSIONAL_SLICES_PER_CODE']))
    total_sql='SELECT COUNT(*) FROM ('+summary_sql+') total_slices'
    total=int(sql_run(total_sql,baseparams,asset_id=a['asset_id'],code=code,stage='SLICE_COUNT',preview=True)[0])
    valid_sql='SELECT COUNT(*) FROM ('+summary_sql+') grouped WHERE '+predicate
    eligible=int(sql_run(valid_sql,baseparams,asset_id=a['asset_id'],code=code,stage='SLICE_ELIGIBLE_COUNT',preview=True)[0])
    take=min(eligible,budget)
    DIMENSION_AUDIT.append({'asset_id':a['asset_id'],'indicator_code':code,'slices_found':total,'semantically_eligible':eligible,'inspected':take,'remaining':max(eligible-take,0),'rejected_semantically':total-eligible,'status':'NO_VALID_TOTAL_SLICE' if 'ndfl' in a['asset_id'].lower() and eligible==0 else 'CANDIDATE_BUDGET_PARTIAL' if eligible>budget else 'EXHAUSTIVE_SQL_FILTERED'})
    if not eligible:
        LAST_SLICE_REASON='NO_VALID_TOTAL_SLICE' if 'ndfl' in a['asset_id'].lower() else 'NO_DIMENSION_SLICES'
        return None,None,None
    top_sql='SELECT * FROM ('+summary_sql+') grouped WHERE '+predicate+' ORDER BY n_recent_keys DESC,n_train_keys DESC LIMIT ?'
    top=sql_run(top_sql,baseparams+[take],asset_id=a['asset_id'],code=code,stage='SLICE_RANKING')
    if top.empty:return None,None,None
    for row in top.to_dict('records'):
        sl={d:str(row['dim_'+str(i).zfill(2)]) for i,d in enumerate(dims)}
        ok,why=_slice_dimensions_supported(a,sl)
        if not ok:
            CANDIDATE_AUDIT.append({'asset_id':a['asset_id'],'code':code,'name':name,'decision':why,'slice':json.dumps(sl,ensure_ascii=False)})
            continue
        if int(row['n_train_keys'])<CFG['MIN_INDICATOR_OBS_TRAIN'] or int(row['n_train_years'])<CFG['MIN_INDICATOR_YEARS_TRAIN']:continue
        pred=' AND '.join(f'{d_alias[i]}=?' for i in range(len(dims))) if dims else 'TRUE'

        # FIX: DuckDB requires the output alias "value" to be quoted.
        audit_sql=f"""WITH v AS ({raw}), sliced AS
          (SELECT entity_id,observation_year,val FROM v WHERE {pred})
          SELECT entity_id,observation_year,
            MIN(val) AS "value",
            COUNT(DISTINCT val) AS n_different,
            COUNT(*) AS n_raw
          FROM sliced GROUP BY entity_id,observation_year"""

        audit=sql_run(audit_sql,baseparams+[sl[d] for d in dims],asset_id=a['asset_id'],code=code,stage='EXTRACT_SLICE')
        if audit.empty:continue
        conflict=audit.n_different.gt(1);train_mask=audit.observation_year.le(TRAIN_CATALOG_MAX_YEAR)
        train_conf=conflict&train_mask;share=float(train_conf.sum()/max(1,train_mask.sum()))
        if conflict.any():
            conflict_rows.append({'asset':a['asset_id'],'code':code,'n_collisions':int(conflict.sum()),'share':share,'slice':json.dumps(sl,ensure_ascii=False),'decision':'REJECT_NONUNIQUE_SLICED_KEY' if share>CFG['MAX_CONFLICT_SHARE'] else 'EXCLUDE_CONFLICTED_ENTITY_YEARS'})
        if share>CFG['MAX_CONFLICT_SHARE']:continue
        audit=audit.loc[~conflict,['entity_id','observation_year','value']].copy()
        train_values=audit.observation_year.le(TRAIN_CATALOG_MAX_YEAR)&audit['value'].notna()
        if int(train_values.sum())<CFG['MIN_INDICATOR_OBS_TRAIN']:continue
        ensure(not audit.duplicated(['entity_id','observation_year']).any(),'NON_UNIQUE_FEATURE_OUTPUT')
        LAST_SLICE_REASON='ELIGIBLE'
        return audit,sl,row
    LAST_SLICE_REASON='PARTIAL_CANDIDATES_INSPECTED' if eligible>budget else 'NO_VALID_SLICE_AFTER_SUPPORT_CHECK'
    return None,None,None

def run_duckdb_sql_contracts():
    tmp=WORK/'cache'/'ml03_stage3_sql_contract.parquet';tmp_ndfl=WORK/'cache'/'ml03_stage3_ndfl_contract.parquet'
    audit0=len(SQL_EXECUTION_AUDIT);dim0=len(DIMENSION_AUDIT);conf0=len(conflict_rows)
    selected=base_keys.loc[base_keys.observation_year.between(TRAIN_CATALOG_MIN_YEAR,TRAIN_CATALOG_MAX_YEAR),['entity_id','observation_year']].drop_duplicates()
    eligible_ids=selected.groupby('entity_id').observation_year.nunique()
    eligible_ids=eligible_ids[eligible_ids>=2].index[:100]
    selected=selected[selected.entity_id.isin(eligible_ids)].groupby('entity_id').head(2)
    ensure(len(selected)>=100 and selected.observation_year.nunique()>=2 and selected.entity_id.nunique()>=CFG['MIN_INDICATOR_MUNICIPALITIES'],'SQL_UNIT_FIXTURE_INSUFFICIENT_BASE_KEYS')
    fixture=selected.rename(columns={'entity_id':'oktmo','observation_year':'year'}).copy()
    fixture['indicator_code']='Y48010001';fixture['indicator_name']='Площадь жилищного фонда';fixture['indicator_unit']='кв. м'
    fixture['indicator_value']='1 234,5';fixture['group']='Все';fixture['order']='Вся территория';fixture['index"quoted']='Данные'
    con.register('ml03_fixture_df',fixture)
    try:
        con.execute('COPY (SELECT * FROM ml03_fixture_df) TO '+qstr(str(tmp))+' (FORMAT PARQUET)')
        toy={'asset_id':'02::housing::SYNTHETIC_SQL_CONTRACT','columns_manifest':list(fixture.columns)}
        inv=inventory_asset(toy,tmp,'G2')
        ensure(not inv.empty and inv.indicator_code.eq('Y48010001').any(),'SQL_UNIT_INVENTORY_EMPTY')
        dat,sl,support=select_dimensional_slice(toy,tmp,'Y48010001','Площадь жилищного фонда','кв. м','G2')
        ensure(dat is not None and len(dat)>=100,'SQL_UNIT_EXTRACT_NO_ROWS')
        ensure(dat['value'].notna().all() and np.allclose(dat['value'].to_numpy(),1234.5),'SQL_UNIT_LOCALE_DECIMAL')
        ensure(not dat.duplicated(['entity_id','observation_year']).any(),'SQL_UNIT_DUPLICATE_KEYS')
        ensure(sl['group']=='Все' and sl['order']=='Вся территория' and sl['index"quoted']=='Данные','SQL_UNIT_DIMENSION_ROUNDTRIP')

        # Municipal-level contract: a D05-like region row with a matching OKTMO
        # must still be excluded, even if its identifier happens to coincide.
        lev_fixture=fixture.iloc[:2].copy().drop(columns=['group','order','index"quoted'])
        lev_fixture['object_oktmo']=lev_fixture.pop('oktmo')
        lev_fixture['object_level']=['Регион','Муниципальный район']
        con.register('ml03_level_fixture_df',lev_fixture)
        level_path=WORK/'cache'/'ml03_level_contract.parquet'
        con.execute('COPY (SELECT * FROM ml03_level_fixture_df) TO '+qstr(str(level_path))+' (FORMAT PARQUET)')
        try:
            lev_asset={'asset_id':'02::labor::SYNTHETIC_LEVEL','columns_manifest':list(lev_fixture.columns)}
            lvi=inventory_asset(lev_asset,level_path,'G2')
            ensure(len(lvi)<=1,'LEVEL_INVENTORY_UNEXPECTED')
            raw_level,_,hasunit=factor_raw_sql(lev_asset)
            pp=[str(level_path),'Y48010001','Площадь жилищного фонда']+(['кв. м'] if hasunit else [])
            lv=sql_run('SELECT COUNT(*) AS n FROM ('+raw_level+') filtered',pp,asset_id=lev_asset['asset_id'],stage='OBJECT_LEVEL_UNIT',preview=True)
            ensure(int(lv[0])==1,'REGIONAL_OBJECT_ROW_LEAKED_IN_MUNICIPAL_JOIN')
        finally:
            con.unregister('ml03_level_fixture_df');level_path.unlink(missing_ok=True)
        ndfl=fixture.iloc[:1].copy().drop(columns=['group','order','index"quoted'])
        ndfl['indicator_code']='SYNTHETIC_NDFL_TOTAL';ndfl['indicator_name']='Общая сумма денежных доходов'
        ndfl['indicator_unit']='рублей';ndfl['income']='9999';ndfl['tax_rate']='13'
        sparse=pd.concat([ndfl.assign(income=str(10000+i)) for i in range(2101)],ignore_index=True)
        total=fixture.drop(columns=['group','order','index"quoted']).copy()
        total['indicator_code']='SYNTHETIC_NDFL_TOTAL';total['indicator_name']='Общая сумма денежных доходов'
        total['indicator_unit']='рублей';total['income']='всего';total['tax_rate']='всего'
        ndfl=pd.concat([sparse,total],ignore_index=True)
        con.register('ml03_ndfl_fixture_df',ndfl)
        con.execute('COPY (SELECT * FROM ml03_ndfl_fixture_df) TO '+qstr(str(tmp_ndfl))+' (FORMAT PARQUET)')
        toy_ndfl={'asset_id':'02::5ndfl::SYNTHETIC_SQL_CONTRACT','columns_manifest':list(ndfl.columns)}
        a,sl2,z=select_dimensional_slice(toy_ndfl,tmp_ndfl,'SYNTHETIC_NDFL_TOTAL','Общая сумма денежных доходов','рублей','G1')
        ensure(a is not None and len(a)>=100,'SQL_UNIT_NDFL_NO_TOTAL_SLICE')
        ensure(sl2=={'income':'всего','tax_rate':'всего'},'SQL_UNIT_NDFL_NARROW_SLICE_INCLUDED')
        record=DIMENSION_AUDIT[-1]
        ensure(record['slices_found']>2000 and record['semantically_eligible']==1 and record['inspected']==1 and record['remaining']==0,'SQL_UNIT_NDFL_LIMIT_ORDER')
        mark('SQL_DUCKDB_UNIT_CONTRACT','PASS',f'real DuckDB queries on Parquet; quoted columns; numeric parsing; >{record["slices_found"]} NDFL slices -> 1 eligible total')
        return {'status':'PASS','housing_rows':len(dat),'ndfl_total_rows':len(a),'ndfl_slices_found':record['slices_found'],'ndfl_semantically_eligible':1,'real_duckdb':True,'synthetic_only':True}
    finally:
        try:con.unregister('ml03_fixture_df');con.unregister('ml03_ndfl_fixture_df')
        except Exception:pass
        tmp.unlink(missing_ok=True);tmp_ndfl.unlink(missing_ok=True)
        del SQL_EXECUTION_AUDIT[audit0:];del DIMENSION_AUDIT[dim0:];del conflict_rows[conf0:]

SQL_DUCKDB_CONTRACT_REPORT=run_duckdb_sql_contracts()
REQUIRED_SQL_PREFLIGHT={'02::housing::00::00':['Y48010001','Y48011010'],'02::education::00::00':['Y48014001','Y48014003'],'02::budget::00::00':['Y48013001','Y48013002']}

def execute_real_sql_preflight():
    for aid,codes in REQUIRED_SQL_PREFLIGHT.items():
        if aid not in LOCAL_ASSETS:
            for code in codes:REAL_SQL_PREFLIGHT.append({'asset_id':aid,'code':code,'status':'NOT_DOWNLOADED','n_rows':0})
            continue
        a=get_asset(aid)
        for code in codes:
            try:
                row=sql_run('SELECT COUNT(*) FROM read_parquet(?) WHERE TRIM(CAST('+qident('indicator_code')+' AS VARCHAR))=?',[str(LOCAL_ASSETS[aid]),code],asset_id=aid,code=code,stage='REAL_PREFLIGHT_COUNT',preview=True)
                n=int(row[0]);REAL_SQL_PREFLIGHT.append({'asset_id':aid,'code':code,'status':'PASS' if n>0 else 'NO_CODE_ROWS','n_rows':n})
                raw,_,with_unit=factor_raw_sql(a)
                header_cols='indicator_name, indicator_unit' if 'indicator_unit' in a['columns_manifest'] else 'indicator_name'
                hdr=sql_run('SELECT '+header_cols+' FROM read_parquet(?) WHERE TRIM(CAST('+qident('indicator_code')+' AS VARCHAR))=? LIMIT 1',[str(LOCAL_ASSETS[aid]),code],asset_id=aid,code=code,stage='REAL_PREFLIGHT_HEADER')
                if len(hdr):
                    name=str(hdr.iloc[0]['indicator_name']).strip()
                    unit=str(hdr.iloc[0]['indicator_unit']).strip() if with_unit and pd.notna(hdr.iloc[0]['indicator_unit']) else ''
                    p=[str(LOCAL_ASSETS[aid]),code,name]+([unit] if with_unit else [])
                    observed=sql_run('SELECT COUNT(*) FROM ('+raw+') source_example',p,asset_id=aid,code=code,stage='REAL_PREFLIGHT_RAW_JOINED',preview=True)
                    joined=int(observed[0]);REAL_SQL_PREFLIGHT[-1]['n_joined']=joined
                    if joined==0:REAL_SQL_PREFLIGHT[-1]['status']='NO_MATCHED_MUNICIPAL_YEARS'
                    else:
                        dat,chosen_dims,validation=select_dimensional_slice(a,LOCAL_ASSETS[aid],code,name,unit,source_group(a))
                        if dat is None or dat.empty:
                            REAL_SQL_PREFLIGHT[-1]['status']='NO_VALIDATED_DIMENSIONAL_SLICE'
                            REAL_SQL_PREFLIGHT[-1]['slice_rejection_reason']=LAST_SLICE_REASON
                        elif dat.duplicated(['entity_id','observation_year']).any():
                            REAL_SQL_PREFLIGHT[-1]['status']='DUPLICATE_ENTITY_YEAR'
                        else:
                            REAL_SQL_PREFLIGHT[-1]['selected_keys']=int(len(dat))
                            REAL_SQL_PREFLIGHT[-1]['selected_nonmissing_values']=int(dat['value'].notna().sum())
                            REAL_SQL_PREFLIGHT[-1]['selected_dimensions_json']=json.dumps(chosen_dims,ensure_ascii=False)
                            if not dat['value'].notna().any():REAL_SQL_PREFLIGHT[-1]['status']='NO_NUMERIC_VALUES'
                else:REAL_SQL_PREFLIGHT[-1]['status']='NO_HEADER'
            except Exception as ex:
                REAL_SQL_PREFLIGHT.append({'asset_id':aid,'code':code,'status':'SCAN_ERROR','n_rows':0,'exception_type':type(ex).__name__,'message':str(ex)[:300]})
    return pd.DataFrame(REAL_SQL_PREFLIGHT)

REAL_SQL_PREFLIGHT_DF=execute_real_sql_preflight()
if CFG['MODE']=='FULL' and CFG['SQL_PREFLIGHT_REAL_CODES']:
    failed=REAL_SQL_PREFLIGHT_DF[~REAL_SQL_PREFLIGHT_DF.status.eq('PASS')]
    if len(failed):
        out=WORK/'ml03_sql_preflight_failures.json';out.write_text(json.dumps(REAL_SQL_PREFLIGHT,ensure_ascii=False,indent=2),encoding='utf-8')
        mark('REAL_STAGE1_SQL_PREFLIGHT','FAIL',f'Failed real SQL tests: {len(failed)}/6. Details: {out.name}')
        raise RuntimeError('FAIL_PIPELINE_SQL_PREFLIGHT: '+str(failed[['asset_id','code','status']].to_dict('records')))
mark('REAL_STAGE1_SQL_PREFLIGHT','PASS' if REAL_SQL_PREFLIGHT_DF.status.eq('PASS').all() else 'PARTIAL',f"real source code checks={len(REAL_SQL_PREFLIGHT_DF)}, passed={int(REAL_SQL_PREFLIGHT_DF.status.eq('PASS').sum())}; DuckDB fixture={SQL_DUCKDB_CONTRACT_REPORT['status']}")

progress('INVENTORY',0,len(FACTOR_ASSETS),'Аудит схем и индикаторов')
for _fi,a in enumerate(FACTOR_ASSETS,1):
    aid=a['asset_id'];groups=_grp_eligible_from_source(a)
    progress('INVENTORY',_fi,len(FACTOR_ASSETS),f'{_fi} из {len(FACTOR_ASSETS)} массивов')
    if aid not in LOCAL_ASSETS:
        SOURCE_QA.append({'asset_id':aid,'group':source_group(a),'status':'NOT_DOWNLOADED'});continue
    if not eligible_municipal_long_schema(a):
        SOURCE_QA.append({'asset_id':aid,'group':source_group(a),'status':'SCHEMA_NOT_ADAPTED','reason':'WIDE_OR_NON_MUNICIPAL_UNVALIDATED'});continue
    for g in groups:
        try:
            inv=inventory_asset(a,LOCAL_ASSETS[aid],g)
            SOURCE_QA.append({'asset_id':aid,'group':g,'status':'INVENTORIED_EXHAUSTIVE','n_candidates':len(inv),'cutoff_year':TRAIN_CATALOG_MAX_YEAR})
            if len(inv):
                inv['asset_id']=aid;inv['group']=g;CATALOG_ROWS.append(inv)
        except Exception as ex:
            SOURCE_QA.append({'asset_id':aid,'group':g,'status':'INVENTORY_ERROR','error_type':type(ex).__name__,'error_message':str(ex)[:1000]})
            log('INVENTORY_ERROR '+aid+': '+type(ex).__name__+' '+str(ex)[:400],'WARN')

progress('INVENTORY',1,1,'Инвентаризация завершена')
CATALOG=pd.concat(CATALOG_ROWS,ignore_index=True) if CATALOG_ROWS else pd.DataFrame()
if len(CATALOG):
    result=[classify_indicator(x['group'],x['asset_id'],x['indicator_name'],x['indicator_unit']) for x in CATALOG.to_dict('records')]
    CATALOG['domain']=[v[0] for v in result];CATALOG['semantic_rejection_reason']=[v[1] for v in result]
    for x in CATALOG.loc[CATALOG.domain.isna()].to_dict('records'):
        CANDIDATE_AUDIT.append({'asset_id':x['asset_id'],'code':x['indicator_code'],'name':x['indicator_name'],'decision':x['semantic_rejection_reason'],'slice':''})
    inspection_by_group=collections.Counter()
    _scan_total=int(CATALOG.loc[CATALOG.group.isin(('G1','G2','G3','G4'))&CATALOG.domain.notna()].shape[0])
    _scan_done=0
    progress('SCAN',0,_scan_total,'Размерные разрезы показателей')
    for g in ('G1','G2','G3','G4'):
        ranked=CATALOG.loc[CATALOG.group.eq(g)&CATALOG.domain.notna()].sort_values(['recent_keys','n_municipalities','n_years'],ascending=False)
        for idx,row in enumerate(ranked.to_dict('records')):
            _scan_done+=1
            if _scan_done%5==0 or _scan_done==_scan_total:
                progress('SCAN',_scan_done,_scan_total,f'{g}: {_scan_done}/{_scan_total}')
            if inspection_by_group[g]>=CFG['MAX_DIMENSIONAL_SCANS']:
                SCAN_AUDIT.append({'group':g,'asset_id':row['asset_id'],'code':row['indicator_code'],'status':'NOT_INSPECTED_CAP','reason':'RESOURCE_BUDGET','rank':idx})
                continue
            inspection_by_group[g]+=1;a=get_asset(row['asset_id'])
            try:
                dat,dims,support=select_dimensional_slice(a,LOCAL_ASSETS[a['asset_id']],row['indicator_code'],row['indicator_name'],row['indicator_unit'],g)
                reason='ELIGIBLE' if dat is not None else LAST_SLICE_REASON
                SCAN_AUDIT.append({'group':g,'asset_id':a['asset_id'],'code':row['indicator_code'],'status':reason,'rank':idx})
                if dat is None:continue
                feature_score=(int(support['n_recent_keys']),int(support['n_train_keys']))
                FEATURE_CANDIDATES.append({'group':g,'domain':row['domain'],'asset_id':a['asset_id'],'code':row['indicator_code'],'name':row['indicator_name'],'unit':row['indicator_unit'],'dims':dims,'support':support,'data':dat,'score':feature_score})
            except Exception as ex:
                SCAN_AUDIT.append({'group':g,'asset_id':a['asset_id'],'code':row['indicator_code'],'status':'SCAN_ERROR','reason':type(ex).__name__+': '+str(ex)[:400],'rank':idx})
                log('DIMENSION_SCAN '+a['asset_id']+': '+type(ex).__name__+' '+str(ex)[:400],'WARN')
    incomplete=any(r['status'] in ('NOT_INSPECTED_CAP','SCAN_ERROR') for r in SCAN_AUDIT)
else:incomplete=False
progress('SCAN',1,1,'Предметный отбор завершён')

SCAN_AUDIT=pd.DataFrame(SCAN_AUDIT);DIMENSION_AUDIT=pd.DataFrame(DIMENSION_AUDIT)
if len(CATALOG):
    g1=CATALOG.loc[CATALOG.group.eq('G1')].copy()
    if g1.empty:
        G1_DISCOVERY=pd.DataFrame([{'asset_id':a['asset_id'],'domain':'UNRESOLVED','discovered_codes':0,'discovered_rows':0,'reason_code':'UNADAPTED_MUNICIPAL_WIDE' if a['asset_id'] in UNADAPTED_MUNICIPAL_WIDE else 'NO_MATCHING_G1_INDICATORS_AT_VALIDATED_MUNICIPAL_GRAIN'} for a in ASSETS if a['asset_id']!='01::0'])
    g1['scan_key']=g1.asset_id.astype(str)+'|'+g1.indicator_code.astype(str)+'|'+g1.indicator_name.astype(str)
    insp=SCAN_AUDIT.loc[SCAN_AUDIT.group.eq('G1')].copy() if len(SCAN_AUDIT) else pd.DataFrame()
    if not g1.empty:
        G1_DISCOVERY=g1.groupby(['asset_id','domain'],dropna=False).agg(discovered_codes=('indicator_code','nunique'),discovered_rows=('indicator_name','size'),earliest_year=('min_year','min'),latest_year=('max_year','max'),max_municipalities=('n_municipalities','max')).reset_index()
    checked=insp.groupby('asset_id',dropna=False).status.value_counts().unstack(fill_value=0) if len(insp) else pd.DataFrame()
    if len(checked):
        for label in ('ELIGIBLE','NO_VALID_TOTAL_SLICE','NO_VALID_SLICE_AFTER_SUPPORT_CHECK','SCAN_ERROR','NOT_INSPECTED_CAP'):
            G1_DISCOVERY['n_'+label.lower()]=G1_DISCOVERY.asset_id.map(checked.get(label,pd.Series(dtype=int))).fillna(0).astype(int)
    G1_DISCOVERY['data_grain']='MUNICIPALITY_YEAR_EXACT_OKTMO_OR_NOT_VALIDATED'
    if not g1.empty:G1_DISCOVERY['reason_code']=np.where(G1_DISCOVERY.domain.isna(),'NO_SEMANTIC_MATCH','AUDIT_SLICES_IN_SCAN_AUDIT')
else:
    G1_DISCOVERY=pd.DataFrame([{'asset_id':a['asset_id'],'domain':None,'discovered_codes':0,'discovered_rows':0,'reason_code':'NO_MATCHED_MUNICIPAL_INDICATORS'} for a in ASSETS])

G1_EMPTY_CAUSE='NOT_FULLY_INSPECTED' if incomplete else 'UNADAPTED_MUNICIPAL_WIDE' if UNADAPTED_MUNICIPAL_WIDE else 'NO_VALIDATED_ECONOMIC_FEATURE_ON_STAGE_I'
DISCOVERY_EXHAUSTIVE=not incomplete and not UNADAPTED_MUNICIPAL_WIDE and (DIMENSION_AUDIT.empty or not DIMENSION_AUDIT.status.eq('CANDIDATE_BUDGET_PARTIAL').any()) and not any(r.get('status')=='INVENTORY_ERROR' for r in SOURCE_QA) and not SQL_FAILURES
mark('DISCOVERY_EXHAUSTIVENESS','PASS' if DISCOVERY_EXHAUSTIVE else 'PARTIAL',f'inspected={int(len(SCAN_AUDIT))}; unadapted_municipal_wide={len(UNADAPTED_MUNICIPAL_WIDE)}; incomplete={not DISCOVERY_EXHAUSTIVE}')

SOURCE_GRAIN_AUDIT=[]
for a in ASSETS:
    aid=a['asset_id'];records=[r for r in SOURCE_QA if r.get('asset_id')==aid]
    rawstage=next((r.get('status') for r in reversed(records) if r.get('status') in ('INVENTORY_ERROR','SCHEMA_NOT_ADAPTED','INVENTORIED_EXHAUSTIVE','NOT_DOWNLOADED')),None)
    stage=('NOT_DOWNLOADED' if aid not in LOCAL_ASSETS else 'TARGET_D01' if aid=='01::0' else 'SCANNED' if any(r.get('asset_id')==aid and r.get('status')=='ELIGIBLE' for r in SCAN_AUDIT.to_dict('records')) else 'SCAN_ERROR' if any(r.get('asset_id')==aid and r.get('status')=='SCAN_ERROR' for r in SCAN_AUDIT.to_dict('records')) else 'INVENTORIED_NO_MODEL_SLICE' if rawstage=='INVENTORIED_EXHAUSTIVE' else 'SOURCE_SCHEMA_ERROR' if rawstage=='INVENTORY_ERROR' else 'NOT_APPLICABLE_GRAIN' if not eligible_municipal_long_schema(a) else 'NOT_SEMANTICALLY_REVIEWED')
    SOURCE_GRAIN_AUDIT.append({'asset_id':aid,'downloaded_sha':aid in LOCAL_ASSETS,'columns_manifest_n':len(a.get('columns_manifest') or []),'source_grain':'MUNICIPAL_LONG_SCHEMA_AND_LEVEL_FILTER' if eligible_municipal_long_schema(a) and 'object_level' in (a.get('columns_manifest') or []) else 'MUNICIPAL_LONG_SCHEMA_NO_EXPLICIT_LEVEL' if eligible_municipal_long_schema(a) else 'NOT_VALIDATED_MUNICIPAL_LONG','stage':stage,'reason':rawstage or source_group(a)})

SOURCE_GRAIN_AUDIT=pd.DataFrame(SOURCE_GRAIN_AUDIT);SCAN_SQL_ERRORS=pd.DataFrame(SQL_FAILURES)
# The 52-source inventory is exhaustive at file/metadata level, not necessarily at validated content grain.
SOURCE_GRAIN_AUDIT['scientific_join_role']=SOURCE_GRAIN_AUDIT.stage.map(lambda s:'TARGET' if s=='TARGET_D01' else 'MUNICIPAL_LONG_CANDIDATE' if s in ('SCANNED','INVENTORIED_NO_MODEL_SLICE') else 'INDEX_ONLY_NOT_JOINED')
SCAN_SQL_AUDIT=pd.DataFrame(SQL_EXECUTION_AUDIT);REAL_SQL_PREFLIGHT_QA=pd.DataFrame(REAL_SQL_PREFLIGHT)
DATA_SMOKE_REPORT={'version':CFG['VERSION'],'asset_count':len(ASSETS),'scan_error_count':len(SQL_FAILURES),'inventory_errors':sum(r.get('status')=='INVENTORY_ERROR' for r in SOURCE_QA),'scanner_errors':int(SCAN_AUDIT.status.eq('SCAN_ERROR').sum()) if len(SCAN_AUDIT) else 0,'real_sql_preflight':REAL_SQL_PREFLIGHT,'dimension_audit':DIMENSION_AUDIT.to_dict('records') if len(DIMENSION_AUDIT) else []}
(WORK/'ml03_data_smoke.json').write_text(json.dumps(DATA_SMOKE_REPORT,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
fatal_sql=len(SQL_FAILURES)>0 or DATA_SMOKE_REPORT['inventory_errors']>0 or DATA_SMOKE_REPORT['scanner_errors']>0
mark('DATA_SMOKE','FAIL' if fatal_sql else 'PASS',f"SQL runtime errors={len(SQL_FAILURES)}, inventory errors={DATA_SMOKE_REPORT['inventory_errors']}, scanner errors={DATA_SMOKE_REPORT['scanner_errors']}")
if fatal_sql and CFG['MODE']=='FULL' and CFG['FAIL_FAST_SQL']:
    raise RuntimeError('FAIL_PIPELINE_SQL_SCAN: see '+str(WORK/'ml03_data_smoke.json')+' and SCAN_SQL_ERRORS')

selected_by_group={g:[] for g in PATTERNS};SELECTION_BY_H=[]
for g in ('G1','G2','G3','G4'):
    selected_candidates={}
    for h in sorted(FACTOR_CUTOFF_BY_H):
        cutoff=int(FACTOR_CUTOFF_BY_H[h]);candidates=[]
        for c in FEATURE_CANDIDATES:
            if c['group']!=g:continue
            z=c['data'].loc[c['data'].observation_year.le(cutoff)&c['data']['value'].notna()]
            years=int(z.observation_year.nunique());num=int(len(z));mun=int(z.entity_id.nunique())
            recent=int(z.observation_year.ge(cutoff-CFG['CATALOG_RECENT_WINDOW']).sum())
            if num<CFG['MIN_INDICATOR_OBS_TRAIN'] or years<CFG['MIN_INDICATOR_YEARS_TRAIN'] or mun<CFG['MIN_INDICATOR_MUNICIPALITIES']:continue
            candidates.append({**c,'_h_score':(recent,num,mun),'_h_rows':num})
        candidates.sort(key=lambda c:c['_h_score'],reverse=True)
        source_counts=collections.Counter();domain_counts=collections.Counter();h_selected=[];pool=candidates.copy()
        for diversified in (True,False):
            while pool and len(h_selected)<CFG['MAX_FEATURES_PER_GROUP']:
                opts=[x for x in pool if source_counts[x['asset_id']]<CFG['MAX_FEATURES_FROM_ONE_ASSET'] and (not diversified or domain_counts[x['domain']]==0)]
                if not opts:break
                chosen=max(opts,key=lambda z:(z['_h_score'][0]/(1+source_counts[z['asset_id']]),z['_h_score'][1],-domain_counts[z['domain']]))
                pool.pop(next(i for i,x in enumerate(pool) if x is chosen))
                source_counts[chosen['asset_id']]+=1;domain_counts[chosen['domain']]+=1
                key=(chosen['asset_id'],str(chosen['code']),chosen['name'],chosen['unit'],json.dumps(chosen['dims'],ensure_ascii=False,sort_keys=True))
                if key not in selected_candidates:
                    feature=f'{g}_{len(selected_by_group[g])+1:02d}';data=chosen['data'].rename(columns={'value':feature})
                    ensure(not data.duplicated(['entity_id','observation_year']).any(),'FEATURE_UNIQUE_KEY_'+feature)
                    meta={'feature':feature,'group':g,'domain':chosen['domain'],'asset_id':chosen['asset_id'],'indicator_code':chosen['code'],'indicator_name':chosen['name'],'unit':chosen['unit'],'dimensions_json':key[-1],'observed_grain':'MUNICIPAL_OKTMO_YEAR_DIMENSION_SLICE','report_available_year':'NOT_VERIFIED','availability_provenance':'CONSERVATIVE_OBSERVATION_PLUS_LAG_NOT_OFFICIAL_PUBLICATION_DATE','effective_available_year_rule':f'observation_year+{CFG["FEATURE_PUBLICATION_LAG"]}','n_observed':int(data[feature].notna().sum()),'n_recent_train_keys':int(chosen['_h_score'][0]),'first_year':int(data.observation_year.min()),'last_year':int(data.observation_year.max()),'transform':'RAW_PRESERVED; UNIT_QA_REQUIRED','selected_horizons':[]}
                    selected_candidates[key]=meta;selected_by_group[g].append(meta);FEATURE_PARTS.append(data)
                meta=selected_candidates[key];meta['selected_horizons'].append(h);h_selected.append(meta['feature'])
            if not diversified:break
        SELECTION_BY_H.append({'group':g,'horizon':h,'max_observation_year':cutoff,'n_eligible_candidates':len(candidates),'n_selected':len(h_selected),'selected_features_json':json.dumps(h_selected,ensure_ascii=False),'domain_coverage_json':json.dumps(sorted(domain_counts),ensure_ascii=False),'status':'COMPLETE_SELECTION' if h_selected else 'NO_VALIDATED_CANDIDATES'})
    for meta in selected_by_group[g]:meta['selected_horizons_json']=json.dumps(sorted(set(meta.pop('selected_horizons'))))
SELECTION_BY_H=pd.DataFrame(SELECTION_BY_H)

from sklearn.neighbors import BallTree
SPATIAL_GEOMETRY_QA=[];SPATIAL_FEATURE_AUDIT=[];SPATIAL_FUNDAMENTAL_GATE='NOT_FEASIBLE'

def _is_municipal_grain(v):
    x=str(v).casefold().strip().replace('ё','е')
    return bool(re.search(r'(^municipality$|^municipal$|муниципальн.{0,15}(?:район|округ|образован)|городск.{0,8}округ)',x))

def _true_flag(v):return str(v).strip().casefold() in ('true','1','да','yes','истина')

def _spatial_snapshot_candidates(source,demography):
    audit={'n_source':int(len(source)),'n_matched_municipal':0,'n_verified':0,'n_distinct':0,'n_conflicting_codes':0,'n_ambiguous_regions':0,'status':'NOT_FEASIBLE'}
    required={'source_oktmo','object_level','region_name','latitude','longitude','coordinates_valid','census_year'}
    if not required.issubset(source.columns):
        audit['reason']='SPATIAL_COLUMN_CONTRACT_MISSING';return pd.DataFrame(),audit
    z=source[list(required)+(['population_original'] if 'population_original' in source else [])].copy()
    z=z.rename(columns={'source_oktmo':'entity_id','census_year':'snapshot_year'})
    z['entity_id']=z.entity_id.astype('string').str.strip()
    z=z[z.entity_id.notna()&z.entity_id.str.fullmatch(r'\d{5,11}',na=False)].copy()
    z=z[z.object_level.map(_is_municipal_grain)&z.coordinates_valid.map(_true_flag)].copy()
    for c in ['latitude','longitude','snapshot_year']:z[c]=pd.to_numeric(z[c],errors='coerce')
    z=z[z.latitude.between(-89.999,89.999)&z.longitude.between(-179.999,179.999)&z.snapshot_year.between(1990,max(YEARS))].copy()
    audit['n_matched_municipal']=int(len(z))
    if z.empty:
        audit['reason']='NO_VERIFIED_MUNICIPAL_GEOMETRY_OR_VINTAGE';return pd.DataFrame(),audit
    z['snapshot_year']=z.snapshot_year.astype(int)
    d=demography[['entity_id','region_name']].drop_duplicates()
    region_conflicts=d.groupby('entity_id').region_name.nunique(dropna=False)
    ambiguous=set(region_conflicts[region_conflicts.ne(1)].index);audit['n_ambiguous_regions']=len(ambiguous)
    d=d[~d.entity_id.isin(ambiguous)]
    z=z.merge(d,on='entity_id',how='inner',validate='many_to_one',suffixes=('_ref','_d01'))
    def norm_region(x):return re.sub(r'\s+',' ',str(x).casefold().replace('ё','е')).strip()
    z=z[z.region_name_ref.map(norm_region).eq(z.region_name_d01.map(norm_region))].copy()
    if z.empty:
        audit['reason']='NO_EXACT_REGION_AND_OKTMO_MATCH';return pd.DataFrame(),audit
    counts=z.groupby(['entity_id','snapshot_year']).agg(n_lat=('latitude','nunique'),n_lon=('longitude','nunique'))
    conflicted=set(counts.index[counts.n_lat.gt(1)|counts.n_lon.gt(1)].tolist())
    audit['n_conflicting_codes']=len(conflicted)
    if conflicted:z=z[~pd.MultiIndex.from_frame(z[['entity_id','snapshot_year']]).isin(conflicted)]
    z=z.drop_duplicates(['entity_id','snapshot_year','latitude','longitude']).sort_values(['entity_id','snapshot_year'])
    audit['n_verified']=len(z);audit['n_distinct']=z.entity_id.nunique()
    if z.entity_id.nunique()<max(30,int(CFG['MIN_INDICATOR_MUNICIPALITIES'])):
        audit['reason']='TOO_FEW_EXACT_VERIFIED_MUNICIPALITY_COORDS';return pd.DataFrame(),audit
    audit['status']='PASS';audit['reason']='TIME_STAMPED_VERIFIED_GEO_EXACT_MUNICIPAL_OKTMO_REGION'
    return z[['entity_id','snapshot_year','latitude','longitude','region_name_d01']],audit

def _snapshot_knn(z):
    out=[];earth=6371.0088
    for year in sorted(z.snapshot_year.unique()):
        available=z[z.snapshot_year.le(year)].sort_values('snapshot_year').drop_duplicates('entity_id',keep='last')
        this=z[z.snapshot_year.eq(year)].copy()
        if len(available)<5:continue
        latlon=np.radians(available[['latitude','longitude']].to_numpy(dtype=float))
        query=np.radians(this[['latitude','longitude']].to_numpy(dtype=float))
        tree=BallTree(latlon,metric='haversine');dist,idx=tree.query(query,k=min(5,len(available)))
        for i,v in enumerate(this.itertuples(index=False)):
            neighbours=[(float(dd)*earth,available.iloc[int(j)].entity_id) for dd,j in zip(dist[i],idx[i]) if available.iloc[int(j)].entity_id!=v.entity_id]
            if not neighbours:continue
            nearby=tree.query_radius(query[i:i+1],r=50/earth)[0]
            n_within_50=sum(available.iloc[int(j)].entity_id!=v.entity_id for j in nearby)
            out.append({'entity_id':v.entity_id,'snapshot_year':int(year),'nearest_municipality_km':neighbours[0][0],'municipalities_50km_n':int(n_within_50),'known_neighbour_universe_n':len(available)})
    return pd.DataFrame(out)

_SPATIAL_SOURCE_ID='06::D04::D04_member00::spatial_reference::derived'
if _SPATIAL_SOURCE_ID in LOCAL_ASSETS:
    try:
        spatial_source=pd.read_parquet(LOCAL_ASSETS[_SPATIAL_SOURCE_ID],columns=['source_oktmo','object_level','region_name','latitude','longitude','coordinates_valid','census_year'])
        geo,qa=_spatial_snapshot_candidates(spatial_source,base)
        SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,**qa})
        if not geo.empty:
            near=_snapshot_knn(geo)
            if not near.empty:
                pubyear=CFG.get('SPATIAL_PUBLICATION_YEAR')
                pubref=str(CFG.get('SPATIAL_PUBLICATION_EVIDENCE','')).strip()
                if pubyear is None or not pubref or not CFG.get('SPATIAL_BOUNDARY_CROSSWALK_VERIFIED',False) or not str(CFG.get('SPATIAL_BOUNDARY_CROSSWALK_EVIDENCE','')).strip():
                    SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,'status':'NOT_FEASIBLE','reason':'PUBLICATION_YEAR_OR_BOUNDARY_CROSSWALK_UNVERIFIED','n_verified_geometry':int(len(near))})
                    near=pd.DataFrame()
            if not near.empty and int(pubyear)>max(YEARS):
                SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,'status':'NOT_FEASIBLE','reason':'VERIFIED_PUBLICATION_AFTER_LAST_TARGET_YEAR'})
                near=pd.DataFrame()
            if not near.empty:
                yrs=pd.DataFrame({'observation_year':YEARS})
                snapshots=near.loc[near.snapshot_year.le(max(YEARS))].copy().merge(yrs,how='cross')
                snapshots=snapshots.loc[snapshots.observation_year.ge(np.maximum(snapshots.snapshot_year,int(pubyear)))]
                snapshots=snapshots.sort_values('snapshot_year').drop_duplicates(['entity_id','observation_year'],keep='last')
                for f,un,domain,col in (('G5_01_km','км','spatial_distance','nearest_municipality_km'),('G5_02_neighbors','число муниципалитетов','spatial_density','municipalities_50km_n')):
                    data=snapshots[['entity_id','observation_year',col]].rename(columns={col:f})
                    data[f]=pd.to_numeric(data[f],errors='coerce')
                    ensure(not data.duplicated(['entity_id','observation_year']).any(),'SPATIAL_TIME_KEY_DUPLICATE')
                    FEATURE_PARTS.append(data)
                    selected_by_group['G5'].append({'feature':f,'group':'G5','domain':domain,'asset_id':_SPATIAL_SOURCE_ID,'indicator_code':'D04_VERIFIED_GEO_'+domain.upper(),'indicator_name':'Подтверждённое пространственное расстояние' if domain=='spatial_distance' else 'Муниципалитеты в радиусе 50 км','unit':un,'dimensions_json':'{}','observed_grain':'MUNICIPAL_GEOMETRY_VINTAGE','report_available_year':f'max(snapshot_year,{pubyear})+publication_lag','effective_available_year_rule':f'max(census_year,{pubyear})+{CFG["FEATURE_PUBLICATION_LAG"]}<=origin_year','n_observed':int(data[f].notna().sum()),'n_recent_train_keys':int(data[data.observation_year.le(TRAIN_CATALOG_MAX_YEAR)][f].notna().sum()),'first_year':int(data.observation_year.min()),'last_year':int(data.observation_year.max()),'transform':'SOURCE_VINTAGE_GEOMETRY; NO FUTURE_COORDINATES'})
                    SPATIAL_FEATURE_AUDIT.append({'feature':f,'n_rows':len(data),'n_municipalities':data.entity_id.nunique(),'snapshot_min':int(near.snapshot_year.min()),'snapshot_max':int(near.snapshot_year.max()),'model_eligible':'PRELIMINARY_TIMING_GATE'})
                SPATIAL_FUNDAMENTAL_GATE='PASS'
    except Exception as ex:
        SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,'status':'NOT_FEASIBLE','reason':'SPATIAL_REFERENCE_REJECTED_'+type(ex).__name__})
        log('SPATIAL_DATA_GATE: '+type(ex).__name__+': '+str(ex)[:250],'WARN')
else:
    SPATIAL_GEOMETRY_QA.append({'asset_id':_SPATIAL_SOURCE_ID,'status':'NOT_FEASIBLE','reason':'SPATIAL_VINTAGE_FILE_NOT_DOWNLOADED'})

SPATIAL_GEOMETRY_QA.append({'asset_id':'06::TRANSPORT::TRANSPORT_member00::full','status':'NOT_FEASIBLE','reason':'TRANSPORT_OBSERVATION_YEAR_NOT_PROVIDED'})
SPATIAL_FEATURE_AUDIT=pd.DataFrame(SPATIAL_FEATURE_AUDIT);SPATIAL_GEOMETRY_QA=pd.DataFrame(SPATIAL_GEOMETRY_QA)
G5_DIAGNOSTICS.extend(SPATIAL_GEOMETRY_QA.to_dict('records'))

# V3.00: independent region-level economic CONTEXT. Never claim these are
# municipality-specific wages/employment. Strict object-level and exact region/year.
REGIONAL_CONTEXT_PARTS=[];REGIONAL_CONTEXT_META=[];REGIONAL_CONTEXT_AUDIT=[]
D05_ECONOMIC_SOURCES=('02::labor::','02::socio1::','02::socio2::','02::socio3::','02::social::')
def _region_norm(s):return re.sub(r'\s+',' ',str(s).casefold().replace('ё','е')).strip()
def _region_context_collect():
    if not CFG.get('ENABLE_REGIONAL_ECONOMIC_CONTEXT'):return
    for asset in ASSETS:
        aid=asset['asset_id']
        if aid not in LOCAL_ASSETS or not aid.startswith(D05_ECONOMIC_SOURCES):continue
        cols=set(asset.get('columns_manifest') or [])
        if not {'object_level','object_name','indicator_name','indicator_value','year','indicator_code','indicator_unit'}.issubset(cols):continue
        sql="""SELECT CAST(object_name AS VARCHAR) region, TRY_CAST(year AS INTEGER) yr,
             CAST(indicator_code AS VARCHAR) code, CAST(indicator_name AS VARCHAR) title,
             CAST(indicator_unit AS VARCHAR) unit, CAST(indicator_value AS VARCHAR) val,
             CAST(object_level AS VARCHAR) object_level
             FROM read_parquet(?) WHERE TRY_CAST(year AS INTEGER) BETWEEN ? AND ?
             AND regexp_matches(lower(coalesce(cast(indicator_name AS VARCHAR),'')),?)"""
        try:
            x=sql_run(sql,[str(LOCAL_ASSETS[aid]),min(YEARS),max(YEARS),PATTERNS['G1']],asset_id=aid,stage='REGION_CONTEXT_DISCOVERY')
        except Exception as exc:
            REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'SQL_FAILURE','reason':type(exc).__name__});continue
        if x.empty:
            REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'NO_G1_CANDIDATES'});continue
        lev=x.object_level.astype('string').str.casefold().fillna('')
        x=x[lev.str.contains(r'регион|субъект',regex=True,na=False)].copy()
        if x.empty:
            REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'NO_VALID_REGION_ROWS'});continue
        x['region_key']=x.region.map(_region_norm);x['yr']=pd.to_numeric(x.yr,errors='coerce')
        x=x[x.yr.notna()].copy();x['yr']=x.yr.astype(int)
        regions={_region_norm(v) for v in base.region_name.dropna().unique()}
        x=x[x.region_key.isin(regions)].copy()
        if x.empty:
            REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'REGION_NAMES_NOT_EXACT'});continue
        x['value']=pd.to_numeric(x.val.astype('string').str.replace(' ','',regex=False).str.replace(',','.',regex=False),errors='coerce')
        x=x[x.value.notna()];candidates=0
        for (code,title,unit),s in x.groupby(['code','title','unit'],dropna=False,sort=False):
            domain,_=classify_indicator('G1',aid,title,unit)
            if domain is None:continue
            # Only intensive regional quantities: an explicit rate, percentage,
            # average wage or average per-capita income; no regional totals.
            text=(str(title)+' '+str(unit)).casefold()
            intensive=bool(re.search(r'средн|на душу|на одного|в расчете на|уровень безработ|коэффициент|процент|%|дол[яи]',text))
            if not intensive:continue
            key=['region_key','yr'];span=s.groupby(key)['value'].agg(['size','nunique','first']).reset_index()
            clashes=int(span['nunique'].gt(1).sum())
            if clashes:
                REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'code':str(code),'status':'CONFLICTING_REGION_YEAR','n_conflicts':clashes});continue
            z=span[['region_key','yr','first']].rename(columns={'yr':'observation_year'})
            before=z[z.observation_year.le(TRAIN_CATALOG_MAX_YEAR)]
            if before.region_key.nunique()<CFG['REGIONAL_ECONOMIC_MIN_REGIONS'] or before.observation_year.nunique()<2:continue
            f=f'G1R_{len(REGIONAL_CONTEXT_META)+1:02d}'
            z=z.rename(columns={'first':f})
            REGIONAL_CONTEXT_META.append({'feature':f,'group':'G1R','domain':domain,
                 'source_grain':'REGION_YEAR_ONLY','source_id':aid,'indicator_code':str(code),
                 'indicator_name':str(title),'unit':str(unit),'n_regions':int(z.region_key.nunique()),
                 'n_years':int(z.observation_year.nunique()),'n_conflicts':clashes,
                 'note':'NOT_MUNICIPAL_G1; no region-to-municipality economic reconstruction'})
            REGIONAL_CONTEXT_PARTS.append(z)
            candidates+=1
            if len(REGIONAL_CONTEXT_META)>=CFG['REGIONAL_ECONOMIC_MAX_FEATURES']:break
        REGIONAL_CONTEXT_AUDIT.append({'asset_id':aid,'status':'AUDITED_REGION_CONTEXT',
            'candidate_rows':int(len(x)),'accepted_features_from_source':candidates})
        if len(REGIONAL_CONTEXT_META)>=CFG['REGIONAL_ECONOMIC_MAX_FEATURES']:break
_region_context_collect()
REGIONAL_CONTEXT_AUDIT=pd.DataFrame(REGIONAL_CONTEXT_AUDIT)
REGIONAL_CONTEXT_METADATA=pd.DataFrame(REGIONAL_CONTEXT_META)
mark('G1_REGIONAL_CONTEXT_DISCOVERY','PASS' if REGIONAL_CONTEXT_META else 'NOT_FEASIBLE',
     f'{len(REGIONAL_CONTEXT_META)} exact-match region-year economic indicators, strictly separate from municipal G1')

FEATURE_DICT=pd.DataFrame(sum(selected_by_group.values(),[]))
SOURCE_QA=pd.DataFrame(SOURCE_QA);CONFLICT_QA=pd.DataFrame(conflict_rows)
SPATIAL_SOURCE_QA=pd.DataFrame(G5_DIAGNOSTICS);SEMANTIC_QA=pd.DataFrame(CANDIDATE_AUDIT)
progress('FACTORS',1,1,'Срезы и факторы рассчитаны')
DOMAIN_COVERAGE=pd.DataFrame([{'group':g,'domain':d,'n_selected':sum(x['domain']==d for x in selected_by_group[g])} for g in SEMANTIC_DOMAINS for d,_ in SEMANTIC_DOMAINS[g]])
for g in PATTERNS:
    domains={r['domain'] for r in selected_by_group[g]}
    required=2 if g=='G3' else 1
    status='PASS' if len(domains)>=required else 'PARTIAL' if domains else 'NOT_FEASIBLE'
    if g=='G1' and not DISCOVERY_EXHAUSTIVE:status='PARTIAL'
    mark('FEATURE_'+g,status,f'{len(selected_by_group[g])} valid indicators; domains={sorted(domains)}')

#### Построение панели, проверка размерностей и документирование пространственной неопределённости.

In [ ]:
# 06 · V3.01 · RESEARCH PANEL — VERIFIED G1R, UNITS, TIME AND LEVELS
PANEL=base[['entity_id','observation_year','region_name','mun_type','municipality','population','average_population','migration','migration_rate','births','deaths','migration_per_1000','boundary_uncertainty','source_vintage']].copy()
merge_audit=[]
for data in FEATURE_PARTS:
    f=data.columns[-1]; n=len(PANEL)
    ensure(not data.duplicated(['entity_id','observation_year']).any(),'SOURCE_JOIN_MANY_TO_MANY_'+f)
    PANEL=PANEL.merge(data,on=['entity_id','observation_year'],how='left',validate='one_to_one')
    ensure(len(PANEL)==n,'PANEL_CARDINALITY_CHANGED_'+f)
    merge_audit.append(dict(feature=f,before_rows=n,after_rows=len(PANEL),nonmissing=int(PANEL[f].notna().sum()),status='ONE_TO_ONE'))
PANEL=PANEL.sort_values(['entity_id','observation_year']).reset_index(drop=True); QA_KEYS=pd.DataFrame(merge_audit)

HISTORY_ID='06::BDMO6::BDMO6_member00::municipality_history_observed::derived'
PANEL['history_observed']=False; PANEL['observed_name_change']=False; PANEL['observed_type_change']=False; PANEL['boundary_change_not_verified']=True
if HISTORY_ID in LOCAL_ASSETS:
    try:
        hh=pd.read_parquet(LOCAL_ASSETS[HISTORY_ID],columns=['source_oktmo','observation_year','observed_name_change','observed_type_change','boundary_change_not_verified'])
        hh['entity_id']=hh.source_oktmo.astype('string').str.strip().astype(str); hh['observation_year']=pd.to_numeric(hh.observation_year,errors='coerce')
        hh=hh[hh.observation_year.notna()].copy(); hh['observation_year']=hh.observation_year.astype(int)
        flags=['observed_name_change','observed_type_change','boundary_change_not_verified']
        for c in flags: hh[c]=hh[c].astype('string').str.lower().isin(['true','1','да','yes'])
        hh=hh.groupby(['entity_id','observation_year'],as_index=False)[flags].max(); hh['history_observed']=True
        PANEL=PANEL.drop(columns=['history_observed']+flags).merge(hh,on=['entity_id','observation_year'],how='left',validate='one_to_one')
        for c in ['history_observed','observed_name_change','observed_type_change']: PANEL[c]=PANEL[c].astype('boolean').fillna(False).astype(bool)
        PANEL['boundary_change_not_verified']=PANEL['boundary_change_not_verified'].astype('boolean').fillna(True).astype(bool)
    except Exception as exc:
        PANEL['history_observed']=False; PANEL['observed_name_change']=False; PANEL['observed_type_change']=False; PANEL['boundary_change_not_verified']=True
        mark('HISTORY_BOUNDARY_QA','PARTIAL','Audit failed: '+type(exc).__name__)
PANEL['boundary_year_at_risk']=~PANEL.history_observed | PANEL.boundary_change_not_verified | PANEL.observed_name_change | PANEL.observed_type_change
PANEL['boundary_evidence_status']=np.select([~PANEL.history_observed,PANEL.observed_name_change|PANEL.observed_type_change,PANEL.boundary_change_not_verified],['NO_HISTORY_RECORD','OBSERVED_NAME_OR_TYPE_CHANGE','BOUNDARY_NOT_VERIFIED'],default='NO_OBSERVED_CHANGE_NOT_PROOF_OF_STABILITY')
BOUNDARY_QA=pd.DataFrame([{'metric':c,'n_yearly_observations_flagged':int(PANEL[c].sum())} for c in ['history_observed','boundary_change_not_verified','boundary_year_at_risk','observed_name_change','observed_type_change']])
mark('PASS_BOUNDARY_AUDIT','PASS',f'Unverified entity-years={int(PANEL.boundary_year_at_risk.sum())}/{len(PANEL)}; NEVER coerced to stable')

def mark_boundary_forecast_window(t,reference):
    z=reference[['entity_id','observation_year','boundary_year_at_risk']].sort_values(['entity_id','observation_year']).copy()
    z['cum_obs']=z.groupby('entity_id').cumcount()+1; z['cum_bad']=z.boundary_year_at_risk.astype(int).groupby(z.entity_id).cumsum()
    a=z.rename(columns={'observation_year':'origin_year','cum_obs':'start_obs','cum_bad':'start_bad'}); b=z.rename(columns={'observation_year':'target_year','cum_obs':'end_obs','cum_bad':'end_bad'})
    out=t.merge(a[['entity_id','origin_year','start_obs','start_bad']],on=['entity_id','origin_year'],how='left',validate='many_to_one')
    out=out.merge(b[['entity_id','target_year','end_obs','end_bad']],on=['entity_id','target_year'],how='left',validate='many_to_one')
    ensure(len(out)==len(t),'BOUNDARY_JOIN_CARDINALITY_CHANGE')
    miss=out[['start_obs','start_bad','end_obs','end_bad']].isna().any(axis=1)
    out['boundary_period_uncertain']=(miss | (out.end_obs-out.start_obs).ne(out.horizon) | (out.end_bad-out.start_bad).gt(0)).astype('boolean').fillna(True).astype(bool)
    return out.drop(columns=['start_obs','start_bad','end_obs','end_bad'])

RATIO_REGEX=r'на\s*(?:одного|одну|одно|1\b|1000\b|10\s*тыс|душу|жител|человек|учащ|предприяти)|в\s+расч[её]те\s+на|в\s+среднем\s+на|удельн|на\s+единиц'
RATE_REGEX=r'(?:%|процент|коэффициент|индекс|дол[яи]\s|отношение|промилле)'
def unit_policy(indicator_name,unit,group,year=2020):
    title=str(indicator_name).casefold().replace('ё','е'); raw=str(unit).casefold().replace('ё','е'); both=title+' '+raw
    m=re.search(r'с\s*(\d{4})\s*г?\.?\s*[-–—:, ]*',raw); eff=(raw[m.end():] if year>=int(m.group(1)) else raw[:m.start()]) if m and re.search(r'рубл|руб',raw) else raw
    rub=bool(re.search(r'рубл|руб\b',eff)); scale=1e6 if re.search(r'млн\.?\s*руб|миллион.{0,4}руб',eff) else 1e3 if re.search(r'тыс\.?\s*руб|тысяч.{0,5}руб',eff) else 1.
    def result(action,mult=1.,den='NONE',family='UNCLASSIFIED',status='PASS'):
        return dict(action=action,multiplier=mult,denominator=den,effective_unit=eff,unit_family=family,status=status)
    if group=='G5' and re.search(r'км|километр|число муниципалитетов|муниципалитет',raw): return result('KEEP_SPATIAL_MEASUREMENT',family='SPATIAL_REFERENCE')
    wage=bool(re.search(r'(средн.{0,20}(?:заработн|начисленн.{0,9}зарплат)|заработн.{0,9}плат.{0,8}одн.{0,9}работник)',both))
    if wage and rub:return result('KEEP_RAW_AVERAGE',scale,family='AVERAGE_WAGE_PER_WORKER')
    if re.search(RATIO_REGEX,both):return result('KEEP_ALREADY_NORMALIZED',scale,family='RATIO_OR_PER_ENTITY')
    if re.search(RATE_REGEX,both):return result('KEEP_RATE',family='RATE')
    if rub:
        action='PER_RESIDENT' if group in ['G1','G2','G3','G4'] else 'KEEP_RAW'
        return result(action,scale,'D01_population_same_year' if action=='PER_RESIDENT' else 'NONE','MONEY_NOMINAL_RUB')
    if re.search(r'человек|чел\b|единиц|ед\.|штук|штука|кв\.?\s*м|м2|квадрат|метр',eff) and group in ['G1','G2','G3']:
        if group=='G1' and re.search(r'средн|уровень|размер',title):return result('KEEP_RAW_AVERAGE',family='AVERAGE')
        return result('PER_1000_RESIDENTS',1000.,'D01_population_same_year','COUNT_OR_AREA')
    return result('KEEP_RAW_UNCLASSIFIED',family='UNCLASSIFIED',status='REVIEW')

ensure(unit_policy('Расходы ОМСУ на одного жителя','2008 г. — тысяча рублей, с 2009 г. — рубль','G4',2020)['action']=='KEEP_ALREADY_NORMALIZED','UNIT_CANARY_DOUBLE_DENOMINATOR')
ensure(unit_policy('Расходы местного бюджета','2008 г. — тыс. руб.; с 2009 г. — рубли','G4',2020)['multiplier']==1,'UNIT_CANARY_HISTORICAL_RUBLES')
ensure(unit_policy('Средняя начисленная заработная плата','тыс. рублей','G1',2020)['multiplier']==1000,'UNIT_CANARY_AVERAGE_WAGE_THOUSANDS')
ensure(unit_policy('Средняя начисленная заработная плата','рублей','G1',2020)['action']=='KEEP_RAW_AVERAGE','UNIT_CANARY_WAGE_NOT_PER_RESIDENT')
ensure(unit_policy('Расходы на одно малое предприятие','рублей','G4',2020)['denominator']=='NONE','UNIT_CANARY_PER_ENTERPRISE')
mark('UNIT_SEMANTICS_CANARY','PASS','No double ratio, mixed-year rubles correctly resolved')


# V3.00: add region-level G1R context under an explicit separate namespace.
# Exact region_key joins only; no organizational aggregation or apportionment to municipalities.
PANEL['region_key']=PANEL.region_name.map(_region_norm)
for part in REGIONAL_CONTEXT_PARTS:
    rows_before=len(PANEL)
    PANEL=PANEL.merge(part,on=['region_key','observation_year'],how='left',validate='many_to_one')
    ensure(len(PANEL)==rows_before,'REGIONAL_CONTEXT_ROW_MULTIPLICATION')
GROUP_FEATURES={g:[r['feature'] for r in selected_by_group[g]] for g in PATTERNS}
GROUP_FEATURES['G1R']=[r['feature'] for r in REGIONAL_CONTEXT_META]
# Regional quantities are not normalized by local municipal population.
# Unit conversion happens in the actual source observation year; noncomparable
# units are removed instead of filling unknown levels with guessed municipal values.
G1R_UNIT_QA=[]
for meta in REGIONAL_CONTEXT_META:
    feature=meta['feature'];unit=meta['unit'];name=meta['indicator_name']
    policies={int(y):unit_policy(name,unit,'G1R',int(y)) for y in PANEL.observation_year.unique()}
    accepted=all(p['status']=='PASS' for p in policies.values()) and len({p['action'] for p in policies.values()})==1
    action=next(iter(policies.values()))['action']
    # A regional total can never masquerade as a municipal income rate.
    accepted=bool(accepted and action in ('KEEP_RAW_AVERAGE','KEEP_ALREADY_NORMALIZED','KEEP_RATE'))
    if accepted:
        scale=PANEL.observation_year.map({y:p['multiplier'] for y,p in policies.items()})
        PANEL[feature]=pd.to_numeric(PANEL[feature],errors='coerce')*scale
    else:
        GROUP_FEATURES['G1R']=[c for c in GROUP_FEATURES['G1R'] if c!=feature]
    G1R_UNIT_QA.append({'feature':feature,'source_id':meta['source_id'],'unit':unit,
        'action':action,'decision':'ACCEPT_REGIONAL_CONTEXT' if accepted else 'EXCLUDE_REGIONAL_UNIT_AMBIGUITY',
        'source_grain':'REGION_YEAR_ONLY','n_values':int(PANEL[feature].notna().sum())})
G1R_UNIT_QA=pd.DataFrame(G1R_UNIT_QA)
audit=[]
for row in FEATURE_DICT.to_dict('records'):
    f,g,unit,title=row['feature'],row['group'],row['unit'],row['indicator_name']
    years=pd.to_numeric(PANEL.observation_year,errors='coerce'); policies={int(y):unit_policy(title,unit,g,int(y)) for y in years.dropna().unique()}
    pol=policies[int(years.max())]; ok={p['status'] for p in policies.values()}=={'PASS'} and len({p['action'] for p in policies.values()})==1
    suffix='_per_resident' if pol['action']=='PER_RESIDENT' else '_per_1000' if pol['action']=='PER_1000_RESIDENTS' else '_scaled_rub' if any(p['multiplier']!=1 for p in policies.values()) else ''
    derived=f+suffix
    if ok and suffix:
        value=pd.to_numeric(PANEL[f],errors='coerce')*years.map({y:p['multiplier'] for y,p in policies.items()})
        if pol['action'] in ['PER_RESIDENT','PER_1000_RESIDENTS']: value=value/pd.to_numeric(PANEL.population,errors='coerce').where(PANEL.population.gt(0))
        PANEL[derived]=value
    if ok: GROUP_FEATURES[g]=[derived if x==f else x for x in GROUP_FEATURES[g]]
    else: GROUP_FEATURES[g]=[x for x in GROUP_FEATURES[g] if x!=f]
    examples={int(y):float(pd.to_numeric(PANEL.loc[PANEL.observation_year.eq(y),f],errors='coerce').dropna().iloc[0]) for y in sorted(policies) if PANEL.loc[PANEL.observation_year.eq(y),f].notna().any()}
    audit.append(dict(parent_feature=f,model_feature=derived if ok else None,group=g,domain=row['domain'],indicator_name=title,asset_id=row['asset_id'],action=pol['action'],ratio_denominator=pol['denominator'],source_units=unit,effective_unit_recent=pol['effective_unit'],effective_unit_by_year_json=json.dumps({y:p['effective_unit'] for y,p in policies.items()},ensure_ascii=False),multiplier_by_year_json=json.dumps({y:p['multiplier'] for y,p in policies.items()}),yearwise_raw_value_exemplar=json.dumps(examples),unit_family=pol['unit_family'],decision='ACCEPT_DIMENSIONAL_CONVERSION' if ok else 'EXCLUDE_UNIT_AMBIGUITY',notes='Raw field preserved; per-resident conversion only where explicitly required.'))
TRANSFORM_QA=pd.DataFrame(audit,columns=['parent_feature','model_feature','group','domain','indicator_name','asset_id','action','ratio_denominator','source_units','effective_unit_recent','effective_unit_by_year_json','multiplier_by_year_json','yearwise_raw_value_exemplar','unit_family','decision','notes'])
for c in ['feature','group','domain','model_feature','unit_status']:
    if c not in FEATURE_DICT.columns: FEATURE_DICT[c]=pd.Series(dtype='object')
if len(FEATURE_DICT):
    lookup=TRANSFORM_QA.set_index('parent_feature'); FEATURE_DICT['model_feature']=FEATURE_DICT.feature.map(lookup.model_feature); FEATURE_DICT['unit_status']=FEATURE_DICT.feature.map(lookup.decision)
MODEL_DOMAIN_COVERAGE=pd.DataFrame([dict(group=g,domain=d,n_model_features=int(((TRANSFORM_QA.group==g)&(TRANSFORM_QA.domain==d)&TRANSFORM_QA.model_feature.notna()).sum())) for g in SEMANTIC_DOMAINS for d,_ in SEMANTIC_DOMAINS[g]])
for g in ['G1','G2','G3','G4']:
    valid=TRANSFORM_QA.loc[(TRANSFORM_QA.group==g)&TRANSFORM_QA.model_feature.notna(),'domain'].dropna().unique().tolist(); threshold=2 if g=='G3' else 1
    mark('FEATURE_'+g,'PASS' if len(valid)>=threshold else 'PARTIAL' if valid else 'NOT_FEASIBLE',f'Usable domains after strict unit QA: {sorted(valid)}; features={len(GROUP_FEATURES[g])}')
actual=set(MODEL_DOMAIN_COVERAGE.loc[(MODEL_DOMAIN_COVERAGE.group=='G1')&MODEL_DOMAIN_COVERAGE.n_model_features.gt(0),'domain'])
# Distinguish nominal domains from valid labour/income proxies; do not call sales 'employment'.
G1_DOMAIN_QA=pd.DataFrame([{'domain':d,'n_model_features':int(n),'adequate_for_claim':int(n)>0,
  'claim':'DESCRIPTIVE_ONLY' if d in ('business','retail','investment') else 'REQUIRES_SEMANTICALLY_VALID_DIRECT_INDICATOR'}
  for d,n in zip(MODEL_DOMAIN_COVERAGE.loc[MODEL_DOMAIN_COVERAGE.group.eq('G1'),'domain'],
                 MODEL_DOMAIN_COVERAGE.loc[MODEL_DOMAIN_COVERAGE.group.eq('G1'),'n_model_features'])])
econ_core=bool(actual & {'wages','employment','income'})
mark('G1_DOMAIN_COMPLETENESS','PASS' if len(actual)>=CFG['MIN_ECON_DOMAINS'] and econ_core else 'PARTIAL' if actual else 'NOT_FEASIBLE',
   f'observed={sorted(actual)}; direct wages/employment/income present={econ_core}; sales not employment')
HEALTH_DOMAIN_QA=pd.DataFrame([{
   'feature':r.model_feature,'name':r.indicator_name,'domain':r.domain,
   'hospital_specific':bool(re.search(r'больниц|больнич|стационар|койк|поликлиник|амбулатор',str(r.indicator_name).casefold())),
   'source_grain':'VALIDATED_MUNICIPAL' if pd.notna(r.model_feature) else 'NOT_SELECTED',
   'n_observed':int(PANEL[r.parent_feature].notna().sum()) if r.parent_feature in PANEL else 0}
    for r in TRANSFORM_QA.loc[(TRANSFORM_QA.group=='G3')&(TRANSFORM_QA.domain=='health')].itertuples()])
health=int(((HEALTH_DOMAIN_QA.hospital_specific)&(HEALTH_DOMAIN_QA.source_grain=='VALIDATED_MUNICIPAL')).sum()) if len(HEALTH_DOMAIN_QA) else 0
HEALTH_AVAILABLE=bool(health)
mark('HEALTH_INFRASTRUCTURE_GATE','PASS' if HEALTH_AVAILABLE else 'PARTIAL' if len(HEALTH_DOMAIN_QA) else 'NOT_FEASIBLE',
    f'Hospital/ambulatory indicators selected={health}, broad health candidates={len(HEALTH_DOMAIN_QA)}; horizon checks below')
mark('PASS_UNIT_AUDIT','PASS' if not TRANSFORM_QA.decision.eq('EXCLUDE_UNIT_AMBIGUITY').any() else 'PARTIAL',f"Excluded ambiguous units={int(TRANSFORM_QA.decision.eq('EXCLUDE_UNIT_AMBIGUITY').sum())}")

FACTOR_ELIGIBILITY=[]; GROUP_FEATURES_BY_H={}
for h,cutoff in FACTOR_CUTOFF_BY_H.items():
    GROUP_FEATURES_BY_H[h]={}
    for g,ff in GROUP_FEATURES.items():
        eligible=[]
        for f in ff:
            parent=next((r['parent_feature'] for r in audit if r['model_feature']==f),f)
            valid=PANEL.loc[PANEL.observation_year.le(cutoff)&PANEL[f].notna(),['entity_id','observation_year']]
            n,ny,nm=len(valid),valid.observation_year.nunique(),valid.entity_id.nunique()
            meta=next((r for r in selected_by_group.get(g,[]) if r['feature']==parent),None)
            selected=True if g in ('G5','G1R') else bool(meta is not None and h in json.loads(meta.get('selected_horizons_json','[]')))
            ok=bool(selected and n>=CFG['MIN_INDICATOR_OBS_TRAIN'] and ny>=CFG['MIN_INDICATOR_YEARS_TRAIN'] and nm>=CFG['MIN_INDICATOR_MUNICIPALITIES'])
            if ok:eligible.append(f)
            FACTOR_ELIGIBILITY.append(dict(horizon=h,group=g,feature=f,parent_feature=parent,observation_cutoff_year=cutoff,source_published_year='NOT_VERIFIED',effective_available_year_rule=f'observation_year+{CFG["FEATURE_PUBLICATION_LAG"]}',n_keys_before_cutoff=n,n_years_before_cutoff=int(ny),n_municipalities_before_cutoff=int(nm),eligible=ok,reason='TRAIN_SUPPORT_VERIFIED' if ok else 'INSUFFICIENT_PRE_ORIGIN_SUPPORT'))
        GROUP_FEATURES_BY_H[h][g]=eligible
FACTOR_ELIGIBILITY=pd.DataFrame(FACTOR_ELIGIBILITY,columns=['horizon','group','feature','parent_feature','observation_cutoff_year','source_published_year','effective_available_year_rule','n_keys_before_cutoff','n_years_before_cutoff','n_municipalities_before_cutoff','eligible','reason'])

ORIGINS=PANEL[['entity_id','region_name','mun_type','municipality','observation_year','boundary_uncertainty','source_vintage']].rename(columns={'observation_year':'origin_year'}).copy()
prior=PANEL[['entity_id','observation_year','population','migration_per_1000']].rename(columns={'observation_year':'known_year'})
lag0=CFG['FEATURE_PUBLICATION_LAG']
for lag in range(lag0,lag0+4):
    tmp=prior.rename(columns={'population':f'hist_population_l{lag}','migration_per_1000':f'hist_migration_l{lag}'})
    ORIGINS['known_year']=ORIGINS.origin_year-lag; ORIGINS=ORIGINS.merge(tmp,on=['entity_id','known_year'],how='left',validate='many_to_one').drop(columns='known_year')
G0_FEATURES=[c for c in ORIGINS if c.startswith('hist_')]+['origin_year']
ORIGINS['hist_growth_lagged_pct']=100*(ORIGINS[f'hist_population_l{lag0}']/ORIGINS[f'hist_population_l{lag0+2}']-1)
ORIGINS['hist_growth_lagged_pct']=ORIGINS.hist_growth_lagged_pct.replace([np.inf,-np.inf],np.nan); G0_FEATURES.append('hist_growth_lagged_pct')
all_factors=list(dict.fromkeys(sum(GROUP_FEATURES.values(),[])))
if all_factors:
    raw=PANEL[['entity_id','observation_year']+all_factors].copy()
    # last observed value carries forward at most MAX_FEATURE_STALENESS_YEARS,
    # with a separate age ledger and with the required publication lag.
    STALENESS_QA=[]
    for f in all_factors:
        observed=pd.to_numeric(raw[f],errors='coerce')
        lastyear=raw['observation_year'].where(observed.notna()).groupby(raw.entity_id).ffill()
        lag_age=raw.observation_year-lastyear
        stale=lag_age.gt(CFG['MAX_FEATURE_STALENESS_YEARS']) | lastyear.isna()
        filled=observed.groupby(raw.entity_id).ffill()
        raw[f]=filled.where(~stale)
        STALENESS_QA.append({'feature':f,'n_observed':int(observed.notna().sum()),
             'n_carried_valid':int((raw[f].notna()&observed.isna()).sum()),
             'max_age_years':CFG['MAX_FEATURE_STALENESS_YEARS'],'publication_lag':lag0})
    STALENESS_QA=pd.DataFrame(STALENESS_QA)
    ensure(STALENESS_QA.n_carried_valid.ge(0).all(),'STALE_QA_INVALID')
    raw=raw.rename(columns={'observation_year':'known_year'})
    ORIGINS['known_year']=ORIGINS.origin_year-lag0
    ORIGINS=ORIGINS.merge(raw,on=['entity_id','known_year'],how='left',validate='many_to_one').drop(columns='known_year')
else:STALENESS_QA=pd.DataFrame(columns=['feature','n_observed','n_carried_valid','max_age_years','publication_lag'])
LABELS=PANEL[['entity_id','observation_year','population','migration_per_1000']].rename(columns={'observation_year':'target_year','population':'future_population','migration_per_1000':'future_migration_per_1000'})
CURRENT=PANEL[['entity_id','observation_year','population']].rename(columns={'observation_year':'origin_year','population':'population_at_origin'})
ORIGINS=ORIGINS.merge(CURRENT,on=['entity_id','origin_year'],how='left',validate='one_to_one'); frames=[]
for h in CFG['HORIZONS']:
    t=ORIGINS.copy(); t['horizon']=h; t['target_year']=t.origin_year+h
    t=t.merge(LABELS,on=['entity_id','target_year'],how='left',validate='many_to_one')
    t['migration_per_1000']=t.future_migration_per_1000
    t['population_change_pct']=np.where(t.population_at_origin.gt(0)&t.future_population.gt(0),100*(t.future_population/t.population_at_origin-1),np.nan)
    t['information_cutoff']=t.origin_year; t['effective_available_year']=t.origin_year-lag0
    t['target_origin_unit']='ENTITY_YEAR'; t['source_vintage']=source_sha
    frames.append(mark_boundary_forecast_window(t,PANEL).drop(columns=['future_population','future_migration_per_1000']))
TASK_PANEL=pd.concat(frames,ignore_index=True)
ensure(not TASK_PANEL.duplicated(['entity_id','origin_year','horizon']).any(),'FORECAST_KEY_COLLISION')
ensure((TASK_PANEL.effective_available_year<TASK_PANEL.origin_year).all(),'FUTURE_COVARIATE_LEAK')
if CFG['MODE']=='SMOKE': TASK_PANEL=TASK_PANEL[TASK_PANEL.entity_id.isin(sorted(TASK_PANEL.entity_id.unique())[:CFG['SMOKE_MUNICIPALITIES']])].copy()

TARGET_AVAILABILITY=[]
for target in CFG['TARGETS']:
    for y in YEARS:
        src=base.loc[base.observation_year.eq(y)]
        valid_raw=(src.migration.notna()&src.average_population.gt(0)) if target=='migration_per_1000' else src.population.gt(0)
        for h in CFG['HORIZONS']:
            tp=TASK_PANEL.loc[TASK_PANEL.horizon.eq(h)&TASK_PANEL.target_year.eq(y)]
            available=tp[target].notna(); g0=tp[[c for c in G0_FEATURES if c!='origin_year']].notna().any(axis=1)
            g1=tp[GROUP_FEATURES['G1']].notna().any(axis=1) if GROUP_FEATURES['G1'] else pd.Series(False,index=tp.index)
            TARGET_AVAILABILITY.append(dict(target=target,target_year=int(y),horizon=int(h),raw_d01_rows=len(src),raw_d01_target_available=int(valid_raw.sum()),label_rows=int(available.sum()),g0_ready=int((available&g0).sum()),factor_rows_g1=int((available&g1).sum()),reason='D01_MIGRATION_UNOBSERVED' if target=='migration_per_1000' and not valid_raw.any() else 'OBSERVED_OR_OTHER_SUPPORT'))
TARGET_AVAILABILITY=pd.DataFrame(TARGET_AVAILABILITY)
GROUP_SUPPORT={g:TASK_PANEL[ff].notna().any(axis=1) if ff else pd.Series(False,index=TASK_PANEL.index) for g,ff in GROUP_FEATURES.items()}
G0_SUPPORT=TASK_PANEL[[c for c in G0_FEATURES if c!='origin_year']].notna().any(axis=1)
TASK_PANEL['common_support']=G0_SUPPORT.copy()
for g in ['G1','G2','G3','G4','G5']:
    if GROUP_FEATURES[g]: TASK_PANEL['common_support'] &= GROUP_SUPPORT[g]
TASK_PANEL['available_factor_groups']=sum(v.astype(int) for v in GROUP_SUPPORT.values())
coverage=pd.DataFrame([dict(group=g,n_features=len(ff),raw_panel_coverage=float(PANEL[ff].notna().any(axis=1).mean()) if ff else 0.,forecast_coverage=float(GROUP_SUPPORT[g].mean()),status='PRESENT' if ff else 'NOT_FEASIBLE') for g,ff in GROUP_FEATURES.items()])
mark('G1R_NONMUNICIPAL_GRAIN','PASS','Regional economic context is labeled G1R, NOT municipal G1; zero municipal claims')
mark('PASS_DATA',('PASS' if sum(map(len,GROUP_FEATURES.values()))>0 and not fatal_sql else 'PARTIAL' if len(PANEL) and not PANEL[['entity_id','observation_year']].duplicated().any() else 'FAIL'),f'panel={len(PANEL)}, features={sum(map(len,GROUP_FEATURES.values()))}, common_support={int(TASK_PANEL.common_support.sum())}')
quiet_print('Панель:',len(PANEL),'строк | label candidates:',len(TASK_PANEL),'| complete support:',int(TASK_PANEL.common_support.sum()))
def audited_stability_share(sample):
    if sample.empty:return {'n':0,'n_unverified':0,'n_verified':0,'verified_share':0.}
    flag=sample.boundary_period_uncertain.astype('boolean').fillna(True).astype(bool)
    return dict(n=len(sample),n_unverified=int(flag.sum()),n_verified=int((~flag).sum()),verified_share=float((~flag).mean()))
BOUNDARY_WINDOW_QA=TASK_PANEL.groupby(['horizon','target_year'],as_index=False).agg(n=('entity_id','size'),uncertain=('boundary_period_uncertain','sum'))

progress('PANEL',1,1,f'Наблюдений: {len(PANEL)}; прогнозных строк: {len(TASK_PANEL)}')

# CHECKPOINT I: полный срез фактических наблюдений и прогнозной панели.
if CFG.get('ENABLE_CHECKPOINTS'):
    cp=WORK/'checkpoints';cp.mkdir(parents=True,exist_ok=True)
    checkpoint_i=[]
    for label,frame in [('municipal_panel',PANEL),('supervised_origin_panel',TASK_PANEL)]:
        file=cp/(label+'.parquet');frame.to_parquet(file,index=False,compression='zstd')
        checkpoint_i.append({'file':file.name,'rows':len(frame),'columns':len(frame.columns),
                             'sha256':digest(file),'source_commit':source_sha})
    (cp/'stage_I_checkpoint.json').write_text(json.dumps({'version':CFG['VERSION'],
       'status':'PHYSICAL_PARQUET_CHECKPOINT','assets':checkpoint_i},ensure_ascii=False,indent=2),encoding='utf-8')
    progress('PANEL',1,1,'Исходная панель и checkpoint I зафиксированы')

### Этап II / IV. Дизайн эксперимента, модели и ансамбли

Фиксированные исторические окна по каждому целевому показателю, горизонты 1 и 3 года, хронологические train/development/test, независимые пространственные тесты, M0–M4, Ridge/ElasticNet/HGB/CatBoost/LightGBM, SuperLearner. Группы G5 и горизонт 5 лет учитываются в диагностике доступности, но не замещают обязательные тесты.

In [ ]:
# 07 · PREREGISTERED ABLATIONS, COMMON-SUPPORT STUDIES, SPATIAL/TEMPORAL SPLITS
GROUP_ORDER=('G1','G2','G3','G4','G5')
PRIMARY_TEST_YEARS=sorted(set(sum(TARGET_TEST_WINDOWS.values(),[])))
SPATIAL_YEAR_COUNTS={int(y):int((TASK_PANEL.horizon.eq(1)&TASK_PANEL.target_year.eq(y)&
    GROUP_SUPPORT['G5']).sum()) for y in PRIMARY_TEST_YEARS}
G5_CANONICAL_ELIGIBLE=bool(GROUP_FEATURES.get('G5')) and all(
    SPATIAL_YEAR_COUNTS[y]>=CFG['MIN_SPATIAL_CANONICAL_TEST_ROWS'] for y in PRIMARY_TEST_YEARS)
mark('SPATIAL_CANONICAL_GATE','PASS' if G5_CANONICAL_ELIGIBLE else 'NOT_FEASIBLE',
     f'vintage-gated support={SPATIAL_YEAR_COUNTS}; undated transport excluded')
CANONICAL=[{'arm':'M0','groups':[]}];prefix=[]
for j,g in enumerate(GROUP_ORDER,1):
    if not GROUP_FEATURES[g] or (g=='G5' and not G5_CANONICAL_ELIGIBLE):
        for k in range(j,6):mark('ARM_M'+str(k),'NOT_FEASIBLE','Missing or time-ineligible group '+g)
        break
    prefix=prefix+[g];CANONICAL.append({'arm':'M'+str(j),'groups':prefix.copy()})
AVAILABLE_GROUPS=[g for g in GROUP_ORDER if GROUP_FEATURES[g] and
                  (g!='G5' or G5_CANONICAL_ELIGIBLE)]
STUDIES=[]
if len(CANONICAL)>1:
    STUDIES.append({'id':'CANONICAL','required_groups':prefix.copy(),'arms':CANONICAL,
        'protocols':['ROLLING_TIME','TERRITORY_ONLY','CROSSED'],'confirmatory':len(CANONICAL)>=5})
else:
    STUDIES.append({'id':'CANONICAL','required_groups':[],'arms':CANONICAL,
        'protocols':['ROLLING_TIME'],'confirmatory':False})
# Separate, self-contained temporal experiments for the two nominal test years.
if CFG['MODE']=='FULL' and len(CANONICAL)>1:
    for target,years in TARGET_TEST_WINDOWS.items():
        for year in years:
            STUDIES.append({'id':f'YEAR_{target}_{year}','test_year':int(year),
               'allowed_targets':[target],'required_groups':prefix.copy(),'arms':CANONICAL,
               'protocols':['ROLLING_TIME'],'confirmatory':False})
for g in AVAILABLE_GROUPS:
    STUDIES.append({'id':'AUX_'+g,'required_groups':[g],
      'arms':[{'arm':'M0','groups':[]},{'arm':'ADD_'+g,'groups':[g]}],
      'protocols':['ROLLING_TIME'],'confirmatory':False})
if GROUP_FEATURES.get('G5') and not G5_CANONICAL_ELIGIBLE:
    STUDIES.append({'id':'AUX_G5_VINTAGED','required_groups':['G5'],
      'arms':[{'arm':'M0','groups':[]},{'arm':'ADD_G5','groups':['G5']}],
      'protocols':['ROLLING_TIME'],'confirmatory':False})
# Secondary observed migration window only if it differs from the predeclared migration calendar.
if CFG['MODE']=='FULL' and CFG.get('EXPLORE_SECONDARY_MIGRATION_WINDOW'):
    mig_years=sorted(base.loc[base.migration_per_1000.notna(),'observation_year'].unique())
    secondary=mig_years[-CFG['TEST_YEARS']:] if len(mig_years)>=CFG['TEST_YEARS'] else []
    if secondary and list(map(int,secondary))!=TARGET_TEST_WINDOWS['migration_per_1000']:
        STUDIES.append({'id':'MIGRATION_OBSERVED_LAST_TWO_YEARS','test_years':list(map(int,secondary)),
          'allowed_targets':['migration_per_1000'],'required_groups':[],
          'arms':[{'arm':'M0','groups':[]}], 'protocols':['ROLLING_TIME'],
          'confirmatory':False,'label':'SECONDARY_OBSERVED_WINDOW_NOT_PRIMARY'})
if GROUP_FEATURES.get('G1R'):
    STUDIES.append({'id':'AUX_G1_REGION_CONTEXT','required_groups':['G1R'],
       'arms':[{'arm':'M0','groups':[]},{'arm':'ADD_G1_REGION','groups':['G1R']}],
       'protocols':['ROLLING_TIME','TERRITORY_ONLY','CROSSED'],
       'confirmatory':False,'label':'REGION_CONTEXT_NOT_MUNICIPAL_G1'})
if len(AVAILABLE_GROUPS)>=2:
    all_g=AVAILABLE_GROUPS.copy()
    specs=[{'arm':'M0','groups':[]},{'arm':'FULL_AVAILABLE','groups':all_g}]
    specs += [{'arm':'WITHOUT_'+g,'groups':[x for x in all_g if x!=g]} for g in all_g]
    STUDIES.append({'id':'MAX_AVAILABLE','required_groups':all_g,'arms':specs,
       'protocols':['ROLLING_TIME','TERRITORY_ONLY','CROSSED'],'confirmatory':False})
for study in STUDIES:
    for arm in study['arms']:
        arm['features']=list(dict.fromkeys(G0_FEATURES+sum((GROUP_FEATURES[g] for g in arm['groups']),[])))
        arm['n_factors']=sum(len(GROUP_FEATURES[g]) for g in arm['groups'])
# ПРЕРЕГИСТРАЦИЯ до обучения: H3.1–H3.8 из научного протокола, без просмотра TEST.
_HYPOTHESES=[
 ('H3.1','CANONICAL','M1 → M3','Последовательный вклад жилищной и социальной инфраструктуры'),
 ('H3.2','INTERACTION','G1 × G2','Экономические условия × жилищная среда'),
 ('H3.3','AUX_G3','M0 → ADD_G3','Дополнительная ценность социальной инфраструктуры'),
 ('H3.4','AUX_G4','M0 → ADD_G4','Дополнительная ценность муниципальных финансов'),
 ('H3.5','SENSITIVITY','LAG_AND_PROXY','Лаги публикации и коррелированные прокси'),
 ('H3.6','HEALTH','HEALTH_VS_EDUCATION','Независимая идентифицируемость медицинских факторов'),
 ('H3.7','BOUNDARIES','STABLE_VS_UNVERIFIED','Устойчивость к изменениям муниципальных границ'),
 ('H3.8','SIZE','SIZE_STRATA','Неоднородность эффекта по размеру муниципалитетов')]
HYPOTHESIS_REGISTRY=pd.DataFrame([{
    'hypothesis_id':h,'study_id':study,'contrast':contrast,'definition':definition,
    'test_family':('CONFIRMATORY_BY_TARGET_HORIZON_PROTOCOL' if h=='H3.1' else
                   'EXPLORATORY_BY_TARGET_HORIZON_PROTOCOL'),
    'sample_contract':'PAIRED_IDENTICAL_ENTITY_ORIGIN_TARGET_YEAR',
    'primary_metric':'MAE','multiplicity_method':('HOLM' if h=='H3.1' else 'BH_FDR'),
    'preregistered_before_test':True,'status':'PLANNED_CONDITIONAL',
    'version':CFG['VERSION'],
    'reason':'Requires verified source domain, sufficient paired support and temporal labels'}
    for h,study,contrast,definition in _HYPOTHESES])
STUDY_HYPOTHESIS_FAMILIES=pd.DataFrame([{
    'study_id':s['id'],'confirmatory_design':bool(s['confirmatory']),
    'family_scope':'TARGET_HORIZON_PROTOCOL',
    'multiplicity_method':'HOLM' if s['confirmatory'] else 'BH_FDR',
    'preregistered_before_test':True,'status':'DESIGN_FROZEN'} for s in STUDIES])
TASKS=[];TASK_RECORDS=[]
ALL_REGIONS=sorted(base.region_name.dropna().astype(str).unique())
rng=np.random.default_rng(CFG['SEED']);HELD_REGIONS=set(rng.choice(
   np.asarray(ALL_REGIONS,dtype=object),size=max(2,int(0.2*len(ALL_REGIONS))),replace=False)) if len(ALL_REGIONS)>8 else set()

def splits_for(d,protocol,h,test_year=None,test_years=None):
    # Real-time forecasting: all training AND DEV outcome labels must have been observed
    # before the earliest issuance (origin year) of the held-out test forecasts.
    yrs=[y for y in YEARS if y>=min(YEARS)+h]
    nt=1 if CFG['MODE']=='SMOKE' else CFG['TEST_YEARS']
    nd=1 if CFG['MODE']=='SMOKE' else CFG['DEV_YEARS']
    if len(yrs)<nt+nd+CFG['MIN_TRAIN_YEARS']:return None,'TOO_FEW_TARGET_YEARS'
    test_yrs=([int(test_year)] if test_year is not None else
              sorted(int(v) for v in test_years) if test_years else yrs[-nt:])
    first_test_origin=test_yrs[0]-h
    latest_known_label=first_test_origin-int(CFG['TARGET_LABEL_AVAILABILITY_LAG'])
    potential_dev=[y for y in yrs if y<=latest_known_label]
    if len(potential_dev)<CFG['MIN_TRAIN_YEARS']+nd:
        return None,'LONG_HORIZON_TARGET_LABELS_NOT_YET_KNOWN_AT_FORECAST_ORIGIN'
    dv=potential_dev[-nd:]
    tr=[y for y in potential_dev if y<min(dv)]
    if len(tr)<CFG['MIN_TRAIN_YEARS']:return None,'TOO_FEW_REALTIME_TRAIN_YEARS'
    held=d.region_name.astype(str).isin(HELD_REGIONS)
    if protocol=='ROLLING_TIME':
        return {'train':d.target_year.isin(tr),'dev':d.target_year.isin(dv),
                'test':d.target_year.isin(test_yrs)},''
    if not HELD_REGIONS:return None,'INSUFFICIENT_REGIONS_FOR_GROUP_HOLDOUT'
    if protocol=='TERRITORY_ONLY':
        # Pure geography generalization (historical evaluation on held regions).
        return {'train':(~held)&d.target_year.isin(tr),
                'dev':(~held)&d.target_year.isin(dv),
                'test':held&d.target_year.isin(dv)},''
    if protocol=='CROSSED':
        return {'train':(~held)&d.target_year.isin(tr),
                'dev':(~held)&d.target_year.isin(dv),
                'test':held&d.target_year.isin(test_yrs)},''
    raise ValueError('UNKNOWN_SPLIT_PROTOCOL')

# A missing late test year must NOT move the test window backwards.
_test_stub=pd.DataFrame({'target_year':[y for y in YEARS if y<max(YEARS)-2],
                         'region_name':['unit_test']*len([y for y in YEARS if y<max(YEARS)-2])})
_stub_masks,_=splits_for(_test_stub,'ROLLING_TIME',min(CFG['HORIZONS']))
if _stub_masks is not None:
    ensure(int(_stub_masks['test'].sum())==0,'UNIT_TEST_CALENDAR_SLIPPAGE')
mark('UNIT_SPLIT_CONTRACT','PASS','D01-frozen target years and issued-forecast cutoff; source gaps do not shift test years')

for st in STUDIES:
    for target in CFG['TARGETS']:
        if target not in st.get('allowed_targets',CFG['TARGETS']):continue
        if target=='migration_per_1000' and not valid_rate:
            for h in (CFG['HORIZONS'] if CFG['MODE']=='FULL' else [1]):
                for protocol in st['protocols']:
                    TASK_RECORDS.append({'study_id':st['id'],'target':target,'horizon':h,
                      'protocol':protocol,'status':'NOT_FEASIBLE','scientific_status':'NOT_FEASIBLE',
                      'reason':'MIGRATION_TARGET_SEMANTICS_FAILED','n_support':0,'n_train':0,'n_dev':0,'n_test':0,
                      'requested_test_years':','.join(map(str,TARGET_TEST_WINDOWS[target])),
                      'actual_test_year_counts_json':'{}','missing_test_years':','.join(map(str,TARGET_TEST_WINDOWS[target]))})
            continue
        for h in (CFG['HORIZONS'] if CFG['MODE']=='FULL' else [1]):
            support=G0_SUPPORT.copy()
            missing_required=[g for g in st['required_groups'] if not GROUP_FEATURES_BY_H[h].get(g)]
            for g in st['required_groups']:
                if GROUP_FEATURES_BY_H[h].get(g):
                    support &= TASK_PANEL[GROUP_FEATURES_BY_H[h][g]].notna().any(axis=1)
            d=TASK_PANEL[TASK_PANEL.horizon.eq(h)&TASK_PANEL[target].notna()&support].copy()
            local_st=dict(st)
            local_st['arms']=[{**a,'features':list(dict.fromkeys(
                G0_FEATURES+sum((GROUP_FEATURES_BY_H[h][g] for g in a['groups']),[]))),
                'n_factors':sum(len(GROUP_FEATURES_BY_H[h][g]) for g in a['groups'])}
               for a in st['arms']]
            for protocol in st['protocols']:
                mm,issue=splits_for(d,protocol,h,st.get('test_year'),st.get('test_years',TARGET_TEST_WINDOWS[target]))
                if missing_required:issue='GROUP_NOT_ELIGIBLE_AT_HORIZON_'+','.join(missing_required);mm=None
                counts={k:int(v.sum()) for k,v in mm.items()} if mm is not None else {}
                if mm is not None and (counts['train']<CFG['MIN_TRAIN_ROWS'] or
                      min(counts['dev'],counts['test'])<CFG['MIN_TEST_ROWS']):
                    issue='INSUFFICIENT_TRAIN_DEV_TEST';mm=None
                requested=([int(st['test_year'])] if 'test_year' in st else
                           st['test_years'] if 'test_years' in st else
                           TARGET_TEST_WINDOWS[target])
                test_by_year=(d.loc[mm['test']].groupby('target_year').size().to_dict()
                              if mm is not None else {})
                missing_test_years=[y for y in requested if test_by_year.get(y,0)==0] if protocol!='TERRITORY_ONLY' else []
                task_status=('FEASIBLE_COMPLETE' if mm is not None and not missing_test_years else
                             'FEASIBLE_PARTIAL_YEARS' if mm is not None else 'NOT_FEASIBLE')
                TASK_RECORDS.append({'study_id':st['id'],'target':target,'horizon':h,'protocol':protocol,
                  'status':task_status,'scientific_status':('COMPLETE_YEARS' if task_status=='FEASIBLE_COMPLETE' else 'PARTIAL_YEARS' if task_status=='FEASIBLE_PARTIAL_YEARS' else 'NOT_FEASIBLE'),
                  'reason':issue,'n_support':len(d),
                  'n_train':counts.get('train'),'n_dev':counts.get('dev'),'n_test':counts.get('test'),
                  'requested_test_years':','.join(map(str,requested)),
                  'actual_test_year_counts_json':json.dumps({int(k):int(v) for k,v in test_by_year.items()}),
                  'missing_test_years':','.join(map(str,missing_test_years)),
                  'evaluation_window':('HISTORICAL_DEV_YEARS_GEO_ONLY' if protocol=='TERRITORY_ONLY' else
                     'PRIMARY_TARGET_YEARS' if 'test_years' not in st else 'SECONDARY_OBSERVED_YEARS')})
                if mm is not None:
                    TASKS.append({'study':local_st,'target':target,'horizon':h,'protocol':protocol,
                                 'data':d,'masks':mm})
TASK_STATUSES=pd.DataFrame(TASK_RECORDS)
ensure({'status','scientific_status','study_id','target','horizon','protocol','n_test'}.
       issubset(TASK_STATUSES.columns),'TASK_STATUS_SCHEMA_INCOMPLETE')
ensure((TASK_STATUSES.loc[TASK_STATUSES.status.eq('FEASIBLE_PARTIAL_YEARS'),
       'scientific_status']=='PARTIAL_YEARS').all(),'INCOMPLETE_TEST_YEARS_MISLABELLED')

def _study_target_years(st,target):
    if 'test_years' in st:return list(map(int,st['test_years']))
    if 'test_year' in st:return [int(st['test_year'])]
    return TARGET_TEST_WINDOWS[target]

# Audit target availability per study with explicit source/target coverage.
TEST_YEAR_AUDIT_ROWS=[]
for rec in TASK_RECORDS:
    if rec['protocol']=='TERRITORY_ONLY':continue  # historical geo-only protocol, not the primary time calendar
    st=next(s for s in STUDIES if s['id']==rec['study_id'])
    required=list(st['required_groups'])
    for year in _study_target_years(st,rec['target']):
        mask=TASK_PANEL.target_year.eq(year)&TASK_PANEL.horizon.eq(rec['horizon'])&TASK_PANEL[rec['target']].notna()
        supported=mask&G0_SUPPORT
        for g in required:
            if GROUP_FEATURES_BY_H[int(rec['horizon'])].get(g):
                supported=supported&TASK_PANEL[GROUP_FEATURES_BY_H[int(rec['horizon'])][g]].notna().any(axis=1)
            else:supported=supported&False
        by_year=json.loads(rec.get('actual_test_year_counts_json','{}') or '{}')
        TEST_YEAR_AUDIT_ROWS.append({'study_id':rec['study_id'],'target':rec['target'],
            'horizon':rec['horizon'],'protocol':rec['protocol'],'target_year':int(year),
            'n_target_observations':int(mask.sum()),'n_with_study_factor_support':int(supported.sum()),
            'n_test':int(by_year.get(str(year),0)),
            'status':rec['status'],'reason':rec['reason']})
TEST_YEAR_AUDIT=pd.DataFrame(TEST_YEAR_AUDIT_ROWS)
CANONICAL_YEAR_QA=TASK_STATUSES.loc[TASK_STATUSES.study_id.eq('CANONICAL') &
    TASK_STATUSES.protocol.isin(['ROLLING_TIME','CROSSED'])].copy()
if CFG['MODE']=='FULL':
    primary=CANONICAL_YEAR_QA.loc[CANONICAL_YEAR_QA.horizon.eq(1) &
        CANONICAL_YEAR_QA.status.str.startswith('FEASIBLE')]
    missing_any=primary.missing_test_years.ne('').any() if len(primary) else True
    mark('TWO_YEAR_CALENDAR_QA','PARTIAL' if missing_any else 'PASS',
         'Two-year target-specific observed primary windows: migration 2021–2022, population 2022–2023; no silent sliding')
else: mark('TWO_YEAR_CALENDAR_QA','SMOKE_ONLY','Year coverage in FULL checked separately')
H5_CALENDAR_GAPS=TASK_STATUSES.loc[TASK_STATUSES.horizon.eq(5),
    ['study_id','target','protocol','horizon','status','reason','requested_test_years','actual_test_year_counts_json']].copy()
if len(H5_CALENDAR_GAPS):
    mark('H5_LONG_HORIZON','PASS' if H5_CALENDAR_GAPS.status.eq('FEASIBLE_COMPLETE').all() else 'NOT_FEASIBLE',
         f'{int(H5_CALENDAR_GAPS.status.eq("FEASIBLE_COMPLETE").sum())}/{len(H5_CALENDAR_GAPS)}; unmet label availability is never bypassed')
# A historical time-window redefinition is NOT substituted for preregistered h5 tests.
ARM_SPECS=CANONICAL
if not TASKS:mark('PASS_SPLITS','NOT_FEASIBLE','No feasible task after paired common-support gating')
else:mark('PASS_SPLITS','PASS',f'{len(TASKS)} dataset/protocol tasks, {len(STUDIES)} studies')
# One fixed work budget before the first model; never grow denominator during run.
RUN_COMPLETION_STATUS='IN_PROGRESS'
_TICK_PHASE=None;_TICK_DONE=0;_TICK_TOTAL=0
def set_tick(phase,total):
    global _TICK_PHASE,_TICK_TOTAL,_TICK_DONE
    _TICK_PHASE=phase;_TICK_TOTAL=max(0,int(total));_TICK_DONE=0
    if _TICK_TOTAL==0:progress(phase,1,1,'Нет доступных единиц работы')
    else:progress(phase,0,_TICK_TOTAL,'Выполнение')
def tick(stage,n=1,detail=''):
    global _TICK_DONE
    ensure(_TICK_PHASE is not None,'PROGRESS_PHASE_NOT_SET')
    _TICK_DONE+=int(n)
    ensure(_TICK_DONE<=_TICK_TOTAL,'PROGRESS_WORK_COUNT_OVERFLOW')
    progress(_TICK_PHASE,_TICK_DONE,_TICK_TOTAL,detail)

quiet_print('Studies:',[(s['id'],len(s['arms'])) for s in STUDIES], '| feasible tasks:',len(TASKS),
      '| frozen regions held out:',len(HELD_REGIONS))

progress('TASKS',1,1,f'Задач в плане: {len(TASKS)}')

#### Обучение, DEV-подбор, заморозка моделей, тестовые предсказания и ансамбли.

In [ ]:
# 08 · DEV-LOCKED MODEL BANK; TRUE PAIRED TEST PREDICTIONS, SEEDS, NO OUTCOME-BASED SELECTION
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
PRED_ROWS=[];METRIC_ROWS=[];DEV_ROWS=[];SPLIT_ROWS=[];IMPORTANCE_ROWS=[];MODEL_REGISTRY={}
try:
    from catboost import CatBoostRegressor
    CB_AVAILABLE=True
except ImportError:CB_AVAILABLE=False
try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE=True
except ImportError:LGBM_AVAILABLE=False
BANK=[f for f in CFG['MODEL_FAMILIES'] if f in ('RIDGE','HGB','ELASTICNET') or (f=='CATBOOST' and CB_AVAILABLE) or
      (f=='LIGHTGBM' and LGBM_AVAILABLE)]
mark('MODEL_BANK','PASS' if 'RIDGE' in BANK and 'HGB' in BANK else 'FAIL',
      f'available={BANK}; CatBoost={CB_AVAILABLE}; LightGBM={LGBM_AVAILABLE}')

def design(train,test,features,keep_reference=None):
    refs=sorted(train.region_name.dropna().astype(str).unique()) if keep_reference is None else list(keep_reference)
    usable=[f for f in dict.fromkeys(features) if f in train and
            pd.to_numeric(train[f],errors='coerce').notna().any()]
    ensure('origin_year' in usable,'ORIGIN_YEAR_MISSING')
    def mx(df):
        xx=df[usable].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).reset_index(drop=True)
        for f in usable:
            if f.startswith(('G1_','G2_','G3_','G4_','G5_')):
                xx[f]=np.sign(xx[f])*np.log1p(np.abs(xx[f])) # fixed transform; no fitted test statistics
        reg=df.region_name.astype(str).reset_index(drop=True)
        # One-shot matrix construction: no repeated __setitem__ / pandas frame fragmentation.
        region_columns={'region_'+hashlib.sha256(name.encode()).hexdigest()[:10]:
                        reg.eq(name).to_numpy(dtype=float) for name in refs}
        if region_columns:
            region_matrix=pd.DataFrame(region_columns,index=xx.index,dtype=float)
            xx=pd.concat([xx,region_matrix],axis=1,copy=False)
        ensure(not xx.columns.duplicated().any(),'DUPLICATE_REGION_DUMMY_COLUMNS')
        return xx.astype(float)
    return mx(train),mx(test),usable,refs

def estimator(family,seed,params):
    if family=='RIDGE':
        return make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler(),
                             Ridge(alpha=float(params['alpha'])))
    if family=='ELASTICNET':
        return make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler(),
                ElasticNet(alpha=float(params['alpha']),l1_ratio=float(params['l1_ratio']),
                           max_iter=5000,random_state=int(seed)))
    if family=='HGB':
        return HistGradientBoostingRegressor(max_iter=int(params['iterations']),
            max_leaf_nodes=int(params['leaves']),learning_rate=0.045,min_samples_leaf=18,
            l2_regularization=2.0,early_stopping=False,random_state=int(seed))
    if family=='CATBOOST':
        return CatBoostRegressor(iterations=int(params['iterations']),depth=int(params['depth']),
             learning_rate=0.05,l2_leaf_reg=4,loss_function='RMSE',verbose=False,
             thread_count=int(CFG['DUCKDB_THREADS']),random_seed=int(seed),allow_writing_files=False)
    if family=='LIGHTGBM':
        return LGBMRegressor(n_estimators=int(params['iterations']),num_leaves=int(params['leaves']),
             max_depth=6,learning_rate=0.04,reg_lambda=3.0,min_child_samples=18,
             n_jobs=int(CFG['DUCKDB_THREADS']),verbosity=-1,random_state=int(seed))
    raise ValueError('UNKNOWN_MODEL_FAMILY')
PARAM_GRID={
 'RIDGE':[{'alpha':2},{'alpha':20}],
 'ELASTICNET':[{'alpha':0.2,'l1_ratio':0.25},{'alpha':2.,'l1_ratio':0.55}],
 'HGB':[{'iterations':125,'leaves':15},{'iterations':175,'leaves':23}],
 'CATBOOST':[{'iterations':220,'depth':4},{'iterations':300,'depth':6}],
 'LIGHTGBM':[{'iterations':150,'leaves':15},{'iterations':200,'leaves':23}]}
def fit_predict(family,train,test,target,features,seed=1729,params=None):
    Xtr,Xte,used,refs=design(train,test,features)
    model=estimator(family,seed,params or PARAM_GRID[family][0]);model.fit(Xtr,train[target].to_numpy(float))
    pred=np.asarray(model.predict(Xte),dtype=float)
    ensure(len(pred)==len(test) and np.isfinite(pred).all(),'PREDICTION_NOT_FINITE')
    return pred,model,used,refs

def baseline_predict(train,test,target,h):
    la=CFG['FEATURE_PUBLICATION_LAG'];hist=test.get('hist_migration_l'+str(la))
    if target=='migration_per_1000':
        return pd.to_numeric(hist,errors='coerce').fillna(float(train[target].median())).to_numpy(float)
    p1=pd.to_numeric(test[f'hist_population_l{la}'],errors='coerce')
    p3=pd.to_numeric(test[f'hist_population_l{la+2}'],errors='coerce')
    recent_growth=np.power(p1/p3,0.5*h)
    series=100*(recent_growth-1)
    return series.replace([np.inf,-np.inf],np.nan).fillna(float(train[target].median())).to_numpy(float)

def regional_baseline_predict(train,test,target):
    """Train-only regional tendency; unknown region falls back to overall median."""
    med=train.groupby('region_name')[target].median()
    glob=float(train[target].median())
    return test.region_name.map(med).fillna(glob).to_numpy(float)

def deterministic_family(family):
    # HGB and LightGBM used here have neither row nor column sampling, and no early stopping.
    return family in ('RIDGE','ELASTICNET','HGB','LIGHTGBM')

def family_seeds(family):
    return [CFG['SEED']] if deterministic_family(family) else list(CFG['SEEDS_FINAL'])

def metrics(y,p,w):
    y=np.asarray(y,float);p=np.asarray(p,float);w=np.asarray(w,float)
    ensure(len(y)==len(p) and np.isfinite(y).all() and np.isfinite(p).all(),'INVALID_EVALUATION_VECTORS')
    w=np.where(np.isfinite(w)&(w>0),w,0)
    err=y-p
    return {'mae':float(np.mean(np.abs(err))), 'rmse':float(np.sqrt(np.mean(err**2))),
       'weighted_mae':float(np.average(np.abs(err),weights=w)) if w.sum()>0 else np.nan,
       'n_test':int(len(y))}

def add_result(task,arm,model,seed,train,test,pred,n_feat):
    target,h,protocol=task['target'],task['horizon'],task['protocol'];study=task['study']['id']
    y=test[target].to_numpy(float)
    w=test[f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'].to_numpy(float)
    met=metrics(y,pred,w)
    METRIC_ROWS.append({'study_id':study,'target':target,'horizon':h,'protocol':protocol,
         'arm':arm,'model':model,'seed':seed,'n_train':len(train),'n_features_used':n_feat,**met})
    rows=test[['entity_id','origin_year','target_year','region_name','mun_type','information_cutoff',
       'source_vintage','horizon','population_at_origin']].copy()
    rows['target']=target;rows['study_id']=study;rows['split_id']=protocol
    rows['arm']=arm;rows['model']=model;rows['seed']=seed;rows['y_true']=y;rows['y_pred']=pred
    rows['population_weight']=w
    rows['absolute_error']=np.abs(y-pred);rows['squared_error']=(y-pred)**2
    PRED_ROWS.append(rows)

# CPU-only, isolated contract test (synthetic values never added to evidence or metrics).
def _model_unit_check():
    rng=np.random.default_rng(CFG['SEED']);n=100
    fake=pd.DataFrame({'region_name':np.repeat(['R1','R2'],n//2),
       'origin_year':np.tile(np.arange(2010,2020),n//10),
       'G1_01_per_resident':rng.lognormal(size=n),'G2_01_per_1000':rng.random(n)*20,
       '_test_target':rng.normal(size=n),'hist_population_l1':rng.integers(100,10000,n)})
    for fam in ('RIDGE','HGB','ELASTICNET'):
        yp,_,_,_=fit_predict(fam,fake.iloc[:80],fake.iloc[80:],'_test_target',
            ['origin_year','G1_01_per_resident','G2_01_per_1000'],CFG['SEED'],PARAM_GRID[fam][0])
        ensure(np.isfinite(yp).all() and len(yp)==20,'MODEL_UNIT_TEST_FAIL_'+fam)
_model_unit_check()
mark('UNIT_MODEL_CONTRACT','PASS','Ridge and HGB synthetic function-only smoke; excluded from scientific results')

# DEV count is known after TASKS and exact parameter grids are registered.
DEV_RUNS=[(t['study']['id'],t['target'],t['horizon']) for t in TASKS
     if t['protocol']=='ROLLING_TIME']
DEV_WORK=len(set(DEV_RUNS))

DEV_TOTAL=len(set(DEV_RUNS))*sum(len(PARAM_GRID[f]) for f in BANK
                  if f in PARAM_GRID and (CFG['MODE']=='FULL'))
if CFG['MODE']!='FULL':DEV_TOTAL=len(set(DEV_RUNS))*len(BANK)
set_tick('DEV',DEV_TOTAL)
# Tune on a held-out historical DEV block, prior to FINAL test. No use of final target values.
SELECTED_FAMILY={}
for st in STUDIES:
    for target in CFG['TARGETS']:
        for h in CFG['HORIZONS'] if CFG['MODE']=='FULL' else [1]:
            runs=[t for t in TASKS if t['study']['id']==st['id'] and t['target']==target and
                  t['horizon']==h and t['protocol']=='ROLLING_TIME']
            if not runs:continue
            task=runs[0];df=task['data'];tr=df[task['masks']['train']];dv=df[task['masks']['dev']]
            maxarm=max(task['study']['arms'],key=lambda a:len(a['features']))
            for family in BANK:
                for params in PARAM_GRID[family] if CFG['MODE']=='FULL' else PARAM_GRID[family][:1]:
                    try:
                        yp,_,used,_=fit_predict(family,tr,dv,target,maxarm['features'],CFG['SEED'],params)
                        sc=metrics(dv[target],yp,dv[f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
                        DEV_ROWS.append({'study_id':st['id'],'target':target,'horizon':h,
                            'family':family,'params_json':json.dumps(params,sort_keys=True),
                            'dev_mae':sc['mae'],'dev_rmse':sc['rmse'],'n_dev':len(dv),
                            'features_used':len(used)})
                    except Exception as ex:
                        log(f'DEV model failure {st["id"]} h{h} {family}: {type(ex).__name__}', 'WARN')
                    finally:tick('DEV model',1,f'{st["id"]} h{h} {family}')
            v=[r for r in DEV_ROWS if (r['study_id'],r['target'],r['horizon'])==(st['id'],target,h)]
            if v:
                choice=min(v,key=lambda r:(r['dev_mae'],r['dev_rmse']))
                SELECTED_FAMILY[(st['id'],target,h)]=(choice['family'],json.loads(choice['params_json']))
            # Fits, not tasks, are the exact DEV work units.

ensure(_TICK_DONE==_TICK_TOTAL,'DEV_PROGRESS_MISMATCH')
TRAIN_TOTAL=sum(2+(len(t['study']['arms'])*len(family_seeds(SELECTED_FAMILY[(t['study']['id'],t['target'],t['horizon'])][0]))
                   if (t['study']['id'],t['target'],t['horizon']) in SELECTED_FAMILY else 0)
                for t in TASKS)
set_tick('TRAIN',TRAIN_TOTAL)
for task in TASKS:
    study=task['study'];d=task['data'];m=task['masks'];protocol=task['protocol']
    tr=d[m['train']|m['dev']].copy();te=d[m['test']].copy()
    if protocol in ('ROLLING_TIME','CROSSED'):
        ensure(tr.target_year.max()<te.origin_year.min(),'TRAIN_LABEL_NOT_KNOWN_AT_EARLIEST_TEST_ORIGIN')
    if protocol in ('TERRITORY_ONLY','CROSSED'):
        ensure(set(tr.region_name.astype(str)).isdisjoint(set(te.region_name.astype(str))),'REGIONAL_CROSS_FOLD_LEAK')
        ensure(set(tr.entity_id).isdisjoint(set(te.entity_id)),'ENTITY_CROSS_FOLD_LEAK')
    SPLIT_ROWS.append(pd.concat([tr[['entity_id','origin_year','target_year','region_name']].assign(
           study_id=study['id'],target=task['target'],horizon=task['horizon'],protocol=protocol,part='train'),
           te[['entity_id','origin_year','target_year','region_name']].assign(
           study_id=study['id'],target=task['target'],horizon=task['horizon'],protocol=protocol,part='test')]))
    bp=baseline_predict(tr,te,task['target'],task['horizon'])
    add_result(task,'M0','HISTORICAL',CFG['SEED'],tr,te,bp,1)
    rb=regional_baseline_predict(tr,te,task['target'])
    add_result(task,'M0','REGIONAL_MEDIAN',CFG['SEED'],tr,te,rb,1)
    tick('historical baselines',2,study['id'])
    choice=SELECTED_FAMILY.get((study['id'],task['target'],task['horizon']))
    if choice is None:continue
    family,params=choice
    for arm in study['arms']:
        seeds=family_seeds(family)
        for si,seed in enumerate(seeds):
            try:
                yp,model,used,refs=fit_predict(family,tr,te,task['target'],arm['features'],seed,params)
                add_result(task,arm['arm'],family,seed,tr,te,yp,len(used))
                if si==0 and protocol=='ROLLING_TIME' and arm==max(study['arms'],key=lambda a:len(a['features'])):
                    key=(study['id'],task['target'],task['horizon'])
                    MODEL_REGISTRY[key]={'model':model,'features':arm['features'],
                       'used':used,'refs':refs,'train':tr,'test':te,'pred':yp,
                       'family':family,'params':params,'arm':arm['arm']}
                    name=f'{study["id"]}_{task["target"]}_h{task["horizon"]}_{family}.joblib'
                    joblib.dump({'model':model,'features':used,'refs':refs,'target':task['target'],
                        'horizon':task['horizon'],'study':study['id'],'source_commit':source_sha,
                        'model_family':family,'params':params},WORK/'models'/name,compress=3)
            except Exception as ex:
                mark(f'FIT_{study["id"]}_{task["target"]}_{task["horizon"]}_{protocol}_{arm["arm"]}_{seed}',
                     'FAIL',type(ex).__name__+': '+str(ex)[:100])
                log(f'FIT {study["id"]} {arm["arm"]} {family}: {type(ex).__name__}','ERROR')
            tick('training',1,
                 f'{study["id"]} {arm["arm"]} h={task["horizon"]} s={seed}')
ensure(_TICK_DONE==_TICK_TOTAL,'TRAIN_PROGRESS_MISMATCH')
ensure(DEV_WORK==len(set(DEV_RUNS)),'DEV_WORK_COUNT_CONTRACT')
# Единственный _PROGRESS_BAR сохраняется на весь прогон.
RUN_COMPLETION_STATUS='MODELS_COMPLETED_EXPORT_PENDING'
METRICS=pd.DataFrame(METRIC_ROWS);DEV_METRICS=pd.DataFrame(DEV_ROWS)
PREDICTIONS=pd.concat(PRED_ROWS,ignore_index=True) if PRED_ROWS else pd.DataFrame()
SPLITS=pd.concat(SPLIT_ROWS,ignore_index=True) if SPLIT_ROWS else pd.DataFrame()
SEED_QA_ROWS=[]
if len(PREDICTIONS):
    for (sid,targ,h,proto,arm,model),g in PREDICTIONS[PREDICTIONS.model.isin(BANK)].groupby(
          ['study_id','target','horizon','split_id','arm','model']):
        hashes=[]
        for seed,z in g.groupby('seed'):
            arr=z.sort_values(['entity_id','origin_year','target_year']).y_pred.to_numpy(float)
            hashes.append(hashlib.sha256(np.asarray(arr,dtype='<f8').tobytes()).hexdigest())
        SEED_QA_ROWS.append({'study_id':sid,'target':targ,'horizon':h,'protocol':proto,
            'arm':arm,'model':model,'n_requested_seeds':len(family_seeds(model)),
            'n_configured_seeds':len(CFG['SEEDS_FINAL']),
            'n_executed_seeds':len(hashes),'n_distinct_prediction_hashes':len(set(hashes)),
            'n_effectively_distinct_runs':len(set(hashes)),
            'seeds_are_independent_samples':False})
SEED_QA=pd.DataFrame(SEED_QA_ROWS)
if len(PREDICTIONS):
    cols=['entity_id','origin_year','target_year','horizon','target','study_id','split_id','arm','model','seed']
    ensure(not PREDICTIONS.duplicated(cols).any(),'DUPLICATE_TEST_PREDICTIONS')
    for keys,p in PREDICTIONS[PREDICTIONS.model.ne('HISTORICAL')].groupby(
           ['study_id','target','horizon','split_id','model','seed']):
        ensure(p.groupby('arm').size().nunique()==1,'ABLA_SAMPLE_SIZE_MISMATCH')
        key_hashes=[hashlib.sha256(a[['entity_id','origin_year','target_year']].sort_values(['entity_id','origin_year']).to_csv(index=False).encode()).hexdigest() for _,a in p.groupby('arm')]
        ensure(len(set(key_hashes))==1,'ABLA_TEST_KEYS_MISMATCH')
mark('PASS_TRAINING','PASS' if len(METRICS) and len(SELECTED_FAMILY)==DEV_WORK and DEV_WORK>0 and not any(v['status']=='FAIL' for k,v in STATUS.items() if k.startswith('FIT_')) else 'NOT_FEASIBLE' if not DEV_WORK else 'PARTIAL',
     f'{len(METRICS)} model evaluations; {len(PREDICTIONS)} test predictions; dev-selected families={len(SELECTED_FAMILY)}')
quiet_print('Bank:',BANK,'| evaluated models:',len(METRICS),'| test forecasts:',len(PREDICTIONS))


# ADVANCED · V4.00 · DEV-LOCKED NON-NEGATIVE SUPER LEARNER, NO TEST-TUNING
# Additional experiment: all feasible TASKS; legacy preregistered comparisons remain intact.
from scipy.optimize import minimize as _minimize
from sklearn.ensemble import ExtraTreesRegressor as _ExtraTreesRegressor
try:
    from xgboost import XGBRegressor as _XGBRegressor
    XGB_AVAILABLE=True
except Exception:
    XGB_AVAILABLE=False

def _advanced_estimator(family,seed,params=None):
    z=CFG['ADVANCED_N_JOBS'];p=params or {}
    if family=='XGBOOST':
        ensure(XGB_AVAILABLE,'XGBOOST_NOT_INSTALLED')
        return _XGBRegressor(n_estimators=p.get('n_estimators',260),max_depth=p.get('max_depth',5),
            learning_rate=p.get('learning_rate',0.045),min_child_weight=6,subsample=1.,colsample_bytree=1.,
            reg_lambda=7.,reg_alpha=0.3,tree_method='hist',n_jobs=z,random_state=seed,objective='reg:squarederror')
    if family=='EXTRA_TREES':
        return make_pipeline(SimpleImputer(strategy='median',add_indicator=True),
          _ExtraTreesRegressor(n_estimators=p.get('n_estimators',240),max_features=0.84,
             min_samples_leaf=8,max_depth=16,n_jobs=z,random_state=seed))
    if family=='RIDGE':return estimator('RIDGE',seed,{'alpha':20.})
    if family=='HGB':return estimator('HGB',seed,{'iterations':175,'leaves':23})
    if family=='CATBOOST':return estimator('CATBOOST',seed,{'iterations':320,'depth':6})
    if family=='LIGHTGBM':return estimator('LIGHTGBM',seed,{'iterations':250,'leaves':23})
    raise ValueError('ADVANCED_FAMILY_NOT_REGISTERED '+str(family))

_ADV_REQUEST=['XGBOOST','CATBOOST','LIGHTGBM','HGB','EXTRA_TREES','RIDGE']
_ADV_READY={'XGBOOST':XGB_AVAILABLE,'CATBOOST':CB_AVAILABLE,'LIGHTGBM':LGBM_AVAILABLE,
            'HGB':True,'EXTRA_TREES':True,'RIDGE':True}
ADV_BANK=[x for x in _ADV_REQUEST if _ADV_READY[x]]
if CFG['MODE']=='FULL' and CFG['ADVANCED_REQUIRE_ALL_FAMILIES']:
    ensure(all(_ADV_READY.values()),'ADVANCED_FULL_MODEL_DEPENDENCIES_INCOMPLETE')
ensure(len(ADV_BANK)>=3,'ADVANCED_ENSEMBLE_REQUIRES_THREE_MODEL_FAMILIES')
ADVANCED_DEV_ROWS=[];ADVANCED_TEST_ROWS=[];ADVANCED_INTERVAL_ROWS=[];ADVANCED_ENSEMBLE_REGISTRY={};ADVANCED_MODEL_REGISTRY={}

def _advance_design_model(family,tr,te,target,features,seed=1729):
    xtr,xte,used,refs=design(tr,te,features)
    # Missing predictors are supported by all configured estimators through native missing
    # handling (XGBoost, HGB, LGBM, CatBoost) or train-only median imputation.
    mod=_advanced_estimator(family,int(seed));mod.fit(xtr,tr[target].to_numpy(float))
    yp=np.asarray(mod.predict(xte),float).reshape(-1)
    ensure(len(yp)==len(te) and np.isfinite(yp).all(),'ADVANCED_PREDICTION_NONFINITE_'+family)
    return {'family':family,'model':mod,'prediction':yp,'used':used,'refs':refs,'cols':list(xte.columns)}

def _simplex_weights(y,prediction_columns):
    P=np.column_stack(prediction_columns).astype(float);v=np.asarray(y,float)
    ensure(np.isfinite(P).all() and np.isfinite(v).all(),'ENSEMBLE_DEV_NONFINITE')
    n=P.shape[1];baseline=int(np.argmin(np.mean(np.abs(P-v[:,None]),axis=0)))
    if n==1:return np.ones(1,dtype=float)
    # Smoothed absolute loss with weak shrinkage toward equal weights; DEV only.
    fun=lambda w:float(np.mean(np.sqrt((P@w-v)**2+0.03**2))+CFG['ADVANCED_WEIGHT_RIDGE']*np.sum(w**2))
    res=_minimize(fun,x0=np.full(n,1./n),method='SLSQP',bounds=[(0.,1.)]*n,
        constraints={'type':'eq','fun':lambda w:float(np.sum(w)-1.)},options={'maxiter':200,'ftol':1e-9})
    w=np.maximum(0,np.asarray(res.x if res.success else np.eye(n)[baseline],float))
    if not np.isfinite(w).all() or w.sum()<=0:w=np.eye(n)[baseline]
    w=w/w.sum()
    if mean_absolute_error(v,P@w)>mean_absolute_error(v,P[:,baseline])+1e-9:
        w=np.eye(n)[baseline]  # no regressive weights chosen on DEV
    ensure(np.all(w>=0) and abs(w.sum()-1)<1e-9,'ENSEMBLE_WEIGHT_SIMPLEX_BROKEN')
    return w

def _adv_fit_one_task(task):
    sid=task['study']['id'];target=task['target'];h=int(task['horizon']);proto=task['protocol']
    d=task['data'];m=task['masks'];tr=d[m['train']].copy();dv=d[m['dev']].copy();te=d[m['test']].copy()
    ensure(len(tr)>=CFG['MIN_TRAIN_ROWS'] and len(dv)>=CFG['MIN_TEST_ROWS'],'ADVANCED_DEV_TOO_SMALL')
    ensure(len(te)>=CFG['MIN_TEST_ROWS'],'ADVANCED_TEST_TOO_SMALL')
    if proto in ('ROLLING_TIME','CROSSED'):
        ensure(tr.target_year.max()<te.origin_year.min(),'ADVANCED_ORIGIN_CAUSAL_ORDER_BROKEN')
    if proto in ('TERRITORY_ONLY','CROSSED'):
        ensure(set(tr.region_name.astype(str)).isdisjoint(set(te.region_name.astype(str))),'ADVANCED_REGIONAL_LEAK')
        ensure(set(dv.entity_id).isdisjoint(set(te.entity_id)),'ADVANCED_GEOGRAPHICAL_DEV_LEAK')
    arms=task['study']['arms'];full_arm=max(arms,key=lambda a:len(a['features']));features=full_arm['features']
    supported=[f for f in features if f in tr and pd.to_numeric(tr[f],errors='coerce').notna().any()]
    if len(supported)<2:return {'status':'NOT_FEASIBLE','reason':'TOO_FEW_TRAIN_PREDICTORS'}
    seed=int(CFG['SEED']);devmods=[]
    for fam in ADV_BANK[:CFG['ADVANCED_MAX_CANDIDATE_FAMILIES']]:
        try:
            r=_advance_design_model(fam,tr,dv,target,features,seed)
            loss=float(mean_absolute_error(dv[target],r['prediction']))
            devmods.append((loss,fam,r))
            ADVANCED_DEV_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                'family':fam,'dev_mae':loss,'n_train':len(tr),'n_dev':len(dv),'n_columns':len(r['cols']),
                'selection':'PRE_TEST_HISTORICAL_DEV_ONLY','status':'PASS'})
        except Exception as ex:
            ADVANCED_DEV_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                'family':fam,'dev_mae':np.nan,'n_train':len(tr),'n_dev':len(dv),
                'selection':'PRE_TEST_HISTORICAL_DEV_ONLY','status':'FIT_FAILED_'+type(ex).__name__})
            log(f'ADV DEV {sid}/{h}/{fam}: {type(ex).__name__}: {str(ex)[:120]}','WARN')
    if len(devmods)<2:return {'status':'NOT_FEASIBLE','reason':'LESS_THAN_TWO_DEV_FAMILIES_FIT'}
    devmods.sort(key=lambda z:(z[0],z[1]));top=devmods[:min(CFG['ADVANCED_ENSEMBLE_K'],len(devmods))]
    weights=_simplex_weights(dv[target], [v[2]['prediction'] for v in top]);dev_mix=sum(float(w)*entry[2]['prediction'] for w,entry in zip(weights,top))
    dev_abs=np.abs(dv[target].to_numpy(float)-dev_mix)
    q90=float(np.quantile(dev_abs,min(1.,math.ceil((len(dv)+1)*0.9)/len(dv)),method='higher'))
    full=pd.concat([tr,dv],ignore_index=True);members=[];testpred=[]
    for (dev_mae,fam,_),weight in zip(top,weights):
        r=_advance_design_model(fam,full,te,target,features,seed)
        members.append({'family':fam,'weight':float(weight),'model':r['model'],'used':r['used'],
                        'refs':r['refs'],'columns':r['cols'],'dev_mae':dev_mae})
        testpred.append(r['prediction'])
    best_single_family=str(top[0][1]);best_single_prediction=np.asarray(testpred[0],float)
    pred=sum(member['weight']*v for member,v in zip(members,testpred))
    ensure(np.isfinite(pred).all(),'ADVANCED_FINAL_NONFINITE')
    arm='SUPERLEARNER_DEV';add_result(task,arm,'SUPERLEARNER',seed,full,te,pred,len(supported))
    for member in members:
        r=next(v for v,memb in zip(testpred,members) if memb is member)
        if CFG['ADVANCED_EXPORT_MEMBERS']:add_result(task,arm,'SL_'+member['family'],seed,full,te,r,len(member['used']))
    n_test=len(te);covers=np.abs(te[target].to_numpy(float)-pred)<=q90
    ADVANCED_INTERVAL_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
        'n_dev':len(dv),'n_test':n_test,'q90_abs_dev':q90,'empirical_test_coverage':float(covers.mean()),
        'avg_width':2*q90,'method':'DEV_RESIDUAL_QUANTILE_AFTER_REFIT_HEURISTIC','nominal':0.9,
        'formal_coverage_guarantee':False,'reason':'Temporal shift and refitting after DEV; not formal split-conformal coverage'})
    result={'study_id':sid,'target':target,'horizon':h,'protocol':proto,
        'n_train':len(tr),'n_dev':len(dv),'n_test':len(te),'arm':arm,
        'members':[m['family'] for m in members], 'weights':[round(m['weight'],8) for m in members],
        'dev_mae':float(mean_absolute_error(dv[target],dev_mix)),
        'test_mae':float(mean_absolute_error(te[target],pred)),
        'best_single_family_dev_locked':best_single_family,
        'best_single_dev_mae':float(top[0][0]),
        'best_single_test_mae':float(mean_absolute_error(te[target],best_single_prediction)),
        'ensemble_minus_single_test_mae':float(mean_absolute_error(te[target],pred)-mean_absolute_error(te[target],best_single_prediction)),
        'status':'PASS','selection':'DEV_ONLY','n_factors':len(supported)}
    if proto=='ROLLING_TIME':
        key=(sid,target,h)
        ADVANCED_ENSEMBLE_REGISTRY[key]={'members':members,'study_id':sid,'target':target,'horizon':h,
            'train':full,'test':te,'features':features,'pred':pred,'dev_predictions':dev_mix,
            'dev_target':dv[target].to_numpy(float),'q90_dev':q90,
            'best_single_family':best_single_family,'best_single_pred':best_single_prediction}
        base=WORK/'models'/f'SUPERLEARNER_{sid}_{target}_h{h}.joblib'
        joblib.dump({'members':members,'source_sha':source_sha,'model_stage':'DEV_FROZEN_FULL_REFIT',
                     'target':target,'horizon':h,'features':features},base,compress=3)
    return result

if CFG['ADVANCED_ENABLE']:
    progress('ADVANCED',0,len(TASKS),'Ансамбль: проверка по DEV')
    _adv_task_done=0
    for t in TASKS:
        sid=t['study']['id'];h=t['horizon'];target=t['target'];proto=t['protocol']
        try:ADVANCED_TEST_ROWS.append(_adv_fit_one_task(t))
        except Exception as ex:
            ADVANCED_TEST_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                'status':'FAIL','reason':type(ex).__name__+': '+str(ex)[:250]})
            log('ADVANCED ENSEMBLE '+sid+'/'+target+': '+type(ex).__name__+': '+str(ex)[:200],'ERROR')
        _adv_task_done+=1
        progress('ADVANCED',_adv_task_done,len(TASKS),f'Задач: {_adv_task_done}/{len(TASKS)}')
    progress('ADVANCED',1,1,'DEV-ансамбль завершён')
if not CFG['ADVANCED_ENABLE']:progress('ADVANCED',1,1,'Ансамбль отключён настройкой')
ADVANCED_DEV=pd.DataFrame(ADVANCED_DEV_ROWS);ADVANCED_ENSEMBLES=pd.DataFrame(ADVANCED_TEST_ROWS)
ADVANCED_INTERVALS=pd.DataFrame(ADVANCED_INTERVAL_ROWS)
if CFG['ADVANCED_ENABLE']:
    allpass=bool(len(ADVANCED_ENSEMBLES)==len(TASKS) and ADVANCED_ENSEMBLES.status.eq('PASS').all()) if len(TASKS) else False
    mark('ADVANCED_ENSEMBLE','PASS' if allpass else 'PARTIAL',
        f'completed={int(ADVANCED_ENSEMBLES.status.eq("PASS").sum()) if len(ADVANCED_ENSEMBLES) else 0}/{len(TASKS)}; ' +
        f'families={ADV_BANK}; dev-only weight calibration')
else:mark('ADVANCED_ENSEMBLE','NOT_FEASIBLE','Explicitly disabled')
# Rebuild canonical output structures; the next cell computes all independent metrics/CI from combined predictions.
METRICS=pd.DataFrame(METRIC_ROWS)
PREDICTIONS=pd.concat(PRED_ROWS,ignore_index=True) if PRED_ROWS else pd.DataFrame()
if len(PREDICTIONS):
    _id_cols=['entity_id','origin_year','target_year','horizon','target','study_id','split_id','arm','model','seed']
    ensure(not PREDICTIONS.duplicated(_id_cols).any(),'ADVANCED_OUTPUT_DUPLICATE_TEST_KEYS')
quiet_print('Advanced:',len(ADVANCED_ENSEMBLES),'tasks | pass',int(ADVANCED_ENSEMBLES.status.eq('PASS').sum()) if len(ADVANCED_ENSEMBLES) else 0,
      '| full prediction rows:',len(PREDICTIONS))

# CHECKPOINT II: полные модельные предсказания и реестр выборки, включая ансамбли.
if CFG.get('ENABLE_CHECKPOINTS'):
    cp=WORK/'checkpoints';cp.mkdir(parents=True,exist_ok=True)
    checkpoint_ii=[]
    for label,frame in [('all_test_predictions',PREDICTIONS),('model_test_metrics',METRICS),
                        ('dev_model_selection',DEV_METRICS),('test_split_registry',SPLITS),
                        ('ensemble_model_tests',ADVANCED_ENSEMBLES),('ensemble_dev_selection',ADVANCED_DEV),
                        ('seed_independence_QA',SEED_QA)]:
        file=cp/(label+'.parquet');frame.to_parquet(file,index=False,compression='zstd')
        checkpoint_ii.append({'file':file.name,'rows':len(frame),'columns':len(frame.columns),'sha256':digest(file)})
    (cp/'stage_II_checkpoint.json').write_text(json.dumps({'version':CFG['VERSION'],
       'status':'PHYSICAL_PARQUET_CHECKPOINT','source_commit':source_sha,
       'assets':checkpoint_ii,'saved_models':[x.name for x in sorted((WORK/'models').glob('*.joblib'))]},
       ensure_ascii=False,indent=2),encoding='utf-8')
    progress('ADVANCED',1,1,'Модели и checkpoint II зафиксированы')

### Этап III / IV. Статистика, интерпретация и проверка устойчивости

Парные сравнения на общих муниципалитетах и годах, региональные bootstrap-интервалы, поправки Холма и FDR, SHAP, ALE, PDP, ICE, перестановочные тесты, временной replay/canary, анализ лагов публикации и сензитивности, визуализации и Excel-книга.

In [ ]:
# 09 · COMMON-KEY ABLATIONS, REGION-BLOCK BOOTSTRAP, HOLM/FDR, ALE, INTERACTIONS
progress('STATS',0,1,'Парные сравнения и статистика')
ABLA_ROWS=[];LOO_ROWS=[];CORR_ROWS=[];INTERACTION_ROWS=[];TRANSFER_ROWS=[]
GEO_ROWS=[];IMP_ROWS=[];ALE_ROWS=[];SHAP_ROWS=[];FE_ROWS=[];SENSITIVITY_ROWS=[];ERROR_YEAR_ROWS=[]
def holm_adjust(p):
    arr=np.asarray(p,float);order=np.argsort(arr);out=np.empty(len(arr));running=0.
    for rank,j in enumerate(order):
        running=max(running,min(1.0,float(arr[j])*(len(arr)-rank)));out[j]=running
    return out
def bh_adjust(p):
    arr=np.asarray(p,float);ordr=np.argsort(arr);out=np.empty(len(arr));running=1.
    for rank in reversed(range(len(arr))):
        j=ordr[rank];running=min(running,float(arr[j])*len(arr)/(rank+1));out[j]=running
    return out
def block_delta(z,old,new,boots,seed):
    v=z[['region_name','target_year',old,new]].dropna()
    groups=[g for _,g in v.groupby('region_name',dropna=False)]
    if len(groups)<CFG['MIN_BOOTSTRAP_REGIONS']:return None
    d=np.array([(g[old]-g[new]).sum() for g in groups]);n=np.array([len(g) for g in groups])
    point=float(d.sum()/n.sum())
    rng=np.random.default_rng(seed);ids=rng.integers(0,len(groups),(boots,len(groups)))
    vals=(d[ids].sum(axis=1)/n[ids].sum(axis=1))
    ci=np.quantile(vals,[0.025,0.975])
    # Monte Carlo finite-sample p with +1 correction; no p=0 due to bootstrap discretization.
    # Random region-level sign-flips approximate a paired null test; CI remains a block bootstrap.
    signs=rng.choice(np.array([-1.,1.]),size=(boots,len(groups)),replace=True)
    null_dist=(signs*d).sum(axis=1)/n.sum()
    p=(1+int((np.abs(null_dist)>=abs(point)).sum()))/(boots+1)
    return {'delta_mae':point,'ci_low':float(ci[0]),'ci_high':float(ci[1]),
         'p_cluster_signflip':float(p),'n_regions':len(groups),'n_common_keys':len(v),
         'bootstrap_unit':'region','n_bootstraps':boots}

# Isolated, inexpensive sign-flip / regional block unit test.
_check=pd.DataFrame({'region_name':list('ABCDEFGH')*2,'target_year':[2022]*16,
                      '_baseline_error':[2.0]*16,'_improved_error':[1.0]*16})
_st=block_delta(_check,'_baseline_error','_improved_error',99,CFG['SEED'])
ensure(_st is not None and abs(_st['delta_mae']-1.0)<1e-12 and _st['n_regions']==8,
       'UNIT_CLUSTER_INFERENCE_FAILURE')
mark('UNIT_STAT_CONTRACT','PASS','Synthetic sign-flip/bootstrap arithmetic; excluded from scientific findings')

if len(PREDICTIONS):
    selected=PREDICTIONS[~PREDICTIONS.model.isin(['HISTORICAL','REGIONAL_MEDIAN'])].copy()
    keys=['entity_id','origin_year','target_year','region_name']
    for (sid,target,h,proto,model),p in selected.groupby(['study_id','target','horizon','split_id','model']):
        # Average losses over stochastic seeds BEFORE block resampling; seeds are not independent municipalities.
        grp=p.groupby(keys+['arm'],dropna=False).agg(absolute_error=('absolute_error','mean')).reset_index()
        wide=grp.pivot(index=keys,columns='arm',values='absolute_error').reset_index()
        st=next(s for s in STUDIES if s['id']==sid)
        if st['id']=='CANONICAL' or st['id'].startswith('YEAR_'):
            pairs=[(a['arm'],b['arm']) for a,b in zip(st['arms'][:-1],st['arms'][1:])]
            pairs+=([('M1','M3')] if 'M3' in wide and 'M1' in wide else [])
        elif st['id'].startswith('AUX_'):
            pairs=[('M0',st['arms'][-1]['arm'])]
        else:
            pairs=[('M0','FULL_AVAILABLE')]
            pairs += [('WITHOUT_'+g,'FULL_AVAILABLE') for g in AVAILABLE_GROUPS if 'WITHOUT_'+g in wide]
        for old,new in pairs:
            if old not in wide or new not in wide:continue
            stat=block_delta(wide,old,new,CFG['BOOTSTRAPS'],CFG['SEED']+h)
            if stat:
                row={'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                     'model':model,'old_arm':old,'new_arm':new,
                     'comparison':old+' → '+new,'confirmatory':(st['confirmatory'] and
                     len(TASK_STATUSES.loc[(TASK_STATUSES.study_id==sid)&
                      (TASK_STATUSES.target==target)&(TASK_STATUSES.horizon==h)&
                      (TASK_STATUSES.protocol==proto)&
                      (TASK_STATUSES.scientific_status=='COMPLETE_YEARS')])>0)}
                ABLA_ROWS.append({**row,**stat})
        best=p[p.arm.eq(st['arms'][-1]['arm'])].drop_duplicates(
            ['entity_id','origin_year','target_year','arm'],keep='first')
        if len(best):
            for reg,v in best.groupby('region_name'):
                if len(v)>5:TRANSFER_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                    'protocol':proto,'region':reg,'n':len(v),'mae':float(v.absolute_error.mean())})
            for (reg,mt),v in best.groupby(['region_name','mun_type'],dropna=False):
                if len(v)>5:GEO_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                    'protocol':proto,'region':reg,'mun_type_original':mt,'n':len(v),
                    'mae':float(v.absolute_error.mean())})
            for year,v in best.groupby('target_year'):
                ERROR_YEAR_ROWS.append({'study_id':sid,'target':target,'horizon':h,'protocol':proto,
                   'year':int(year),'n':len(v),'mae':float(v.absolute_error.mean())})
ABLA=pd.DataFrame(ABLA_ROWS)
if len(ABLA):
    ABLA['p_holm']=np.nan;ABLA['q_fdr_bh']=np.nan
    for keys,inds in ABLA.groupby(['target','horizon','protocol']).groups.items():
        for is_confirm in (True,False):
            ix=[i for i in inds if bool(ABLA.loc[i,'confirmatory'])==is_confirm]
            if ix:
                p=ABLA.loc[ix,'p_cluster_signflip'].tolist()
                if is_confirm:ABLA.loc[ix,'p_holm']=holm_adjust(p)
                else:ABLA.loc[ix,'q_fdr_bh']=bh_adjust(p)
    ABLA['finding']=np.where((ABLA.ci_low>0)&(
          np.where(ABLA.confirmatory,ABLA.p_holm,ABLA.q_fdr_bh)<0.05),
          'PREDICTIVE_IMPROVEMENT',np.where(ABLA.ci_high<0,'DEGRADATION_OR_NEGATIVE','NOT_CONFIRMED'))
    mark('PASS_VALIDATION','PASS',f'{len(ABLA)} paired group contrasts; region-block CI; Holm confirmatory/FDR exploratory')
else:mark('PASS_VALIDATION','NOT_FEASIBLE','Not enough regions/pairs for block bootstrap')
LOO=ABLA[ABLA.study_id.eq('MAX_AVAILABLE')&ABLA.old_arm.str.startswith('WITHOUT_',na=False)].copy() if len(ABLA) else pd.DataFrame()
REGIONAL_ERRORS=pd.DataFrame(TRANSFER_ROWS);MUNICIPAL_TYPES=pd.DataFrame(GEO_ROWS)
ERROR_YEARS=pd.DataFrame(ERROR_YEAR_ROWS)

# Непосредственно из сохранённых test predictions: независимый пересчёт каждой MAE/RMSE.
PREDICTION_METRIC_QA=pd.DataFrame()
if len(PREDICTIONS) and len(METRICS):
    groups=['study_id','target','horizon','split_id','arm','model','seed']
    pp=PREDICTIONS.copy()
    pp['weight_eff']=pd.to_numeric(pp.population_weight,errors='coerce').fillna(0).clip(lower=0)
    pp['weighted_loss']=pp.absolute_error*pp.weight_eff
    calc=pp.groupby(groups,dropna=False).agg(
      independent_mae=('absolute_error','mean'),
      independent_rmse=('squared_error',lambda z:float(np.sqrt(np.mean(z)))),
      independent_n=('entity_id','size'),sum_weight=('weight_eff','sum'),
      sum_weighted_loss=('weighted_loss','sum')).reset_index().rename(columns={'split_id':'protocol'})
    calc['independent_weighted_mae']=np.where(calc.sum_weight.gt(0),
         calc.sum_weighted_loss/calc.sum_weight,np.nan)
    cmp=METRICS.merge(calc,on=['study_id','target','horizon','protocol','arm','model','seed'],
        how='outer',validate='one_to_one',indicator=True)
    cmp['mae_abs_difference']=(cmp.mae-cmp.independent_mae).abs()
    cmp['rmse_abs_difference']=(cmp.rmse-cmp.independent_rmse).abs()
    cmp['n_difference']=(cmp.n_test-cmp.independent_n).abs()
    cmp['weighted_mae_abs_difference']=(cmp.weighted_mae-cmp.independent_weighted_mae).abs()
    PREDICTION_METRIC_QA=cmp[['study_id','target','horizon','protocol','arm','model','seed',
       '_merge','mae_abs_difference','rmse_abs_difference',
       'weighted_mae_abs_difference','n_difference']].copy()
    passed=(cmp['_merge'].eq('both').all() and
       cmp[['mae_abs_difference','rmse_abs_difference','weighted_mae_abs_difference']].max().max()<1e-10 and
       cmp.n_difference.eq(0).all())
    mark('INDEPENDENT_METRIC_RECONCILIATION','PASS' if passed else 'FAIL',
         f'rows={len(cmp)}; max MAE diff={cmp.mae_abs_difference.max()}')
    ensure(passed,'INDEPENDENT_METRIC_RECONCILIATION_FAILED')

# Correlation diagnostics use TRAIN ONLY and the ORIGINAL training measurements (no post-hoc test selection).
for task in [t for t in TASKS if t['protocol']=='ROLLING_TIME' and t['study']['id']=='MAX_AVAILABLE']:
    tr=task['data'][task['masks']['train']]
    ff=list(dict.fromkeys(G0_FEATURES+sum(GROUP_FEATURES.values(),[])))
    for i,one in enumerate(ff):
        for two in ff[i+1:]:
            if one=='origin_year' or two=='origin_year':continue
            sample=tr[[one,two]].apply(pd.to_numeric,errors='coerce').dropna()
            if len(sample)<50 or min(sample[one].nunique(),sample[two].nunique())<5:continue
            r=sample[one].corr(sample[two],method='spearman')
            if np.isfinite(r) and abs(r)>=0.7:
                CORR_ROWS.append({'target':task['target'],'horizon':task['horizon'],
                  'feature_A':one,'feature_B':two,'n_train_pairs':len(sample),'spearman_r':float(r)})
CORRELATIONS=pd.DataFrame(CORR_ROWS)

# Interpretation of the fitted predictive function, not causal effects.
for (sid,target,h),registry in MODEL_REGISTRY.items():
    if sid not in ('MAX_AVAILABLE','CANONICAL'):continue
    tr=registry['train'];te=registry['test'];model=registry['model']
    Xtr,Xte,used,_=design(tr,te,registry['features'],registry['refs'])
    n=min(len(te),CFG['MAX_IMPORTANCE_ROWS'])
    if n<40:continue
    indices=np.sort(np.random.default_rng(CFG['SEED']).choice(len(te),n,replace=False))
    x=Xte.iloc[indices].copy();y=te[target].to_numpy(float)[indices]
    try:
        benchmark=mean_absolute_error(y,model.predict(x))
        for col in [u for u in used if u!='origin_year'][:CFG['MAX_IMPORTANCE_FEATURES']]:
            losses=[]
            for s in (11,37,71):
                z=x.copy();z[col]=np.random.default_rng(s).permutation(z[col].to_numpy())
                losses.append(float(mean_absolute_error(y,model.predict(z))-benchmark))
            IMP_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':col,
                'mae_increase':float(np.mean(losses)),'sd_permutation':float(np.std(losses)),
                'n_test_sample':n,'scope':'predictive_unconditional_permutation'})
        # First-order ALE on observed TRAIN support; percentiles computed without final test outcomes.
        for col in [u for u in used if u.startswith(('G1_','G2_','G3_','G4_'))][:4]:
            observed=Xtr[col].dropna()
            if observed.nunique()<12:continue
            edges=np.unique(np.quantile(observed,np.linspace(0,1,11)))
            if len(edges)<5:continue
            accumulated=0.;raw_rows=[]
            for k in range(len(edges)-1):
                lo,hi=edges[k:k+2]
                mask=Xtr[col].between(lo,hi,inclusive='both')
                if mask.sum()<15:continue
                xl=Xtr.loc[mask].copy();xh=xl.copy();xl[col]=lo;xh[col]=hi
                inc=float(np.mean(model.predict(xh)-model.predict(xl)))
                accumulated+=inc
                raw_rows.append((k,(lo+hi)/2.,inc,accumulated,int(mask.sum())))
            if raw_rows:
                center=float(np.average([v[3] for v in raw_rows],weights=[v[4] for v in raw_rows]))
                for k,middle,inc,total,nn in raw_rows:
                    ALE_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':col,
                      'bin':k,'feature_mid_logscale':middle,'ale_centered':total-center,'n_train':nn,
                      'scope':'train_support_first_order_ALE_not_causal'})
    except Exception as ex:log('importance/ALE '+sid+': '+type(ex).__name__,'WARN')
IMPORTANCE=pd.DataFrame(IMP_ROWS);ALE=pd.DataFrame(ALE_ROWS)

# Additional interpretable evidence: SHAP for best available tree model, on held-out data.
try:
    import shap
    SHAP_AVAILABLE=True
except Exception as ex:
    SHAP_AVAILABLE=False;log('SHAP import unavailable: '+type(ex).__name__,'WARN')
if SHAP_AVAILABLE:
    for (sid,target,h),r in MODEL_REGISTRY.items():
        if sid!='MAX_AVAILABLE' or r['family'] not in ('HGB','CATBOOST','LIGHTGBM'):continue
        try:
            xtr,xtest,used,_=design(r['train'],r['test'],r['features'],r['refs'])
            take=min(len(xtest),CFG['MAX_SHAP_ROWS']);xtest=xtest.iloc[:take]
            tree=shap.TreeExplainer(r['model'])
            values=tree.shap_values(xtest)
            if isinstance(values,list):values=values[0]
            values=np.asarray(values)
            ensure(values.shape==(len(xtest),xtest.shape[1]),'SHAP_SHAPE_MISMATCH')
            for j,name in enumerate(xtest.columns):
                SHAP_ROWS.append({'study_id':sid,'target':target,'horizon':h,'model':r['family'],
                  'feature':name,'mean_abs_shap':float(np.abs(values[:,j]).mean()),
                  'n_test_sample':take,'scope':'explanation_of_prediction_not_causal'})
        except Exception as ex:log('SHAP computation '+sid+': '+type(ex).__name__+' '+str(ex)[:120],'WARN')
SHAP_IMPORTANCE=pd.DataFrame(SHAP_ROWS)
mark('SHAP_QA','PASS' if len(SHAP_IMPORTANCE) else 'NOT_FEASIBLE',
     f'tree SHAP cells={len(SHAP_IMPORTANCE)}; optional where compatible')

# Train-only two-way fixed effects: identifiability / cluster diagnostics, never causal.
FE_QA_ROWS=[]
try:
    from linearmodels.panel import PanelOLS
    PANEL_OLS_AVAILABLE=True
except ImportError:
    PANEL_OLS_AVAILABLE=False

def _fit_fe_train_only(task):
    tr=task['data'].loc[task['masks']['train']].copy()
    report={'target':task['target'],'horizon':task['horizon'],
      'n_initial':int(len(tr)),'n_complete':0,'n_entities':0,'n_regions':0,
      'n_years':0,'n_regressors':0,'status':'NOT_FEASIBLE','reason':''}
    if not PANEL_OLS_AVAILABLE:
        report['reason']='LINEARMODELS_NOT_INSTALLED';return pd.DataFrame(),report
    available=[]
    for g in ('G1','G2','G3','G4','G5'):
        ff=[f for f in GROUP_FEATURES.get(g,[]) if f in tr]
        ff.sort(key=lambda f:tr[f].notna().sum(),reverse=True)
        if ff:available.append(ff[0])
    if not available:
        report['reason']='NO_FE_COVARIATES';return pd.DataFrame(),report
    columns=['entity_id','origin_year','region_name',task['target']]+available
    w=tr[columns].copy()
    for f in available:w[f]=pd.to_numeric(w[f],errors='coerce')
    w=w.dropna()
    counts=w.groupby('entity_id').origin_year.nunique()
    w=w[w.entity_id.isin(counts[counts.ge(CFG['MIN_FE_ENTITY_OBSERVATIONS'])].index)].copy()
    report.update(n_complete=int(len(w)),n_entities=int(w.entity_id.nunique()),
        n_regions=int(w.region_name.nunique()),n_years=int(w.origin_year.nunique()))
    if len(w)<400 or w.entity_id.nunique()<20 or w.region_name.nunique()<8 or w.origin_year.nunique()<4:
        report['reason']='INSUFFICIENT_CONNECTED_FE_PANEL';return pd.DataFrame(),report
    # Drop regressors absorbed by entity or time effects: residual variance of two-way demeaned x.
    keys=['entity_id','origin_year']
    X=pd.DataFrame(index=w.index)
    for f in available:
        x=np.sign(w[f])*np.log1p(np.abs(w[f]))
        residual=x-x.groupby(w.entity_id).transform('mean')-x.groupby(w.origin_year).transform('mean')+x.mean()
        if residual.std()>1e-8:X[f]=x
    if X.empty:
        report['reason']='ALL_COVARIATES_ABSORBED_BY_FIXED_EFFECTS';return pd.DataFrame(),report
    residualized=pd.DataFrame(index=w.index)
    for f in X:
        a=X[f]
        residualized[f]=a-a.groupby(w.entity_id).transform('mean')-a.groupby(w.origin_year).transform('mean')+a.mean()
    rank=np.linalg.matrix_rank(residualized.to_numpy(float),tol=1e-9)
    if rank<len(residualized.columns):
        selected=[]
        for f in residualized.columns:
            candidate=selected+[f]
            if np.linalg.matrix_rank(residualized[candidate].to_numpy(float),tol=1e-9)==len(candidate):
                selected.append(f)
        X=X[selected]
    report['n_regressors']=len(X.columns)
    if X.empty:
        report['reason']='COLLINEAR_WITHIN_FE';return pd.DataFrame(),report
    aligned=w.set_index(keys).sort_index()
    x=X.loc[aligned.reset_index().index] if False else pd.DataFrame(
       X.to_numpy(float),index=pd.MultiIndex.from_frame(w[keys]),columns=X.columns).sort_index()
    y=aligned[task['target']].astype(float)
    groups=pd.DataFrame({'region_cluster':pd.factorize(aligned.region_name.astype(str))[0]},index=aligned.index)
    try:
        model=PanelOLS(y,x,entity_effects=True,time_effects=True,drop_absorbed=True,check_rank=False)
        fit=model.fit(cov_type='clustered',clusters=groups)
        rows=pd.DataFrame([{'target':task['target'],'horizon':task['horizon'],'parameter':f,
           'coefficient':float(fit.params[f]),'std_error_region_cluster':float(fit.std_errors[f]),
           'p_value':float(fit.pvalues[f]),'n_train':int(len(w)),
           'n_municipalities':int(w.entity_id.nunique()),'n_clusters':int(w.region_name.nunique()),
           'interpretation':'train-only two-way FE; predictive association, NOT causal'}
          for f in fit.params.index])
        report.update(status='PASS',reason='TWO_WAY_FE_TRAIN_ONLY_REGION_CLUSTERED')
        return rows,report
    except Exception as ex:
        report['reason']='FIT_EXCEPTION_'+type(ex).__name__+': '+str(ex)[:200]
        return pd.DataFrame(),report

if PANEL_OLS_AVAILABLE:
    for task in TASKS:
        if task['study']['id']!='MAX_AVAILABLE' or task['protocol']!='ROLLING_TIME' or task['horizon']!=1:
            continue
        rows,qa=_fit_fe_train_only(task)
        FE_QA_ROWS.append(qa)
        if not rows.empty:FE_ROWS.extend(rows.to_dict('records'))
else:
    FE_QA_ROWS.append({'status':'NOT_FEASIBLE','reason':'LINEARMODELS_UNAVAILABLE'})
FE_QA=pd.DataFrame(FE_QA_ROWS)
FE_DIAGNOSTICS=pd.DataFrame(FE_ROWS)
mark('FE_DIAGNOSTICS','PASS' if len(FE_DIAGNOSTICS) else 'NOT_FEASIBLE',
     f'train-only FE parameter rows={len(FE_DIAGNOSTICS)}, audited attempts={len(FE_QA)}')
# Exploratory interactions (H3.2), identical test labels/keys, no post-hoc causal claim.
# Non-arbitrary G6: semantically identified wage/income by housing access or stock.
def _feature_for_domain(group,domains):
    rows=FEATURE_DICT[FEATURE_DICT.group.eq(group)&
       FEATURE_DICT.domain.isin(domains)&FEATURE_DICT.model_feature.notna()]
    for record in rows.to_dict('records'):
        if record['model_feature'] in GROUP_FEATURES[group]:return record['model_feature']
    return None
G6_ECON_FEATURE=_feature_for_domain('G1',('income','wages'))
G6_HOUSING_FEATURE=_feature_for_domain('G2',('housing_access','housing_stock','housing_construction'))
if G6_ECON_FEATURE and G6_HOUSING_FEATURE:
    g1,g2=G6_ECON_FEATURE,G6_HOUSING_FEATURE
    for (sid,target,h),reg in MODEL_REGISTRY.items():
        if sid!='MAX_AVAILABLE':continue
        tr=reg['train'].copy();te=reg['test'].copy()
        inter='G6_income_housing_interaction'
        signed_log=lambda x:np.sign(pd.to_numeric(x,errors='coerce'))*np.log1p(np.abs(pd.to_numeric(x,errors='coerce')))
        tr[inter]=signed_log(tr[g1])*signed_log(tr[g2]);te[inter]=signed_log(te[g1])*signed_log(te[g2])
        if tr[inter].notna().sum()<CFG['MIN_TRAIN_ROWS']:continue
        try:
            pred,_,_,_=fit_predict(reg['family'],tr,te,target,reg['features']+[inter],CFG['SEED'],reg['params'])
            ref=metrics(te[target],reg['pred'],te[f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
            cur=metrics(te[target],pred,te[f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
            INTERACTION_ROWS.append({'target':target,'horizon':h,'study_id':sid,
              'factor_A':g1,'factor_B':g2,'base_mae':ref['mae'],'interaction_mae':cur['mae'],
              'delta_mae':ref['mae']-cur['mae'],'n_test':len(te),
              'status':'EXPLORATORY_PREDICTIVE_NOT_CAUSAL'})
        except Exception as ex:log('interaction '+str(h)+': '+type(ex).__name__,'WARN')
INTERACTIONS=pd.DataFrame(INTERACTION_ROWS)
mark('H3.2','PASS' if len(INTERACTIONS) else 'NOT_FEASIBLE',
     f'{len(INTERACTIONS)} semantically anchored income/wages × housing comparisons')
mark('CAUSAL_MODULE','CAUSAL_NOT_IDENTIFIED','No validated treatment register with dates, exposure, and controls')


# ADVANCED · V4.00 · ENSEMBLE SHAP, ADDITIVITY, GROUP PERMUTATION, ALE/PDP/ICE, REGION STABILITY
# All explanations refer to frozen predictors trained without TEST labels. Predictive, never causal.
ADVANCED_SHAP_ROWS=[];ADVANCED_SHAP_LOCAL_ROWS=[];ADVANCED_SHAP_QA_ROWS=[]
ADVANCED_GROUP_PERM_ROWS=[];ADVANCED_PDP_ROWS=[];ADVANCED_ICE_ROWS=[];ADVANCED_SHAP_CI_ROWS=[];ADVANCED_SHAP_STABILITY_ROWS=[];ADVANCED_SHAP_INTERACTION_ROWS=[];ADVANCED_LOO_ROWS=[]
try:import shap as _shap
except Exception as _e:
    _shap=None;log('Advanced SHAP import failed: '+str(_e)[:150],'WARN')

def _explain_one(m,x,bg):
    model=m['model'];family=m['family'];cols=list(x.columns)
    if family=='RIDGE':
        prep=model[:-1];head=model[-1];tx=np.asarray(prep.transform(x),float);tb=np.asarray(prep.transform(bg),float)
        mean=tb.mean(axis=0);co=np.asarray(head.coef_,float)
        v=(tx-mean)*co;bias=np.full(len(x),float(head.intercept_+mean@co))
        names=list(cols)+[f'missing_indicator_{int(k)}' for k in prep[0].indicator_.features_] if v.shape[1]>len(cols) else list(cols)
        return v,bias,names,'ANALYTICAL_LINEAR'
    if family=='EXTRA_TREES':
        prep=model[:-1];tree=model[-1];xx=np.asarray(prep.transform(x),float)
        names=list(cols)+[f'missing_indicator_{int(k)}' for k in prep[0].indicator_.features_] if xx.shape[1]>len(cols) else list(cols)
        if _shap is None:raise RuntimeError('SHAP_UNAVAILABLE_FOR_EXTRA_TREES')
        expl=_shap.TreeExplainer(tree,feature_perturbation='tree_path_dependent')
        values=np.asarray(expl.shap_values(xx),float)
        bias=np.broadcast_to(np.asarray(expl.expected_value,float).reshape(-1)[0],len(x))
        return values,bias,names,'TREE_PATH_DEPENDENT'
    if family=='XGBOOST':
        import xgboost
        dm=xgboost.DMatrix(x,feature_names=cols)
        raw=np.asarray(model.get_booster().predict(dm,pred_contribs=True),float)
        ensure(raw.shape==(len(x),len(cols)+1),'XGB_SHAP_CONTRIBS_WRONG_SHAPE')
        return raw[:,:-1],raw[:,-1],cols,'NATIVE_TREE_SHAP'
    if family=='LIGHTGBM':
        raw=np.asarray(model.predict(x,pred_contrib=True),float)
        ensure(raw.shape==(len(x),len(cols)+1),'LGB_SHAP_CONTRIBS_WRONG_SHAPE')
        return raw[:,:-1],raw[:,-1],cols,'NATIVE_TREE_SHAP'
    if family=='CATBOOST':
        from catboost import Pool
        raw=np.asarray(model.get_feature_importance(Pool(x),type='ShapValues'),float)
        ensure(raw.shape==(len(x),len(cols)+1),'CATBOOST_SHAP_CONTRIBS_WRONG_SHAPE')
        return raw[:,:-1],raw[:,-1],cols,'NATIVE_TREE_SHAP'
    if _shap is None:raise RuntimeError('SHAP_NOT_INSTALLED_FOR_'+family)
    expl=_shap.TreeExplainer(model,feature_perturbation='tree_path_dependent')
    values=np.asarray(expl.shap_values(x),float)
    return values,np.broadcast_to(float(np.asarray(expl.expected_value).reshape(-1)[0]),len(x)),cols,'TREE_PATH_DEPENDENT'

def _adv_predict(reg,x):
    pp=[]
    for m in reg['members']:
        pp.append(m['weight']*np.asarray(m['model'].predict(x),float))
    return np.sum(pp,axis=0)

def _adv_feature_group(feature):
    if feature.startswith('G1R_'):return 'G1R_REGIONAL_NOT_MUNICIPAL'
    for g in ('G1','G2','G3','G4','G5'):
        if feature.startswith(g+'_'):return g
    if feature.startswith('region_'):return 'REGION_FIXED_DUMMIES'
    if feature.startswith('hist_'):return 'G0_DEMOGRAPHIC_HISTORY'
    if feature=='origin_year':return 'G0_TIME'
    return 'OTHER'

def _within_strata_permute(X,source,cols,seed):
    rng=np.random.default_rng(seed);res=X.copy();groups=source[['region_name','target_year']].reset_index(drop=True)
    ensure(len(groups)==len(X),'SHUFFLE_STRATA_ALIGNMENT_INVALID')
    # Joint permutation preserves correlations WITHIN each named feature block.
    # Within region-year strata when >=2; isolated observations remain unchanged.
    for _,group in groups.groupby(['region_name','target_year'],dropna=False).groups.items():
        ids=np.asarray(list(group),dtype=int)
        if len(ids)<2:continue
        shuffle=rng.permutation(ids)
        res.iloc[ids,res.columns.get_indexer(cols)]=X.iloc[shuffle,X.columns.get_indexer(cols)].to_numpy()
    return res

def _adv_explain_registry(reg):
    sid=reg['study_id'];target=reg['target'];h=reg['horizon'];tr=reg['train'];te=reg['test']
    cols=reg['members'][0]['columns']
    # Design by the exact fitted train population; no fitted statistics from target TEST.
    _,XX,_,_=design(tr,te,reg['features'],reg['members'][0]['refs'])
    indices=np.sort(np.random.default_rng(CFG['SEED']+h).choice(len(XX),
                  size=min(CFG['ADVANCED_SHAP_ROWS'],len(XX)),replace=False))
    x=XX.iloc[indices].reset_index(drop=True);meta=te.iloc[indices].reset_index(drop=True)
    background=tr.sample(n=min(CFG['ADVANCED_BACKGROUND_ROWS'],len(tr)),random_state=CFG['SEED'])
    # Design shares the same fitted train population for test and background.
    _,BACK,_,_=design(tr,background,reg['features'],reg['members'][0]['refs'])
    ensure(list(x.columns)==list(BACK.columns),'SHAP_BACKGROUND_COLUMN_DIFFERENCE')
    contributions=np.zeros((len(x),len(cols)),float);bline=np.zeros(len(x),float);ok_members=0
    fam_rows=[]
    for member in reg['members']:
        fam=member['family'];weight=member['weight']
        try:
            XX_m=X if False else x
            vals,baseval,names,method=_explain_one(member,XX_m,BACK)
            ensure(vals.shape==(len(x),len(names)),'SHAP_MEMBER_MATRIX_SHAPE')
            reconstructed=np.asarray(baseval,float)+vals.sum(axis=1)
            direct=np.asarray(member['model'].predict(XX_m),float)
            discrepancy=np.max(np.abs(reconstructed-direct))
            ensure(discrepancy<CFG['ADVANCED_SHAP_TOLERANCE'],'SHAP_MEMBER_ADDITIVITY_'+fam)
            for j,feature in enumerate(names):
                fam_rows.append({'study_id':sid,'target':target,'horizon':h,'method':'SHAP_MEMBER',
                    'model':fam,'feature':feature,'group':_adv_feature_group(feature),
                    'mean_abs_shap':float(np.abs(vals[:,j]).mean()),
                    'mean_signed_shap':float(vals[:,j].mean()),'ensemble_weight':weight,
                    'n_explained':len(x),'mode':method,'scope':'PREDICTIVE_NOT_CAUSAL'})
                if feature in cols:contributions[:,cols.index(feature)]+=weight*vals[:,j]
                else:
                    # Imputer missing indicators contribute jointly to their original input.
                    if feature.startswith('missing_indicator_'):
                        nmissing=int(feature.split('_')[-1]);
                        if nmissing<len(cols):contributions[:,nmissing]+=weight*vals[:,j]
                    else:raise RuntimeError('UNRECOGNIZED_SHAP_INTERNAL_FEATURE_'+feature)
            bline+=weight*np.asarray(baseval,float)
            ADVANCED_SHAP_QA_ROWS.append({'study_id':sid,'target':target,'horizon':h,'member':fam,
                'max_additivity_abs_error':float(discrepancy),'n_explained':len(x),'status':'PASS','mode':method})
            ok_members+=1
        except Exception as ex:
            ADVANCED_SHAP_QA_ROWS.append({'study_id':sid,'target':target,'horizon':h,'member':fam,
                'max_additivity_abs_error':np.nan,'n_explained':len(x),'status':'FAIL',
                'reason':type(ex).__name__+': '+str(ex)[:160]})
            log('SHAP member '+sid+'/'+fam+': '+str(ex)[:160],'ERROR')
    if ok_members!=len(reg['members']):return 0
    direct=_adv_predict(reg,x)
    diff=float(np.max(np.abs(bline+contributions.sum(axis=1)-direct)))
    ensure(diff<CFG['ADVANCED_SHAP_TOLERANCE'],'SHAP_ENSEMBLE_ADDITIVITY_FAILED')
    ADVANCED_SHAP_QA_ROWS.append({'study_id':sid,'target':target,'horizon':h,'member':'WEIGHTED_ENSEMBLE',
        'max_additivity_abs_error':diff,'n_explained':len(x),'status':'PASS','mode':'WEIGHTED_MEMBER_SUM'})
    for j,feature in enumerate(cols):
        rho=np.nan
        if x[feature].nunique(dropna=True)>5:
            rho=float(pd.Series(x[feature]).corr(pd.Series(contributions[:,j]),method='spearman'))
        fam_rows.append({'study_id':sid,'target':target,'horizon':h,'method':'SHAP_ENSEMBLE',
            'model':'SUPERLEARNER','feature':feature,'group':_adv_feature_group(feature),
            'mean_abs_shap':float(np.abs(contributions[:,j]).mean()),
            'mean_signed_shap':float(contributions[:,j].mean()),'shap_value_spearman_r':rho,
            'ensemble_weight':1.,'n_explained':len(x),'mode':'WEIGHTED_MEMBER_SUM',
            'scope':'PREDICTIVE_NOT_CAUSAL'})
        for i in range(len(x)):
            if abs(contributions[i,j])>1e-9:
                ADVANCED_SHAP_LOCAL_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                    'entity_id':meta.entity_id.iloc[i],'target_year':int(meta.target_year.iloc[i]),
                    'region_name':meta.region_name.iloc[i],'feature':feature,
                    'feature_value':float(x[feature].iloc[i]) if pd.notna(x[feature].iloc[i]) else np.nan,
                    'shap_value':float(contributions[i,j]),'baseline':float(bline[i]),
                    'prediction':float(direct[i]),'method':'WEIGHTED_MEMBER_SUM'})
    ADVANCED_SHAP_ROWS.extend(fam_rows)
    # Region-clustered confidence intervals for mean |SHAP|; seeds never treated as independent units.
    rng=np.random.default_rng(CFG['SEED']+h+19)
    region=meta.region_name.fillna('UNKNOWN').astype(str).to_numpy()
    ids=[np.flatnonzero(region==s) for s in np.unique(region)]
    if len(ids)>=CFG['MIN_BOOTSTRAP_REGIONS']:
        boot=np.empty((CFG['ADVANCED_EXPLAIN_BOOTSTRAPS'],len(cols)))
        for j in range(len(boot)):
            draw=rng.integers(0,len(ids),len(ids));part=np.concatenate([ids[k] for k in draw]);boot[j]=np.mean(np.abs(contributions[part]),axis=0)
        center=np.abs(contributions).mean(axis=0);top_k=min(5,len(center));global_top=set(np.argsort(center)[-top_k:].tolist())
        for j,feature in enumerate(cols):
            probability=float(np.mean([j in np.argsort(boot[b])[-top_k:] for b in range(len(boot))]))
            ADVANCED_SHAP_STABILITY_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                'feature':feature,'top_k':top_k,'bootstrap_in_top_k_share':probability,
                'base_rank':int(len(center)-np.argsort(center).tolist().index(j)),
                'n_region_resamples':len(boot),'n_regions':len(ids),'scope':'SHAP_RANK_STABILITY_NONCAUSAL'})
        for j,feature in enumerate(cols):
            ADVANCED_SHAP_CI_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':feature,
                'mean_abs_shap':float(np.abs(contributions[:,j]).mean()),
                'lo95':float(np.quantile(boot[:,j],0.025)),'hi95':float(np.quantile(boot[:,j],0.975)),
                'n_regions':len(ids),'n_bootstrap':len(boot),'unit':'REGION'})
    # Model-native interactions. XGBoost returns true pairwise SHAP interactions;
    # CatBoost's Interaction metric is a different, structural split score (marked explicitly).
    for member in reg['members']:
        try:
            if member['family']=='XGBOOST':
                import xgboost
                matrix=xgboost.DMatrix(x.iloc[:min(45,len(x))],feature_names=cols)
                shap_pairs=np.asarray(member['model'].get_booster().predict(matrix,pred_interactions=True),float)
                ensure(shap_pairs.shape[1:]==(len(cols)+1,len(cols)+1),'XGB_INTERACTION_SHAPE')
                for jj in range(len(cols)):
                    for kk in range(jj+1,len(cols)):
                        strength=float(2*np.mean(np.abs(shap_pairs[:,jj,kk])))
                        if strength>0:
                            ADVANCED_SHAP_INTERACTION_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                                'family':'XGBOOST','feature_a':cols[jj],'feature_b':cols[kk],
                                'interaction_strength':strength,'definition':'TWO_TIMES_MEAN_ABS_PAIRWISE_SHAP',
                                'n_sample':len(shap_pairs),'predictive_only':True})
            elif member['family']=='CATBOOST':
                rank=np.asarray(member['model'].get_feature_importance(type='Interaction'),float)
                if rank.ndim==2 and rank.shape[1]>=3:
                    for pair in rank[:100]:
                        jj,kk=int(pair[0]),int(pair[1])
                        if 0<=jj<len(cols) and 0<=kk<len(cols) and jj!=kk:
                            ADVANCED_SHAP_INTERACTION_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                                'family':'CATBOOST','feature_a':cols[jj],'feature_b':cols[kk],
                                'interaction_strength':float(pair[2]),
                                'definition':'CATBOOST_NATIVE_SPLIT_INTERACTION_NOT_SHAP',
                                'n_sample':len(x),'predictive_only':True})
        except Exception as ex:log('Interaction audit '+sid+'/'+member['family']+': '+type(ex).__name__,'WARN')
    # Grouped conditional permutation on held-out TEST; no re-training and no test selection.
    n=min(len(te),CFG['ADVANCED_PERM_ROWS']);take=np.sort(np.random.default_rng(CFG['SEED']+37).choice(len(te),n,replace=False))
    sample=te.iloc[take].reset_index(drop=True)
    _,mx,_,_=design(tr,sample,reg['features'],reg['members'][0]['refs'])
    before=np.abs(sample[target].to_numpy(float)-_adv_predict(reg,mx))
    groups={g:[f for f in mx.columns if _adv_feature_group(f)==g] for g in
            ('G1','G2','G3','G4','G5','G1R_REGIONAL_NOT_MUNICIPAL','G0_DEMOGRAPHIC_HISTORY')}
    for g,ff in groups.items():
        if not ff:continue
        altered=[];effective_counts=[]
        for rep in range(CFG['ADVANCED_PERM_REPEATS']):
            perm=_within_strata_permute(mx,sample,ff,CFG['SEED']+h*31+rep)
            changed=int((perm[ff].fillna(-999998)!=mx[ff].fillna(-999998)).any(axis=1).sum())
            effective_counts.append(changed)
            if changed:altered.append(np.abs(sample[target].to_numpy(float)-_adv_predict(reg,perm)))
        if not altered:
            ADVANCED_GROUP_PERM_ROWS.append({'study_id':sid,'target':target,'horizon':h,'group':g,
                'status':'NOT_FEASIBLE','reason':'NO_EXCHANGEABLE_REGION_YEAR_STRATA','n_permuted':0});continue
        pert=np.mean(np.column_stack(altered),axis=1)
        z=pd.DataFrame({'region_name':sample.region_name,'target_year':sample.target_year,
                        'perturbed_error':pert,'base_error':before})
        delta=float(np.mean(pert-before))
        stat=block_delta(z,'perturbed_error','base_error',CFG['ADVANCED_EXPLAIN_BOOTSTRAPS'],CFG['SEED']+h)
        ADVANCED_GROUP_PERM_ROWS.append({'study_id':sid,'target':target,'horizon':h,'group':g,
            'n_test_sample':len(sample),'n_permuted':int(np.mean(effective_counts)),
            'n_features':len(ff),'delta_mae':delta,'ci_low':stat['ci_low'] if stat else np.nan,
            'ci_high':stat['ci_high'] if stat else np.nan,'n_regions':stat['n_regions'] if stat else 0,
            'repetitions':len(altered),'status':'PASS','method':'JOINT_REGION_YEAR_STRATIFIED_PERMUTATION',
            'interpretation':'Predictive reliance, not causal importance'})
    # PDP/ICE on training domain only to minimize post-test interpretation bias.
    strongest=np.argsort(np.abs(contributions).mean(axis=0))[::-1]
    train_sample=tr.sample(n=min(CFG['ADVANCED_PDP_TRAIN_ROWS'],len(tr)),random_state=CFG['SEED'])
    _,xt,_,_=design(tr,train_sample,reg['features'],reg['members'][0]['refs'])
    for j in strongest[:CFG['ADVANCED_PDP_TOP']]:
        feature=cols[j]
        if feature.startswith('region_') or not xt[feature].notna().any() or xt[feature].nunique()<8:continue
        knots=np.unique(np.quantile(xt[feature].dropna().to_numpy(float),np.linspace(.05,.95,CFG['ADVANCED_PDP_BINS'])))
        chosen=np.sort(np.random.default_rng(CFG['SEED']).choice(len(xt),min(12,len(xt)),replace=False))
        for k,v in enumerate(knots):
            xx=xt.copy();xx[feature]=float(v);yp=_adv_predict(reg,xx)
            ADVANCED_PDP_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':feature,
                'bin':k,'feature_value_transformed':float(v),'mean_prediction':float(yp.mean()),
                'n_train_background':len(xt),'method':'TRAIN_BACKGROUND_PDP_MARGINAL_NOT_CAUSAL'})
            for ix in chosen:
                ADVANCED_ICE_ROWS.append({'study_id':sid,'target':target,'horizon':h,'feature':feature,
                    'train_sample_index':int(ix),'feature_value_transformed':float(v),
                    'prediction':float(yp[ix]),'method':'ICE_TRAIN_BACKGROUND_NONCAUSAL'})
    return len(x)

progress('STATS',1,1,'Статистика и факторные вклады рассчитаны')
if CFG['ADVANCED_EXPLAIN_ENABLE']:
    keys=list(ADVANCED_ENSEMBLE_REGISTRY)
    if CFG['MODE']=='SMOKE':keys=keys[:2]
    else:keys=[k for k in keys if k[0] in ('MAX_AVAILABLE','CANONICAL') or k[0].startswith('AUX_')]
    progress('SHAP',0,len(keys),'SHAP, permutation, PDP/ICE')
    _shap_done=0
    for key in keys:
        try:_adv_explain_registry(ADVANCED_ENSEMBLE_REGISTRY[key])
        except Exception as ex:
            ADVANCED_SHAP_QA_ROWS.append({'study_id':key[0],'target':key[1],'horizon':key[2],
                'member':'ENSEMBLE_OVERALL','status':'FAIL','reason':type(ex).__name__+': '+str(ex)[:200]})
            log('ADV EXPLANATION '+str(key)+': '+type(ex).__name__+': '+str(ex)[:200],'ERROR')
        _shap_done+=1
        progress('SHAP',_shap_done,len(keys),f'Моделей объяснено: {_shap_done}/{len(keys)}')
    progress('SHAP',1,1,'Объяснения сохранены')
if not CFG['ADVANCED_EXPLAIN_ENABLE']:progress('SHAP',1,1,'Объяснения отключены настройкой')
ADVANCED_SHAP=pd.DataFrame(ADVANCED_SHAP_ROWS);ADVANCED_SHAP_LOCAL=pd.DataFrame(ADVANCED_SHAP_LOCAL_ROWS)
ADVANCED_SHAP_QA=pd.DataFrame(ADVANCED_SHAP_QA_ROWS);ADVANCED_GROUP_PERM=pd.DataFrame(ADVANCED_GROUP_PERM_ROWS)
ADVANCED_PDP=pd.DataFrame(ADVANCED_PDP_ROWS);ADVANCED_ICE=pd.DataFrame(ADVANCED_ICE_ROWS)
ADVANCED_SHAP_CI=pd.DataFrame(ADVANCED_SHAP_CI_ROWS)
ADVANCED_SHAP_STABILITY=pd.DataFrame(ADVANCED_SHAP_STABILITY_ROWS)
ADVANCED_SHAP_INTERACTIONS=pd.DataFrame(ADVANCED_SHAP_INTERACTION_ROWS)
# V4.02: independent, paired test against the strongest SINGLE family selected on DEV.
# Only predictions with the same municipality, target year and task enter inference.
ADVANCED_SINGLE_DUEL_ROWS=[]
if len(PREDICTIONS) and len(ADVANCED_ENSEMBLES):
    for row in ADVANCED_ENSEMBLES.loc[ADVANCED_ENSEMBLES.status.eq('PASS')].itertuples():
        family=getattr(row,'best_single_family_dev_locked',None)
        if not family:continue
        subset=PREDICTIONS.loc[
          PREDICTIONS.study_id.eq(row.study_id)&PREDICTIONS.target.eq(row.target)&
          PREDICTIONS.horizon.eq(row.horizon)&PREDICTIONS.split_id.eq(row.protocol)&
          PREDICTIONS.model.isin(['SUPERLEARNER','SL_'+family])]
        if subset.empty:continue
        z=subset.pivot_table(index=['entity_id','origin_year','target_year','region_name'],
              columns='model',values='absolute_error',aggfunc='first').reset_index()
        single='SL_'+family
        if single not in z.columns or 'SUPERLEARNER' not in z.columns:continue
        stat=block_delta(z,single,'SUPERLEARNER',CFG['BOOTSTRAPS'],
                         CFG['SEED']+int(row.horizon)+sum(ord(c) for c in str(row.study_id)))
        if stat:ADVANCED_SINGLE_DUEL_ROWS.append({
            'study_id':row.study_id,'target':row.target,'horizon':int(row.horizon),
            'protocol':row.protocol,'best_single_family_dev_locked':family,
            'comparison':'DEV_BEST_SINGLE → SUPERLEARNER','confirmatory':False,
            'model_selected_without_test_labels':True,**stat})
ADVANCED_SINGLE_DUEL=pd.DataFrame(ADVANCED_SINGLE_DUEL_ROWS)
if len(ADVANCED_SINGLE_DUEL):
    ADVANCED_SINGLE_DUEL['q_fdr_bh']=np.nan
    for _,inds in ADVANCED_SINGLE_DUEL.groupby(['target','horizon','protocol']).groups.items():
        ADVANCED_SINGLE_DUEL.loc[list(inds),'q_fdr_bh']=bh_adjust(
             ADVANCED_SINGLE_DUEL.loc[list(inds),'p_cluster_signflip'])
mark('ADVANCED_VS_BEST_SINGLE','PASS' if len(ADVANCED_SINGLE_DUEL)==int(ADVANCED_ENSEMBLES.status.eq('PASS').sum()) else 'PARTIAL',
     f'paired={len(ADVANCED_SINGLE_DUEL)}/{int(ADVANCED_ENSEMBLES.status.eq("PASS").sum())}; regional bootstrap; DEV locked')

# Independently test SuperLearner vs historical M0 on identical keys/regions; exploratory only.
if len(PREDICTIONS):
    subset=PREDICTIONS[PREDICTIONS.model.isin(['HISTORICAL','SUPERLEARNER'])]
    for (sid,targ,h,proto),grp in subset.groupby(['study_id','target','horizon','split_id']):
        z=grp.pivot_table(index=['entity_id','origin_year','target_year','region_name'],columns='model',
                          values='absolute_error',aggfunc='first').reset_index()
        if 'HISTORICAL' not in z or 'SUPERLEARNER' not in z:continue
        stat=block_delta(z,'HISTORICAL','SUPERLEARNER',CFG['BOOTSTRAPS'],CFG['SEED']+h)
        if stat:ADVANCED_LOO_ROWS.append({'study_id':sid,'target':targ,'horizon':h,'protocol':proto,
            'comparison':'M0_HISTORICAL → SUPERLEARNER','confirmatory':False,**stat})
ADVANCED_COMPARISONS=pd.DataFrame(ADVANCED_LOO_ROWS)
if len(ADVANCED_COMPARISONS):
    ADVANCED_COMPARISONS['q_fdr_bh']=np.nan
    for _,inds in ADVANCED_COMPARISONS.groupby(['target','horizon','protocol']).groups.items():
        ADVANCED_COMPARISONS.loc[list(inds),'q_fdr_bh']=bh_adjust(ADVANCED_COMPARISONS.loc[list(inds),'p_cluster_signflip'])
    ADVANCED_COMPARISONS['significant_improvement']=(ADVANCED_COMPARISONS.delta_mae.gt(0)&
        ADVANCED_COMPARISONS.ci_low.gt(0)&ADVANCED_COMPARISONS.q_fdr_bh.lt(0.05))
mark('ADVANCED_SHAP_QA','PASS' if len(ADVANCED_SHAP_QA) and ADVANCED_SHAP_QA.status.eq('PASS').all() else 'PARTIAL',
     f'additive explainers={int(ADVANCED_SHAP_QA.status.eq("PASS").sum()) if len(ADVANCED_SHAP_QA) else 0}; '
     f'failures={int(ADVANCED_SHAP_QA.status.eq("FAIL").sum()) if len(ADVANCED_SHAP_QA) else 0}; '
     f'region-block CI={len(ADVANCED_SHAP_CI)}; interactions={len(ADVANCED_SHAP_INTERACTIONS)}; perm={len(ADVANCED_GROUP_PERM)}')
quiet_print('Advanced explanations: SHAP global',len(ADVANCED_SHAP),'local',len(ADVANCED_SHAP_LOCAL),
      'perm group',len(ADVANCED_GROUP_PERM),'PDP',len(ADVANCED_PDP),'ICE',len(ADVANCED_ICE))

#### Устойчивость по датам публикаций, лагам, пересчёт признаков и canary против будущей утечки.

In [ ]:
# 10 · V3.02 · REPLAY-CONSISTENT PUBLICATION LAG / PROXY SENSITIVITY / FUTURE CANARY

# Functions must exist BEFORE the sensitivity loop; previous V3.00 swallowed NameError.
def shifted_predictors(frame,year_lag,features,panel_override=None):
    """Rebuild the exact factor availability transform used in cell 06.

    Origin t uses the factor value at t-publication_lag. A missing value at that
    source year is carried forward from the latest *past* observation only if
    its age <= MAX_FEATURE_STALENESS_YEARS. Future data are never backfilled.
    Unlike the former direct-year join, this mirrors cell 06's ffill+age guard.
    """
    source=PANEL if panel_override is None else panel_override
    ff=list(dict.fromkeys(features))
    if not ff:return frame.copy()
    ensure(set(ff).issubset(source.columns),'CANARY_FEATURE_NOT_IN_SOURCE_PANEL')
    ensure(not source.duplicated(['entity_id','observation_year']).any(),'CANARY_SOURCE_DUPLICATE_KEYS')
    known=source[['entity_id','observation_year']+ff].sort_values(['entity_id','observation_year']).copy()
    max_age=int(CFG['MAX_FEATURE_STALENESS_YEARS'])
    for f in ff:
        observed=pd.to_numeric(known[f],errors='coerce')
        source_year=known.observation_year.where(observed.notna())
        last=source_year.groupby(known.entity_id,sort=False).ffill()
        filled=observed.groupby(known.entity_id,sort=False).ffill()
        age=known.observation_year-last
        known[f]=filled.where(last.notna()&age.le(max_age))
    known=known.rename(columns={'observation_year':'available_year'})
    result=frame.drop(columns=ff,errors='ignore').copy()
    result['available_year']=result.origin_year.astype(int)-int(year_lag)
    result=result.merge(known,on=['entity_id','available_year'],how='left',validate='many_to_one',sort=False)
    ensure(len(result)==len(frame),'CANARY_JOIN_CARDINALITY_CHANGED')
    return result.drop(columns='available_year')

def _temporal_replay_unit_test():
    """Small real-pandas contract: bounded carry, future isolation, past sensitivity."""
    yrs=[2014,2018,2019,2020,2021]
    toy=pd.DataFrame({'entity_id':['TEST_A']*len(yrs),'observation_year':yrs,
                      'G2_TEST':[1.,10.,np.nan,333.,444.]})
    frame=pd.DataFrame({'entity_id':['TEST_A'],'origin_year':[2020],'G2_TEST':[10.]})
    f=['G2_TEST'];replay=shifted_predictors(frame,1,f,toy)
    ensure(float(replay.G2_TEST.iloc[0])==10.,'UNIT_CANARY_CARRY_FORWARD_MISSING')
    future=toy.copy();future.loc[future.observation_year.ge(2020),'G2_TEST']=1e12
    after=shifted_predictors(frame,1,f,future)
    ensure(float(after.G2_TEST.iloc[0])==10.,'UNIT_CANARY_FUTURE_MUTATION_LEAK')
    past=toy.copy();past.loc[past.observation_year.eq(2018),'G2_TEST']=12.
    changed=shifted_predictors(frame,1,f,past)
    ensure(float(changed.G2_TEST.iloc[0])==12.,'UNIT_CANARY_PAST_MUTATION_NOT_OBSERVED')
    stale=toy.copy();stale.loc[stale.observation_year.eq(2018),'G2_TEST']=np.nan
    ensure(pd.isna(shifted_predictors(frame,1,f,stale).G2_TEST.iloc[0]),'UNIT_CANARY_STALE_VALUE_NOT_EXCLUDED')
    mark('UNIT_TEMPORAL_REPLAY','PASS','Bounded ffill, lag, future invariance, past mutation and stale guard')
_temporal_replay_unit_test()

def rebuild_historical_g0(sample,source):
    """Reconstruct demographic lags from source observations, never from target labels."""
    out=sample.copy();lag=CFG['FEATURE_PUBLICATION_LAG']
    history=source[['entity_id','observation_year','population','migration_per_1000']].copy()
    for k in range(lag,lag+4):
        p='hist_population_l'+str(k);m='hist_migration_l'+str(k)
        values=history.rename(columns={'observation_year':'lookback_year','population':p,
                                        'migration_per_1000':m})
        out=out.drop(columns=[p,m],errors='ignore')
        out['lookback_year']=out.origin_year-k
        out=out.merge(values,on=['entity_id','lookback_year'],how='left',validate='many_to_one')
        out=out.drop(columns='lookback_year')
    out['hist_growth_lagged_pct']=100*(out['hist_population_l'+str(lag)]/
                                         out['hist_population_l'+str(lag+2)]-1)
    out['hist_growth_lagged_pct']=out.hist_growth_lagged_pct.replace([np.inf,-np.inf],np.nan)
    ensure(len(out)==len(sample),'G0_HISTORY_REBUILD_CHANGED_ROWS')
    return out



for (sid,target,h),r in MODEL_REGISTRY.items():
    if sid!='MAX_AVAILABLE' or not AVAILABLE_GROUPS:continue
    features=list(dict.fromkeys(sum((GROUP_FEATURES[g] for g in AVAILABLE_GROUPS),[])))
    try:
        alt_train=shifted_predictors(r['train'],CFG['FEATURE_PUBLICATION_LAG']+1,features)
        alt_test=shifted_predictors(r['test'],CFG['FEATURE_PUBLICATION_LAG']+1,features)
        pred,_,_,_=fit_predict(r['family'],alt_train,alt_test,target,r['features'],CFG['SEED'],r['params'])
        alt=metrics(r['test'][target],pred,r['test'][f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
        original=metrics(r['test'][target],r['pred'],r['test'][f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
        SENSITIVITY_ROWS.append({'study_id':sid,'target':target,'horizon':h,'test_keys':len(r['test']),
           'check':'publication_lag_plus_1_year','original_mae':original['mae'],
           'alternative_mae':alt['mae'],'delta_mae':original['mae']-alt['mae'],
           'interpretation':'different information timing; test keys held fixed'})
        # Remove correlated proxies, preferentially retaining more complete training measurements.
        ff=[f for f in r['features'] if f.startswith(('G1_','G2_','G3_','G4_','G5_'))]
        tr=r['train'];removed=[];retained=[]
        for f in sorted(ff,key=lambda c:tr[c].notna().sum(),reverse=True):
            redundant=False
            for stay in retained:
                sample=tr[[stay,f]].dropna()
                if len(sample)>50 and abs(sample[stay].corr(sample[f],method='spearman'))>=CFG['CORRELATED_PROXY_CUTOFF']:
                    redundant=True;break
            if redundant:removed.append(f)
            else:retained.append(f)
        if removed:
            feats=[f for f in r['features'] if f not in removed]
            pred,_,_,_=fit_predict(r['family'],tr,r['test'],target,feats,CFG['SEED'],r['params'])
            teststat=metrics(r['test'][target],pred,r['test'][f'hist_population_l{CFG["FEATURE_PUBLICATION_LAG"]}'])
            SENSITIVITY_ROWS.append({'study_id':sid,'target':target,'horizon':h,
               'test_keys':len(r['test']),'check':'remove_correlated_proxies_on_train',
               'original_mae':original['mae'],'alternative_mae':teststat['mae'],
               'delta_mae':original['mae']-teststat['mae'],
               'excluded_features':','.join(removed),'interpretation':'exploratory; no causal attribution'})
    except Exception as ex:
        mark('SENSITIVITY_'+sid+'_'+target+'_h'+str(h),'FAIL',type(ex).__name__+': '+str(ex)[:180])
SENSITIVITY=pd.DataFrame(SENSITIVITY_ROWS)

# Two independent canary contracts:
# (1) SOURCE_REPLAY reproduces *exactly* the model's actual feature matrix.
# (2) FUTURE_INVARIANCE changes all source rows t >= issue origin and reruns
#     the SAME transformation. A breach is genuine only after replay parity.
CANARY_RECORDS=[]

def _canary_model_matrix(registry,frame):
    return design(registry['train'],frame,registry['features'],registry['refs'])[1]

def _canary_comparison(before,after,rtol=0.,atol=1e-10):
    same_shape=(list(before.columns)==list(after.columns) and before.shape==after.shape)
    if not same_shape:
        return False,['SHAPE_OR_COLUMN_DIFFERENCE'],float('inf')
    lhs=before.to_numpy(dtype=float);rhs=after.to_numpy(dtype=float)
    matches=np.isclose(lhs,rhs,equal_nan=True,rtol=rtol,atol=atol)
    changed=[str(c) for c in before.columns[np.any(~matches,axis=0)].tolist()]
    different=np.abs(lhs-rhs);good=np.isfinite(different)
    max_delta=float(np.max(different[good])) if np.any(good) else 0.
    return bool(matches.all()),changed,max_delta

for (sid,target,h),r in MODEL_REGISTRY.items():
    if sid not in ('MAX_AVAILABLE','CANONICAL','AUX_G1_REGION_CONTEXT'):continue
    covariates=[f for f in dict.fromkeys(r['features']) if f in PANEL.columns]
    sample=r['test'].drop_duplicates('entity_id').head(CFG['MAX_FUTURE_MUTATION_ROWS']).copy()
    if sample.empty:continue
    record={'study_id':sid,'target':target,'horizon':h,'n_test_rows':len(sample),
            'n_model_source_factors':len(covariates),'status':'NOT_RUN'}
    try:
        train=r['train'];refs=r['refs']
        x_original=_canary_model_matrix(r,sample)
        # Fidelity contract: replay without modifying a single source value.
        replay=shifted_predictors(sample,CFG['FEATURE_PUBLICATION_LAG'],covariates)
        replay=rebuild_historical_g0(replay,PANEL)
        x_replay=_canary_model_matrix(r,replay)
        faithful,replay_cols,replay_delta=_canary_comparison(x_original,x_replay)
        record.update(source_replay_exact=faithful,
          replay_mismatch_columns_json=json.dumps(replay_cols,ensure_ascii=False),
          replay_max_finite_delta=replay_delta)
        if not faithful:
            record['status']='FAIL_SOURCE_REPLAY_NOT_IDENTICAL'
            raise AssertionError('CANARY_SOURCE_REPLAY_MISMATCH: '+','.join(replay_cols[:8]))
        y_original=np.asarray(r['model'].predict(x_original),float)
        y_replay=np.asarray(r['model'].predict(x_replay),float)
        ensure(np.allclose(y_original,y_replay,rtol=0,atol=1e-9,equal_nan=True),
               'CANARY_UNMUTATED_MODEL_REPLAY_DIFFERENCE')
        mutated=PANEL.copy(deep=True)
        origins=dict(zip(sample.entity_id,sample.origin_year))
        at_or_after=mutated.observation_year.ge(mutated.entity_id.map(origins))
        mask=mutated.entity_id.isin(origins)&at_or_after.fillna(False)
        mutated_source_rows=int(mask.sum())
        ensure(mutated_source_rows>0,'CANARY_HAS_NO_FUTURE_SOURCE_ROWS')
        for f in ('population','migration_per_1000'):
            if f in mutated.columns:mutated.loc[mask,f]=1e12
        for f in covariates:mutated.loc[mask,f]=1e12
        after=shifted_predictors(sample,CFG['FEATURE_PUBLICATION_LAG'],covariates,mutated)
        after=rebuild_historical_g0(after,mutated)
        x_after=_canary_model_matrix(r,after)
        invariant_x,changed_columns,changed_max=_canary_comparison(x_replay,x_after)
        y_after=np.asarray(r['model'].predict(x_after),float)
        invariant_y=bool(np.allclose(y_replay,y_after,rtol=0,atol=1e-9,equal_nan=True))
        record.update(n_future_source_rows_mutated=mutated_source_rows,
          n_model_features=len(x_original.columns),invariant_features=invariant_x,
          invariant_predictions=invariant_y,
          mutated_mismatch_columns_json=json.dumps(changed_columns,ensure_ascii=False),
          mutated_max_finite_delta=changed_max,
          status='PASS' if invariant_x and invariant_y else 'FAIL_FUTURE_DEPENDENCE')
        if not invariant_x or not invariant_y:
            raise AssertionError('FUTURE_MUTATION_CHANGES_FINAL_FORECAST: '+','.join(changed_columns[:8]))
    except Exception as ex:
        if record['status']=='NOT_RUN':record['status']='FAIL_CANARY_RUNTIME'
        record['exception_type']=type(ex).__name__;record['reason']=str(ex)[:400]
        CANARY_RECORDS.append(record)
        CANARY_QA=pd.DataFrame(CANARY_RECORDS)
        (WORK/'ml03_canary_failure.json').write_text(json.dumps(CANARY_RECORDS,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
        mark('TEMPORAL_CANARY','FAIL',record['status']+' '+record['reason'])
        raise
    CANARY_RECORDS.append(record)
CANARY_QA=pd.DataFrame(CANARY_RECORDS)
mark('TEMPORAL_CANARY','PASS' if len(CANARY_QA) and CANARY_QA.status.eq('PASS').all() else 'NOT_FEASIBLE',
     f'Source replay + future mutation; passing models={int(CANARY_QA.status.eq("PASS").sum()) if len(CANARY_QA) else 0}/{len(CANARY_QA)}')

# Same fitted models, objectively restricted boundary and municipality-size subsets.
BOUNDARY_ROWS=[];SIZE_ROWS=[]
for (sid,target,h),r in MODEL_REGISTRY.items():
    if sid not in ('MAX_AVAILABLE','CANONICAL'):continue
    test=r['test'];y_true=test[target].to_numpy(float)
    err=np.abs(y_true-np.asarray(r['pred'],float))
    if 'boundary_period_uncertain' in test:
        safe=~test.boundary_period_uncertain.fillna(True).astype(bool)
        verified=test.loc[safe]
        if safe.sum()>=CFG['MIN_TEST_ROWS'] and verified.region_name.nunique()>=CFG['MIN_BOUNDARY_SENSITIVITY_REGIONS']:
            BOUNDARY_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                'n_test_all':len(test),'n_boundary_verified':int(safe.sum()),
                'n_regions_verified':int(verified.region_name.nunique()),
                'mae_all':float(err.mean()),'mae_verified_boundary':float(err[safe.to_numpy()].mean()),
                'status':'EXPLORATORY_FIXED_MODEL_STRATUM_NONCAUSAL'})
    pop='hist_population_l'+str(CFG['FEATURE_PUBLICATION_LAG'])
    train_pop=pd.to_numeric(r['train'][pop],errors='coerce')
    train_pop=train_pop[train_pop.gt(0)].dropna()
    thresholds=np.quantile(train_pop,[1/3,2/3]) if len(train_pop)>100 else None
    if thresholds is not None and thresholds[0]<thresholds[1]:
        values=pd.to_numeric(test[pop],errors='coerce').to_numpy(float)
        bins=np.digitize(values,thresholds)
        for group in range(3):
            flag=np.isfinite(values)&(bins==group)
            if flag.sum()>=15:
                SIZE_ROWS.append({'study_id':sid,'target':target,'horizon':h,
                    'size_group':('small','medium','large')[group],
                    'n_test':int(flag.sum()),'mae':float(err[flag].mean()),
                    'train_size_thresholds_json':json.dumps(thresholds.tolist()),
                    'scope':'train-defined lagged-population strata; prediction not causal'})
BOUNDARY_SENSITIVITY=pd.DataFrame(BOUNDARY_ROWS)
SIZE_SENSITIVITY=pd.DataFrame(SIZE_ROWS)
mark('BOUNDARY_SENSITIVITY','PASS' if len(BOUNDARY_SENSITIVITY) else 'NOT_FEASIBLE',
    f'Boundary-confirmed test strata={len(BOUNDARY_SENSITIVITY)}; undocumented does not imply stable')
mark('SIZE_HETEROGENEITY','PASS' if len(SIZE_SENSITIVITY) else 'NOT_FEASIBLE',
    f'Train-derived municipality-size strata={len(SIZE_SENSITIVITY)}')
# H3.5 can only be marked tested when publication lag AND correlated-proxy comparisons are present.
lag_exists=(len(SENSITIVITY)>0 and SENSITIVITY.check.eq('publication_lag_plus_1_year').any())
proxy_exists=(len(SENSITIVITY)>0 and SENSITIVITY.check.eq('remove_correlated_proxies_on_train').any())
mark('H3.5','PASS' if lag_exists and proxy_exists else 'NOT_FEASIBLE',
     f'lag sensitivity={lag_exists}; correlated-proxy sensitivity={proxy_exists}')
# Scientific interpretation is predictive only; lack of causal data is not, by itself, a fatal gate.
if len(ABLA) and len(SENSITIVITY):
    mark('PASS_INTERPRETATION','PASS',
       f'factor tests={len(ABLA)}; improved={int(ABLA.finding.eq("PREDICTIVE_IMPROVEMENT").sum())}; uncertainty and negative findings retained')
else:
    mark('PASS_INTERPRETATION','NOT_FEASIBLE','No paired factor tests or inadequate sensitivity checks')
quiet_print('Independent factor comparisons:',len(ABLA),'| ALE rows:',len(ALE),
      '| sensitivity:',len(SENSITIVITY),'| future-mutation tests:',len(CANARY_QA))

CELL10_COMPLETE=True
mark('CELL10_PIPELINE','PASS','Sensitivity, temporal canary, boundary and size tables constructed')


# V4 · ADVANCED temporal anti-leakage replay for the actual fitted SuperLearner.
# Validates source replay, model matrix and future-mutation invariance, fails closed.
ADVANCED_TEMPORAL_CANARY_ROWS=[]
for (sid,target,h),r in ADVANCED_ENSEMBLE_REGISTRY.items():
    if sid not in ('MAX_AVAILABLE','CANONICAL','AUX_G1_REGION_CONTEXT'):continue
    source_cols=[f for f in dict.fromkeys(r['features']) if f in PANEL.columns]
    small=r['test'].drop_duplicates('entity_id').head(CFG['MAX_FUTURE_MUTATION_ROWS']).copy()
    if small.empty:continue
    rec={'study_id':sid,'target':target,'horizon':h,'n_test':len(small),'n_source_features':len(source_cols)}
    try:
        orig=design(r['train'],small,r['features'],r['members'][0]['refs'])[1]
        replay=shifted_predictors(small,CFG['FEATURE_PUBLICATION_LAG'],source_cols)
        replay=rebuild_historical_g0(replay,PANEL)
        rebuilt=design(r['train'],replay,r['features'],r['members'][0]['refs'])[1]
        faithful,missing,delta=_canary_comparison(orig,rebuilt)
        ensure(faithful,'ADVANCED_SOURCE_REPLAY_MISMATCH: '+','.join(missing[:8]))
        mutated=PANEL.copy(deep=True)
        first=dict(zip(small.entity_id,small.origin_year))
        mask=mutated.entity_id.isin(first)&mutated.observation_year.ge(mutated.entity_id.map(first)).fillna(False)
        ensure(mask.sum()>0,'ADVANCED_FUTURE_MUTATION_NO_ROWS')
        for f in ['population','migration_per_1000']+source_cols:mutated.loc[mask,f]=1e12
        altered=shifted_predictors(small,CFG['FEATURE_PUBLICATION_LAG'],source_cols,mutated)
        altered=rebuild_historical_g0(altered,mutated)
        after=design(r['train'],altered,r['features'],r['members'][0]['refs'])[1]
        stable,changed,change=_canary_comparison(rebuilt,after)
        y0=_adv_predict(r,rebuilt);y1=_adv_predict(r,after)
        ok=bool(stable and np.allclose(y0,y1,rtol=0,atol=1e-9))
        rec.update({'n_mutated_source_rows':int(mask.sum()),'source_replay_exact':True,
            'prediction_invariant':ok,'matrix_invariant':stable,
            'max_predictive_diff':float(np.abs(y0-y1).max()),'status':'PASS' if ok else 'FAIL'})
        ensure(ok,'ADVANCED_ENSEMBLE_FUTURE_DEPENDENCE')
    except Exception as ex:
        rec.update({'status':'FAIL','exception':type(ex).__name__+': '+str(ex)[:250]})
        ADVANCED_TEMPORAL_CANARY_ROWS.append(rec)
        mark('ADVANCED_TEMPORAL_CANARY','FAIL',rec['exception'])
        raise
    ADVANCED_TEMPORAL_CANARY_ROWS.append(rec)
ADVANCED_TEMPORAL_CANARY=pd.DataFrame(ADVANCED_TEMPORAL_CANARY_ROWS)
mark('ADVANCED_TEMPORAL_CANARY', 'PASS' if len(ADVANCED_TEMPORAL_CANARY) and ADVANCED_TEMPORAL_CANARY.status.eq('PASS').all() else 'NOT_FEASIBLE',
     f'passing weighted-ensemble replay/future tests={len(ADVANCED_TEMPORAL_CANARY)}')

PUBLICATION_AVAILABILITY_QA=pd.DataFrame([{
  'feature':r.get('feature'),'indicator_name':r.get('indicator_name'),'group':r.get('group'),
  'documented_publication_date':r.get('report_available_year','NOT_VERIFIED'),
  'availability_provenance':r.get('availability_provenance','UNKNOWN'),
  'lag_assumption_years':CFG['FEATURE_PUBLICATION_LAG'],
  'status':'DOCUMENTED_SOURCE_DATE' if str(r.get('report_available_year','')).isdigit() and str(r.get('availability_provenance','')).startswith('OFFICIAL') else 'LAG_ASSUMPTION_ONLY'}
  for r in FEATURE_DICT.to_dict('records')])
if len(PUBLICATION_AVAILABILITY_QA):
    real=int(PUBLICATION_AVAILABILITY_QA.status.eq('DOCUMENTED_SOURCE_DATE').sum())
    mark('HISTORICAL_PUBLICATION_DATES','PASS' if real==len(PUBLICATION_AVAILABILITY_QA) else 'PARTIAL',
         f'documented={real}/{len(PUBLICATION_AVAILABILITY_QA)}; future-mutation and lag sensitivity do not prove original release dates')

progress('SENSITIVITY',1,1,'Canary и чувствительность завершены')

#### Таблицы, диаграммы, паспорта расчётов и Excel.

In [ ]:
# 11 · V3.01 · ANALYTICAL WORKBOOK + OBSERVED FIGURES · STAGE CONTRACTS
ensure(globals().get('CELL10_COMPLETE',False),
       'CELL10_NOT_COMPLETE: execute cell 10 before workbook; do not manufacture missing diagnostics')
_REQUIRED_11=('SENSITIVITY','BOUNDARY_SENSITIVITY','SIZE_SENSITIVITY','CANARY_QA','MODEL_REGISTRY','REGIONAL_CONTEXT_METADATA')
_MISSING_11=[k for k in _REQUIRED_11 if k not in globals()]
ensure(not _MISSING_11,'CELL11_INPUTS_MISSING: '+','.join(_MISSING_11))
# H3.5 tests are derived from the actual sensitivity ledger, never stale variables.
_SENS_CHECKS=set(SENSITIVITY['check'].dropna().astype(str)) if isinstance(SENSITIVITY,pd.DataFrame) and 'check' in SENSITIVITY.columns else set()
lag_exists='publication_lag_plus_1_year' in _SENS_CHECKS
proxy_exists='remove_correlated_proxies_on_train' in _SENS_CHECKS
CHARTS=[]
plt.rcParams.update({'figure.dpi':125,'savefig.dpi':160,'font.size':9,
    'axes.spines.top':False,'axes.spines.right':False})
def figure_file(name):
    path=WORK/'charts'/(name+'.png');plt.savefig(path,bbox_inches='tight',dpi=160)
    plt.close();CHARTS.append(path)
def safe_plot(label,fn):
    try:fn()
    except Exception as ex:log('PLOT '+label+': '+type(ex).__name__+' '+str(ex)[:150],'WARN');plt.close('all')

def chart_ablation():
    if ABLA.empty:return
    p=ABLA[(ABLA.protocol=='ROLLING_TIME')&(ABLA.target=='population_change_pct')].head(16)
    if p.empty:return
    plt.figure(figsize=(11,5));x=np.arange(len(p));plt.bar(x,p.delta_mae,yerr=[p.delta_mae-p.ci_low,p.ci_high-p.delta_mae],capsize=3)
    plt.axhline(0,color='black',lw=.7);plt.xticks(x,[f"h{r.horizon}:{r.comparison}" for r in p.itertuples()],rotation=45,ha='right')
    plt.ylabel('Δ MAE, п.п. (положительное = улучшение)');plt.title('Парный прирост качества: общие тестовые наблюдения и региональные интервалы')
    figure_file('01_ablation_region_CI')
safe_plot('ablation',chart_ablation)
def chart_loo():
    if LOO.empty:return
    p=LOO[(LOO.target=='population_change_pct')&(LOO.protocol=='ROLLING_TIME')].head(12)
    if p.empty:return
    plt.figure(figsize=(9,4));plt.barh(np.arange(len(p)),p.delta_mae)
    plt.yticks(np.arange(len(p)),[f'{r.horizon} г. {r.old_arm}' for r in p.itertuples()])
    plt.xlabel('Вклад исключённой группы в MAE, п.п.');plt.title('Leave-one-group-out: прогнозный вклад группы')
    figure_file('02_leave_one_group_out')
safe_plot('loo',chart_loo)
def chart_importance():
    if IMPORTANCE.empty:return
    p=IMPORTANCE.sort_values('mae_increase',ascending=False).head(15)
    plt.figure(figsize=(9,5));plt.barh(p.feature[::-1],p.mae_increase[::-1])
    plt.xlabel('Ухудшение MAE при перестановке');plt.title('Важность для прогностической функции, не причинный эффект')
    figure_file('03_permutation_importance')
safe_plot('importance',chart_importance)
def chart_ale():
    if ALE.empty:return
    p=ALE[ALE.feature.eq(ALE.feature.iloc[0])];plt.figure(figsize=(8,4))
    for h,a in p.groupby('horizon'):
        plt.plot(a.feature_mid_logscale,a.ale_centered,marker='o',label=f'{h} год(лет)')
    plt.legend();plt.ylabel('Центрированная ALE');plt.xlabel('Признак в логарифмической шкале')
    plt.title('Условная нелинейность на обучающей области наблюдений')
    figure_file('04_train_ALE')
safe_plot('ale',chart_ale)
def chart_heterogeneity():
    if REGIONAL_ERRORS.empty:return
    p=REGIONAL_ERRORS[(REGIONAL_ERRORS.protocol=='ROLLING_TIME')&(REGIONAL_ERRORS.horizon==1)]
    p=p.groupby('region',as_index=False).agg(mae=('mae','mean')).sort_values('mae').head(18)
    if p.empty:return
    plt.figure(figsize=(10,5));plt.barh(p.region,p.mae)
    plt.xlabel('MAE');plt.title('Территориальная неоднородность качества')
    figure_file('05_regional_errors')
safe_plot('regional',chart_heterogeneity)
def chart_times():
    if ERROR_YEARS.empty:return
    p=ERROR_YEARS[(ERROR_YEARS.study_id=='MAX_AVAILABLE')]
    if p.empty:p=ERROR_YEARS
    p=p.groupby(['target','horizon','year'],as_index=False).agg(mae=('mae','mean'))
    plt.figure(figsize=(9,4))
    for (target,h),x in p.groupby(['target','horizon']):plt.plot(x.year,x.mae,marker='o',label=target[:10]+' h'+str(h))
    plt.legend(fontsize=7);plt.xlabel('Год целевого наблюдения');plt.ylabel('MAE')
    plt.title('Стабильность ошибки по временным отложениям')
    figure_file('06_out_of_time_stability')
safe_plot('time',chart_times)
def chart_coverage():
    if coverage.empty:return
    plt.figure(figsize=(7,3));plt.bar(coverage.group,coverage.forecast_coverage)
    plt.ylim(0,1);plt.ylabel('Доля прогнозных наблюдений с доступными факторами')
    plt.title('Наблюдаемость блоков после публикационного лага')
    figure_file('07_factor_availability')
safe_plot('coverage',chart_coverage)
def chart_sensitivity():
    if SENSITIVITY.empty:return
    p=SENSITIVITY.groupby('check',as_index=False).agg(delta_mae=('delta_mae','mean'))
    plt.figure(figsize=(8,3));plt.barh(p.check,p.delta_mae)
    plt.axvline(0,color='black',lw=.8);plt.xlabel('Средняя разница MAE (оригинал − вариант)')
    plt.title('Проверка задержки публикации и коррелированных показателей')
    figure_file('08_sensitivity')
safe_plot('sensitivity',chart_sensitivity)
def chart_migration_gate():
    if MIGRATION_QA.empty:return
    p=MIGRATION_QA.head(8);plt.figure(figsize=(8,3))
    plt.barh([f'{r.denominator}, ×{r.numerator_scale:g}' for r in p.itertuples()][::-1],
             p.agreement_5pct.iloc[::-1]);plt.xlim(0,1)
    plt.xlabel('Доля согласованных значений');plt.title('Аудит семантики миграционного показателя D01')
    figure_file('09_migration_semantics')
safe_plot('migration',chart_migration_gate)
def chart_interactions():
    if INTERACTIONS.empty:return
    p=INTERACTIONS.head(8);plt.figure(figsize=(8,3))
    plt.bar([str(h) for h in p.horizon],p.delta_mae)
    plt.axhline(0,color='black',lw=.8);plt.ylabel('ΔMAE взаимодействия, п.п.')
    plt.title('G1 × G2: исследовательская проверка добавочной прогностической ценности')
    figure_file('10_income_housing_interaction')
safe_plot('interactions',chart_interactions)

# Advanced figures from real fitted ensembles/explanations; no decorative plots.
# Scientific visualization labels from source dictionary and transformations (never generic ID alone).
_VIS_META={str(r.model_feature):str(r.indicator_name) for r in TRANSFORM_QA.itertuples()
           if pd.notna(r.model_feature)}
_VIS_META.update({str(r.feature):str(r.indicator_name) for r in FEATURE_DICT.itertuples()})
def feature_label(feature):
    f=str(feature);label=_VIS_META.get(f)
    if not label and f.startswith('hist_'):
        lab=re.sub(r'_l(\d+)$',r' (лаг \1 г.)',f[5:]).replace('_',' ')
        label='История: '+lab
    if not label and f.startswith('G1R_'):label='Региональный контекст: '+f
    if not label:label=f.replace('_',' ')
    return str(label).strip()[:74]
def _uniquify_axis_labels(features):
    labs=[feature_label(f) for f in features]
    if len(labs)!=len(set(labs)):labs=[f'{lab} [{f}]' for lab,f in zip(labs,features)]
    return labs

def chart_adv_global():
    if ADVANCED_SHAP.empty:return
    p=ADVANCED_SHAP[(ADVANCED_SHAP.method=='SHAP_ENSEMBLE') & ADVANCED_SHAP.study_id.eq('MAX_AVAILABLE')]
    if p.empty:return
    ss=p.groupby(['feature','group'],as_index=False).mean_abs_shap.mean().sort_values('mean_abs_shap').tail(18)
    plt.figure(figsize=(11,8));plt.barh(_uniquify_axis_labels(ss.feature),ss.mean_abs_shap)
    plt.xlabel('Среднее абсолютное SHAP, единицы целевого прогноза');plt.title('SuperLearner: объяснение предсказаний на тестовых наблюдениях')
    figure_file('11_ensemble_shap_global')
safe_plot('advanced_shap_global',chart_adv_global)

def chart_adv_permutation():
    if ADVANCED_GROUP_PERM.empty:return
    p=ADVANCED_GROUP_PERM[(ADVANCED_GROUP_PERM.status=='PASS')&ADVANCED_GROUP_PERM.study_id.eq('MAX_AVAILABLE')]
    if p.empty:return
    p=p.groupby('group',as_index=False).delta_mae.mean().sort_values('delta_mae')
    plt.figure(figsize=(9,4));plt.barh(p.group,p.delta_mae);plt.axvline(0,color='black',lw=.8)
    plt.xlabel('Рост MAE после совместной перестановки блока');plt.title('Зависимость прогноза от групп факторов (не причинный эффект)')
    figure_file('12_conditional_group_permutation')
safe_plot('advanced_group_permutation',chart_adv_permutation)

def chart_adv_comparison():
    if ADVANCED_COMPARISONS.empty:return
    p=ADVANCED_COMPARISONS[(ADVANCED_COMPARISONS.protocol=='ROLLING_TIME') & ADVANCED_COMPARISONS.study_id.eq('MAX_AVAILABLE')]
    if p.empty:return
    lab=[('миграция' if 'migration' in x.target else 'население')+f' {int(x.horizon)} г.' for x in p.itertuples()]
    plt.figure(figsize=(9,4));xx=np.arange(len(p));plt.bar(xx,p.delta_mae)
    plt.axhline(0,color='black',lw=.8);plt.xticks(xx,lab,rotation=25,ha='right')
    plt.ylabel('ΔMAE (M0 − SuperLearner); положительное = улучшение');plt.title('Независимая проверка ансамбля по регионам')
    figure_file('13_superlearner_vs_historical')
safe_plot('advanced_superlearner',chart_adv_comparison)

def chart_adv_pdp():
    if ADVANCED_PDP.empty:return
    p=ADVANCED_PDP[ADVANCED_PDP.study_id.eq('MAX_AVAILABLE')]
    if p.empty:return
    row=p.groupby(['target','horizon','feature']).size().sort_values(ascending=False).index[0]
    p=p[(p.target==row[0])&(p.horizon==row[1])&(p.feature==row[2])]
    plt.figure(figsize=(8,4));plt.plot(p.feature_value_transformed,p.mean_prediction,marker='o')
    plt.xlabel(feature_label(row[2])+' (преобразованная шкала)');plt.ylabel('Прогноз модели')
    plt.title('PDP на обучающей области, не причинная зависимость')
    figure_file('14_ensemble_train_PDP')
safe_plot('advanced_pdp',chart_adv_pdp)

def chart_adv_best_single():
    if ADVANCED_SINGLE_DUEL.empty:return
    d=ADVANCED_SINGLE_DUEL[(ADVANCED_SINGLE_DUEL.protocol=='ROLLING_TIME') &
                          ADVANCED_SINGLE_DUEL.study_id.eq('CANONICAL')].copy()
    if d.empty:return
    labels=[('Миграция' if 'migration' in v.target else 'Численность')+f' · {int(v.horizon)} г.'
            for v in d.itertuples()]
    plt.figure(figsize=(10,5));x=np.arange(len(d));plt.bar(x,d.delta_mae)
    plt.errorbar(x,d.delta_mae,yerr=[d.delta_mae-d.ci_low,d.ci_high-d.delta_mae],
        fmt='none',ecolor='black',capsize=4)
    plt.axhline(0,color='black',lw=.8);plt.xticks(x,labels,rotation=20,ha='right')
    plt.ylabel('MAE одиночной − MAE ансамбля (плюс означает выигрыш ансамбля)')
    plt.title('Сравнение с лучшим одиночным методом по DEV · один тестовый набор')
    figure_file('15_superlearner_vs_best_single')
safe_plot('advanced_vs_best_single',chart_adv_best_single)

PASSPORT=pd.DataFrame([
 {'Параметр':'Исследование','Значение':'ML-03: экономика, инфраструктура и демографическая устойчивость'},
 {'Параметр':'Версия','Значение':CFG['VERSION']},{'Параметр':'Режим','Значение':CFG['MODE']},
 {'Параметр':'Источник Stage I','Значение':CFG['REPO']+'@'+source_sha},
 {'Параметр':'SHA-256 handoff','Значение':hashlib.sha256(raw_index).hexdigest()},
 {'Параметр':'Верифицированных Stage I файлов','Значение':sum(a['asset_id'] in LOCAL_ASSETS for a in ASSETS)},
 {'Параметр':'Муниципалитетов D01','Значение':int(base.entity_id.nunique())},
 {'Параметр':'Муниципалитет×год','Значение':len(PANEL)},
 {'Параметр':'Диапазон лет','Значение':str(min(YEARS))+'–'+str(max(YEARS))},
 {'Параметр':'Дополнительные БДМО Parquet','Значение':len(BDMO_EXTERNAL_ASSETS)},
 {'Параметр':'Advanced SuperLearner experiments','Значение':int(ADVANCED_ENSEMBLES.status.eq('PASS').sum()) if len(ADVANCED_ENSEMBLES) else 0},
 {'Параметр':'SHAP additivity checks','Значение':int(ADVANCED_SHAP_QA.status.eq('PASS').sum()) if len(ADVANCED_SHAP_QA) else 0},
 {'Параметр':'Факторных признаков','Значение':sum(len(x) for x in GROUP_FEATURES.values())},
 {'Параметр':'Тестовых прогнозов','Значение':len(PREDICTIONS)},
 {'Параметр':'Модельных оценок','Значение':len(METRICS)},
 {'Параметр':'Вариантов факторных исследований','Значение':len(STUDIES)},
 {'Параметр':'Миграционная цель проходит проверку','Значение':bool(valid_rate)},
 {'Параметр':'Причинная идентификация','Значение':'НЕ ПРОВЕДЕНА — нет реестра вмешательств'},
 {'Параметр':'Прогнозные признаки доступны','Значение':f'не позднее origin_year−{CFG["FEATURE_PUBLICATION_LAG"]}'},
 {'Параметр':'Разбиение','Значение':'rolling-time / territory-only / crossed; раздельное DEV и TEST'},
 {'Параметр':'Семантика оценок','Значение':'ассоциации и прогнозы; НЕ причинные эффекты'}])
HYPOTHESES=pd.DataFrame([
 {'Гипотеза':'H3.1','Критерий':'M1→M3; region-block CI, Holm; same test keys',
    'Статус':'TESTED' if len(ABLA) and ABLA.comparison.eq('M1 → M3').any() else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.2','Критерий':'G1×G2 interaction; exploratory',
    'Статус':'PRELIMINARY_PENDING_SEMANTICS' if len(INTERACTIONS) else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.3','Критерий':'G3 incremental on common support',
    'Статус':'PRELIMINARY_PENDING_HEALTH_COVERAGE' if len(ABLA) and ABLA.study_id.eq('AUX_G3').any() else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.4','Критерий':'G4/G5 availability and transfer',
    'Статус':'EXPLORATORY_PARTIAL' if len(ABLA) and ABLA.study_id.eq('AUX_G4').any() else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.5','Критерий':'publication lag + correlated-proxy sensitivity',
    'Статус':'EXPLORATORY_TESTED' if lag_exists and proxy_exists else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.6','Критерий':'health separately identified from education',
    'Статус':'EXPLORATORY_ELIGIBLE' if HEALTH_AVAILABLE else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.7','Критерий':'municipality boundary-consistency strata',
    'Статус':'EXPLORATORY_TESTED' if len(BOUNDARY_SENSITIVITY) else 'NOT_FEASIBLE'},
 {'Гипотеза':'H3.8','Критерий':'municipality size strata using train thresholds',
    'Статус':'EXPLORATORY_TESTED' if len(SIZE_SENSITIVITY) else 'NOT_FEASIBLE'}])
LIMITATIONS=pd.DataFrame([
 {'Ограничение':'Неполнота предметного поиска','Следствие':'Если сохранены неразобранные коды, SCIENTIFIC_PASS запрещен'},
 {'Ограничение':'Муниципальные границы','Следствие':'ОКТМО не означает неизменных границ; spatial history requires validated crosswalk'},
 {'Ограничение':'Отбор источников по семантике','Следствие':'Владение всеми 52 активами не означает применимость всех; широкие таблицы без проверенной единицы остаются в аудите'},
 {'Ограничение':'Региональные и организационные источники','Следствие':'Indexed and downloaded but not joined without verified grain/crosswalk'},
 {'Ограничение':'Срок публикации','Следствие':'Отсутствует конкретный available_at; применяется conservative publication lag'},
 {'Ограничение':'Миграционная цель','Следствие':'Defined only when independent numerical audit validates scale and denominator'},
 {'Ограничение':'Финансы','Следствие':'Local budget spending is not treated as a treatment assignment'},
 {'Ограничение':'Инфраструктура','Следствие':'Counts are different from access; age-cohort denominator not inferred'},
 {'Ограничение':'Денежные показатели','Следствие':'Nominal source-year amounts; no unreliable municipal deflator imputed'},
 {'Ограничение':'Персональные траектории','Следствие':'Population change is municipal stock variation, not individual retention'},
 {'Ограничение':'Коррелированные признаки','Следствие':'Permutation and ALE explain prediction, not structural/causal mechanisms'},
 {'Ограничение':'Пространственные признаки','Следствие':'Static transport/settlement crosswalk withheld if source-vintage unavailable'},
 {'Ограничение':'Множественная проверка','Следствие':'Holm for preregistered, BH FDR for exploratory; regional cluster bootstrap'},
 {'Ограничение':'Устойчивость модели','Следствие':'Stochastic seeds not counted as independent test municipalities'}])
SHEETS={
 'Паспорт':PASSPORT,'Гипотезы':HYPOTHESES,'Источники':input_map,'Загрузки_52':DOWNLOAD_QA,
 'Аудит_миграции':MIGRATION_QA,'Окна_целей_V3':TARGET_CALENDAR_PREFLIGHT,
 'Реестр_физ_52':STAGE1_PHYSICAL_AUDIT,'Манифест_экономики':ECONOMY_SOURCE_MANIFEST_QA,
 'Региональный_G1R':REGIONAL_CONTEXT_METADATA,'G1R_аудит':REGIONAL_CONTEXT_AUDIT,
 'Единицы_G1R':G1R_UNIT_QA,
 'Давность_факторов':STALENESS_QA,'Наличие_целей_по_годам':TARGET_AVAILABILITY,'Факторы':FEATURE_DICT,'Преобразования':TRANSFORM_QA,
 'Каталог_индикаторов':CATALOG,'Поиск_Г1_все':SCAN_AUDIT,'Паспорт_поиска_Г1':G1_DISCOVERY,
 'Измерения_кодов':DIMENSION_AUDIT,'Уровни_источников':pd.DataFrame(SOURCE_ADAPTER_AUDIT),
 'Паспорт_52_источников':SOURCE_GRAIN_AUDIT,'SQL_ошибки':SCAN_SQL_ERRORS,'SQL_запросы':SCAN_SQL_AUDIT,'SQL_предпроверка':REAL_SQL_PREFLIGHT_QA,
 'Факторы_по_горизонтам':FACTOR_ELIGIBILITY,'Отбор_по_горизонтам':SELECTION_BY_H,'Семантика_отбора':SEMANTIC_QA,'Домены':DOMAIN_COVERAGE,'Домены_модели':MODEL_DOMAIN_COVERAGE,'Коллизии_разрезов':CONFLICT_QA,'QA_источников':SOURCE_QA,
 'Дата_публикации':PUBLICATION_AVAILABILITY_QA,'Экономика_домены':G1_DOMAIN_QA,'Медицина_покрытие':HEALTH_DOMAIN_QA,'Пятилетний_аудит':H5_CALENDAR_GAPS,'Исключения_организаций':pd.DataFrame(ORGANIZATION_GRAIN_EXCLUSIONS),
 'Пространственные_данные':SPATIAL_SOURCE_QA,'Геометрия_G5':SPATIAL_GEOMETRY_QA,'Геопризнаки_G5':SPATIAL_FEATURE_AUDIT,'История_муниципалитетов':BOUNDARY_QA,'Неопределенность_границ':BOUNDARY_WINDOW_QA,'Покрытие':coverage,'JOIN_аудит':QA_KEYS,
 'План_задач':TASK_STATUSES,'Годы_раздельно':TEST_YEAR_AUDIT,'Годы_CANONICAL':CANONICAL_YEAR_QA,'Модели_DEV':DEV_METRICS,'Метрики_TEST':METRICS,
 'Абляции_CI':ABLA,'Вклад_LOO':LOO,'Реальные_сиды':SEED_QA,
 'Сверка_метрик':PREDICTION_METRIC_QA,'Реестр_гипотез':HYPOTHESIS_REGISTRY,'Семейства_тестов':STUDY_HYPOTHESIS_FAMILIES,'Корреляции':CORRELATIONS,'Взаимодействия':INTERACTIONS,
 'Региональные_ошибки':REGIONAL_ERRORS,'Типы_муниципалитетов':MUNICIPAL_TYPES,
 'ALE':ALE,'SHAP':SHAP_IMPORTANCE,'FE_Панель':FE_DIAGNOSTICS,'FE_QA':FE_QA,'Важность':IMPORTANCE,'Сенситивити':SENSITIVITY,'Ошибки_по_годам':ERROR_YEARS,
 'Canary_контроль':CANARY_QA,'Границы_сенситивити':BOUNDARY_SENSITIVITY,'Размер_территорий':SIZE_SENSITIVITY,'Приемка':pd.DataFrame([{'Этап':k,**v} for k,v in STATUS.items()]),
 'БДМО_доп_план':BDMO_EXTERNAL_DESIGN,'БДМО_доп_загрузки':BDMO_EXTERNAL_QA,'БДМО_неуспехи':BDMO_EXTERNAL_RECOVERY,
 'Ансамбль_DEV':ADVANCED_DEV,'Ансамбль_TEST':ADVANCED_ENSEMBLES,'Ансамбль_интервалы':ADVANCED_INTERVALS,
 'Ансамбль_сравнения':ADVANCED_COMPARISONS,'Ансамбль_лучший_один':ADVANCED_SINGLE_DUEL,'SHAP_ансамбли':ADVANCED_SHAP,
 'SHAP_локальные':ADVANCED_SHAP_LOCAL,'SHAP_аддитивность':ADVANCED_SHAP_QA,'SHAP_регионы_CI':ADVANCED_SHAP_CI,
 'SHAP_устойчивость':ADVANCED_SHAP_STABILITY,'SHAP_взаимодействия':ADVANCED_SHAP_INTERACTIONS,
 'Canary_ансамбль':ADVANCED_TEMPORAL_CANARY,'Важность_групп':ADVANCED_GROUP_PERM,'Ансамбль_PDP':ADVANCED_PDP,'Ансамбль_ICE':ADVANCED_ICE,
 'Ограничения':LIMITATIONS}

def write_xlsx(path,tables):
    # One-pass streaming workbook. No head() truncation and no multi-gigabyte cell cache.
    from openpyxl.cell import WriteOnlyCell
    wb=Workbook(write_only=True);count=0;sheet_index=[];used=set()
    for name,df in tables.items():
        if df is None or len(df)==0:continue
        maxrows=min(int(CFG['EXCEL_MAX_ROWS_PER_SHEET']),1048575)
        chunks=max(1,math.ceil(len(df)/maxrows))
        for chunk in range(chunks):
            offset=chunk*maxrows
            d=df.iloc[offset:offset+maxrows].copy().replace([np.inf,-np.inf],np.nan)
            sheetname=(name[:25]+f'_{chunk+1:02d}') if chunks>1 else name[:31]
            ensure(sheetname not in used,'XLSX_DUPLICATE_SHEET_NAME_'+sheetname);used.add(sheetname)
            ws=wb.create_sheet(sheetname);ws.freeze_panes='A2';ws.sheet_view.showGridLines=False
            cols=list(d.columns)
            for j,col in enumerate(cols,1):
                sample=d[col].dropna().head(90)
                widths=[len(str(v)) for v in sample]
                w=max(13,len(str(col))+2,int(np.percentile(widths,80))+2 if widths else 13)
                ws.column_dimensions[get_column_letter(j)].width=min(48,w)
            hdr=[]
            for c in cols:
                cell=WriteOnlyCell(ws,value=str(c));cell.fill=PatternFill('solid',fgColor='173653')
                cell.font=Font(color='FFFFFF',bold=True,size=10)
                cell.alignment=Alignment(vertical='center',wrap_text=True);hdr.append(cell)
            ws.append(hdr)
            for row in d.itertuples(index=False,name=None):
                vals=[]
                for v in row:
                    if isinstance(v,(list,dict,tuple)):v=json.dumps(v,ensure_ascii=False,default=str)
                    elif isinstance(v,np.integer):v=int(v)
                    elif isinstance(v,np.floating):v=float(v)
                    elif isinstance(v,(pd.Timestamp,datetime.datetime)):v=str(v)
                    elif v is pd.NA or v is pd.NaT:v=None
                    elif isinstance(v,float) and not np.isfinite(v):v=None
                    if isinstance(v,str) and v.startswith(('=','+','@')):v="'"+v
                    vals.append(v)
                ws.append(vals)
            ws.auto_filter.ref=f'A1:{get_column_letter(len(cols))}{len(d)+1}'
            sheet_index.append({'source':name,'sheet':sheetname,'rows':len(d),
                                'offset':offset,'full_source_rows':len(df)})
            count+=1
    ensure(count>5,'XLSX_HAS_TOO_FEW_REAL_SHEETS')
    index=wb.create_sheet('Индекс_листов')
    index.append(['Источник','Лист','Строк','Смещение','Всего строк'])
    for r in sheet_index:index.append([r['source'],r['sheet'],r['rows'],r['offset'],r['full_source_rows']])
    wb.save(path)
    exported=sum(r['rows'] for r in sheet_index if r['source']=='SHAP_локальные')
    ensure(exported==len(ADVANCED_SHAP_LOCAL),'XLSX_SHAP_LOCAL_TRUNCATED')
    return sheet_index
# Единый сквозной бар создан в ячейке 02. Второй tqdm запрещён.
XLSX=WORK/'ML3_FACTORS_RESULTS.xlsx';EXCEL_SHEET_INDEX=write_xlsx(XLSX,SHEETS)
progress('EXCEL',1,1,'Книга записана');quiet_print('Excel:',XLSX.name,'sheets=',sum(len(v)>0 for v in SHEETS.values() if v is not None),
                   'graphs=',len(CHARTS))

### Этап IV / IV. Evidence, причинный анализ, полная приёмка и GitHub

Криптографический реестр первичных и производных файлов, полный Evidence (Parquet, модели, предсказания, локальные SHAP), верифицированные вмешательства, когортный DR-DID с региональным cross-fitting и bootstrap только при подтверждении применимости. Финальный архив без утраты расчётов, GitHub Release и сквозной QA.

In [ ]:
# 12 · REPRODUCIBLE EVIDENCE, REAL PARQUET QA, XLSX QA, NO PREMATURE GITHUB WRITE
EVD=WORK/'evidence'
TABLE_EXPORTS={
 'bdmo_external_source_plan':BDMO_EXTERNAL_DESIGN,'bdmo_external_download_qa':BDMO_EXTERNAL_QA,
 'advanced_single_vs_ensemble_paired':ADVANCED_SINGLE_DUEL,
 'bdmo_section_recovery':BDMO_EXTERNAL_RECOVERY,'economic_domain_claims':G1_DOMAIN_QA,
 'health_hospital_coverage':HEALTH_DOMAIN_QA,'publication_availability_audit':PUBLICATION_AVAILABILITY_QA,
 'h5_calendar_feasibility':H5_CALENDAR_GAPS,
 'organization_grain_exclusions':pd.DataFrame(ORGANIZATION_GRAIN_EXCLUSIONS),
 'scientific_blockers_v402':pd.DataFrame([{'blocker':k,'status':v['status'],'reason':v['reason']} for k,v in STATUS.items() if v['status'] not in ('PASS','NOT_FEASIBLE','CAUSAL_NOT_IDENTIFIED')]),
 'excel_sheet_index':pd.DataFrame(EXCEL_SHEET_INDEX),'one_widget_progress_ledger':pd.DataFrame(PROGRESS_LEDGER),
 'advanced_dev_family_evaluation':ADVANCED_DEV,'advanced_superlearner_test':ADVANCED_ENSEMBLES,
 'advanced_interval_diagnostics':ADVANCED_INTERVALS,'advanced_test_comparisons':ADVANCED_COMPARISONS,
 'advanced_shap_global':ADVANCED_SHAP,'advanced_shap_local':ADVANCED_SHAP_LOCAL,
 'advanced_shap_additivity_audit':ADVANCED_SHAP_QA,'advanced_shap_region_ci':ADVANCED_SHAP_CI,
 'advanced_shap_topk_region_stability':ADVANCED_SHAP_STABILITY,
 'advanced_shap_interactions':ADVANCED_SHAP_INTERACTIONS,
 'advanced_ensemble_temporal_canary':ADVANCED_TEMPORAL_CANARY,
 'advanced_grouped_permutation':ADVANCED_GROUP_PERM,'advanced_pdp':ADVANCED_PDP,'advanced_ice':ADVANCED_ICE,
 'municipal_source_panel':PANEL,'forecast_supervised_panel':TASK_PANEL,
 'stage1_physical_qa':STAGE1_PHYSICAL_AUDIT,'economy_manifest_structural':ECONOMY_SOURCE_MANIFEST_QA,
 'regional_economic_dictionary':REGIONAL_CONTEXT_METADATA,'regional_economic_qa':REGIONAL_CONTEXT_AUDIT,
 'regional_economic_unit_qa':G1R_UNIT_QA,
 'target_evaluation_calendar_preflight':TARGET_CALENDAR_PREFLIGHT,'carry_forward_age_qa':STALENESS_QA,
 'test_predictions':PREDICTIONS,'split_membership':SPLITS,
 'factor_provenance':FEATURE_DICT,'factor_transforms':TRANSFORM_QA,
 'factor_eligibility_by_task':FACTOR_ELIGIBILITY,'feature_selection_by_horizon':SELECTION_BY_H,'target_availability_matrix':TARGET_AVAILABILITY,
 'G1_source_discovery':G1_DISCOVERY,'G1_per_code_scan':SCAN_AUDIT,'dimension_scan_qa':DIMENSION_AUDIT,
 'stage1_source_grain_full_audit':SOURCE_GRAIN_AUDIT,'sql_error_audit':SCAN_SQL_ERRORS,
 'sql_execution_audit':SCAN_SQL_AUDIT,'real_stage1_sql_preflight':REAL_SQL_PREFLIGHT_QA,
 'all_stage1_source_adapters':pd.DataFrame(SOURCE_ADAPTER_AUDIT),
 'hypothesis_registry_preregistered':HYPOTHESIS_REGISTRY,'study_hypothesis_test_families':STUDY_HYPOTHESIS_FAMILIES,
 'seed_distinctness_qa':SEED_QA,'independent_metric_qa':PREDICTION_METRIC_QA,
  'factor_semantic_selection':SEMANTIC_QA,'factor_domain_coverage':DOMAIN_COVERAGE,'factor_model_domain_coverage':MODEL_DOMAIN_COVERAGE,
 'source_asset_catalog':input_map,'source_download_qa':DOWNLOAD_QA,
 'source_semantics_audit':MIGRATION_QA,'source_qa':SOURCE_QA,
 'raw_indicator_catalog_train_only':CATALOG,'dimension_conflicts':CONFLICT_QA,
 'spatial_source_qa':SPATIAL_SOURCE_QA,'spatial_geometry_gate':SPATIAL_GEOMETRY_QA,'spatial_feature_audit':SPATIAL_FEATURE_AUDIT,'municipal_history_qa':BOUNDARY_QA,'boundary_window_qa':BOUNDARY_WINDOW_QA,'panel_join_qa':QA_KEYS,
 'common_support_coverage':coverage,'task_plan':TASK_STATUSES,'separate_test_years':TEST_YEAR_AUDIT,'canonical_test_year_qa':CANONICAL_YEAR_QA,
 'dev_model_selection':DEV_METRICS,'holdout_metrics':METRICS,
 'ablation_block_bootstrap':ABLA,'leave_one_group_out':LOO,
 'factor_correlations_train':CORRELATIONS,'factor_importance':IMPORTANCE,
 'ale_on_train':ALE,'shap_tree_explanations':SHAP_IMPORTANCE,
 'two_way_FE_train_only':FE_DIAGNOSTICS,'FE_diagnostic_qa':FE_QA,'regional_transfer':REGIONAL_ERRORS,
 'municipality_types':MUNICIPAL_TYPES,'exploratory_interactions':INTERACTIONS,
 'lag_correlation_sensitivity':SENSITIVITY,'target_year_errors':ERROR_YEARS,
 'future_mutation_canary':CANARY_QA,'boundary_sensitivity':BOUNDARY_SENSITIVITY,'size_strata':SIZE_SENSITIVITY}
EVID_FILES=[];EVIDENCE_INDEX=[]
for name,frame in TABLE_EXPORTS.items():
    if frame is None or frame.empty:continue
    fp=EVD/(name+'.parquet');frame.to_parquet(fp,index=False,compression='zstd')
    verify=pq.ParquetFile(fp)
    ensure(verify.metadata.num_rows==len(frame),f'PARQUET_ROW_MISMATCH {name}')
    EVID_FILES.append(fp)
    EVIDENCE_INDEX.append({'file':name+'.parquet','sha256':digest(fp),
       'n_rows':len(frame),'columns':list(frame.columns),'bytes':fp.stat().st_size})
methods={'version':CFG['VERSION'],'stage1_source_sha':source_sha,'stage1_assets':len(ASSETS),
   'downloaded_SHA_verified_assets':sum(a['asset_id'] in LOCAL_ASSETS for a in ASSETS),
   'publication_lag_years':CFG['FEATURE_PUBLICATION_LAG'],
   'feature_selection_cutoffs_by_horizon':FACTOR_CUTOFF_BY_H,
   'feature_selection_cutoff_year':TRAIN_CATALOG_MAX_YEAR,
   'target_specific_test_windows':TARGET_TEST_WINDOWS,
   'regional_economic_context':'Separate group G1R, region/year exact matched; NOT municipal economic G1',
   'max_feature_staleness_years':CFG['MAX_FEATURE_STALENESS_YEARS'],
   'discovery_exhaustive':DISCOVERY_EXHAUSTIVE,
   'data_smoke_report':DATA_SMOKE_REPORT,
   'sources_measured_level':'D01 municipality-year exact OKTMO, BDMO sliced on all named dimensions; no region-to-municipality cloning',
   'target_semantics_confirmed':valid_rate,'targets':CFG['TARGETS'],'horizons':CFG['HORIZONS'],
   'supplementary_official_bdmo_sources':BDMO_EXTERNAL_RECEIPTS,
   'ensemble_method':'DEV selected non-negative SuperLearner, fixed sum-to-one weights; TEST never used for tuning',
   'shap_method':'native TreeSHAP / linear analytical; additivity tested per member and weighted ensemble',
   'conditional_feature_importance':'joint within-region-year block permutation, region bootstrap',
   'predictive_intervals':'DEV residual quantile, descriptive only after train+DEV refit; no formal guarantee',
   'group_features':GROUP_FEATURES,'group_features_by_horizon':GROUP_FEATURES_BY_H,
   'paired_study_design':STUDIES,
   'independent_split_holdouts':sorted(HELD_REGIONS),
   'model_selection':'Train→DEV frozen family/parameters; TEST is evaluation only',
   'statistics':'MAE, RMSE, population weighted MAE; region-block bootstrap; Holm confirmatory, BH exploratory',
   'causal_inference':'CAUSAL_NOT_IDENTIFIED; predictive statements only',
   'github_colab_secret_name':CFG['GITHUB_COLAB_SECRET'],
   'historical_publication_dates_documented':int(PUBLICATION_AVAILABILITY_QA.status.eq('DOCUMENTED_SOURCE_DATE').sum()) if len(PUBLICATION_AVAILABILITY_QA) else 0,
   'shap_excel_export_rows':sum(r['rows'] for r in EXCEL_SHEET_INDEX if r['source']=='SHAP_локальные'),
   'causal_inference_extension':'separate optional notebook, not run inside forecasting study',
   'evidence_index':EVIDENCE_INDEX}
# Avoid serialising internal live study objects to JSON; keep only prescribed static definitions.
methods['paired_study_design']=[{'study_id':s['id'],'required_groups':s['required_groups'],
 'arms':[{'name':a['arm'],'groups':a['groups']} for a in s['arms']],
 'confirmatory':s['confirmatory'],'protocols':s['protocols']} for s in STUDIES]
with open(EVD/'methods_and_evidence_index.json','w',encoding='utf-8') as f:
    json.dump(methods,f,ensure_ascii=False,indent=2,default=str)
EVID_FILES.append(EVD/'methods_and_evidence_index.json')
if TRACE.exists():
    shutil.copy2(TRACE,EVD/'ml03_technical.log');EVID_FILES.append(EVD/'ml03_technical.log')
EVID_FILES.extend(CHARTS)
EVID_FILES.extend(sorted((WORK/'models').glob('*.joblib')))
ZIP=WORK/'ML3_EVIDENCE.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    for f in EVID_FILES:
        sub='charts' if f.parent.name=='charts' else 'models' if f.parent.name=='models' else 'evidence'
        z.write(f,sub+'/'+f.name)
ensure(ZIP.stat().st_size<CFG['MAX_EVIDENCE_GB']*2**30,'EVIDENCE_ASSET_LIMIT_EXCEEDED')
with zipfile.ZipFile(ZIP) as z:
    ensure(z.testzip() is None,'EVIDENCE_CRC_FAIL')
    required_names={'evidence/municipal_source_panel.parquet','evidence/forecast_supervised_panel.parquet',
                    'evidence/methods_and_evidence_index.json'}
    ensure(required_names.issubset(set(z.namelist())),'EVIDENCE_REQUIRED_FILE_MISSING')
    with z.open('evidence/municipal_source_panel.parquet') as f:
        ensure(pq.ParquetFile(io.BytesIO(f.read())).metadata.num_rows==len(PANEL),'EVIDENCE_PANEL_SIZE_FAIL')
    # Independently check that bytes in the archived Parquet match the saved evidence index.
    for item in EVIDENCE_INDEX:
        arc='evidence/'+item['file']
        ensure(arc in z.namelist(),'EVIDENCE_INDEX_MISSING_FILE_'+item['file'])
        h=hashlib.sha256()
        with z.open(arc) as stream:
            for piece in iter(lambda: stream.read(4*1024*1024),b''):
                h.update(piece)
        ensure(h.hexdigest()==item['sha256'],'EVIDENCE_INDEX_SHA_MISMATCH_'+item['file'])
    if len(PREDICTIONS):
        ensure('evidence/test_predictions.parquet' in z.namelist(),'PREDICTIONS_NOT_ARCHIVED')
        with z.open('evidence/test_predictions.parquet') as f:
            ensure(pq.ParquetFile(io.BytesIO(f.read())).metadata.num_rows==len(PREDICTIONS),'EVIDENCE_PRED_SIZE_FAIL')
w=load_workbook(XLSX,read_only=True,data_only=True)
try:
    expected={'Паспорт','Метрики_TEST' if len(METRICS) else 'План_задач'}
    ensure(expected.issubset(set(w.sheetnames)),'EXCEL_CRITICAL_SHEETS_MISSING')
    actual_empty=[]
    for sn in w.sheetnames:
        vals=w[sn].iter_rows(min_row=1,max_row=2,values_only=True)
        header=next(vals,None);data=next(vals,None)
        if header is None or data is None or not any(v is not None for v in data):actual_empty.append(sn)
    ensure(not actual_empty,'EXCEL_EMPTY_SHEET: '+', '.join(actual_empty[:8]))
finally:w.close()
mark('PASS_OUTPUTS','PASS',f'Excel={XLSX.stat().st_size:,} B; ZIP={ZIP.stat().st_size:,} B; charts={len(CHARTS)}; parquet tables={len(EVIDENCE_INDEX)}')
ensure(not PANEL.duplicated(['entity_id','observation_year']).any(),'PANEL_NOT_UNIQUE')
ensure(not TASK_PANEL.duplicated(['entity_id','origin_year','horizon']).any(),'TASK_NOT_UNIQUE')
ensure((TASK_PANEL.information_cutoff<TASK_PANEL.target_year).all(),'FORECAST_ORDER_INVALID')
ensure((TASK_PANEL.effective_available_year==TASK_PANEL.origin_year-CFG['FEATURE_PUBLICATION_LAG']).all(),'CUTOFF_INCONSISTENT')
progress('EVIDENCE',1,1,'Архив, Parquet и контрольные суммы')

#### Проверка исходного кода, самостоятельный полный репродуктор, статус Scientific QA и техническая публикация.

In [ ]:
# 13 · V4.03 · COMPLETE SOURCE SNAPSHOT, FINAL SCIENCE GATES, RUN MANIFEST, SAFE GITHUB
# Архивируется ПОЛНАЯ версия из 21 исполняемой ячейки: четыре этапа и причинный контур.
NB_SNAPSHOT_B85 = 'c-qvxYjYb{mLT|jenplX-3drZ5PazYI^}>!h(ehJSO6$lMPaj(K!VI72*Ah$C7R-dALT07xTd|;TN~Ta(OuoWvmZ8MV<TjhWXrN7NAxcs|HAx|J&$`cZ@xgvc1+Axg{eg5eV%*ndEIl*IsQ-DrH<3@dzVVu_J7i}WB8BXSpLz-fwz|q?D2kg_;7H^pl@y$U(PLdb{0B4N4z|6e7kG=Hhelh;aA>pGVVB{S@`{qwy{1_T+sgGTkY;*0UmBwYvpzA_SX8k)~GZq<;KdL2ci=w{f>6}|DL`)eRKL6{(E)$0-m238vOhE?6+qhpFTPJ<m}V4kIz0keF^`5hHv2Ov(x8L_RZ;Esl;zjU!48^^!eE*SoiEJsH{;-{PlP6gss2kZ_lvl(<ibkC<$%9h8D2g5Afl8c>WYx`xI(D4F?)NUA}z!zjHHZU!Hyky}=Mafojj;B~*cKp!(_eXMdyty?`QX!{Kh<(N>0iduIkG3KRMC^lO}yq0!JkJ^K;{`&>Kwukhgsjss@-hI`RYzr|Nj2inte|MCBv{sVOU1AP9T2L9>UAGOn`@bUS7e4j5|zMMAZ(|<%bLdU<yDyJ_?8Z`JZJn?XOCTp&LXR@RHU3uUS2dl%6_PXx=@Njn3IT-Ga?VjZg4;_Et?*APO`nT8$^!Wl>p-Cfzsg=&~;K22@rf=^$S{0`81O|s)V|U;3g1|!1v|-Ap-!fdZrqdZazNh_Nt5Gel&NQ}W=>6XnG@9?X*p0Tfxuro*0H{x?GQtp{_sQ8;Q0^b&D?yWeE^1(TLbL#-UeI5t53r_2KBK>%!hk<J`}?y$$jVPnzlQbvdqTU{G<rlaY!pg<39p}={Rtbv<$7}ZS6X3gCPYhW=m`z)4PhGN<qKL{u@+GHMF2%Wgx>;?pCj5AQdjg}3G%O?$Tx&ew6;+9IU>#v1YFSxqSDzH@boo~gb`N31p+L-|3R!JR>gwqA`?Ie&Kj2fhPL8s+AhzrGA|`S6Y>7jpay{O>~oE9`fJ)*zk@-2zD#Z42EvW@G0pQkL|~23`WXTm8vKUwQ!cP}`LY-wz&G3-1mCCd^IIYz-^qF4?thLDpMkEQoPB)xvcROj(yD)pQ!Ht6A2HBzIG@uJhYR<eRylOL&Y<JW63sy*_#VOaRBjmj@sid8>Bq}pKM~3KOf1WH+OMmdIM_G%=P5OW?|}|!l}A8`XIGkcVWk3+hSUka8cFncCFV#p>LpD8dN;IRHtRL*SLI+pKOo-mKopX5`lOIS*8UZD>DgDbdtOG!&gqlV<;#M~XlGv`cuv1#B%#fO@Rl>f75)6|GmRDswmg9S&wv(R;a8l|8-adbNjfRl4R;1^Fl@HmYH2)Jz^X5aOa=J&1I-2J$Y}S38)3jO`o18h@Vf+gdc`%Ki%~z}8i+g37zc6l)1r&15#qj(9RM!#0x`!phk3xnag)ACP#4<y)b>l-AL%<Xvg*vdc-0X~&zKj+2#~UaVHtr(GFsC-h<$#d{avlzvZ||<TB~}yT3H2J&eZV}0R_?03tV-qAV&P00f>WrCZS0q7j*EA02Y#jrw9hQbE7*<?oC?qZv$FKREBBR3nFPi848yWFWc~k1dWRdEc2bEx#Hr|)wd8>NCsiPt5vmGt$J7R5_L&luu#IFAb3Xk83`S6Y#`j`l_S^?x=*3zlJ-8)mgY4<OAr?XGbLDR%{%3p`K7BQ2E9?`0J)pQ%B11cpI|Hg^XXp<rVq~qq7fB&LX)9Ja4Q&^AkH6C)6Bk~sM{4uAnm4Cj2$8Ig~bQ-BMJ+U{J*I3tH@p?e)trzMqmVj@~NQduS81{WV99OH(?3Vs85+TYQ*Ya$lZvIyb@vs@o7UKGIDRCS+C{Zrk3AO=g*m>pZ$p^!JPYi?CT7>QYv;HIh~2`4hL3eI2rixbuj7oRjlkRb$ibB;!-wN4k!L-;*)sZ&JwJPb0}4Btb<U^qBZ;?xtj0-kPjjEJxrA8)888^btaIrxbR;D1%Doj3@jKiO+-oi%MdqLZm(&kb{xLIKl(;}wW61F>|Qstl>fb>>CKJ$`xPDW6v;nupD%;)e3ukAW;JdtDE}f&oICp}HjvG&TkF;49SiGPo8@LxFWt8Lo?`^#P=G*tF<{G*sD8%638@caH#6@;3E~fgHb7~=CSg_3;LoACZxn%$mf=_B*w`AC%{l<-Jb_v_V)dH!t;R~lYE*7R?WOCB*A{v=ZY(V>7Q43HnZL2ru@~l-7U1DVZ>~7E)3fIm9Xrv=opNopejA(Evj^Q_uQyw~W*yEad#kK%06&0o9@3fl*@J#@!Rpx^e>nCMrSDc6&1$_Cl%6^0FU%YQIu;gYdhWpPCu-cQH{M^ZHlW6AXE^Ykfj_;6W~H*Km*%d`--v%%t>3Gy*UPI`8P>YlDz8=M^ipd)5sUqflu&?rB9%OJ#%|Abx*`1pYWKaMc2Z^ljm7TlIm_BF>(!b@1Q$QOL>ZH$QCjO~uw_`7RY^{KPn18fn$_Ccdc^`fthB7PTYBkwp%|aWMzv<GR-5lzw;PoT6}p;I2pX&0MNYlKW#$SAc-*Nss=uz+0OhymjD>dmbF17~tF-W|e&7Pe5s4Z(V{5KhEb4|n8jdD?e%k>Q*>$W@$JZ0h+^&>cTaAiEJ8h-hg4MUy%WE*A#QIe3TK7T2hc4$6gQ+xInI91%twtICiV_RiB^vd6P+)F86|B??msm_KaI0Q#HKCzR*j;N?g5C8r(2Geot<6fqT5Hs|HnGpCC2u$C8&<t07GEzdrIlWRO<7$nw<^{OP*84Ol%FrAS7?>j>ubs4wc!Bq{+&R8dkKt@MP)JxycRkj646&Af_^N;I7#uJsdPT<5qh$ku+(aQ)@pqNpf9>j0c>Rj(1HtPt#GrxQZ28aQwRW5uK}8^Tjyd-a}$se3T~8JD|af0H3<mHA}a1;lj-wFD=qgRkS{%}{H^*{ZMEFEZ>==y&E~!G`ui4;NfNLnH@bMuX&tJwjsYqY=@CV@XajF-wRlckb|GVfh!Hg_wPv+dy<2VFpSD+^%I%7U`UI4#-)dGGcd3)CO<V>12hi3VFdP_ub+fz<z2NH2FHJ8GjWY>=omebRFH2}6OQ#}Fqp|{nRtIrkBHOh@<8l*<O820n`n`-DzFOTt$kl7GGEG<{Bvwl>UfQb9Num+PmglIwoqa*O@{RW2KmN0Z%30-uJLRn=0KOu0vzQ$LWt{W64L9!!$!YUNF#r<qq?D{sf>)&Gz7FkD5mhqKy_>&UpCRJ}nR(a(MS6%v0@N$g6`rJlJAg9**=lZWZq^$>#jY09xfXDkTa|UIe79VMrz$N^I@7_)*H&+fSzC9@>suAt2?FGA6qQ}-0MF6>_=48r2`f6ENUIgA&8+CYVfzPu5-%BkZ0$n$7=ix#DeF-5DC7{bi3m;WNt~&|ysX=0*th^wA;iRK#IJx7feg4_UAqJOyMlWjnLv8!$_Jb4b=c%hB*e6VH>zt5-tQ7$)9P(DsvB6EhzJ*i9*)x_hSK(YaemH#KlAM@>4D13W0{5a$rN^p;@f(qR%tdfS$?HfzO`Nvv<D`S6RU4jR{#pNR^@|M`h1ebRVJOXXHzN=c@PwDu9#UtjKiunE!?;vim%P3k4*@Et!ixr_Bmo|$dOW+9C70sU;vZ{iM9b-xmYL`69rf5jYef1q>5F*nED66PPSSk`WFi~l0*uS2yt%%c2$L>!$x&=tpWn!9gs5(Bz7PrHCymVsJC`&10JDy82CyJM2~iw#18Tg9`v9nijw=>wS@)vvv>{uEaB5Ne7cTLH{k2_;vD>$!$0#2sqMiEY*v7&5WlK0u}H;x#u5+!R^HzP5x$kNKLWuIX~T^hX`rrdU^itHxC0@xcuv6(vF1~k4_0kcv^Jllb)+FD7~xW~*VV0+_g8OO8<mZE<330RS8=7%%e3wQsaVBXCP~CfWu3MI4v5%8^G*e2$vNEL@lM~-nq-2|oWa=b>>YsOR?_Tl*Y({)M+1G*?YMSdyS2Je*F2|#l)%%56W?>Yj^^(<ntO0C@$H?y6DVLy^M%-$)~c;LTekqqfyyK8YT$76=oygCgdAxwS&{Kn<YTh3K9ThITjE^-d0!R>62#`75^EJ5wE*C?_=es>v2&}5G~fe(N|n{&G%S*0T_`5qf&C0?0<y>ER*Pgd80&C^&;a0!Y=6(t*pxGS<k(|RJD7OBwgb=^be)j{e+GX4Nb5LbpOn?C4po`-E}%Imzs-zJp=GyYgKkYdYNN?c-|ftdoxWpxjy85Wjyv+btd697DLTpMW6$-V1L&r&?K(p=hIyJj=;Dt9#~;Il#zW70X!rMd?SwV*<Jj3s+)QamJ4?!O-Ak5D;s~?^<e>%9^JZlQg(iy`AxeSw1LMFuq4|cmG258GjN7B1Nb}AYAmGq(0^t?o)ak3Z_PqpH13V1Sz$$O8R$FR!Y=|H;BOfrlB8!m_xqhJ2j~8g%!s8+~&Y{ie8Da4Y7Cv5*NssN#ku$h-bCHT6iSaxmweZUna~B;6B4|>&R1Z%wjLN}^QGJw>J|P>+({wrw{Ib$O=80q@gHpqnupALlo*-F4=98w9WQDETW_jg(KxV8mU%W^aAg#A54M7zVmC(iof_s7}HXILjRcCJo<Rc(-&ZAM^b_Yb}oPp;Yz(Wb>VIL@>(}k_ycVK((z=qfx4+q2kaF;gCYUOU66qZ--B1>7d7*y%N2?{kTzuKyT9EGN^ke5%bQnP*uI#dc{2uegP6k)ShjqI^D0dTOe@25y`VW|(0VM-XEGt7Gx7I;Z0L3w9>TD$9L6S@$@6jluKM`p2GfH`RJsjX(4nxbvC>g$z8nN;=#XJ+x-(W;`zd{Pt%$^$BHh%sC%7SAaL(o(tAoVEy?tD6xM+Tv1idfEEsG+e<Vx5$n&w~*AAiJP`S5@{hbC2_h5yN&sXw9KzG?~0=xv_*X?4g;iJT^^5zCyxB{4IUk#2K$BTYeUu9xj=A`PXZT%Px%NheUJesPaC5$$T)cn4#UnJzj9>E=T>o$HdgKc=R$VAvc9zn(r5Xeq<Jf4kZgk^jd!%M{V<TG$Z!pR2PjS)I>2=M_ITHcw+^a+kgufuTwx4RgZBm881Urt>~lQhTShOOuTH-?`x9}???OW)Gcnb9BVFoiRaWk>2}_*f$br2hnx@75Qt`piu4oZ2jOg&6rnxmWgxo_C#Sg@bgA>ifz@&$oCDw#l!!6t_2A?nlb`JF*iQOO2!z1Ic3wgWZc#_rYdL4m5M5S5Y*hG>xm!b^_@R7)NU)X1%l2yGe4JKK{u(?v@qyH5^_w`D-NerZZuM)wDk9l*^`F-RF@jGqT_8mfaKAMYoU<feaS=e&m9788*Bslz$%_Pr4uMc)#P>*L7Gs#K#XnP)r$J)y6wQW9lY=exe<-`G_!RLngM0va4&+XkTl3eO~?!fbHK%v~8;ST(~)*FtsJ#2@X4USUU`SgCXPC?o?%{j%NQ9JzyvYCA%y#fe8=*sj;6COKX6FK-sb!j1QAW;Us|A8OTAoolNYK*XCZhNZ9V^QH_g+4xtQ_gb~EPj;pM8~;mFU;>;e=FySceI4}0@*dCf$c@i&j)5qhwf)z8jKxJso;yiHRy$=oa{4UL6@0lNDkPx7P>!*^VgsKxPIxHef8STPdu)l#{>FG?QXSEuOSySw2f-5x=~)&h#Z3qxCz=rr3QPlX=o&}YwPv35S!eC(Gc*@JMs)|2>*F|Fz&vgfyC(p`X0cW$<Amz>^L5L<<fn3r@*Ct;|&Lf=DP>5tM_bg555?n{TbQ;DW$FtxIG;9q2@z-Ja7lQ@br;8>bZR=;|>jN2gv5tMQCDdcbpx&vk(3Gwhx-0+kvtG5{Fyw4Eudy05G)4xQ}BPfvyXmY|n>Qsq4{b3>z<;U2i-*AO#t|iot7}@G~rdC&wnf%Fz~-2g&XHGCmd#_Pg#l2OWV5?U_WroJTM^Yq(Fpl%0&5y~{*{xF!${N3axl?CjZN--Cw(z>>bF=Sx}+dzkI!MJHM9PCQ`6Ca+7O>w5bZy~%n0m|Do^Bedb)g}=!Y_AevV@N2RA-v|c{R<Sf9nqLT~G4i_1a8Fd7G11*tdZzM5A*Y&Y9)O^;Ova9dV>fB+h2Ge4X6G+oUMSu)ndHXLtJ?~S_O!v!*OXRDW7`FG+kmYF$Wa-OhvS?Ma^lu);5x{Cq%eXlu9QJhtG4c|)Vo%xtX5XdW4$}+?00uS9ODNdw1qk2-FZB@KC+?Rhww=}!MAJhX%W8xxwk$1L=QL3d91Y%H<uAR983;IM^Kdh#rEf~Qc>OOxb%VkrH)0<-abeZ;{ku#cl{aqi6s`PKuRaD-A4T(ZcL~Z{<vu_6s{ZZ;*f_UXD~W?)Q9i#5x>sS*R)~p;O@759JzSBX)YCR81G(%*2d0<6UXxr1o9Dqu#gD_nBqgXyX#Pu@W)LPhIa!R{LOHO2hR^U*#(@e!lCP%EwYKjs&#CCXE^k5aPpBL%kb;FyL<lb&H;Rtk2g&K-BQ}fU~GMN000*IvK0Aw)4a;#^Y-i!E(rW{(<~Mi;q@H6ezYscA)XKlMTDs`@B`Na$za#qJ|ux)-#A1n!6;Y&5_#a|@){s{l(-gp?x1UVBd3$whtHmdr^(S>AsQ}4XEk=bN#8d^vRxQY2DxpZ@C9Z|Xh$&gnFB;*cLaZDK7>CcyY&0IaalCmHahmm$1TU4h^e3NJo4id3je@xxZ}dufl32phN;_h19X3~oLQUIO)KC}_boiG0_jKHrqUE>rGrpn#7_cZypNw7n(_932Cycv7>X&9t7!^1H<U{0ecbNTPWhB-?P-_NwhVwJWo3hw$WKmLS)j}<Zne5zSuJV$6)`8z?>gi0_6!~kl-l`ZS0fOQNhSJkflW}Jt|I?>&5nDdKYRw<Eby%)b3940p<goHeMxfzj+z+1B_Gf?p@(PMH3F5p39*9cJIEs1U<7gq@df-f>VgzQyafd#CYA7=Sqc@fgP$nSKnb5-f0*bYr0dJ!V`1Vs0Q%isb>Ac^6h=q*L=E@j@)0p0DIKBo<;4pyLn;7%J4hY1p#6Qvk}osrK+%aG)M&!LeWyzA##z0L9kJD5w*(b^gy^{2?AZOxnoPMZBHglL0~w7`>Dwq#)*L^hsx+(0YIi^uTN~S57v#QMNZ4U44QChd)*EJZkB1T<wFb~}?15$Pc*8zS)^c!<81&Bja02pVmtV!&lDt_qa&I`^2gb5*^oMrWdMJKnRD%6<)OUgb+#U{m14LT@_29M(>PH#a;Q-Irvg#wr45|Zv2JNLkIT%=d$B(R+M0p_?7|gW8@1t&k$(~oh#VB-!qazEHp@(Cj)I{dOvOpSITTS!$WcgmXQLEP0n&x(Uxz#AIR7?_(b@Jr&ozC6>&Dw?ja93Z3T8);8#0Po+>2L_&a`^~hrzQ#{RReD*@z2QskYryi*>!pvG?_c_b`8%#e@_3%1Yz=a9c1(<MIs`NVmFk{r+gYa1>h68v7UR+&F9`L?e6JU^pW*=*Y)=%JMe6QDq`ig?U|2?GdHaE6<`$Gjmm0yrBzvN>%;_N?LLI6v_3f+zoFZ@;UE`4Zm3WE-pqA9kK4MJLa2KM0(eepQH9`PFnk!k#XlbbUxelF<O{Ah<W<e(uk>_n`&a_yqz!}{hj>MQ@L-_FK#0nIZeW9|h7@Gt8Dqx=<<Qg<P~-*Q#<FdY<+!+B64i9Bhrh=;e$H=~7OxlEC(&kb&9IFtIx-ZlEuUjGlRTA+lUhLv8##3wI=Wcgu9TZq<dKH7-bSypB<6$vlc-y;M|j?v!)ZAnFcLuCVu~@KvdWW#?T;8&N=yUCKa|=~nJh`L>6jfyaTHtYy1T%Aa;Of++uAdQ_Fe!9cYf(=E)xyuv{@hTOu;%3&~^~4sQu^q_Q6iqF7-fuvAem&%X7tf{*yO$5I6P;lM!NKZYLiXyinfJ_6mE>BQbKN1g^eb863LfAyG7~{(ec@Jo5Jt<*uM~21x4xFbdwN??Tn>B5XHpWpm491Q~jT&S=8snOr^(-%uwd?b6zHcSNZ%eD3DrI;%04k1E}_ZMU!OI52ZMtb=7k_XVWZxJu}!fUMsIqOXlvWI|aKSI<6#swhGx)a4uyQOgpuuq-hTy`bY%7;8b<$r~CVFG$o)Yoq+jdczXF^MthMZ-oepgV4EE9c%9PhdXv31!hoj9FmFCZVaV8$*b3NIi#eW{@dwaPXFQbUrzs`!NZ^V0WE2GPct;x(uL8N;vge>b@KFymJ9X4!kyN}dcF`db}ELsn0Yg}xEbEn{Fq(jo@!{D(~9#2?ew3suB4!4<TaTUzYnE7mI}!sQ4<b!hju?==YA)Cqm51j=-$U0GE9E1-3Es3Z;po~G8mfwVfO$8K^IohJ_5kg!8~0e7bb1;HxVZacnlsGR_Fp1`KT0PY`e<%UJ$KHzAb;k_Owuh>JwJutFXR5K8jNlt{g4})fdkSZa(TbBVVghX(BYSc4QSZidEL;9Xtr2gD_zBdZl(3Xmze=X!BPQhs#^7JNRP$s-Z1j!#BbwM9VE+Gqkzu^o|eowH*9}igQIScNa)Ad165IIc#y^I(|UMSbDc`-Ov__bNKEyu-ZDF4r@7h1FyLa*g=(bdWrQHmZ-B<x%obqT!P*ft}o!r)e4Ope&IOhsT_L%<3Q&PZRt9_l)4bUhgXJn_4*<WszoQLS`L02+O>IV6c1zY9sGoM*9jnsYczej4uH8nPopItD|`vR;pGC&EOhCH_X{w!>kLNWF$XUZj@Or{1;N<(oDM!MVLR8<PmRhNeYu8Tc<ACw0ITabgX=f0E8l?L(3cwk@f-8k)Gr^fd!3ep_waF%I+G`0@^Fg2FG80$uFk99@eq%`T?I(qxUs}5u@0Yj+2Ix72q=;HlLX4Os;%`(h;n*$wt5$h=Aoff5AiSf8o77<gpXozqv{G~28Hn^W!!CrS{3*j_W9?m>y${1#LY=PhSXv0f6Trds)q<4mj72V!&mGA#OEAHLBB@*R*&q5u-5-3i0o6TPM`e|@9B%HqI?XaE1O2?FT8=OD;r3!DkZ+hGFgf^0gbm<q!N-r8nb#bi$_veR@s6|rN5vZT5-nagIuzF>(TwGRQg}h6^zc$$$d#zHY}7<Fo%o^V$DU6j~AK<;_wxFJn7-aR_o%pkQ03po-AopEV1X+TK|p4{(BKzK@%S1Z%)4xhu~tJbcH&o)&Gfr@K+cn#R1Tx^g^%R6$OJMaCyD|wK$j+D=LXeSR;8Z_lBk>&?j_He=TFH=n480Yb_e3LR|R9Bk^xb$QJ3UoF1oCN*n(c-oDScd9Np_+4M0y<E?i3-RWyRPVJ`Dd>#onpr=s7>D|U6FzD$NQ)%O06gP_Jk(e0hmiHnW`r(D1q_AOIf&f;66RZfuRr>r#8sJ|=T$-<BaEQQ#L$!o=`e@Iy>H|kgMltCD)fCHe2d-~fInU|$(v&THUl^FhWqR<3zTG#G`19|s<8@4(XeD2h7swM(N-j1UdVbG+1o|oew$MUk77NNRyp_-a-NKxi%NPX9I;N#cuQ?JHO>Sn-Kj?#YfdaTJlqPd4W?ZFRsUu+7QcRS~DdqAhFr<#N04QWVER$k#?f9k(IuWZr0N2Y5@7WOA`TQ~}wk9@E0C3wQCl|mz#fs}qc9Lbq0;CidnqN+Tx9c8iWYaPqTsp8H&A@#9z0!^O;^@)x2;Ia7yQR6M(Iai{8ob#Vj=RpdGzY)EVc+d)?{pWt9cO7-Pd)bG;xlOEn|P(U1#H3Y?C;XVOYanS<~j>=%bj5#WZZXpOT8Pt-g0j^@Jn-x&_j9b+I_>b2i}Z_cYq&Ux|!ZhWD;WaO1-}GX!$o2&v$!AGvdZ_$s5@nXJ*IoA3DxpdDk9MPs<1Pc-I}w><oQ>cu<;yxo7vab5l{zLVgo?I{YalE^5RlD5zgyW6Ha;J2#;jR(}B(Kr{XudN&L6m^Q`nkY+d%8+*qdKe(jX0Hqn)5QE`E_^LJ=`F!9G;Bzth&VB^%00W_3YQF(k_Z&RuBUD(9F6)ihsxB-oE-vjX4?(Z)^@k5jdu|u#+r<`-u`dFdW!ZDF_%~`lY>6iKU9=$QZWO!DuJO+NoV{fCjCZcib+7Lf^A|$Q8?@dGe(^WDEeJdO;iz;S<|@$DzK-D_2wi79_amn?H=m6*S73kQ!iPQRDI`UhX>jiv?w~Vv(CjjWJj06SLXu&?VuH5E->x8i3EKd2Lhr{;4=`shm(scC_!bhEu}fxU3zvXKf_6r#Ws>`1z7+BRDidqEHw21qieDN12*Pw;ACiImm~Lb_YMu*=rjy{rjT5(XFH0hwPM%X43vv?bijMxE)tQjfy<1qIx7e(iWPl~6$Qet`oD&*MW1Z0ATd_IieIdz6lTA5&XpE;d9OL-^X0B+O#S+>Fd-`Uhz6O#~Q#*z)Cx4?I2iS2Up2aGjq+xugKlB{IL{pYGc3Nx6&a7MIhN;RpuwLOuyy6zKjfCGSt80~px!sQ1Z6!&~jrMHMF}hIMoHK;nV{{$gcKa!IcDC|q(XJCFWNK3y!Xs81VOD<MITGi)K|@=$_iOcgHBnj+L~JJ?gQDl#W8ZKFT{F`1HWh9!^AwOW%%Q(zP6`(^LnatVMeq~8${V6UY@S1CgnA;nl7sGM=x6>i)|yS*6I2I#zy~9;bcI2lPMlwHz(|sAWpu2wFF3A;h@Gg0_W3F>MFzUE>O=J^#itPO`-I^b6W3Y9!Xj|_F%Jj`(nH6|(E*I1YkGdWbbYCv0J`s(xrq5gK%8)gp(yYs{b<Dbg&9Sg6uJ}znxME)b-UPJj((GrvZo(o|0lD@G=dWpJC1o6C;GGwXwOjNiP7Tv-HfeYDf(M5E-X0{U+HLjD4EA9XYeHGnUK_^nUrEXI6MU+u<Iq>xO!)Zr{BKQ)l0MxI@Q)oydXM=Ocv2RJvNBJN&4AHmf;?W#)lIZc^SGx<}QPb;2D=M??1$cIGq4o{|n&Y_jENZ4Dm?DZT1unOydz>3sfneYRSV{x-a=A5%cZu$<doO)N3`U-%BmUzD-A}<0B&DfaZnY3{jeZonW(5wT;w@CKF#mwog~+ACMGE=s}WxkWr!Q48kuJ5Xtd0e@KTl!E`j54iZOv-N!>6_~GCYyvE);g|I6RmgT^PaUHsz8$Hn>m5d-Iy0iSDMTgEn*6m$%Zv_<H9gdF-4Si$j*ib_uaxmmKCWm6dSM(TPMCNzLvYCQqVH=ipQqqn^CnwtWF*&-*9`XhJdr~^)j!xP@Qt%*C<%QC1x8Us!ABrDl+EmhD9561jy&7}ye?97-n&q6$S1A7)SR%jY%arQKBBwF`!j7#^nA*x13(<Qe&m%EPOR^IRCa99{$aRn4d`4b56zY?3@Yw`40uRdv9p@f{qmTkS0J!gr<<Fe%?hb}yG^vP{G4)m3w{MSaJ{Q~rm7;&7^~Uxt%EwN(a9IyA#2j@C<T>IMlD+4DtXn>4MGH6P-YN()x#j;!M5*A&c_Q3d^d`!aWW*~OV6dbK>(+C$3ICo$dxU3tcx6<^><z==2A4UMl26&-ghtZe%J6bOv%|i<*qQ4sEH3;+hkeq@o0XNVMn%Iz2W?ivyYw^l+WLJh2*nXQ=>ut)JmEsmoG4j)M+7+b2w2)cAX4;-`?5AbPY#WILNpg(N^wk(=)wN=47ytcv{h0#1(W$oyY+tMK8YhMx7P|VhFt2+dLw?bUM11g4#=w0P9k~66wkWQp=pU}PdsOgAdeM?AzNbNcq$%_)zH-o`Kf*z*gKC2=OsI!$%!`e2oeyk3sArvrK%vza&l5oHtr%1=f{9K<V(q^E9ww^EbWXPdw=R#YjoL#R>zg2?Q(DE<>0WB1TPpZjn&xVCDUng@iI}MWllfRId>=UGWXz;upaxveFv6A1`fkr$yaUYWZMX16{!&y1TF`0iB7-c%Z~Yy_TbWa14(mCeQ@d4dY!In<Ay7x1ol?RQ1p7AIKtrhK=)y{<;|+udvIyb_eWl7cGe!b1@`j-q&=7gNqy+e9+QVHg=lP_JWyCx6GXdey=F>}&_Wa49ft#TUZPB4&#}7>P<tWA9zVELp7?vivHKB4#(8k51QT@ZF>LJzm#z>P3@Fn<<e`rsjrK8E0~DS;9CXDnui)LC2bYX=|6cWf4=#N$!vL5mkKCEN?2<xFV<4WHxw)D7>)6tzE!aRa<z4hJ#IG_6<&2C%IfH>kA6z<#-dOnuUJ(u>CnEy93<&-YBHZe;=xsJOYZ9f`?zCE)GXZx;nG$c)5*restDnkP*RxxV_0)jzr!Qgx8>}eEX8`FgYOo`aKz-w~f$lwc>)y;^;Ff%*R6QB@O<4_hxO^^<YU-gwRZf@4qYF<Qz^N;;U`*mSB{~R{kwb%i3`BKcJ|>R@^sP}&7w>cf#%bIr=9Q5qJ<JR2nPwqFl(qml(G;qQ#Y{l4l@-)bRx7n?W%a?OWJ^en7mIVoVsSxC@WCZ=v7?*s^@dfeTMcA)&6L`vxmaA3wQzyu1p##*O>swuy?Ko7fCMb45jLV7M>;uKM)!K+(&llvMC@wy9=Td=m*y7G`S4P@4yJg;;-0sLJHIgzbM!Uhg+MuRpQ-y=xOd6YOmsm}JA~b>(<KGIReS05FLbH%897pYB7<{BZyu-JI6nPunI+%ga<H<Sgm)9g8d0ajK(M?$TNysUaF&xTNLr?ZjU||f*=^&^81>GUyjou0*r?LQVKeN2ZoJ=K69W<vm3dG>{PpniuNn!yI9Hky(>ctQp*Uh>>Bj{Aog?4zQXL|r%gAt`QyKt-di${O;RM7b_KNiE9*FX=nNQvuJ3aF_oUsC_#O4c7Ky;8QrwC-@+meNytBYL+(KiQ`w}XmtKi79;RD9yTLT5kwTm$;A?evE`o(P;y8qyO|nAXH68T06w5}@xZ4YdaHLk9>Dg^dK~mZ1oKAm5ZH>~$iZUc_{J>_goqvoNMv8q=(Pg(gM#lnFE}%21Qe(6DoUVQM-<NY9W1q~Q`o91CQETUMpEQeUms*0OeY0^rzjI>a$<V|BA-;)n|fB`>Go7+(Mv(URL&^jVIW#Ifx}xFnivL|h)*4=s%0^vImn6pE~Pa6u(SBPvy*#zi{==aHYwJtC?35fW%I>}?&b0v_Ia?`e;=bwp^r4T51;u&UHv@&?%=QfQmfr)uA)Q-+wUaWKie>gsEnkwyq1pmZWIoDzi^V{=w09GbOJZBoQm6%h0_vOGb@%NhkZXqr@JTgOZJzNgYNF|ocg$Z=5^;L<!z2j3mrqLSry0fvEzip!orVWnou49If>qc45IGH5JTy1b|VoV=!<lr#$T^dce>!Du))fEs!lJxO0l$qE^Vo?I*pVPH%6S~8%uxFQB0`a=rzrMI_rRGejC1lX~35tJfd==u1ZQ)l^);VjrR7-V7LSQpRF$*u8xmI^V|=UdFevJ-c|d!bI0?k4DAoJ$dE6^74#|3KcuWC#SNWhJm2lW{azXbMROv0y$X6{sfHo-TwARPIY!{rzQ4L`XM}8Oa`8I+TKR8^jQ9*)xw7M1j}HITO9Aa|>;3<iaS7Z2zSCETZj1q)bW=BZG#u9~sTz`qGW={9N~K%|`!6%t9oD{HJtg!gw295t%^fHQAd~n_-x>MDao+7pHI46BuZ5lW*7k%yyw2yVF^`u`vG=?Lv!`&1|7iEYgFBlt*FaNtDt8mrz3sf;rC0IH9xk##*^n{WZl5(bmu%GI3kWcMd4<LPF+~(faV-l@`Rm|5cfObGR5m9*<9=LKI3oQ2hWzAWI4dIiiDcAqIqY_<tl~zKDr0_6Z6Af|x7=yU+mG<B5mi%}`+jvVu0M0yh(Fnq85iJcY)flH>x(2~I8_VKUpL`FY@=^4?|4Hq-TdaRZIxIm}{D$#Wpj4=$Mz$rO#ldm#H_Ad-2dU)@cNR_b8Kj}mN!n09R8PBJd(#)^f<QbC)gMAwEKYbiE#gBU@dg%G`j(?&%qhZY$NeN3RH?60SnLYieT_^dr(7!&~LB*loPqS<z0P;EX1_55saG5DBRJ$c(t+`JBAYb<O2!N|k{DBD=xq}OvF<p>Qx{9cBSSvBK_E*-hR0h5Hb=~TOzJj_a6d#EV+s%a8a%~(v7$jlVggBoCQE!05Ln}{hjx}wuy14*?-PZz-D|1K!S2lk2FLeZFCEas=TD#VgB5g#BV1bnjXMrC+wt0qEz<DI9B5l?#NTgdck)$j@&;D@0<3~s>5Tb9svMpecLWbEl>=-fHI>cm8wkS<B1H_M{wEIVDGq(SQ+Zk2q-7(xRQ!gOK?tD<?Vv!zfk>8hP@*zp~I2KsR91O3F{2N9}3Sxt;yoYiC8pz{k83YW6YjpG74+~k+jf(rzp#MtOW1|e}rPwiyNV06-1kbI5<n=0Yg3a|W33wI%zZ2H58&N!E!GT7gkV}ZoW;kYFWG=)aLGaf$loUx_q&D245yL&@mI+0xtKj<MRTbWW@Y#nT*-~r7A#%QvUkz+y72mHe)L_&-gHftDJ)QGK%>s96T>S_?m?G8rmo-%|ep2;!)C<MwxB07`!Iaf0<YDW56z8FT{s@5n-bBK}C5HXz*oJ(Y;kDadE0wLIEc;LlbOVG2(mD|$!#(p-gboMV$G$3tPn6_cH{KcvXM5WsAk|!gFFG<O6)V@r2Zu5rDuhy$*QN&v=7(Yk9V21&bo-#;EE!96N1<Rg7sd?N&#Dg_Tig*U!4~v)~051|UO-k!1ct{s76n2(>yj1i*q3A~*t|xCH!6kforU+)dL$i#%7|Md?l(XG#d>7clD8`|M4nq?L?uA1YK(N+OJ#O=oLc9zwl<=FBR!3>3qSiG)3AdYWb+pswAua0bSRGU`rKXO_Ahz}5e5us8cR&g?=-C@~+~J)1V_yBSAbu<jy1l-5zDCax8=Jm1FB{(*P7vs#ED#vLzX`?U!&g^JCA{w_|FI67gB@oKEWgwPl{wz%f^5|3PFUtp#%3pqFUsOO6Z8#^e_o=`&~>X(t~Kd$6_04OxJZAL(Rd*1LJx?m&KU1Dl}f-p+(Wc^#r0dpWOV8u<?MVZb+s6Kb_siSjnN1ZD7si`d@6fzl#3V()dRs1o6)F=(I^izPv-wX_YjrV-6B~C5y{kxbEQ&Ij~uCH-!uJ=&rUz*n0;XmnP}2l8UMFL!DiJ9ZK(-J`4eZc#@v}-rWtD9(B0F>Fn=J=*O#X+W|Q}4my;P1--J1h@VNL7A`Kb6`m4yn9vKnk#r+sBK60L5HCH>A2tvnly5-Zqoc`14|B1J73yGaqUM+97Dk2Vl#7Ud8y4<%m%C+k4N;B&WKtHetZV!+9XN&U|=pKXN!I6cV!#bQx8vkXhqI=mm8fHe_Lf|o$+dh`mci%bEOZ$=->!m}Mh{Vr`MOH`A)N%)w98*vlKR{Vsg#ID3mtH~!e<;tX@Po&$A=~X}vwOCOhnh~uxBAYZ)7MKmJ@T8R&Wx6lO&v@IZpR(LZl$^H42ON-VZ(j@V2Cjdxncf!^0!FeFVr%w$=HW&ZMzs{Q3O6~a<ZLT%OZ)FOldkP4i}yPA6M&PmY(%G=%qold(j#V_X5obzM)>ARHNU;v5wo~V;tl7BrmB>JD<fCoxZz^o~jC%`$J%=q&nD1g4gMo#6u&a4lzg$KX#J3Z|0OacRD1!N0yG?v-Rv)4^oSe5Kgi5<O@_=A}`oo+R!lxiGET5p<uFuL+o@i&hrp~hWnWip$n>R?}()Z3!n|aum?#{bR8%<cDiPaMI_CGqTnygC@n8ZsYiTIR6KuiL<{<AwYgHqi@Dau)_SWdvM8p`KRSjqC~!moY4F{)<L5fW!&LtD3(0_R`7(v^Y{13=(^fe|d7uUI0pQIppc;EU>7G`9zqL_Eq01)gAz8lbw>8xg&c55D@o?DFhCQ|60I00NAWnv_!n9+c6&^ZcjQ#`ioHK&1kX)l=+MnRrjm&hTa0|76i!AY!VXgWB2yHnX#gPiiF({ub;q#*M41ZK6pVAcQJ?mSjUc=f%g!zadAB)H?<4x(22G=WhE9<eWV&%29Mr931)v{KrOFCFVUmc}ltFF|5wOgq-Rw*?_yk8mjiW0{``PTY<p1>*wiNsW<VLOyDOh1XETT;x<lAWf&J+I)@R*m`is?ztp>S_g0ecBPOG7cC-$b#a8=OXkmAKNJu-1xlRE>B}qHlETHU+wdMGL+Iqo@|8EB^hHv80dhKEJoNbZhToTn=W90WSl#l%ApuRyfj!e{X#TS>6UqFt*+*Y1VV>vZ*A3rT&gMJ3@vpDjTGn0ALp)pw0!`Zi*y9%N8ww9COOa|bXW|Y{Tq-<T9MtUWCf%^DEYX=pdfNPjp&C-ubJ%ULRz1j8do+gTegs2qC0mxRKY{lCEvahnP$|fH`+1hVTh~l{r>Qwc`WvUqRzt4BL1hUwc)!|E~RR?vFZT}ioIuD^djw&S{|#-if-(ss#IU3Vb<|eS{}})y<9lA&9~N>V__NNsH92|v2O`-PM5lHN!w}u4IFmKC>p4X#G1<E#iHo>Kl+W6=P!DyiTfsz7`cPd#D@(!N@!D=L{92e=q<zaEzWIOtqOH2C`oVZjz1xzhJF$H2$C&`98^2ud=o~KWO4u`OR?L9HoF8TZD-t_DuX<Oo=kMEqyY`5tbvc5t^p4q=HSVjR$Q1VE@?SrEeDw9#COJ{G2UcE>vSkM4S@%a4{57-(I|dMHv<1bPjl$*9rcNM%RuzDCw<@09`1qqq74WABaFG{qQiSv!?UkLd*DkS84WK*jYS~BU)HNNuMo|`V7mrQ+!;&`Kzj3?IJy{fiO{a8X8Dns?%|MP#8mtyQjM)?n3TmV!zD%|GW#f9%*?In!bUqvSZL|a80MWiIp@kGE~)dV^gn8fl}4l9c<cEUg9+ib$f5I~yz&?ZfrYo|FcpofDqEtfrE*Ep8mkVU+ekE>ydOze5+_lBf2lO8hwE9=j=jl2?r{6TC5FL+OTgdAH-?t_!KKg+tw20~!uR2h(^`&rgsC1p5z?l%ruv6+WKwfmu+jFFEKxWrlSoM=Hkfuw;se|5aw{$+NC|?WViRsxdb_Bn9ZA<qsb$6r8oIpP!^bYG#W~Vx!X#BmM?k14QjQ7_Kp7uH3|NR;c;6s_m_5YYU1u(g+{!L{>X`B;v+2vU98O)rh6;)w(RJj(hJsa?T1^@C+e>GG&uV1E5eHQ<|IKm(v$Upw0DU3l)&b1P1VA`%W+fP#JWNTesP>YwqmDA1_(0o%-5_b#r^$pDnQn}ZDSo`!^UsitD6N55iz;Znm^`gj36`aat9$4=57lkUYTLx$lO7y`(Wc*I4rG#TNgvy$@=j6d;CJqLtiEDq5$G|r-^OXqVtKre+#lFERiA)u4VrP<o)Ht?ymP-vY4noj@0g$wm(ks>8KEc3BU1%EDNM<uFtH;?DSRk(M5!Ff?Iy(~q7HI}P<l#@!6a`#S!50`*`aM*%5uQN_(>F@3;Mcr|Fon{SG)(YEOf2TJ_^M{dv`E#E+m}v%+ZGh;naE?<fhyxzdRV5qh3Ldq5uLA4R3kSAq0nStQ*tSe=qIen3ZlZe6sh(D=ka;Hd5;Ctir-F48t|kU`yME#m!h>WT}hyZ$?L*|JiURptHk$K@^)ScPblY+%J^iJh*7^bD_~-oQ6b$P8@|&GB1igqp*-^XLPhOiZ~j>F$^gehvG~Oi1~{Nik;HW2<^tNwknvzw^d#(x60NO+3@Gyq3NJZf?Rjx^o3zc2#K9gN<2B2dPc;u)FP^f|K5>@!ALk?h-an)mw{M7<t(?Ijip&IZiA)(QfWf`NbO7&K;ok_i<#0>PbZ-ueZD8aJa=6W6gyHXkw-3TDS}1Np@s<spp0Bkl&B7DA8^<+kI`5c8(CDiN{&2~Bck#}Y7(o|If(p5>8u?nyUBTg3h+1p)D05&afXwENJXa}0YgcVWeVh9bQ*Ce3AiOS4r|9AYo^vicA|6Gsz^@DM0QWgixdgR5s5UE-g6I>6X^Q@wBYfP)prkE-_-d!=<3$W`>VGw={ClyZlCB6F6Ea~>iT=MTD&l7&DI^fAcHAUqjzc0Nf);XR<neAegqQ&ZN-pGU^}qOA|{bP$-zYIQhH<~h3g+lyM6q;B0Tp&Ur0Xqu_7-I+}&~F*+LToyx4Ka{vM`y>^e3+v}4I3JM3|J;ZeqI6bO^;PTLg!7Hp4bkB$)$&^sDKBP7Ek5G3&rz!K?GPQpvtu}eD0;?N%w<X}`K@gR>}30@^#G4NP?^pQTtpBa)_kN^bnhZb3R+)hrsH3TIBQN$s?hbSzzlaokoD5cmrE%U+<r~`-=K!yQPJO8)w*RXE7cpLAiAVR?A=5E|58uP{ZtNAviP|96oD(OuMT*0iw@ixNM7nQ?NidukE5re@9R>f0h+nuoEI0zNn5oAgM;lAUHM6k=`r0_lj%v<yjUv)a{2kgbI{xc((5kvQ2OU7dBbzQo&=O3A3j0Li2?Hrvu@%(l${5FW?0~aV-t~(x%2DSi4s-hZnSOEEcE?>aQn&Pjaqpf45Uc0?sU1{M32uxAbx^LaDlpApyRY(P}Pb*k4@BD#B@uh(G4gNDdh2Spi`Z@H_RTiLa4o09a4AKY@3ZU$|ae?lEYz5bj31IP?I5^#=DTLZ|>=)q9!zS;$c=0R47G7116F%m-_d$2>x_$|7q*2t*u7(C27(0c@7aLV$ha$4-sEEeLv5r3zyw=E_O0`#SRM#4O8Y>m+Wzf`z<1T|L+F&f%urnAQAidpA-vR1sl#)S{gmIitT>Eq#V0*4$%oXuJ{!=WZ-B|0uw8%6c8{#Zc1R7QQa;)`aE4#yC*W{^6BB}jt*tYUcX_gT!^&%Cv$1?1@F!EBmCe5xpecv{7-1RnAYtIDT@^9E-J6<kbKle~b9r$LB+Ibhz{J?(X9zf4?&dicgEEJb6tBrG8`FsJB|D`n=LxE6FqB*+*Ishus4?~Z^6;#9MiwxV<0E8%a?C~*n2r5*OK$4_5%+}JV<Ljj!CN9c_6V0Q_)O9(Rp(Tv&T!<i0;IFxSmf1|y=1qzE8j@hqLIh4hL#j{_0w-?Hj)5&7BP+3a8371V=F>K8uLCeAV>MA@ga0C_X(t(!I;ezf0xGqs7AUw#GK=kHI?<x(o8(A<B@5AEVs7CZMWP$@RgC$jxlmNmo-T|j&Cy^ZqYOVL9M=GiOl*8Rf?w@rLLc!Zj%Do>`AgxdBWG+Oxd5t9RHJw~n=+u50_8mnm<j(<+8zwTgLx)U=M&fMW0aKKyn17s4#u^Idq8w~ys^{_thI*VZ<QNspzSt68wTdM(hTqX>&G;n6C*H?Vs|`W^O*iQ!L;)OAgyR-0Qpc(ldmyp1M^s2Bq$v$ruY(F+)`no7g0h}?)^=HY@DB7aJsRahK&TP(ip&laLe;GBoH#-t;bdtO~Dm!mY@@?)lApvVag6N2_d@-PC1gY_YQ`~w?U1Z0RJ5OHjtwun42DY;8-h-I%uQF)j3>rxq*p@;|+;LJA?-Cth}wn7m5}|&tV~~o+E2Sg}`<@Au7c*b7`>1=|Mlb5j<uDhR{#{Od)gLgt=JGzS2OFE9rD-i1pE%;kC#n^!FU1fucHb>#7zZ1skXkc&<Y+jG9ei5qy}@DfT}9%#eFpfUL0SgYNJly8uXcE))t-!mO?>;}~b?WPXGa8HzuhJ*Ttp;qODl8qetit^f^Hn?{dG%yz9Ih#9d!mI{%tZ|}f1C}>TtJRFE9mChp@X5n}kXSXvv0D@re0*>Jsk|I3`e8)tG*1c+NwSKQ@9{VK{^6-cRW44jGVY5-)z*~B1Nxi-8qc<K3Lv#R1CArCF<SvNlW~IDdL434=P$H3YkarM7S{DNeqvaYU5DW(#ieXk~&mN$Z2DCs6JbmOzqKisDN5339c3*8*aY(fT$&>&}$4kC89J{;jz~nD8dq!}O<{&?`aZy45s)0Q>>zm?&ff-2m>B9ZFn@1Hlld7HClcxOmDf$R498KB4Z+N(<rC@|~gik*T6woBIMGD<j=cTM4-HnwOyHF1k{j%c$w;8_zio>NvgT83IDYEs>k+l~Iv!Pz`+zXKOPQ6k6b-k9IpB>YV(jZJTZB|omcS<=eh7Spy%K0#d|K^QUZY9O{?0yDx?Njz9#L<k10NMBBNhqI;$H;S4tio}tzq5xQF@UtEz=gNikLhFvXS$cacwY<~F)xsdVIgzezk&H}l?}2td}!+KVBo+Glpa&2J{&kGufRWO72LB&7?nROBm!XEHV+VFjAWU2v<3<3nqznNG>SWl2lmd$M!BJXMA2G23@MI$f8sgXu&3?K2~C&rc42mJsgQxw#sIMzN0G5l<^M8?aHudb%IhUT6vS^P9HjXOj1{xscYFCv=OaCE^01gsag6~qf$5I$$f{An7(C4u{IhDc?$udv5$q=0ttmY8s^4l>8h7!;9*?|l%HxxbY7=il>*tP**oGy%F)d^t&?r)*P%7N4G<aK{M;1iiSR2rKYn`hDty!yrmQ`C7R4!um&E|~}Z7s7{ENOe@BI9Pu{wh`!cidd)UDn3m0VqMM9~>pnO9lrM@&WgN5Ws5?K>qn0OWzzj$^n(af>~gi3?wCEFAxCZXgu_Xogv;9LFVTA{UbvNH@sayl=Al6p06ieA1-<hl`(7?DixjBvS9`kf<jmT=#YHstc-h+6DT6R=YdFSRCNJ@KxX4jL$_JSF0^u_?U{2@&@5@kDa`sLc6|1u!jjEh>)4&Ez0TW)CHr@Zo%cEtckYFfnpNB=6$w;RGrgRBPRUI<c??FrR6iLSh3&(PjDO%TX9nlQ5P1s2pncHK*ChLYBXhUX+xU4Nj`C-YXLh~Vo42nPe`-9lB?@D<DB_tlw{HEiveKFf?kosj!Z4~CmHO5CEZs`dR_;{Lr|*Jynx|h&PJ{*wma`PkOb|wq;|RH&2qi6_55-0e!V*Rkmj4@OAQx$YzRXC7s)XkYZ(D1*rTM%jvS28|u4T-_SfEvbi_at^u;nKJhi9AtL)I@OBEexYElTel%>YTN2jyv`fHPwmcOSD_Zt@$f6#i7Mqpxn%t;z?$1#9JX>sMuRJ3w1@^n<(;5PvU%kTxUg6cwr8zFl3Z;yp8t*|=3PgQ(d+UH1mVg6GTFq~7dK_h6_eht()!c7hgNn>z+gdu>iP=J}t662>Ahu9h%}fN{Nq;SP)&{B7<Ae_W7dMd1bUeG%U;>c+Jq)jr9B$7-cnU8^N;ZL`9wmplVqV%*(9cz<d)ylp^Pq-PsL51E>F=-OH^@x%!>Ikw1q&FWz?$v|{D%1}J(p}h|$=!uUkC3$^Ms^l0Os^6ywIUeRAvHQc_30@G8<^Y8=dd`6j9I69)Febynq&TPrCohVi@aGx3$Q)9ZY5L`5kw<pj9qg|R`#4Hqh6-Zl&rd7sO-2|Ls1UpsDg5@vPS1<rp=<!-cY+{Dm+4@?a`HZSFl^r+d-?S6Ztxu#xSgeNb<M8Z@pIflrY6^A-zFTIE>h9aFs0y2`?xR}i1e!6Z7$l*XWnzRVEUsznI_2?H|{6p?r0>ZjNKRpbDCQ+t`_qc4vPd8zRE{pSX>Ka>SD&0FTc$13q8l_=Eeozc@z(9Ghuf`E=CZ@KgKs>r_ckL0#8N9OdZ|>^058CJb2(ec<|uTyEofc-pjpLdhnp|kvq!kMhsq}uYBUlxHswh31yt{*6C#6ZJM7ZtgbU2nB$&43u+DwL~Q)W!9DgoQ7}uqKZbu#tit8@c)%$cL!_V6wi2^uLb0-(f(IMDvzHUS8i(vlnI2N?Fdsw%11uHyjtWOMp0#%R!wLE+_<B1{DZCBP#)PM6naAW<9!3;nsuEBLE%15nqaV)^TZh@IC|I76);8T>#Vh0P0kmy%_AS1IL&>l-j<;Fji2krnqu~I0Ihc5UK-@IfFJ$Jy6!6Q~lw`ui{0$;lwi-o)Lb&C_NSxtah-fj;%`sXbVhDSv)7py#o~2t1Xedf)JD(b0NDa>=hDXgpi@z{|KV{nz4i3x^E5^EtXb&EAuUNul4Gm5Y!**~`3eZm0*e)F$@a<xt)K?;Hvmp$A<ge(KekG^@6_8}*OYJb5p;~Wl!9rCcxgYd=J9RzfTJm<?hO%wS>?qzF`_7Tsw-0u@wuXg5NF3(oivWS+t0zDbygYkyfc6;OM~QMmN7ggaLC`2JKz7kMBPWPpqC_4vgg6}@O)>%kCYMZ@G)I%v=zs9~0=r#cE6W%&o<AeDvy?y>d7&-Jd7wunovjA^S8Fx4nkhjzKcWz(k%+oI9VIpk_8HJByKm7_6VXF%;G0w>HA-tN=cU_qQVWMWtKdIM)3mu#B!DmZonwpPYoz)3g4_|2TjXR=Qd6J!y_xHJ{uj7PX;J|9AmWy#-*Si!B9(`Xv!mG##%6JOFCGFppPub1BRXg-JgAaXhN3iHFXq({j=<&5XY<LJhByl)s9|>s)uM=olOEj``|$w{1$d#MDgGc4!|V9UlsK&^5ex_8cTF<ZME3y&GUONvVF<(cX8Cn>(`r_KUEz?1=S4Eaxs3lNw=_4e%>P|E*b9Vf6eITbK>zO-x~{ixO)&c)`5@;Dy|LrmWV=(fX02A6?_0MUmB{!nct;adhnW6@jWx@+E3NxhvsK?bZ>oDT*A>oexz{~6!0tGV)G1~OBm^aeg6mo0k+3fxUjsZHhjqvr5$!w4r)Zagnt+6WM#Pi=H%o#VWTuHzGm}o))3I=d%KD|uj0YB<+DdulP9<xCO1d8r%u>uwT>g&7ucl2byZ*I1x*hh5d-<bMW~|JQK)Uw)kI*SuMo3KC2rG@1G^n|Q-Vg*1mk5H3<lv(MzGF8&xxrn8M%Nt+0`3-Mic=Z;RY_zL1ba`FC$KwmB+4mr!=uuppvCOu>QpylAEw$z!gGtz65w>y+uuw<Zr2+d<rb}^xV<s$x^x)rrv*~8qU(b+g9&WKu{54hYXCCnStdr%INrmurfXL&u+??YpP=E{mFC^x{-$vJ2#$xubuwBNGE2w-k%!`xN$^lS3)G4c9<)$Iz3${-<mI-*<xR(m?b3X)-8LBiEZgh2u5htE$)X=DycIK%7|ATHZ$6XUZ&WsJRT@n?Y!TWZUW1PoY}M-+ukF4pordx>r|u(zK@g2&z+)&)4V=#P2u5`E|J{rMZ`C%-ba6VHLj^^6xYHR`*NNv#NR{raW0+j2xX?YxNV^c*R@iJr5Th5-=mx(q<CC7k4=MJ)lM(!XO0}thn?o8|AW?_ML$uKwcZ_66=dS0M1S>{Y2^{!v=QlYl44=>yXcvTKRo1zXJc1!|qp&d(u!*m-SlnCpTb1T{tT_^**hvjNoXPOsB7AQI4*=)mH(jagWx5N1pCk8E(_0xh4qa6jH-COqltB^bX)3yMKT0G*7~?eQ?5#KhucTg5R+LhOD=dRA{;8=Dj3B;bcm)(kwOv|RY6D4Jo-P{W{q3HP&2eDgQi--P{3G|Vo<)ve%b3aAo`HdcE!%o<mL3n{73j)SIfds?mTb{l0LpOMN;n>zwh_4_S|apG+?Sr(VQ8!=q3oxJvJ?g3QNv%$cQPjoqV2S{Ww@WxF^Bsp>GJw%NJW{r8l3Z^L?OVj*^`rth#un3y}&P8qIj5bw<`jRXK(Llc;js{TeM}P$-Df@7NH(xz>i2gky7MWbc7d~nLI<vyk;y<R`4I=ZB}v0dI=5|)S}_4fzw4ZnH!gHCW~s(P*GF54K_5&B}8FjF@pkUT6EeFG(nuH>L>Z1K}J2F5{WR2Q}=Fcg{A-_vU*uF5W35ixKp?g@5JZSz#GElqEf@~7a<ggK(#Zk245sh1D>szspl=r7ln9H0c&vbV%0Y*H6rf=YR~GSYc-<;{!SHkDOviCYve@R!r~YeGN&BJB?D&#nsN{XiHSP&GBZaW%_+@cC?xfmTss!yInlF8hbR4mnRwtFe;&R{fQXX^32(G$Qp6AuWS2J3IV^L!s5mgEn1N|5IJIr@qb-(bs%?H?n#B?ZwCWM`15gEEr3-99=C!!KC&JbMava#=#+#lV>Pr1zG~MC!kn_@ih7wvP7QLh$4^HH$DCc1`qu=w>*I5I(aPYF98!rS4q%c3?-v2rFu|P3DFJjsJVtT|fHF8-`uLyZSOJ8wV)ZcLaZtXbHAwoP1kaiuq%UJIVgZIafGATs_6itmllmX123`7e1Mj(>{m2zG=83YcW7mWWFCwrKF6E?%YMzNRkD+HL2pGq9uB8@mkh+sm<p-A6xUy_99h5$G2O%x6kuMz?M=-DQ}RL(W=O2^o9y1CSaO*ewc#zg0N)5NYR7+b4V*&t~kpKvmX%3++u$-}%&6f7TK>74$tV<Xd)$8Yg=@bK3H)?m@GvBi<eEO8A2<uj3c@TWSV#NiQLiSwYvo|F;sLY@;OBCDwAp$hN$o94o~5!wRk5GVTPab~-C6^_@=!z_SwpBhzILZX5<>brjK!KG5^!KM86e4CNQi^t<5POV0qesD?E9ieU><H1pEFDQe~;_TUOV;E8u^M&6rMm4UqM6Qz{itt<H_aT|bR+m_vb-LS*Whe6f$Z-3Xohn?Wj5ekTg-@Hf9~!l$W@?EvK00M7`50K~_vj?}Sp4M7D)94-{8Bi^>}e;MuZ4X6o;=2E?VQ-<F?ov06v;2nen-s!GyaM^AwLoA@(Xk=eGJV%KmAtih%>D`qkmtJ>*Oo)S_JwP`$(<6z@dH3KB=<$8<D9QQx4-mo<8OTE$H?5`0O)DBTrxBT&|yxZ|t|&)nh2m9nhz_r4wl3v$HSsL}DVH9cmwkezlKJU!vzOKm}+1@nezF=1nM+$g#X62VM3YeS&S!NQJ*9!r`0X6**esP@g@fR&Wg><^D?4o$^+bcjxaOpZ!-TgQ>2mceO107(b!BI+T(Iq5nN*M22#o1>^lrPh?EeDR&QZ65*fx`0NXs$oIVb;>Xz^A5(7aPtjeKp^J&b5i?$Kh#>Nmef?OZ()s;k7&%L9&oBo$zzr8u_>~d@v22I4`(EI~%g4~-D;UscjFGtbk5zYO2Jv_J7xC>Gjr=J=^K*KBjm3YUltF+pFR>w5o=*`q^hEwAUG<25OvC;}qWd#iAj<B83;r1t#iT>e#1F<Y`bzWvCel=hl;lp02v<B&H`7Z3_-j3pb4nMZU<c!`7<D;o`)hS(&v?_pP<}vRfH&X|j2B^f!b(IhsPr+REAFNjj|H~<0oRX4cl{*yUI|8o_ecTA5#Q%LF3MB#YkoyL61IgP6*P>GY4g!)zQjyTxJ#1XVen63KHmZ!@vebE{T6-NiL&q*(JlGxQ`H<QuN_;Qf+Ur`ZRhld_AaKV(W9hIY;}OVBz-X=00WGX#t1FErjkM)tN<4@8I%M8B&RzEqyF%SEsOfu?<xJ$_v~BED<Y~flm(f5tPB->v;UyJpzV-<9JKZ3N4KC*#j`)k;m3({te^Rm67uo)g0{Y5W}wi0+4yTjVO(Q^%?<{@2Z0vX=N0Y?*+YWlr>sx1%v3facItm&GASE*g*zw0P2`tEE-%R)DgBq&O6uK>fO{lr(p=raq2u`i50WhNrN|BTsk~1hA@-E0#W$>%LObN*j4AY}YvN#d+klEMo``)+#3ebcrQ~<acE;3zaZ*k~j)glNfplTveH*vKyGdYpuo}opVBDWEuN99Hl)<70lfVD~bnYQ8oFkQdtx^UihO}nDHPgNj+}{u#{~?5yLP%9Y`zBy|Pmqu$ONZ!%f06xT=C@?O;bZFLPG9Kxl!S&kUF3SOJb&2P$9aDvX(cdAK|dacFK?WvbeI?NIT88icrPSasCkxShy}+9xBNx`m;J}`C;|wd%E>G+1yMyDY&Ig`<R4Rw=e%rKYd%sdRya!>638lR{SdIGH_^F8nv2ZC?sS-L5F7vO?Dw!>FMy4F5t%xRXs@BnH%O_G3?{$R>H)<{e4!5U(A4!^#`>RCW=j*z5>jHCtd?B<O!+T?jSGy8j@|e7XhxY5S*i#Y6Vue6zt57o;zD3jj8i2yDjf0_W}!;f&YwXj#TEc#BwOUiKj4g?(>*IB7=NXXMS4)cve>;jfxtP!ypM@|(T?E8nf&3(JY}Zw2iyP(Va(G|L%J3bMins?TL@?yk-Rq~btKnyiPja=@UiQIifi{Rd+gYfQon^f7b|}KBq<{@8^Y8Y*#3nCK;}Up<bAUZt^KJTlYXM5gU5i=tMViDE~1*cha6xtDmJDFj)9E14XgK}ohl?tPWB01f=m+fMFMenZtRUXh!eBoP(Pq`dPaSJBQJHzwqqg~V+YCP0@cIjDbC46@W>d?IY*&La>}n0EMDb)f+Yj?>|zGG0i7K6WzSJnN6~H~x=)I#qbM7{B^>%x2z{8y$l9^{_MjtW018d>JK81!Ta*{ih!l~c@HI`AHJ*s@B^4cI0I-6Jz7XVsqiFgSQ-{_Aa9lqjl7uoSAt3I%7eY39hI;FlbRCn#&`*iNy;S(WFfx!-7!mN3W#STzQkw`aQ6ptoh_{m6GM)ThUJVTx>3p=590JP`(FWd-1cB)GJIRBUdB2o6c3nk|5QL2i-mA50_1Z?I+}vtZHY&B2sgt<>DIq=LG?FjL22e6p+Px<BGLi4VzVm|hQKD=wY0bZ4tbQRbp(?kdc?vIxl#3THiIaRn2=QC>zOsI9;?2P8CdVf((h6tL@iJ^hKask}_4^LpahdGo#`=Q?-W9-LO-G0I9Jy>3I=1KZhW&0XpO*o0mnQ@Aa-bHK0;&G_)CKyUh4(^I0dG*jATuZXx>Cxyu#BGQfhH=CHRC#Tye&%!H<6m@Q#NRtsBhEJbvWwSj!PeJxzVWKvs&d3Fn%}`-df=(5{>!?_s{Q4rxfCp%1bPrbiyrVy3VTCZT=@wKlF6itlsA1V1}Pn-l%USmQ+kkoz&SE#Mpqli1^%6!;BG+$5Gl4WT(~v^V5d1g54_vijCwQO&&$@W%W0F;hiLSN{yCFv*ezAML`KP95ip-(^uNfxuv|W9XK|+8G2f?yk2Q?dew)+@xC+m3U7giu*yGV4C}G-<r~swSVNXyk@@|VB`$fgmiqNuW(S}Ph2MiUO|F;BkE@tJXo}X|`uf%eyU|xRHrMO-Q}*yXTA9P$;AsX1m&EY&ET~H8pzAwbYn=GQ0}z~{ME^+J8+s!b)6cqt-q>a>5WrhV9$<8b2Zp%PnMNgq3s&hIM3k38qo|E->*@J|1FF;9y47r9mv=D}Ii?awqvWl6i>`ZYU~<Z>O4B^n!xZymW1+<mKj=&FAfqBsSuz&AfNzwY7ehSv`%=^R1IEG&e>^$+O8xR&HAvDPY2fV>7*nOjd2@5**?H)8F*AgSyB-)nx~SL)y~ej6$#~HsV|Y-y>pa4nPY0qDCinpnA7i$kn`_(ayB#MNu(hsxknj)1gHNWSf+-{@saPJ)NU>6GRo3c_`&M(avchg(uwpCq_4V>*GfomHfjG9A2Ag#Jq2f4BzHy4Kf$(E{zpe63$yeo&<ET)Y>Uq!6R*6*xu4~)^4Ko;_rC-@kMO20a6~^pOv3XfC0kLQl-3)IvqZ>Jqa4bb6p{Nh3Sw>I)DNf3fGAYCykygV99RY^{%>5i|{@=_RPM_%c?P5E;BNOcLm2!>a?tH-XKH{W(k-<{-QS9p9r9}-0J;fDy60?3k!PhafJinCI#V4d&k=lWkzN0|{k(WmO_^9ud*D4nt@IoXtY8F_fg@n2%1HSO7Ky!;6Phd}p59m&%yxzL=HXsbouW>|E`6;Qk3GGgXeMvnNJ$(XPn9TQD)E>?az6xa)ZJIG%+VM`G<T0jDWTDfJIL`j4#Z_PK@Fl%?L%RLvvP;yqzCcOo>{D%SNsECz9V1j$8e0QqtudaOT#-m(Y^IxOo9|eX?~*=0UtClvcS*mYpWt|sV<1}51ga24Pm%T}S~!StrMK)yBC?&KA<E-*D)fyqnD4`cIq17R9uFT9jn!!2+U~@r*jlOn0MW==k%0D;RD5!fON(nMVqNDAN-Q0=Cp@WFwp#Vux2;=%^PG5PFHa6Z7#$Y>J33+?j-#Wjz?rgx{LJ1A`M=(-lv|+3Sesk70CFOae0fb5-XLiKD%CKmly6-4d(j;voGRVcPYyZkd^uJRo~d4Xb39P_7QMo2azbh(i=coAR20mt+~5dBu&_N?IiKzW;0xiW^Q*F29Pne6B3e<MW@R0e!pbT>2VcU#{%cFXlgewgdJ|(aQw1m$eiuNftXJ2nx2o&a)_r^%GQVW%z4bDz79larwA`%MO!Bt^QH`#CWC<f5A`?Fh#E1u%^aq!&@Vnr454|+tFXr^QLcd~-P{i(BGEE2R)n0HFmtviZFeIRzKf*YJLP$6?^lFV_f=Mr^6%@B?8Nn4`2>O-7Fm43=G;mm#^!ye4kv84tuhwNWJcc;@u6$4dM2qBH0O$cVfN$xceWNq?KI~iL$sh;+8hFp~z%z+uUk<Dlrmh=!uL%F8e4+R-Vn17P7crhDalGr9bbpas6j}3@{i2CC@e2n-fZ1^1cH$<AbZ27`yhcsMgrmh?^2>PD#pDu&p5u4+VDNmIRt%Bd%S4H8FYRW33LH~!UPw7tr)ySpT?L=`!on`Sfq&vD_YB?X+au5ES_dNLW-bne8NZmnj8QAGDOeDB1tS&)0X`MzcD94YpFQW1xQdhN9xoLlfX|OjeMbqSq0`VHm3z<4=XUo-*5h5*-<#}MpoLlaZF}aU;>-=JjmE0&MrE~(grg1U@?lyamM$GJn?DV-L~TS)y$j^A!&@GnCt4cQH@sR~L~aVBgtua@lw?`He(Qz6n@s3D>EnH090d$@n?`dOjwOt9gecs8EBrV~$6iSlB-%2F^ULHRj0Hk)Y5EoA0`l>GiQ_Kq8#E%WD4j(~WBP?j5IRY_gq-bJAG-t@vyjrA9}F#@v*X00(=pFUznM-?`9dS<oAPl0jS274;SPF`TWX=`o3JdBzpR0SvItN>uToT*f6U;Q6FFqF^ku}4q?)|FKV9t&l-y1y*76W@-5o1eL4|{C^3tgEdVaf`rD%PQycjuSci2t6eHzVCpd@RP%?1hOWQMuOa*5X%tOdmB;OCx;fq;&J-xMd1q6kOc_(PEutM42-eev9NQR%1TDzWr>(Uf-dXvIb)1}%QcW`iNy)T4-jh$QYn_~}qSESPjRVws8lio!^JC3?l6rwmw2Tqrn6-67i@Ey!vIxDI&Ik+KV-My7;th=-9M_HzIa+r#*OiQszNe(wPY#rHvqY_@WZ%I11`rIPr!1=>N5N*q&jCxsYpXyvAMx7=8{Q*PujEWM^17@x|}I(y^X+|^=U{DgVKp91VEsIXcG5x$aFg}PX*s@?I(3bZf82#Z%Exr2AjKzHUX+T+AA11OpfR#vPr)FvzaYm0iDOva!$%9~dAt&OeqR#hBwFrC@n<zi_zP*EatP0-tGfVeJQ#SMyJl^SykgMUN_?lMrRx6g8nbj3N6CLJt-$aDOoE}wI{Mdf07C!lYUD+uwJi9lgYK(m!P>f8}@vJ1yThL2OE{7D?AfL&c8IvXpgG6B{VVt_P77XXL+^3mAT1AMm&`W4wfO%8;3?6z8KRn{tve9Ul36v6@lmHKx~P*7k`JC8<|i1e0Ys^1x6ZW=(-j_u)kfo3CniO?F<Y1CQbl@pw4GL-ORVUZ@2jr3d*{tus_TZ#Zpr1=OZZ8f$ki3mI(Blb{v!3?RWHfrar453PFW$;I|MB?7mIjs`DO>LD9UXO9LPof|2RiZnxQ08C{x2m=Hoc5#LP(K4pVI8jHo!8*;(fCAEr0DDCm5077+PyoK8r9TpRa*Bd_#@q>00XS8CO!(wLPCygVYAa_`1mnM-H#tL@y70zk6NX+N)s779lQvNIfD|DfpO)686UVjX9EVLytN<E+j!O6jrxWbNuT>3=-}Ef>(!bTBq`E%wEFZZm^)#<p%fE7feVHL#p2m7;-z5`8Wk%1u9*d4ZZEdglkz*3y3ppo7{c?`e=ncf&(ytGZECf8ORH_Iug5E|HR@ZN+O7NAoH1`KXm`qYtF^T_b`revH_hW9@LILDiiWj%1GrzUx>DUNuUA``0IQt<L%p$DX<!dA3J}}6(GANKCzUI!W{qvs_&BVXysae87~7$dBO;PZJ4X=FU7r$t7)_#<q8Q+XXtsCS7d#PDyG}nA!^5QL=}}o2d}EwtxNoo!XvPoeBJ-nhrMia<_e+s;EN(4P6tho+XK8^(k~{bGF?n8@c^;Xca=5LWk;Tr#_=s-GsbR8(ty;5%=|$vmV|i_@QCTC4aHiR@XJe#inAJlTj(;|(YZ)PGCKS!o_<2Gg!*wH}49${1n?x4cGleaeV{gQHKSpyIfh;7$@u>0{(vs@!R57g^m&;JB*Z=z$pR-6G&?x|fNJQo43m^5|em^&##{}yfTg>Q2<2?{!XO8d+KT;fWeiAoulX52X<U3=OYKU_FB#{%QX1;g6%&DhJiDDo*F_8b?$cYKNxBc(}G9rrBNdYe4+R2;riU@ujmB3F_0UV#>Ox-w^sx6m=U&o9VNiKas2_!|Th|46I6TN_Ra>2=7K!6bv%#RS$0h6-^_twI>`JQ$hZnhICEs38~*gGH-@i2o9WWpvg*)->W2-!3)Jo81?oJrUS+%nS@>D~Tt$L?#1bE;^m)MG4*32AVSrYH6WPlv+zv6>s_FUj1h;<!c;Z$h(q;4j*~lG_e^L&@L}^dXH533ADnp6%%*IE+_0#zcY?R+6c2Jl}-Hzz<o3=pj$Ot!|HHh*T0d8hWN4il6$GpbgB32}3ftbB@kK-CvnQZER|*-sAuiI9ZC?dav5Lqa7l0=8gg8ppFP4bu5RUqzReYS$y-Iqay&_nI?RP>RCV@R9!6IS*l((uzZYIny#2#d?G)u$|>kZZ2CWas?(Iv|AC9vN`+v2ML6vmv6n;b81Oxn>_txnQuT$D09iy5E%*<qL{I0E{@6Xp9g8iv?Y2)~w*V8%DSPWfKJ*TQVhI0@xF1{|c5q&wB8bs8%sQz^30WLk#B>6(6e*R7P@C!t)hv~ggOqNAkXy!_foE&1oWfFuJ~LuA(Ht{Va&;=Bq+XY^J4vW(98-6NLa)WUBmYn$O1dCEjI5XQj32N>3grUO%Nl*UHk;BIrQVdPaL~LFGLJ0%m(Kf%CkjoN0Y&2p5C19LX)+rp=WBnPc@VpGx<V+?uLKZ|uLrBNk!c0J{rG9Ilgj!<=jYq|j)_i3ay$kjK&)8I*Co}XATqkR;8miL>b$%%Bp$6IycDUa(Dgbv+0aYDi1Z-aIc5;u$LNd>=CHyGWhF#ZS#pNHqT_)KvSq1liw<YPhJzFMK&oM~z}AQhP%6?=gQYZOlF-n{=+@RMdT+^qKon|I6u-%k7F8-A+$nE0$&Z^3!K>>n5!5c#v>_Ww+0xU5wzr)gj}5Aa2q0YZGOZcpaTENBZ{$&AFI|4?O>{*M&1(_vsI7uZYpAjTz0oK%w)S3vqKwRtmW*Os?#ebIQu{4;UZY%lpVFBo77ThQI0qyDsB{4sAqPSqMOYK8=MODPMHXfOO>=JPo5x*p27#K}=|Zk>1x#mdDe}y4-*~upWKOpUdj0IU`@{VV-}0CT-UWz>N<v8YDsfQLuG4Wn3aAD{71SjXliusUO}P2&=#efSQ~C^D6~}<_y_;epuHR}}Byj0%w1g$9lAx4Pm%L16?J4O|a7)BQ3tbh3%=h$cwI;4q#3YW?#A4MCO4{nAv%h*vrW4={W1MzWqmH!)m&gkD;1VFOw&Q3YPKJOfDH*>gn&nij#b6KJF1$)xuAHDDp*<yG6G6FFNTcGA1A&&l`scTnjx%@?1pSa4985l=Y)oF4)zD$1*K;tZ_|%G*rA}Z!Oe;n+6(9?=%<TEZMe~d$3dm%zU=c#;^|!liz6m8Q=ohD(6xDA<Fe^Qsl2EsTdS-71=@e{oaRs7<T|YOMU*>&xV0-(bNFpbE-^r!xNT~^%ur=(|ls*4#*u%27XOA7T*B{z`E-YF=ck%ozOV(j&ew|OBy=*--Lw0=f<PjaCrqf9&jH}-#3M#zyg8q0=6exX+euntZ#R;rRnf35S?pCe(tE~zz_p0@N<vz(E)b`Cl%~+||ZbJdU3Q&bXvk2rPx*;2Uh6XCNR<(7X31({a?DHF+f>MnAj4YVr{69<B7~6WVtH?7dib&`RMF$b>Hf4~=W#SJz(!V`E{w>wmmfaM9*$(VnK9g&yy@`cAhiOd#i*f~o2i~1v7IjQzlRF16ZIS4mj{pTSNQeFUR%>%BHHS%xJS&x)Y$=q+2I7eUd*WQ1rjX8+O9KEFytIeD`q?6TAXRf~<=ZVxF~j*CII8=U%bO|EU)0JwMn^tix7{5ix|C&YY<E)f&|%(`NO~9jpkG|Dh{i5N>IR8jmZ6S?qW_eN_&HfK<{XbuE>dQ;6u)yi{^PGP{gDE7_xMwUdc}Ql$XW?<PH-$iTQU5V*fl$j|ImR}(`|Z2#*R8~OT$*~jzLWba6OS!FF2ugyPheB&8aJQj&e%t`Q(@flYtAso!X@OV_W@k)0}S?=%9JKG*>LPGXN+bwu7++Ln;8F4)uR*Gv7`h`JQ8UL6b;5zQv`KZRc;AFcgsoRdkm<YnVZ_XgD&Z%mo}V4Dt?#mn~4AKOm|C+jxs|BjaVGTjgd2xlSUPPS1UW*So`Y8&lGRoH*Auj}<7C6ffQwmXyTFb3T>si%z!VmXWrp-(S35M34A6eL1BRDLhc>^k2hxLuY@~!r)uMMMfDN?es}%Lv%=s4ZS>lTF_2k>E{kma2Qjc)8-czjU}a7j$Dnc{R3SW55Itx>Wt+VyycjPigfl>bbQXE2bWk;plW~SFg0giLbEZKbdWn_PV*d7E76YC?dgVizR*P+^#+c<Qs2BEC=|jd5o<lKwd$IFh0V$EA5c&<8EZMt0;h=#aQQS2<`1L!cIdLC2+vk3HSgD2cPi*Gi|7jy3JJxjd2;9;k{DA#d`{Ed-NB(5cdLa#7?kO1Z9X*(V#wE_CJWr0v=4qoQMTZGm{V{z;OfEd3tWxx8foi(T6bNWxU;Y&n3+eK!KlOjjJMkfMu_j{%pJy*@IWDz+4h;DDN~fH3kH6v-)l~9H6f}MFeN%04+kUs!C5jRMg5^Q01+D@I|rMuu%tqmB70Y?)yhhBqr863dP1{5N=gfQTix2ETczpLQ_W%84+%p+8JM{DL&7kfG6corgds*5)bRz<1*pBX)m9^#S2N9;u_2vXXXuz-lY&l3v$fUY(oEmo$BTA&H-W5ilta$uq}H%&wfg(5je1(cl<eH|-F_d<ADF;-(sjCqh9SyGAoo3u&Z05;`&io}b<IO-NoVK|I+(*WB{_m~C`pkix4(;$P-$K&86=<t97`^PA%Jegp_!lB$#(RLO50Sv6-LmHV1gHHg*|H8Fsjpk<Cw6xJ8@_JbKF-aNKHbQ;J>4C?R4rLV+7LOB4h{3P|^pA&a8>l*J^9$h`$%-Bk{q-xk)&fi!+tXWHvZ&qWMXtLlGxbma|^DTUpQMHR`%g;WyOlp_|}2!AOj7f>}g3CQCwd`FG7ZL#Gk`f7HEucO%D@C-_y;^0X4EN}>RkN|r#SyeJY?#StGRg4ClSY#xwE5O9G2jRdR6V$sKs?)LQB)1IF0In(FtPS5ntoc(Xk$`7?=*_O}Q{Q|77;7_vm5fK>?8JPe}<=z>&y9gvA<8dS6#*O><UF}FY*lB$%u4YA9Lwiq|;9(pNd#TF>MRlwwzwjx$qG<E-HI5+kTggr#a$FJpDsWrLXsXn2IrAf`0+snh4H=CmY*B_?9-d}6ZIK$X%Q<-W9_eV9*2q&W!(r)kp^jnv1Z|5EP8b5Ak&#$bilAj%t>*iLZk7q{G5VW1^Q>adiSb3Tva5PFS+w_qwQS~!a$Dx>ryH@!U+b@qzhdn!=|9_yWX2~Goxf(6R=0qEl;}#H#9X`n%+tZy7#}2qv-#@!7yjbj$jgC&7@~SXe$epiJMf=;svM#5esIUpL<(FpO)wD)f^qcbl@7jG#fY?HOxuf<5+(s|s@knl-VbZFUaJS)5Q<M>UWvp-bH_dca`J9CS;dMvtabTG%N2jsvzBYXJ#DI(oHq^aOa7=&pZ0UXuFxitr!}N%mYW;LX-(QC{jkOd!OI@6R&9>E8Wp+8g^kr?xdT){+}!leMhm~>07XmcYWS@59z;C~<yb=Z5)5B*saa4#AqIjL%lSR$J*k(wB?(wY!|kB#_OKd78fcIKQKvx%Y0s~6jWI|jYh*Wf7**35(@*B~$WlypR_fit{_!gv(}|_~9zrY#97XZF)1L({Fm<C?=GBt>Df4fEpK=5xmxh?!AY3C8NJ;R@#=-jg8~6B4ZH+RL#{g5djip5wq}$mjNj<pksC<9S9tFTby5^C~z~~6`L*YTX=Dl|*I{fKtI!!)(t$U?GET!JcdtP0sGnt3^#V8+_Im{7>WY2j><&``n=araU!Lpxv@(0t`$US+osM`an9ukd&8lDrCg?@T5nGX{v9Zx6dROCuuRw%-t)b12v4B^FD!e*JSjbx|xJ_YvW=tN1vZ^>Vlmzw%;eRgR7J8T?ibl0$ez0c4xAbv6o039+6G`dBQL1IbE)g14&(Ap9e8%uOZ<_``w5uVqC6S_3DaRR)o(J{n*`V7KI4VMg-S2+8*H{R6VfX{d28*Dnt6~ChuhtI1<JfQ_u$>(59YZ8YEB05m+csQ@Pa;stLUKCfz>(xDYiH+jMmxKyrSf8Fru0Sv+JO*fv0%x$<MjpGVFbA~ZBbYhm?Z+FtAMIc+PmxfUw|VX64m`cc%thx;!E5&aWs=B32aB(4&cGWxcqI$Iw_{g03||X4@i`ncmnJ-*dDs0g#8NQ4FKBwy)HwY`%aY|vrzW`et$p2@R=E`?@6|HnDtZ>Pb~T&&Nm&9`3_XdLk2}%-L(^7*gdV4nCBKJWF3!yz`_}8^#Xsr3U|k7HI?|uz7Lb2S>mICvSG|selVtssuBV5sW`TgBu^2qCVZ=eBJVh_kjBbihmM`YDZvL93ux`sMqoq+&8DFU~;9Mij2&FW*3$Y%*yaIQ;j{TodG!@XC!nja<gmo?z@nrwU`hBzjmvSZ6A))q2Bz{fKdnc!eM|66#Y2RypM4BCB*h}&f{0|)%p_9l|WRD9t3ySFINy%@3fXpAts|Uh~7NT0Cs7}F-y5uiSVA^nAX8J1jGb(SVAj-o253OHE!)cnJquNOtlHWgx6H!TUUv7=qk$z(8NZSG2E?C?V^a5q8mjuE=V-b7XEq)vSTZ)+ECeuWptwk~aH*SW!liA%LO;`dosv9kcY;;c^lj9|fWv`4_nW;_lZu6Jv`%(+9v=hSnK{z(kUX*vO;1yLpnNH+|g8k7%J5D1hyQ;mSmNf_V@9+xIycqSp=x}FTt_#BL-EHrsMY7ah**Vy5?QO2hK(6ApY08(XQXItvO8jM0$rqFQX2PC<cBw@ALg331+{>WTfGbljoxBTuE8l6SfjD0*-nbu?UA(e)VgA`ki2~xH$iIKEwZ-OadTv%rEun<l_U-f#gWhNiCzZ1=o$omxSDCuxnL)yIkcVQ#z06g+*wyrGF6^M}1gMWP#@8^tNR_tE8kDU*ox$BPo@2IpZ6oD7=h#n(d;?rHS`9)19tCa?`DgRN%G<^2EUtLJ`Jh!d>xVUjfT5_*x=(}iNUa5Byg3NqX~VnJ27Yz>t*W;{xS;q84g}ZMRUzu9w|6!{1k=`RhPd<CRQzc3QELk%9J3=0k3(*_rH*E06Z=4Ifo|6k)f42f&dY|ZE=)_V2H|jGWv3q?R2!v4LAil%zemI5=yA%~=$(~gI(k$q!3rht@@0F#-27@yY8<6M!)38rCu3t*pT6eYucx!=F|5n9qQM)5H42~?KrbvV1)+ca?<fcH7cYM3QU1BluYc!Vf8h~c|F861SfmfvTBAJmgHhNs<ED)I!+H)k+75qMBe5)v%J3U4zyR0auqTe;4g5>^<qC)MYhLkYfh)>bz^T6|%fT-kacaved=%z-A?s(8Tf-7bi?usfr1f-YZ>)gF43;h)Pccg@EZCD?9<%5b^^$RdC?!I3qxoo`^iV>`Z1^8-ZnQ9I38fNK7lnfzCGQpw3s)7vW%9CU+{D4d<=*GI9vl})E=$wpSfhkogNQ4Y1*_5#egtP6-d_@-f``5V4dKX~4hF(Cwg7&hpo8MH!g@*s5%^i(V0b-Uqi50u?RZVkCLjlkwJuhjmx@7Oi-4bGP~Ui{6iYk&b6(~_rF6H5bcB*^H3cUVbn2FmM^vtzTfR8BsO~ojF5CvfUchkc@IU|hA2?4pWfT335~2K>(_xC-HI&5r`;;yE_n1^Cm-9h0GVAl^Ffro-(TniGRs^>yR+D%kkN8W7{2fP{nRB(2!gIrG<~*-_GM`Bm5bpAGO1{C3I}UzJP!k0^et(KKK=hhJ*P0Axo%`EeQw62*%5@8dckB6g^PI@&BWkqdXYNIHl;Sm}yIeH#67jv~uAsE@yrYE-_jM6mi!nPdrGkoE^YhX4Y=gzphZoHM5$SHs&(i8Trj)@*c=I&^QI0cM3K!+5GLEgNIAE27;B||};;z_5VE)r257oep_NS91suc+ec(+0UNb6vTPM4TO^mm-&7oBI>yYknRLg?2oe&Xd*jdH@Ef+Nd09v8l0Z}J$##A51zcz9ew2ZT}-*=Eotu44V^&@tg@>5&Ml4&koB`GyPc@8oCHKMzHqeJ0g$mtFFC!BLRS%B`qgc$#IU{_HNoM4(8SO=qM{SIK#2Pjgm-G2b8x9$&%bCj0s7_N7+f$}c{39s8kk+?CNcHV6##B7b`|&PkO<Cu0KH$(XSN3Xab+1bUoj35E!%0`)++c%;i_do6KFGXw|z1as>*VA{)UNW~_^0)7GXftg>^4UE=sf|oHQqKcUGQ42#PY{^e=x*NJ_Xgw7i)q2W@SfLcjhbQ5)W&xR+Mp^j>(cydpeeANUruPzShC96ES}?Wajq;^IY$fTDcSZ_KBZ<f{R!kpSEOZXm4MOWz15~1<Q~3;6E-DG>fyp67iI*iFpTX`m<I^f>xR(whkpR(Y9s6ReAjt9u_!Xoz_!U;VPZaYT1_6tv=EfS!Kjnfl)tQp}IHc@GUd&!<5t2Z!!S6tGD9`I%{Th-zGqs~B+WO{6gnH^(rgq`O-o<9MQL~ggqtp+I(0bB9)KhD1(3$MIQShn8mx=!@5+!oJbPfrj1c;ay@%zSo;>F)y|K5FB(XzN1cLS}Pxh!r*K|5@D**nE?y<F%a?@Bj30A|ss7h2#R)MJ%~!|ZCO1U^SH1OH$LPsObdNsG}R!7h7(fdaIsSqZN_REl|C)B?u~>JW+$P2+)j5MupC!?+@fN@rdkWN6*oc@V?cRBUfMZtiPV!|dG$1W<6r4o=trpL)5rzY+PhdZSUxoxaC)Sl%cL0jEV3LA|Py`1VrTLa<|kTmEzZ7W`%fK9-iiO{bU0-!vkP5#hSfzPgu&RD@^?w*0EADa@8#MNm!#dmB1yb!eb3{iG*E>1yxzxMHuIsFKI&7NOgI5L!zrs^l@e)6p|53F7-0WcfJ>@=#7DR={jU(3Kz@qmC*2S!7tipL^%<Gd?h{1yIUyQ=rnAQIQ6VY6Y8yqy;B`g!D9vh0myC(nKh`Id2hXA*}?}vk5#udW?vVBBxPQ3o_MKdb-?P5=H`d#|exH8)4i#OvcQsl^_<nLEpk<P%A?7P|NfGqRWuJuu!J)Z&BAIoLo233uGG)@o&Wt2~eJ13V|+da!RiBaZlkzJAhe3a>D(v7RbGZ&&}?b-9RY=M0HlC_!?*c8`tC3$7pe&fKJxsYMjR}-&$rvI-l%Dlhf+cA*}j}F=j)b!WrCUx+G5D9{RM;CBo(=pZL7_efv}`9`?--2TpGNl81uRVJ73|d4}C}9@sDfiWrAQcF=Bd&Q!Fbeq!OVWYUB<M@kQ+#b;1<(zMohcXqd7HWV(+4%^`3O>He?sjxClc*K?y)iDb0hDU|1V*y*j5OMb+A(j23W>hOgm=wMSfoh*bI?mG|#JV7ga>{5;hLgJOtO`9Vqiv;hkz%JsZu`ne=_-@6MP~63Y*)0-J+vI2O?v&_DSC5cMxdvoe!|wuB0}Svn)rFil+E|(sz8WjdXV7Z$n<Ni>xEW8p-_rCGY83$SYoc!D==GV%*lbQv-{ly4e<8-W;M?eOv{(_MeGVA4%?;a;K*AioTjfrE0AT*3jzW+vhjSV&lS6HiS1G16LJ#ljapq6j--GmO5jr2k#%KD-XbGM+2nRZroM1oWn9WYtx>swUZ!3B{hPI{&jT{w-Hn!x>#EjAR5W991ibXbTxUaZRG?UeWSg)q9+9^i!VR-vzO=Pgt3)1vP2}@?tq(U_ALT=~^1HCm^wys~os6fwKBE5)d-Tv=Rrxr*cp{>Q7G$)MsV{{<(a7Y9cxx9fF@li5okcJ;1Nj;#V#P$ERNT2VBnvwwi0SSMtPYh`h@<_N>q|C4u6r1^;M)m}rS?NE9_*fFTEAMB$C?-R-0uOWXs5U%T$OF>1?|MrTKuNBKTN#4t1H6eMH$gi$CsQBduQo<^aRfCIR?=uw@;FJ55-3YiwJ3HPm&|F$j(ON3oIvGcPUP8GKbncS{c={!Ql;b#1J4y<i`wR6-5X-Ryh%pB;;}L<gnlK(uUxM44w!ZRA-LPm5`<CpjVjTONJSi@sf<)Y_vWGSqOG0t_b1a9ayRsq`GVNT+nb1x1kT3(egffFq~%G-rjw`C1b!YWeQrqLFNiL-cfeS)Sh%R<87|If=Cs>hIZ{|WBFjXxb(H4k}0Y%PhL`};wC4_Ox1<Lj>hvfAx~Gb3IC}=jX+JrDjJS^zQaG1l(K9>h$%$RZ`g8X{@9Vt+6pllgrLQ&RWvU`x_YH2)5i&W4h98ogSp8p^BN5M$)JZOdsMQNc@PeIG6z9!PTwOFT&~$peyY2LMXh;+f;rh(y(GO9DdB_cmJ)#}9QRX<Z$#8bbT{<4QWk8Do?nF;*OH}{8VI~8x?J|2Ju1`l&gj1KY%TJ5C2z_5Wcc}DD`zNN&c6~Ddc#YTMCh{1R@W_V+d3KCV!d(<S9br)<3^{O8M$S8#j7<An88JV@00jpPWoTkB(9pw{H3DHtGtbUsHG_pUVOkickkb)_FcLccC{i`g!~JgVK-RQn(TB7f*-NjymhHjXe>Tb?lIqpju%+Ro<&EjiJ{~IN1g1O3;34D6TGfU<vi-U?q<jXfavh71aUH0o1C4f_=acj9i$-fmIHjU2xf@Pc}0yK%TcV*R#4tbt#yQ={0I)z-H#~BtR%<*h3T`CB>?7W*bBkru_O%i-6fzj7tCh1nv@j&#v4bW9}i<;$g41QLg;@8Z3gt42;n?CaoM}HI&tk|Qn^ToJVy;!2&(RGC~mbyab-f}WJRpooo9`;Ugc$mv4T(Sbk|O&({ZGAvIsJUu+T>&iVQ{xT|FuU9Kg!rg6;Hl2=b<fZ74X6ta%C7-27Oor8xBtp}dzmU`$V8be;3e$BSA(AyB7L$8g8$I$FaRK@k61e1C$aM9M1)chQE5BiHGQ^-41W0$CL8T0Ln5bhvQ{lFZ{07Y&Nmp%g_}Jk(aovqt5azEnE3?(ItbmO0vI#%P1ke52kCSIkj7(?(HdnMc|ZIvy!yMk3wDx#?hpg-Dz_$2$iPrL0o3>s;RGKRTb7{gs7OrwcukWOCO^zFyA1Cmyp(I|$`>Vf&X~g|$ByP}O1kf7d0=k#iS%JMb<p_rk#3utGWHI7Ao3y$PR!i2S1BOC6|Kcj5_d)f=_d{*}LmM^2<hz^7CW5tz>->9BCMAV$nY<Ltc@9H%^*1#tTNtxsbAmS}%L^LUSiK8wPt*P+8<#9lnQe}rR|<nufkOD31Ew_z<I;XY!mB;StKYrqP?z&1%Bn7X_IhD|tR)3MSOWto|KPkN(q4;3E*vY8Ydtv$+@48_FnqcP6N>1+x@bZ<hwqwBjn?bhChO-j)O=P!ST79MRKwBxPj1G*XS652FIL?7IvKu#<{SMShI1_L}~o+MT;;@KIZ%?xsre~Z$OfBKr7dA&|(5OH?M?@`i(?~=H{I%0<HV9GFaL(=?>IQ4^=0)u(N3xm;2m|U~!^WqMs5X0)ny=4jm=f=X&g-J^3u^T?djl&%k$=cdj^9Zx(gJvvoOHkJ&H|=mb8$F|x6`k%?`NDT_iw+FA8{v*vbFl#_&|raJO;NtNx3B!Ha8M;1jy79S!)y3!$1xVw@K6^#^xkyT{_U2?i{_DBbd;J`8kbj;=$PLalD$06qn)#Oiz`Yg6YW5d>^rSIIV%29lW(-i6=Rh9UOQ8=@&zeJ<RJ*7?;R)k3dEJlI4lEqUCO&_eRuoOR%;*bqLN~ChtK-rG9qS|r<}t%$^<x>#!;p%zed6Oa0D%j;h@7QzQnk^`Gpl2p@r66*3{D(zd3Ks7_R1}kH@e8Ce=wYIvyTQXQ?+jLFVe+>y5|zvqZZzwBZ(w-0(qbce{ldpFe1#EG-|)?Iq1>*$)Rh8%<#0=(z9z2q{d#t!CRw`(Yv^#iLYHXwwN%$%PP|6ZprIpnB9xlfiV{uPCe!IOb2sy`!Y!zxZ2(()-+ZhZa+&=2=orfeuH*%FM6)ubPv|#lL!<ef?L@^9jij<QTJ%*Y93czTL=>j-O{q$?@aQ*-IaT^$_<uSdhm&@Wb<<FljXK`?Dk-jC;pO5it%y2s!$UAgt2BSAYzJcgs-f7vw(!WBI|0A5!GxXMS;u=_#-a)D=y7r|EDy*Vsda#oDl+oJ?VwPKOt{_(@8?bBSDoit>3v!W(_(vy4;6YxZFBO_;>M0*L>Kul-R9x7nk)_$m>{5QZT0a+t($hZr57P0!Mpk-z!5;cl*$;FY~FC?i@QdMqUNEKP)F&Otc@YnF~rW1!Idj2x!W{31UeO{cT|Xo8fA$-scmkWr<~d#{{sDY6gMQJS6|B46OHDB#IPL6#%wx<bIFVh678ZXImzWD&XBo9#AcH(YZ?*9RfhV^;qx;^sSqfGIl{AQm#Ps<SblCBUYR5iO5lk)6M=I*bf967$CGJNYbnmKW5G>1Y_jwocN9ua>FE;$hXT%jw-nR}=|9Bp<iY<k+YE<c1=G4t5L`6ANdr>(B7z!P$6BOB-0rr=RvO?}qg{h<eK7tZ?`p)I??LjqFAarid-())*I7Y@(nMr77$bEnbt3-Nv)dQ5O#yPH2D#k`;Uu3MDiw_5}C0b1<?AmDgwG?RTp0yz@@Te{dYC>8wosTU<!|#ZpG6RLS}~@6^z1utIvWpkT;ikGQx^SJxuuHs!NI56s-QL-|46LUeCed2H_P#UF0&pdZprHekn|i^wKKp$lC<%2;#9yrSwHY;j^FQChOxG%Co7q}8f4uZEYUH7Al)M-*^lHPW#kfUHuf2O?2Yu%>1-&+)rtEK<+&Wa)e_mpdsd-OV#3Sn<a&ln}#YG%^mSPZ4Ntf_T)@h8%&UJ-LWsqf95TlhU)nV02DStkAsykUs8Xz7@Mq39|p$i5NoVTpjNyp`0BO4@og2y62Nt=iCr|@*+=A#=Nw7!cGy>&{_}BaA}r;KIcjmJHOv-&#><%BhHi41?vFVm2&x=BGkaoG*4)K#ap|GouDT%FejKAjSFTz&lB{fz#lcfsD2G{NsVZcQkE`E!H9hkDZRuOYKt>jKF=P(O@JO@ST3EhK@sE13p`yT_gd1=Pek@x184qwK*jB|@qDz29jg*<#egXKtW!!YAQx>Ctu4ncvAIZDl?7rG8dHpj(XuHlAe53DnLf`nR-t4_^YAZ7-o`6AU+P^vZSK@4_-hF}Ot6&9w!gazg6T(MH{{O|bah_e-QC-0-(XLORx%(dvQw|+8{gb+p^NPHBWO~!4JViAj;){L&FapKSiY1B#bt}zVA!KIAp2$_>wP@INw#Q=P)KkWg(m02=&d{7s#j~Zx8Kf-LD(tHRBNBD%6#aWLa6{Go@%}~PAd4<3V^m~5#)A^PPoTOavG5?&WUb69HowSfpLp=i%eU>2}E^1c=v9Jp%5kunc(EKI_vdEy-AwA*eMVmyIR^ScKW9zx-RPg0+wIsmE)*hLA~=?l0;IGQQ<L!CkY&gN0C45k!B4JXUuJx&cjjvJi<4s)SU|b8$OOu8+tc1X%!TOp@&aQ3lR<kfthD>m|U1x#=T%6sK|UMR552DPj@Puc(~sW-k>EMsy;ASDYBmLz$~lv!v4bje)t{i$TK+kacrGY7l#Lb$XvCkl5a#$vM=3I{bbhJ1!m?6^#AZeBXEq9eU>tvJEO+vR=pd}qPw-*G}53`D`<HVBW~(w65p*ws0qK(Xox&S8}3kn_7%azlp<A27^&4{b2QRv(53_N0REsOs;)?iAD{S*j9iE_i$R=bKogAV@d^5`VrAMuq0YyX=~Li~vH}dtCCj<;c2%o$84Nubpi|isOPzu6a@*X9{MuWM#zyT;`X@d~P7ae90-sBj5J--go%<+hH2QGlqV2+PGRjxE0F=rdOs_OhqyxZ9sDYYn#LefMM0{15LFQjTspHb>A1BlByS=Dfm1Yc*va>sCa%f^*9@Kd%gm1j@-CjX<Z3>1;rlo+fxUM7f#D!Oa1snp;qXj6mr)P&02nb>ndt%D2>t-^7PYp^dfwf2);nI3qA8u~6cF2-gctNrTBKoI;>*CnBMfok)=snC}*o666-`j1sKWc8hubC(L5RmUWJMDPi-S-7n^Os*Q1qxFQn!}zTRBt%?eXye$W;5A5hUJ3?@<pZ+#`O{ei;VEJ63e133s<2?ggUQ?ISt;+ltyK^1AjsqDoI->CI@p8)P`^hr}bY(n3pF?$-pl_4l(MybYx2=@|AT>@{Ba1M)9Dg4>@t!p}`Ys7w8CUT8?JZH1(HiOnssjvu`dJKafA6@zDv06j?!|f1$o?f-(;%R}XG<16brqvfx^v3#EhbY!b4e6c0f%2UYwBcjH<eWfID%i}`fr*@FTj`AHPZF+s=`tj5Z2h*_t8jK(C)rRe4DWRw>c@gF!tm1+!KL*^HLksuT-;%(&MqlnyDKsO{nH}3%i9~y&2EQftDZ1uNzx<CW|r|=<?oQ5v#=s~3L#uv5zGM*5I2Zl0o!z#$U9dxH8<&K0UGLIy&3>Pp=+YiY{!P8Jj__svx^%*ohs|0!9B+TlG#?>D(w8-^8!{@)ngv0RkbMN{OXnRG=C)s6xPc~2jJQM}+F(RxIjO@>FDP#6y1QDWkqbR>!g!e!`53XGL2+sBkhHFFY+8<%&Gw*KA6Z?<7r(mz<J*NE2E5FoTN}Wp5vQX$GH9!xLipkb41y8#QZowy7h`8Ryz2jv(ov=Pl_SI`!?DO<)#HK;16>>X3-N5yUUU0iA!Rg|LE)BY_CjNz-rNp`y6Q^c!&D+`a?jNuNKkEB7cbrZrMRJPaj^W;+6Eg{0FK|HdHE19X<)?Q2MLZ9_u;x#6NuI25@4UFMAkPWx`mvU~&40Al!n8I!C{ym^hX>?IQc8?s_Y+cnnlCO5L-nT=1LZq=t^2J#6!PKIzh<Bg2O^Yf?1V2RtD%4(9B~1llmTM3de?<0<p8Gmmmq-L3->p-T5%3r+8K1ZT(3x#0X$%*jR8{HM@_AP_IiJb&SvlG4y1Gi!dD*c?c!C<?6BKc%5>av6$#c{&R5R&v;tbgAgpa{HXrQlw)Z#J+tuWJj-+u>%@L;VBKSAG5ARfgT$oQQK{{i!EUQjdNR#5h>yzop=*UC2lh()kRc{AFJ-nlFZ*=0#hY1{(AYJNJ>SE>!0!Uy`t@pHdoTRtmnlheVz_~cDdTrF;&OKJ#i01HCmz?*GWG`+LN71uEh)mX^J1qR5Z6Nmx(4y)5A&(fbQM;SLjPO!#M-@FiyU!T|)9KM@S~tJ0nqTi2UwdQt8d$atshfh6qmnK%4W~CZ4eO0i2=qBd+EP4CPRStdkV<LM+9B)!MKnbgK|Npg*OA>2H)tX5hQi*k6Q18B(V^6{^vElxH=C!ZTdla}FDTJuQ%*U8XvbbG2<0zSvt8OcNAXqY9CMIb*#nMLK;OOI%n$QOT^4HNRpvg1Jue#>hSq-i8vak_6vM=GTF3M}b7GH$9x@FoNZK(d0i3(_pamr^W?{kEOy$j<8+B8P8M{=j2OyA)lO>y=6l?m+lwr#*MYJ`F!IRP!^jS1s)PE?bpfvAa5mE28_CIR1cD!$U&7BSJ+XYqQZ!kg0`Kj=7N-GT7bu{gblN6W$@P&B^aY?rremmgxaFc8X=PETF{4LBth(aOjcr=HtPzLFCF#NV_Whhh$>c~4pGGqn)<1d{B;HaSIZf0KyOp-x{W&z!K%+MoUTw5#Nd>BGjs)&2p-tFun6r<t8UeXO%uhF^bbm;Vl)Q%N_;O4s~;5<6-KrQ?VGzu5B6iz8Z1N|jjj?qh+(whJL#s3D6zI^dRKcsa_VPuOuB>&WOV_-e8;3#uxDecZX(sIOLl;^^4iajy4gF9G*q8I4EXDX7;tH3fBYmtzsSTuuHXX2<@>BmA}z|4&=jV=v&;z^L@7C^g`;l;<^djy2Y*T1Wxo@GQ}S0T~Z{~wbm=57aWe{GV<$%$p*G>Yi--1NY6_jXXCm4{Xtv4j<zr1pCV|Ern>mY#9ym6~)N1EA*iMX1m@%Eu(0cR@&CG3T0JoDdqQj8s6G7@-u}8I?vm3zc3RIg)M=kOEGM){R1o1Xr{qLiabl-O2a@?Kr3@dy<Gn-5Yz~J;U4)sJv*vr2sfSYF6R=M}LgaJ4Gr_2;!)y;vUBw(Y*sDIr$x4T{-HVjws;gXnK;YfiRb~OL+WgwE!O;B<To|WGh68K*2c^NpwNI2&-O<Gs=al00(7w<=;QY7!L3HS1*3_;@`aZk@)r%{sRC0^7^aiLi-4B{FqenpL<y5Ih3bFS(NYPr{49?DU$@MDG{LghtIFSgg^iG#b3Pm3!&AdyfDv&4)w3Z*FSp>Eq!(U+s=!hU4PN_LmYBYZhumvS}@rd)@-oXkXYSXdfJ;rsanwROOSy}pkHT)(P%S&O`+98EqV!JtdjO&MT7K2e&N_wN7d71Z_P!B52Aw71dKQ0ZLANcaBlcy2Dk0p-`ZT?N8OlUd}J@wV`QM5=us&T6agPN;5^k81s!qJ7Z`exvse@<=?-DV&k_;mu)M0akl0+GegI{jn&h6DznY@MCO>U6q=u-Y7;fa<)dg#H5DkFq?ZqPLo<p^ug42sA?+AQhnit!soDD6o%hdPshM5xGvJIA}EiwWV^A(()TCKJnCtJ5=?bw!_jB838iDH~=2N2<14Bq0J{_^%@u9?O{s2TTODzAJxPlOUtv)0tLqhn=0UC|X^nc@sJa1$8o8KIf@OV7Pw)_WE#f_*8=byK{EMIIT)p3<;J{BBXu^2^F}RIH`sa;aM?m;+ypQkXiVlgT+wX731ZhI*W`mcbFGS5fGy*njqw5~oo0R-(pqsF2-d$|}K5qq$%~Jv25O7!B^1xq{if+(>Wgs7`~X%#`dFwFiC)GreN1?+R%;<d!1EiP}esp`;=DS;6K7$uUh%dngim6b7v}poftCE`_($$8W$HYmy0NfShH#DjYq6uw&rNDDXfhDCP&vU`-z6xiXUT&{B376$+|0RsX@7!YH>3TB+UwS6UB7Bx}pe9{-eO9s9xq@$fUe#A94DJW&?(ACvtXjqE?sNxQMTO$R)#ljeg~4&*!!XVN!u2axU>C{YykGiW3yD9lv`2&ag^y=b%7bQrsSgIBwO113)Dt1Rh``vZseEbCha9_i{>QTXvHB@l@s1fk=7QqMs!16Pc)Ek>4HZXbA%tjL1r&hA>NYxm_-m|zd;zCsXp;cYjvu5cq}EzBw87V>W$hg?1sTf1Po$g@`4zH1qA4g7~RxblP7u(>Y$bA=6G;<wk1lJS^wa=zAJc7=Gt+_Be2eSG-#-PP5%-hAt>cql<);it<he#FNCJm?5B75wL+bBXXC`HH+oz96R(T;-H;?~72lm<aEaFX1XC5CBB#K7Jol^HJvFUs3G;kMLJYy7`58`7>2_sl&=|JZktWDu$Oq=!A54;rRaQLi(U&Cdj}^()lPlo6O<Y$=P_U)2;vJx86J)3=WIwHa(l4p3PGvT(^LdUhOUK|M@jp66^Q@A!ASl4O?rcwdMWs`+x6AuOHycdm$mov_lU)o=D407|0roJ-rv9_G+OcnC0gU2LLB;H5}3K&XYy46QmI1kr)q8Mj#X0pJO~+;KAW&HXovQTpxrC`j8blO=dABfW=}!=YDT?5uZ(tk{PN>sc|7uKN*qA0xw;*7L9x(8T>VeW=U|z8W1E2hIWGbh&(*RRReLEKsKd-IU=Sd`MeUpH9n^hcNpZMoloNZUH<Fe8d%NoeA2%%_7+;j@Y{v{k>n{lP6l&5BzHPVWJIoeVA;5%Xo9i}e1tnFdEap8-NWY219Hf)`>IWZa;ZZ@x`HeV`EtpLb0%SLFo)@<FIlVMWI8z+r742oqO3|!9nMLT-6D)1kPrSCc6yy|#c`CbInuQvs0d_4w>9sBCb}3}$lThHoifwI%{Cz@%OH|_8{50`kLGb<c$W@GDIwiTH!xRF2~)+li-HV5n#wg#6i^OxWP&*w_9nQ8iVETOFDj(AC;uM9bm;M=NF*cXB^&um#kvfK<?$_9^`&twIiBYvNa!X(xM0wMtJ@~9#&B5G`&0_9p=aA#qnh%|(^APp^>&#h)agK5sNL~y&Z1J(3)j1)1{Z|jzzKSl<S)VfrREyg`y=cu{OhK?$mGO!noa>iP`8ML1@ZL`R@_TtJ?^nX{p+%~9~0TJlg%w9Th<vHvZqS6`tx<%qK0Jnz;O7dI5#X9m0Txow&%JH!Ej*W<Oel(sQFKN2)r(gt}=@o|BgQg1r%Gzz?Ldi+!Hz^N;<N<DgJ+O)~;{?=|Usfu!?v3ph*f3kp{_c;>qX@dZRI%W0@1C<`od+4mH&s^0Ds0ZGV(LhJJjH7rjjw?>Ud7S~d(-$VIjMj~ca#yO%r}^)cu(R-git(<<NNDxHGPhQ+;jUfhRp@c_GF2Tk^JKK5GcyL%XD#-5p&3ZuE*!pOA!Pad^|3<p1}c{6}RyDBLG)~TxCx1?|9?)@14uf5;Aw~2D0&K9)QkW*@K@VVbOk}1x|7djeaN@z{T()nGn0G9f$7I2!<l^hnVtQ#_@ZchJ)%7<(RaBjO$CPMHJCTbBf<6U7`_e(Xts<U5m$hR*u;rMFJ+kp$;p2vrK-<wZS1%fW(g}`;#$5^$1@;CZE#ZTctnm(<}LlwMio421u>Ziqb$%0MWy`k{!1On^$M`tHE%>-8B#9M*_bWg7Yx9V&1jl&*eo4LuW$t$W(4djICq$#j~L|n$Hd^n(Q26e~*G<<Tsq|c%1q<o5>uRtI;T(S*#3DdOa+JKu^t_6I?_2=P1=0ZT|+4F3SLGO9i-)`3-RA}!N$P+p=l%q$8KEQoCwEAR@IUA`QpQ}ImJ2&r-b}>fGfQT-uXn9jxo-XqFJF;C7UTbk=olXHQ-QyAIHA#8UJ0GP{Jx?(0SULr6Gwo+u6xae4Y#B~D?0;1>v4WpfHv)td!zTi0HPA~#H?-35EkN6UEeKv-91?!pSgR$(-heVrmbZsd2xyOKeH_hv^zQid)AWtXw;QG(2z@~p5%KhS{nJkmpQ9>@|Kdv+ulC{x&og(Df95#0>)$>{_Yn;9Bn9M`=s)t~>(5^N^>d|t#J^v>_!;UQgXh=3K)-b9c+$E4{KemPu_Am*{wJb<@?(L2f2V$Y@gt$l{Hbtop<l4v7aV2h8>4Tj!NZ{c%=S?!@?^oPenl>bKQO#lpsnYWsT|d)|3)f$MXlGrgD<~@|9u|#=AnbO>G*Uy9vxj+Ubf@{6%xY@3GiUGwpz1~9x`?ljDh*`0)PHBO!OBi+xViwhtv5mqNd0h3Es8#Yh-#E(LzE1%Qu0E!sqK>e*4o;tKY_tos~a+|L^|z{l9O7UiS?@gsPzaWH5*-@Ue4(+o%#W2ow|t{T`{dPWarZ)MXw-EZaEYeFnF1LaI68WQ%%75FhvFxCrS3x27tv_SxB?!(Ym*02aI-LSF;ua(dJQ!q=0xC~Knj4ovlLuYWIx%P&4x^G}|vh~WF?mFyH&C`>lV9lJK(SypZQ5cbTEv8KDOnyPuMz*&-lpjzpXhd_9OLTkMW^6La;*By{?2fdTg_#*NTcGkB*Bohb>hT+KN*e5s?DKNS6NNZzsIvypn2tQ(5(-TBOna(2k9A@F7xC$fqN^3qA9pOK+1N<isx5i1FB8(l*EdY#`#f|(0aiHH(VD0yxFCBaSy!dBcOASwj)evp{_gk$;Dx7V*)odT^wJ@GABW^Bm&p!Lml`y{&D;dkfIbW?Ig7o#Tfln&mJHhkR=*ZrKh2Oan1^)m~KYQ^*V6)bXqLlbu0Zji&{-T!!N_-jkA+iJq^eH()(lneMHeAF(y#FSfaba!YCQ_skkK>Q<AIx)uIlo#t7#CxQXonbg$=PUgYp>PZz`NVtHpm@+PV3reo9lsFAGQk<X5G_V?Y0&+yGQvSv?f}vI)PA*AST{OK6YoX)kYu}PW(vAov(HI%wt_)q?MHR3wNu@444-FMy;+{tKt;zcq9r^uoU;m?vk_$BLnfb;SW+7!*5}UAKWXY;mo&%1Nw?$g??z$7vO-;LA3ZG?8h(F2b2fCM2Ud8BkuLzt<ob5n<pvahtK7<7wi$&W)Tft8I^PnQ_pi4&7;<IjeV}0aD8@a4i@hSYcKf0q8X&=Kxs_P33^<>_iD*}hW<K8<omeZo3Jwq(C%}+G!KUI<AM|Vx-Z+}o}i09{`G$$neQ(Nh~X#R^*>(!AJ_kM{a?MDXR?<GmoI(>BU!E0-uAA4StS#2b~2I?olkExoQK|vA9zsoovf(%^~Fzl?}Q{?b$njul-972clea1WW7m2<{P^Q7|aNyr**%-YjCV(#)@m?2P`@A2!H<KZ(zXxOy|&N?lD%y0IF++BfI{>99ec8<LK(<C|Dnc*_d(PZnfGA2k}pG<pa5Y=H-tb_zsV)lZGFPgNDXt4u@w8`&}>UH>_>5>&CgU|8V!9-Q3yum+OCFzwMP|Y;<D;)N|4n>b&)at4QA{xTZcM&MMqNteX~y_4%G>D@7C3sQZGw>s=P-WNYuy-e%k2Q>^NvveG@OE#6YfK7VKG^`~ctW0b^!N+DeN&yzko<IV7S>K#pIvxNNbfDui{sJC=-6GxzVuirNV%pcVEJmDz`H?Dp%Gr~QTe;?kQ(o?woRP{<*TBtZW3wxNlIrXNK@df$jPG`rx$><p|+1}Xe9Untk`bHrkr--crUoajW)sQA?Mm484H$uZ$?Wu=X<zmE4KMoiP1vI-oeeG1N)2C@HeCvzcIP!iK*aWMN42S<^4P?tHkfdjY*2U1z4Qg-qqj>vZYk%{R1dGwDq&$hrhn5n(&a`Yw<(_t~l(0orcgRD&8x>UV^hWLKOKdM6k%WK;jJyVu48(hu^u5CiZ#)HAP4hv`&Ua(Y9$*+Vtay<c+Z&Rb2}O0yhY1REXYjJfuA$8wUAl2qW7HDv3fPKzy~u&MYqR8J`h@aart_rN_of5Pojx8-j-e0%kaG6))Whg`@Axc9Qz)MGoK@XguQ3!t<p0XKC&}DNPEy)#O2UiAN}jbwmrQ=h;cIKSp~Mp_lEAW=6VDwb!WMB6G;~g!DQh@Tf{Sp7<OQ#v%wDDW0TjeXN>}r!Hy&3`H6(87yh+=flX`Nmlml^~l89cOX6H}I3s<ni>UQ%a=W{MgOo*CqMLXNfcfaX1tspmy9q=Xu=5f+X5x=+BPtc^>(^dk9$Whc?wSX3MA(r20l-vVOSmZcfi)FX$YUa4V$|hW(vZV1Tm`mi!mluswQ(ECA5Sp|HqMi`xi!9nQho9xbV*Z%(UU`H1XYtcKCJ+8sz`Tw!o1wD=MX7xwl07i%)i$pymPjU^)WUv5#xwPxo57mFR?7EUuDOP|w1Sj!G)n1--&|jBJ=)I&7x?;=imPht<MpkB4GnyRdfnW<xB1`zZ~d;87l$)Vgyl{*#unsa)F)nNd8Q|R_Jda3?Yap^r0%(iQ3noa9-B#q$IP=>gA8?OtTLppBFQ&@5dpvSxtv$B(blnc7kgl>l{4BbG8j6a<FeH5!sv6TDw2(KBn)>d;KaIX-t=*FsWII$-Wf%g%#p8N0o&!IFOg4<z98_T$Hbi0FT8EsC0zQK%ssg14NFBJ-y^2ck3r<-y)wC|SUl_-QN8rU;fj1)-%Ng((cl)3E5~Q2kBMgdtAj;R{cZ;6V1059Uj+}g>Bo8^gXp1)qTPn*H2r(qwf-|%EjNVf(_sRVuU$@<4~tF;LFO*E=^%2$=H$n~=fztDrVvE~o<S{Fc<z<1<5Y1~Tvi9Q28V}T*Lm**vmL1Y&Pfbh)I(I>g3}jMSOlSCWn}Dt@cwIEjbDhc5V<DEhO~4`il>vK`jsPvis9G|k~9KPX8zp_TPVM4GXfif<^ypJ*!UM63q)RK(kel-$%WZGmjU5WSuJ6+5bKPk9n#%9*3B~XyK(&w#o<Sie?-jJwA%=DO@7t<2~&WZWe<gK31XG%Nlkg*+HaTcU1tW@Z09d%YIznx#@DB#G>K=ur+m!E$vI-o^=5j#aw!Lf5sRpzJs_7cmRlkzwy3PtEV+OT+m;Du<#JGBi!H5S&@if7dr#r|m5lq|DZ1sOGu9fRP{^Z1zYL=0!Xt*GbReW<5osh1xtO2}?}PlD6H#cDYcA#H8uukMXZD@t2^FK~054PPHI!>K^mA)83w5C-X~52E>oNaaUl0_#<6J5NF|A<21@wuWOic(4DSq2@g4TTSO2}Z-i5idsNFGV=NMhC*0-u<NhVfKnU~B_>fy0_J;0ORYT~8vS0rfX+adAAIJ_c5#uZC(C{bA2N2y#X@tRZ$IqwC@^o!{I})hM5<%^=Xiy5tOL1K!@<Xl=#iQKiJUEe6~VC$Z5r#`~yL47vz=(Q)v)`CP(5_k-7s=SGe4V3D_l;ME4AI<1D*T$8HKoN7D~iw3!PENZQH^WwS6VYlYZhd|Kbbi70~Yd%QS3~Z$#1n)(vN=IBPM~u2+$_M^Iii$54`bzOvA|9MnVBQDKh9?6{xv{G?Pb~^^M9qmn7<sDQ<wylb0Fln+$TWBVq*)|`*WE?zh02m-HWPua)!U#PcgJU=bm%dk9jmcZ%sbNxVM@<tw}raqbT&N)wt|7S#F5}8G4@hK4fOmBfz>w@e6pJkcV{DV+w8jlfkqgHJ{(dA(J(-u5JZQSzr9y8F7Ceg(cfsb*Y`FbVc;>})hVQz)kcSBnA0mwDZ+E!8;$8qdXl7b>bbnz?6p2P*n}xm=8Wdn)+Yk<Z3BK--_J7nnJJoigHB|6t>SqunPwC@X`-{aD?<_ax^3iW-wpJ&f<OuJ5!wrcMvnBzuX~q1LGR+Bn@v4Z6PJqNN~@3{R8O`aZMF7WJ1sH{8QYkvmk=3<z=P-*lE;DIMo7N0OO8D*`nM1`VxU~uRk*ll8@6?Sgm$PLws5u&K;Q!T_fmBP-r;?K+d`v<Mums1=GOkh;_YDcF4LpUOa=?nMX)!ss^vPUcGIltLhJE|(-gtU#<4_B_8U2uBQ$xkFC}|n=wE;QoD9~Vkz>?XWETBCc}!ma=K9O$*MGpG=qN=0lFQKZ>(8!#cm4AhKN6<Xuh1(39T4rl>&}Qa8jr1rqdnN%*K=O70z{_Rw&)dTD}kYFQ6p?NVP_TKoA#_~ia54Xk33tfGD`K&t=6vcXvSn7TJEI9v#>~_@P|onJRka)={QRuXlz_AA;cAX^;L800b0rx(uogkx`-d>^uQf#S0ZMhmfH!<51X4?hziY&g}a+$8=~;$&i%b+dw=g>oz8#4_t#j5U8RFDl=Q6>U}oSy6r{S{J2^xx&2)C5!A>YsVsyzvxC*_)*|gX9_^XDc#8N`86Tm!#Yq@tcOpYF>-eEGHKK0$fG<+S@18F2+p1Dwj3`~r^;3_F}85Xg^!Kbga&X2}teYCsCgx72bWTnkZG$a(^pT6dX&mrKLxV5$UV3R|p5MkX?DaH3bi62InSEh_R43Ey{)4_nQM_kSN!T#?3`&3-s=`DHJQE_M3HR_i8^*H3Z%~uNQWnCF3rZANDM#d$I!W|UA{xL&gV_apHyJKrMX1a|N7s}tZqCpT23b4Ic*k7pk8YdN|Z1B2`GY~EcRw~NP8BQ+3$w`#S>+B>@f~qD8EX=~JLN7t8lf)~r_7gVyF`_RC0&1jViU%R;;|f|)TOMOM=;p%WC1o>0%bD0y#Ms!7&7uLN2SgEOh%b;TX$-6)r9uoCulg!@LMn9>ovt4gp-E03GZU3BGaQd5MP-Sncki`hLQfRJ)a1fhj9|CTf+x;`xkT25aB<Ve4Y@g_s<dd0<{5&aTot5=hDu+W=~*qsD}N336~@b&7FxugrN)!;8>K@aKK<w$k)L5{uA^v>+XqV|IPyczP9*$YMAstNC=5GjJ*SJ$1ty)h@}W73odls`MbpP2V~<BZ1824m9#L`<H8?}_2$J@}{rfPhtsM+!#n@Pw2lbZkT7_<wibew2i&A!P2*df`HKn7NdrhsD>Ctiu$%1Lm6pH4IFHC5Kx261i`7&PD^}KARusxZ_NWsIqW4Cw;+9DR;WWdXU6_v%NE<zT%X<X-3K4A59Hk!_y18{WAjiHoTJ(uhf=S9vT&5nD=HD+c7o71-}?(hO~>>0*6SMco+|GD*ME;{SvG@^0ld#8=4F`x)--Bo*h2^C(s$100wKZ(+|HB&5-jy~*eB%W5a=~;o-*JN>3fSvtd2>CEky}~7&{LIKj{cYV9qLz8;>8>cEJEZy6)3GoS;L3BWV}{Y@b_oZ-<5B-~K8!hNK0ZB~12KT0Z&YNlf@!E-`Q6)#%G_FoQY(7A>EgyEq1^>kAYNy33RmPoxWa$CA^e^6CILvq8Oh{<Sk#)XcpQUl*gL}8N<^wxJVdLg=0jda5S5mAYv5Cwb_1<4+qbm$bh+K#y`@#+hW`Jihbi+s1!TeLSB`o!v}@o}dD@$Slr#5Ercg`+PDRv(EmAG92ou`%l5b)!^8m*1rE$_fCJ*60|BMck3LjBRdWdYm@>k26vwRp0ale>oB)v&ZlxYOVkjJHRha#y+V8J&W<QA?5vYE`_AY=RCJw3XAfNq@?XG6y^2CHFtyO|Z)3x^-w9R29T%cz!SF|v5l;0*Mht6_;CZ^vy4x(QTM<GSf^n7|QNBZ%}p>Z~iRx;D{8852!Cl1*<3&sLzzMfhl1-+hrZo+=TmK-TfPs+5Vcu+o#>IZ|DsSTCkL9KBYS&jzu?pk&>fXtmleZf|dcl-8&**K#eWq9-h(0b)rg1@{e7KIP(;faTz9e+^OK#^+|u`L1<{LZ|*^4pDN=DV~f>Zq9__i4F^6FS^iLG5O`7%con-d#$bZzpzYa!d&QLl|MMcF~*A;QUx8q>1$*CU~doZKrb=AmynpPp=bg}Zys|&qjT<sC%XJnz#}@4*_t=N1(fD|Pf2q@Y~&%d{O6P$!4`s5pCne^JiE7bqal4afJdUazRPY6?fN9*39q+TjbZ}5ajiyZwQ!l_D$u4&04%Y{2cjDvEc2>sM%hUFx@yUG1Pn=Db^LtWT^|UfQd-P)5kyH0EYU8*HsNVvif+JmVFxeFrd5<<r>@B7_-f8MLXFrL%t$;)hdoP#qE#pl+(m#*97AN|rntC5F#|9V+4r06_hSx-0={}Qod6@OaG%1_@dU5(j9y5nF1heMQp>9<N&s;UKA^UWLdCtmyVt^`f0W66eRpeXvu#B_i3TfLYef_Ju2x2CSRJ}5uk?q_y-kdI-)c4Ax4ef~j@aF3k-Kqwd-wg8-_V8{Wgk0wmy{ltU?qj&)mcF0)#$L$LT%)<ja^6TFn#v+9<=tQSx6cdOn$}-RxvexCC}586Df&Fu3(Yb5n)S{PMtouT}iu6*XL0mUC*Kl)KXPzQVD(dG!HH<T!rzT%!7cLaI9xm>R}c3t8V+m5)57?t~%A`nFg+X6{{FWJq%vge$be8AxX%oR<7D;e%j%elpMU~9oO)&5yFyPjz!>#dFYugyiT|%`QLFJE6>No6jXxExe7DsGk-kJn#K%cj(2<bP|yR%(lu1zV#fJTsF5^cx9U*p$noniK=6{dU#n-3(y$=1r{evL;`o+Ymhz4PKP6+X0e@Ge<?qL}7*UBY*ygN$e*L(nt|@#8raEFEvRzOm%~JYmL^Rmme6Yuwxq}^vxacoj_A(<uHVoa`sgN7dB(ndowdd4syBLTeUporq!cj%V7>%lN#lfI(DGdf?3rE{YynvRlQ(hTbH`yVrH?MFXX|?)C(~}d#p*W+EXgDjf)^lvT)MGJ3+nU2QHs^AuA42)|z1Jhpm=t)!-Sh0o6ib$r3+R1{3-pN&k(g!1G^)-bX0jHJrp(35JWnT?t10f&!GO2vX>S6XPgTHC6QxTDWai#PcxK2YRjY<dYO?CG=^|gx@SrdQg8-$G^_i{{%PhsN!!v9z-r3#JGGLL_Q6UGYP-&I3M+WwfQjd62(&r2i&>16H&zkovnQ8I&oL)XH@T|ZbM}AnzoGNgp$*nk<gar!HAb78)IL)9(4mIB$7r39WbgGIkrz`F1QlsgyftiJU;Hf^XjYtPW4K>%)rG^1-n43kodSuMT{8@T*=(POq*$57i(;2?#U;k&B@8rc#8-A!|G4NGxlrKrk2NS3{$%wL9!|=j$?Az%$nS&g|%{MscdQtbR?Dga_k@V4%{BT4G1>n8`Qgcdi9F83J{Da6pD?oBCeYl?2P;+vo;A_PL<E#9_ZNQ_d?Op;Gs7SO>2vPa9aU_YiaLx`f%uw$+9l_WWdGKbV;M4hg_@g?O%|QvyA<GDc;X*t9Xme*{_ajtKX;YSA#Ailb^CoYB!&BPMNmMbm3?=cz5FA0Vs8Ry>3B}J&j3eHa$cnnhX{-kR^?xETnBNjY4}(1Zrs0{$!vgd1mjvkZYfNVK;xD~ynTz7un!a))n|#w-f7n`o|IzN|&c3(V@PuC_)OY<GPaseII?JW?1*K~J0pi_3d+3yDca+ixS)JpIKw3JYs(rO%Pn6aUy3)m2kMd}MN2d`&w%zv0aqV;pD<<{Vj!vs5kNcyU2!cyFV|Q|nhN|gf`>IU!sMq6>(~3e|BBYBqi3pX<w#_G_FYf-pzkvk+4;`hjN`>4aN@(<V-DqH6CDGC8?Fuz^%dd*;8h)@w4Wn&FCfvZXp|&(bWKg(xD9Y<uI`6v-HTp0$8v@kjH}KyoyYg~2LOT4h-2gMYk}gAgD8qkBvsaoKI)}a0yKni8{^&SKc?5y7^dtAA<e5qhW9!H(Kp6~@hAZf1oDHN(-bq3A=?pk)jHhx%kG%6m1e&ls$UrxTLwlCR@zKLi+5}4l=jR6pAYErGP)3Yleoc**uE;=|oyj~}4dBP4X&;m3_-FIM%G>tLIQPAO{SOjqDZ_wL2O4gxtOMRAqelG&x;?{o)Y5Z_tadMzee9jLzjb%@&e7_PvX6P!|DE#b{lVMZ^lp2bA6C8VzlVN7|E_;?{d@1lzojnzf%1%fp)!hn79xPuSB$2MJOloZ5{a=BH$HRmtLy)kA2M;P&piazVy|zVP2mr~5E;7a4+vZJCA{;`1grbI7eDd{j2H2#ejy$Z2mF$N^*$3AaDM}R{UuinPrt(czxat)$3EYoUob&Q=)L$$>eKIugChGgyD>+n=G%P)teKvYcRvUp^N>@eA+>%%&|p9!9!^P$H7Y<Tq4@JpLT|hF$M65$_M4%%H|ifJw_9k*93AZ>^V<&}+`GNro8OyG)A{YK(eZHp;NEuVwUK3Q!I?KnW-wuYM?hVFfHr^PJ-ADgVGY=Wr@uwCIFIm^zrc2p{4f3nKK%e!>n}V4RQ!zjKBswup?^-<<fx<Ag8GRD{uPa$7BfNA{WVvJQ}**0Khvh*r@2AD;tX=%y0d!s?on;<H8aS43lqn|y#?_*+WYWl!-F%8bP4UyTi@N@-rZRdS01muf3UIHg69IU_4a;i8-8eR!2`J1x9xlV?%qagFRB#XR`u*1dz;%R1?{)m`)p^8gz6K6!BDgkdIT!KvQn`Vh6I!)G!lY!Y^z{;64~5JxO%w~?sb=ow7d<{?OS>JXD~;(ap{VNyNCg|yF;v;efgxXzjX^CX#*+7cOz6oTgK-6h&vZu%^g)1$Qq<B?Ex8Ou(!Lvxtn0VCOg)^Ij-e0)e3XR$}wK%Jay4>3=d>V6k)L#Fsez41u-cOdFAU%OLZmcT>8Bk2Ec9Cq;n)3lb!BWcMT*Hgc`9`-`@#I-;$8S3A9zw)^%8ST!--1n#Xy7nkvT{9y6745y|B)kNtR@xz>Frd@E@1Of>hl<8A*IMhra$LB%Akzs-5@u<`lmBv~0vq$i~Sd-ZP}J0vhi!f8Pq{&Nckw5EElWt?Wpa(BKHWc20NgfdH)nN%!FOcZNJo#QO9S0l7ji-mM7lj|zS-(Y$}g>F)7Q!V(KG5ta#p<WE$t2b;14|*#j0eYnP>IHH;We=&?NsAC^h8`#K4HwK3h+gw4NV2$4{XMj6?L3GvfhSZC?0RpLGWH0>rS-ktcAHZ0_?Yc@gtpA-?1F`2Z6wj%E89WhYYz?1n6EdUT&GD$y)uGw+rlR;!~lcA3TnAPMnM*koB#tr4cIx}Xi-`rCpdCcn?IeZ5MC5v1T0dCx$geK)|Rgjj7ru(W*;L+3<48$om9oqFvNNh;#0yk6<Cs-Q!?<0BVKYBuHs9q0exg~n8JlZ0y>r4gM2tprcW_f5!HipBdkfUVhz8FsaJf#W*SQ^+~&c@91x#hwa593mKajuX%WgFnj0HjtybHWqrR^*@Eq^UWP$RdWNE^{+aJQkvH5_I0P_?g$>!@3iK2eY3STvykT*aP)M;ftvC8P>vHt_3yZ}i@bFV)d4B+O?vHZv-U|si<qj7I$yogv}SnG1!BGU|g{Ag<zod+>EoAi5`iH><fPE!PJvPYSed`(+<-hh`uNjMx@m2-dvsV0dOspdK>Y}Qv2pWo$NBgpzxDH(9C7h{pmRz{&xSWN}7P;{t9^*V5(Q2i7H--sNX((P!}aCw`$7Ia2sWxm`(uaNzZc15o>-iX?wg1$M~n1f9^d+R<O-8Gi)qHb8eE1Y#Yc1q}z#tvG@7NjDO&^&WyJ`(@^WjGK9@h#6}>H=I+!BDzc4Pt6KG+#GjHnJ&`=U~lOuDV}R!pc`vN_TVgL~PBRSGOsSd(S@VU<<BV1V^v=<H#=Pj2FNt_B2{dLS$xHn7CGm+PdmGYV~eKo~rnxO@y1@w+==P29N3y(06cNW+FfR&|PLlp?kL8#oS7ws-$c}eSCEjzl^Ao&d>UEpBDXhNR7}p3cSUV9qy@%&A|`KGZ}5`j_4H48{rn}AMz+W8jZ0!X{gWXO^ik-G9E)%^xgP9MOEu3^Ywqe{x8@6f&TyRuK(fspRWH;`26p#|1bFbf0MF})n`AVTqg*n@w0~a@J_Y<$M63g{CV3$r5-0Ez#=Lm1nV1rNZL=-V4{Zh=Vbnosz_AWvO3bkGe$eZ<A<L-LZ_BC-ug5GRD5h0=Ucn$Y}dU%{?)(two!KlWBE$tL--Zza}9;xZ#;N}5Feji|BAA5P<o`lmL{byJb3pjLJ0Xx0Eh5%c=<0xinkw<YV%7}Kf<S<_{Ld8{X<3CCgvGvuJ1FSdr<fO-~Z^5hVA;l6FGj*3DyjSE>!pfky-|5jBgk{L>0cu-SU~ET^v6#bNsMfBcc$;CBpnYR4j~fDjCU7Wwn1L@<*^<^{20d3~$QrylHjk%^P<opVW%y6K#}#L2xas>Ic#}I)c(|8_^2)H$Nl*LVv6IAa3q#V5ZxT@%#1vPHJYdG>SYVsIsT#*pz^o^}bKE|E<eZzy?aPBRD$n7Bs=28R|p)VLNVZ9Ebz}A=+pyr9GQ$zF<oznMy@gKO+3+w09(RIQI~B_xH-0s*J2(aeo~h`&O1E7@bimTm{dm4-us!iyQZ^|0|n9u7887dNggwroW`!&sG@89j<>@HeuJlWcvdpH_UhGZS2M7pSQ%HxV^teUM)-^kt>7-^ecV6a@NDoX}rQPqh~WhJ4|WbG+);@N#?WJj#=-%_em+(k>?LDPN#55Nk=K+gby*jB^(^|g+9@-G@td-x!;hV5Q%Xxnv6JJBKvHeg)9&Y^u|WT2N~u5=H7Obb8~+Z3n^Xry5gGN-(BBjr=1~Oe~4q!U7*y^D(*xJtF>ispPe#d9@Wtt?VrufJrG<8WtcC$>^(1{@I}q!OK(?pIvbs!--FB~4YEW_1HL>$rwRCNIPD`T9`0^!JI3kx_a4UgH}>*<g-ayHny2#w#Ssz8j`Z~#Wa8IsxLeJg9n6aj;`ru10iEVLyPRoQhQ))B$#PKYsie^&n}q<pVr=Q75n`X7_9%8!sMzVErj%$Tw5GlT>}448YEw%rTU96nXveB90-zN1Nvy5<`_1jmE!4P{uX=5bcYKD!*M+aR4;$imB5=CWjwzXSbCbhMJWfG8PJmiP3($Vexi+$;k!#mnA}_x}Hh@$QcH;YcyMNx=@mWw#HKDR=xs;;*dW{TNoWR}GXE(RD#J#9(C>ZoSItqzW_)MkqS>Vzn%qGVewljl8Gd-z_bnh`d<+#;$3kBt1?O|(cBWn{bjZmU`G@Oo(64QJIb@kHTY}UJ|Xx$7MoRzUzgAl!#BE+O$CC9c}b@dIpC1o!J<P}X3;=&zDVQ%TA-g{d({6mil6Q8Fr;qw#&Xe#|s-W0-NsFVj0p;2eL1p01Yh%t*zZ@hx`7f*VXjwZ(qyz8)i9W68<DR>*L4>6%Jq37}T^G?pvxi{=RNxZ`(nJBH6-2yKmQei<q#v@d4kJ9w4H-U{`VgDY|EC-m628#`oG38`tWw`8DYQ=3|z?eE0R#*q+pnWeo8BHo|9lSNPZfuizRQC$vUzqMWJ(5*Hf2j(9VD9(Dj6?T>cPH~(lm0Db1Z8L!spJywe>9K&{_ZZ^Yd+G?j<yBh&vOk_sZf#xHT3~$%hXo`$*Am96uXcD2{ae!Y)J)1y(tg=0yTD%dBcVg0tN&HsWi`dB>UT`byuu-!Euj%<bYXPZc*<273`Q6@e@;Mq2b8SI3TvO&~%Q@hrKbd=O@Mrx<LB~te<zIeEaT6H0lc8bx$Nrad8S>p53ioxY|obuMek9nwU@E>-#&qAMM1={a8hku{2?4ad;9Lor1aOH>}RjXU4FN@=O5SKm8|xd(gmi-@VC_-9v$YbGv1fHEYHlAN37hud{dxHPg9uJ_(Ryfv^<~RY@`BFa0?~QuYm33Tg`)Md9>`^G6^|iOZhPXQiuBvEGBL(9XRfEc5yhO<Ed&PND>^1lFj*(%IVBJ=pil$YQCC9EAC%_Xr^{kCW-~tam!RfKQWThI!1`)=)v964-p0%`8FR7LhP-Lh~oGNToq_G07|6!#9H03rJr=U7D1V*O!_ce7oRf=iF#ilzO(t|FnZFS%%PW;NTyk*HQ~HOE4uQ^BwY^((XWR*XO2pBAhyqfo8UB78NMAWBP<e0s2i?y$*Xvj}Z+<KjE3vgMfW#%HWrVe|f_#0KAK07@z+utKDV%YYhahhJdRlI(A8lTc#bud4#$ipY+nlX&6VEw3ZDlw3NEYbgigO3%N(DR6oTVPK*vb9<>)byezuH1uTkWMplw^))9VUqq!Hiw>BR=va%TK*+FR?pp^iXds&3fMy<Xwm`$I794NwZStP(~8wz_#e?`i19!IXP$-Cci?=+Qo<mqrUm}@=|?uEu0>~i(>l0m^@gNxE4Nkckk6OwIC3(cwmQe=!KXGuw&ONKVilLEI270+ug4X?n=%Qnw?H)s?ud&&(Zcp4A5qgflejq(knU*R&thpz9DmtiJ>-w+-sDoCU;T(~Dyf6$>r+-|a~r{zkPlF9%vkmG_P3!76qip)YQ<ghFuE=FsSIxvez<ftLR@aZ%0ZhlH^xVB8&aDh!RtVuWhLOWH=T1)0%5Q9ROHCnjA1dD?Rd<I9)nqbde0Z!WRFB!Uy*!S1y$5lZzbn#Ys<ttozWtK#$8i2E6%+g3_xcT8)wyk$vw<D!Q%w%NRqLBFVnoD+G&dhDKw_wvsJ=*J$9$s6B`FVp#`+fmZL=k#`4FvGI=vudQmMDiV?Cyx7;nMknV$xz6buQm{qZeit%7(*0SxmVFrb~@rnw>Fcz$}y1GAm}oZN${t2;7Za6;@AnT)EhoASKK4<xYf?laLPhs7NCvPt(lhFUUVj{c^Tax-aY<+~5xROh`Nkq|+JRpXD{%zm;(|tAb$d<b?EP#mzGFj4mIO|KbzfvHo#LdXke96}wNJJjp3uyvwWdah>E)rz_lUN8B;G_C;Ly%TH&Ejsp}nWnFk=bwgD%iHZ6ZKXb%i#AO8GjoT`B4Ao&+k-wO<>NpgNEL@A(;_0$FOMAlWb+Lgb12gRC3(u-9y>WeI3%9reYEUFGiz<C`q&qp0<nOA`TI0SGR8h-J8JIX%Y_MePVDSnpsUa^77<wZ$Pf86ahC_S%BCO#hqpRPazmd~IxLx!NI>F^dNOcjVk0OaqrJ(T(HVS^p_ka0G1=g(E5-0#db5%r23rL~3l~*D^yX51tfhpNpqo-4FM*Aw0RvWKA36X0O47h~FFIoj%^eZ2Y6xWI27y>z-p#24JD~X+Xo;Zu>QJ`da{Wp9NWHP<+8a`Rr+oc42RcwJczd)4Nqj!;iKIWjZC&uYy?AI)BW@)oa-461F7L6uTB?=fgvTSd*flH9_Tv<}$2d&*KAy~0$bD3T4t4K24pprWZltw_TQHOgfzZqNwH#p0exMAC@0amt<ZC@#0$f+KC^ByI1Bc+kwI61kpk%8L#Aepvp<G9Y%@~C!~lK;4Pj^-?)#ujRb5^6|l2VUiM%*Y82(PV1$F$=QywdZkr{UO}y<INq#!HR(RR7DCKO+De>Q`VdU#um7O<a5KPbl>KC638#>pce5MZ*R8AIE2cGLIZKkG}WYs6!xdU8FNGi<|QC$fiQj0E#WB$N@}tya!Yeg%##wCszCKjxpagcPe4BrKn!No75=V-3@)?h3iab{@rr~}(iWDYI45f&5jYotIZKW>rYc+gdD=pu>^epBZ5v~xdIXIynw8}^nXXVwh}z4<LO2|R9EB)$QbmDS&d(wXQJQnj5Pl&jWkeD6PVK;M4unRiMQ9l~wHAQ`Xi-Fr6|6#PQtWi+r2@H6R1)lrX6@JA$_A=C1K`v(4@R>}0pUP`WMKjZjs18%QzW^ySI1Hsrpsqx_8ix}9(RP*;UyG0S^w0h*B5AYTn*h!&qHCDu9nG_#Sy40?lxMEB<O(9td|KeA?hLn42-1@`I=oD(<Mk-3p+bEjdf6PJZ6WgNjCk2wf41%eY7yJ5f3wgwVTC9iNvY6?j&N9k8;s`D7KOSvs3ae@l-h%<yi7ce#Lk6yBdZ>b5Co<FUt#ppei3-D(4M9mI-hSnZcYbOKNaTr7Jd`%SiC4(nFc2n~vqoNj~r_bOiG)IfrZ^%|%<N;0gy;ty@Xmup`uoSx33?&<OHR4bTU>>-b;(6(NnlR2msqvI|w$4(DSz9rk@Bx>wE1?$t?B@l~2Z>Gi_J1&AGb1-ddl%aFFrw)jDl42Bt`p4JC-gG6(pV+H#APp2z%s_>p!ky$UZXi9k^@+Bf|Xi<`@wN;GPfIq9Dmx1Ww^HqL+XAMqPFCC51_cR@k5N#-j(qBF@#y1x6IG<owrBFes4PjX@;t+bZia7D{g^^(PZdT)i=A-t)#jY)ryY{~AIE89Wl+NU9C5DUb8@ey7VL99GI`;<U%e2Ooi!9|lp?7254a!IkYE|GAy)#jsrRQY;Js8x3ZpkHWPy#r!kxyS!88L5PI>$)(A15GLGCZOolHyG;fu#Xw<SC+I8F(P*Mxlv-gpDB_B`^Xwb7xM;qXKU|^U?ea^>D1k*Zg<A;a$4v>z);e%F%w*T5o6E$e1ySc;0Sk$eN6w<L5v;cBTkZnG$LT>Uu=lb>Y0th>k1L_H^8vyLOeU8Ri2pH4=3X2<qY86mEF8m7!uz?iCEpCnQ@ahU<Qmq$W*wCy6&0okORdPG^t3!?Qk0{hDto`4@VWhI38K4yI)C`Uyib%rP(c*$kaqW@i(Po_hx>AvRi#d2^x;9NlFe>|u&*%`Ieqa~;IAhs}12wBT#}U}JZu71hL-{oVcMR#db6eL+l^kB%N!PKQ8)a6X;)#`Zos9#0Qp#b|AWT8&}Wl=+(8C#v1(2fDb_$fKFt42&-4t~&awQLUkn*|^(`%l4lL_R2)D;?Nih<KZ9`4qy2>1<3uxmI7(rSasKo!TQ(#{>4vJfE8mYe(qanqC`+a#0w`;J?v9L`N;Q637aRNP#9=*)O9_un8iBa-(t>Wy>E{xtNj+|e+)`m!D8=5#(-5Ti08;4L@{QJzrTgS`~`q{`L%*I8<b?KIQ0w$Tx^@SUBeGM6&Vgen5EM6D9mF<h(PH1&rJ@Z!^gJs!YCGJKO^C8$i9cuv$=TBa1&jC5=w0-l+gc(L6O&=z4$Rful<37Bw^`j(4TFA`uTETfXSQn-qAbBjlclE>0SS$9(sEHd5Es~E3~qIMcd>1p@e(mu&`fa<erWN`1$p3LWT2&v0hoo9_%*?mm_dK4qkHs2fkuxxVP)?B(>U|+Si1JdmEwQkQiIL>+iQVJe0t<ynD@^_t(7ry@Qs=VOVrsL<qT!&2<4AhXEjM1e^AD5B7m~w^r`K2H5a8zdV%DbUX;Zkv<+HvS0OdbefDuaFUEp=s<fUz84fnMA2!gO0SbDav#CKFpWzx)8XY>8&bctw%yv_+gw*)QT;7Hv-_R=wu#_&`^}y8R`xE^APd^l4b1b|h3zMYojvN!4-v~y^-^FP?j^KIY)kU`JuU6KDf%bpN6G2j+oXEPf3#tgw)6xA{TU;q;_=~$?9bMNd)ozl!Jlr@8>_t4A6z(5XON}*Q`fpJ6%WS7;sNK1-rL-GfHKg-2lwFL7UFnpuJ0gYCqT6UdJ8xA@9plk_XYZjMvcH>qo-H4HXl6PhmP_qW|#Fl97|2yJ?y$JFN>~WH}NN`iMpwtXz=4X7CXqG^P@{F4Wg`E715<W2>3xFU`Cf_SFegS(;O97A$uDID9PN<Y4|uvPUBfJ0NHhNlsH3{;8%q}Lsi@$LJVnRHbGLfGU1Yn(=1>eoM$PKMPDYSFOeKdpcV>svKS7|%os_9T{$BSJ)M1*7ovfY<N=%J-iE%2JEc48_GY`ix%0r52ykLf&MW<aOYeAo9`y$uuC0rjw(&*9=y$%mi!z}?0TgKofVqOR<^q*AGH`P9#p>08uJ=IvmzY33>EAo_@0;}R-PjMxl&k0Gm{JnDNvo8;1#iID@$|TUTES4j!&E#Dyl;w2#awcAQX>pTbHaT@KW4l}ju1RHbwvyu17QopUX~>h-ELOWA-X*D=Cjc`dRsybXGeGpH?T)A9SLDk#xc~1_TD)5Zo@ro((k35q6_~$MqoP5FXZekpXxFlo_OHj0{MG59vxO0pt?c>A;@Ojbk$*UE)c7OPNQC1<X$YA!izUn6o_SZdQstcOG2z?QF0={#E~alIi0Z0J{Pns3Nh_W{iON6zzaV==e#YU-XV9@=jU41vCjE<XtdQ0S>qm};zWdm=IhZik)5AQgQg>=MCflEJfalXEeU3{aj?Bj>BbJWciJvhx-hpvy<*|H1|tdqdm#1{g!l>OGYAEPLHZYQdnh-e$m0r!H-~AG^g~pCoum#kBp1RcE;_=B!)#7^kCRvmWR>;^WANdyKbt2r+&w^;<BP~YN&2JS#1DJ@K5ir=*mTDGC~VJh_0Rfkw4Ti>OXIK;By^=WhHIxtpTMJmdk#MH^O!aIxHLK8e~`gt;MS#s3di-B)T!3*@^L<(z2HAgPYN7p7TvAYY9R+{lP#Q+R^j?6r63A2#iV-(%=W>o*GDTmi7cbZv63Gvs%>mZjsPDB<4fTf@I{d72k`{BT57xwkDny;DvAB%NQ#50RcmkF#i*W`5h7BmAgaGz(8PEZ#E#>$ac?F9?xNLdEyS4V3n0(*^b`T`8Mdo{XvH*Bf@m?}a)N&)qe}_7`XFTwjo;}>$!f?^>23%^C*tCP_+~hsrfEDlo6zwEoM5{RV)m0{b_ka#L4bm)=R-_i#uv{>#RqR3tiQi;FW!H+*J^H{7ptOQB66e<`9g~Ki7^iX_72l4q)w)+6IO}q#e_9(FqKKHIAR%D;lzu1VHK>k0t(<d<tcf~u^jREiO0Q@!+tNi1B(k6=h1L9?#m@wygI!}{GI7xT1Hi#qS5>!TB$F$M2iCswR_hm1`vC5uelvRfYTw;(KzIBfiQ2xuNtca87Ds1Ok_QS31_d=s;gnf&hYD9y^^Z=<K4SfrNoz@Hf<ss_0_wfoU}%rsxyB6miF`N9jn4@(^|EzuGT`a%^Gjob>69EKfYzvn5|a3pLc6z{amfp%zj>(hdc<pG2)Y=OR_lTwd<X5Q&YD*+IIDCt-d3~0OvjL<M}N7c%FcW0pbE(`ik$3)~X^UepDuS-~~NfNgXcG?bN!#8ogYF0adUEd>@k$D@qb&DJiE^g<)w>AzYIr3ydWw5g3J#$L3YoC=um*A}Ndgnc@1BlI<tq0@6h+Q*4o9`Tpk4=6=roSa2eIF{Ex~WG#~05PSut4>hm(2_4#VHtsP9t4vM*=-?j7qu4ouVpv1N76Cu79|(TcGWd=eoqgPM9A#zUl*PQ)yj^2+`KzkP#>G;NIpHi}VMZ)_?yae>N7g+ahS(YK__Qp~<I8dH%Fo~Kt=jG^U)!zK#r4gSBM^qrUF#{F3$T(-r%yp9Jgwh8y%Pc>dt;bG;l-Sl5oNA6US6rY7Z$&;_|pG1No}ssnaC;_FJ7wlLjCkL{NFwnWB)o?7u?w~>Cn*yT5k#dL;M6M{&<{HsJb_wdSuuik3H^nmFu&(5O4!lP^nN3J;*%BjXZC2ib@IPoC(@nx<UaRH&2jX2-;)4OOv>tz##xa0x2*A1e;)|c_<+`|5#@!RDvGz7!OmYGfL1T8O`CFhgQ`IrYfbIENAEgDJ2wf8-x;c;Yo?N>KrEb!PLWQhq_eCs`IXJgc5TL&W5yLWYT+8!;T^~a2atleWH4;jm){ZW%x^)ong?ov~m%ihEMHoT<B-ZMaUDkcB(%;l~14QPoIVh4x9^d+MMQ(nhWVnkvoGlALS35Tbmnk3;rB1um7;MzQ4PdQ#eqsp5lV2=v6%pUavfTuNH()Lp!52@U`e-<<vPJm;Ol)v#!w=f%+(Sp-|7kdl+}=`LmN$FOz<EmX(D<Z@jVUjL-C+Cb)=`er&cOQh4jlgj_U@p0a*f<XV2O#!G-XpQ8oc7mp`F59GA6Ucv<*#GJD+W{plChrQW}IL1k)(byfG@SK0vUO)2CR<W58)}{GD^4HAKk-ul|jcepkB;Mkfgf828?S)F`Re{iCFNwml4fhRs-nc}GPhac8xmR8{?50TDQAnAmRuRqA=8nW7<jSufZo$~oIBh=7o8RE+^Kt&59h}81Dv3dsNYmzLxrAe6r+$MUgm1h7L$hN}D6uyY)XGV#*#%4|)zCPT5NC2iHeL7#2lo^)14;i$kR84&oPpB~io8f>*hV&+Jqg<7@)DCh&z7P{ioV8AGY{T~KFY@!Z_JM+>0Naq<+~C5D_&#zcO&-n5mn(VK<)5HJh_0mI7{H!g?fMsh}yCt?zn(9TIL8)_NlW&hbbI<^CTvr%_ma>G85o)6nZSW^xZSe^q>mKpMhhUl`7M*T)_B#QEJjh2PE!<-U#|Y_5g`m%EpRHdNF}B8qPD1bC(HEC}2Qn&w#V}$*50uGdRD+-qP`&DDZ?R8xw*K`8(#)$}9=bI6(o_vJm$kCpI3Cf!u@DlBoloco8RiZxuM%<o4}V7nj%f1-?2TjhVM|Dy!96y%w(4R^e!Z8tOGy6%Xohtxh))*rG@qbT0>~WjsBe;Px3;2p1B!@`F`blrJq<jA>Ar1aGW5fI@tBwN!5F@Pb`wj&Ihv9PG#BIGLptXe-3_YT+aoqeiy1-pL|!w&QwraT><>zsS1zhd4vk5xS~2-mZ1EuW#2H-OwCRfu^Zr@hb<3bHwb1+R_Qlt3BQ8i_I#!v~#M9J9H|af^3A^7S=~)fJtaNkNsOvy$<@6U;0iIC9W)wqC}PjY8z$gY%fY(DBo4mlj-ARP34kfw}~vQ%wtI*EduF6Bu`S1qV1cnIPA1Z9!}JHa*iB<R3#|mddIU-KlOkgd6{dew(H45<zS~Bbt-d9G6?6ofT2g9FiMk*ooZY2gBN})WXen}Vz!PhZ=4`InO9_Y+}>XVL_qF|`SNc_2o8M#iLMT$??jo6L)!ea_ZhXp5g`y1#t?i%#t@Wt6(3=Dv}%-(D1wjrp?ckLgxx=zBna8p%V1VDZ)02>hp=5Ahv-g^Gc@(+*1{J{Z)b|PIs|GRaJn*%p5wzpTN3O^1t$}0a1BSKR}9~!@Plp$A5>%vKAWUbXKpP|u^^4L9Yr!ypV6r``6ZHph~-HD^$ih4Ls`vI%idacXdT9`mtc}fG=&Ju0Z&}(4}>sJLBHL#`3&7W_z{Qyb`{VFlz!s)m7qhqg(fo)yU2mqx-+mpX!tm}h}an6^+@RL>8V7DKwre6C9eOt1r{NlvC7xjmQv+gx=0)v;zoUu`$FEW&GjFhpQMA$9igl|5BpD~nw9COlg2pggeiS-o9U6d4)vd?1bWaD`1X<)uq9fR?u&miq&Oie$@~C<f<BWa4~f9fL-Zhv(PK;4%a?2g);p^N(xVRPAG(AS8?&APAJV6;LXjXiiYH6gZ(a8AZTwV<E<nNzdpox152`DBDLh+ZA%dJ*8Q^)rRukA+qcMWOUc%gb`r4?E^Xv^ThgaSuQ}(LiU6Q9ravoG;61(I0$`3vNqvqa@zo4IVYK$)$>X_nYee%FAZnLP`CsAicdz^xe<x;~<U6~o7pEFGindxFk6jjtJRA!B0Hv6Q6q>LO-*yB%h-ZpDsXXMJE_SKRpg|@N|(T=O9QH7iB$PLg9wSZgso4y$eoBXDCe>6`;ggHmn^dxXQ34}OZ6g`6iu729*uoJWp5i>{ur$_}VY;U%=oBQhz_1R^#D3*Fq8e6?Z0@*xwdawj!sBd%hT3#BZJ3{%|#YJ@k@fO7!vp;;O)PnNUDw8>!k{!{<4>dAbsl6x)OIFda0ReIfytTe51Dr(03BP733p3Fq2Pxj~oG5AG`H2(_gtljGo`?bh-pZu}(m-9rmDC-E>$;GNg*TrWPyr}=1n<t1s&=ES;s!-L5F#Oi#-rs_0j#&0d+6&eE_woGeNk5n6S9?b7A1P<aDn?L!k+g>DT3?FxoA>Y6o$hdf0*|i8;<YqZf(R{t>*iFu&8~dDRi}%{RnM{G&T!b$buB)nXkD(&Sx*Fn}jvfOUVJixql?Dk3_;cjZNBmZm~%tTi>mgy3*tV+nh%V7|Pl1Jo%q*JcL-4V|RzAk;9N0&Xcg*PGD7I+bTGdV+QBs8^S9$aCjQ#49ka6nSD3kt);r5HpzBtW3##A>dWjf*B9oz+AUdjDD}hDP@5p@Js}WS#{hXgRLp5DrEi|~aK$oqq2S6V1YzM~!>_z_nnyZdBbTw%eIZXrWSYXN6Y^r{%@gtfON(6Z%kNEaOR@Y`3a(B0<!axZbpPYx1|23hVM_>gcOq^PdZ#4yau~356h$?<=NBccpdazxOc!TkUKU=M7s&hg`b`#Id2ME>SqCcbMwga(D}&v&XJO&Z#OE6j&9{#eju0b*G?dvDfbt^I0Zf2T#+e4p!{U0~7e&7zE{4ifC#L(Vyhh~Z5nWx7tn&7P8SjCXpT5SlrAI^gOc0pMi3?8kJJZAQ=+G~3j0@34wo*)L2g%g!+L-Ufx5MlnZXYfZlulnx(in(g|08Hld`*YFWtw8$dHL*Ig{bMh&1F;nHc~&tFiUi<jqVg77v1!7Ose{Ue;>q{B_`H6uX20CM_fyvz6SsDFUU8p79!G%*u30L`<CDE*h>zcVmM8$Y@hP&9>(EW<M7O0ExO$MAl>fn72U%5e3a?J@Gf66xZGZ`yE2R%1ohHL5H48$vdI#D>2*_7p=<~aO*>y&wV--sw=t6l{|Ck{pdjmCaQ;?GxSMt0UUQEV%l;*20DbW@1VVrpen#1YF*Ps7X#Z5EQ~f38$z6;zL5#Sqph-63*2fqZ96xI9Y+$5cRsp*^Sip^%4@5x0#?Deww-T5^%Oa#&3vyES=x~G<(1hdHSj&bdp!%J!!$u=|N=8Bg(dx}^6$KS%1s2o{(q8;QGY{t{&}y_{1rEzZB^~ui5SN}bnKw=l5)zzNK}Y*sWq)8om?FbXhVks$E}aFSn9BWuPb}$J;zZNK_`D>ITrkTdtf3_H47+}u`6`l6M3F?%o6VwURnnod>8G;nC8!%z*_EQ)cgO6xyhqw`hMA8W{41QR{dWg%`{>|!cmd*p6@TbxMPCk19W_k1p`I@F|0x1bEAs!D#2M5QiN~z-H3V&5TP2`wl-~3BY-ZNs@H;H7f<YV13Yhi`h>SjtlOrW?<1Lnd=Ay_kMe(vm27?6NeUiY?RE=niTD}tMUfdJA|CF3{=Q4Q3k?dZ~uF1>vDGzS4|1lXtwET5H+pKE;YJ0k10}G!pcZ4^pW_6aL1%YDU9D$&u!_r>y9WEef1_@6Wvo6Gf(=PH07}!lXm^t(kmDy>=ppa+^4}T(K{G+HI`ptV=&A8pf^Dw3)+ODHtyA{QfL0Sx^x3q4j+g*|ycFFhpD1B1l<BQ|V+|klBIdMoCJnVsY3rQrmQZ}wYqYABu_gkO1#UTxfi-ac>A2eJEtXfD1nTw32-hj5GyhKIhm-lK4mme4DlMF)=jjkx{>hR<&+elz?!fh~N4dJNw11H9tw1%r*7})eV!CPRA$zfdyJ6Hxvo~|(fkj@vb*j+B;hES-VJXuMTF>pBa$w3B=F%*|B5*bpg0itxTH+kIfLzE9f|GDf1RLD0TfpH`|i$%;FxuSsHSEm<@0XMhtA!5jEAe_#JZ&qtH3s+}ncV}g%^`N=G`C-dzA3SR9d0VaK-cD<eAarm=t?VD*S}|(v_X(#Cb48w?CbJPj3vgm#h4(}@iZxh8$GxKq3%L$&b-mdrMTn<ag6Y9M^f5h}S8cSNqtWR_bqY7T6W|3BHU|Cj={wx@V)>;2rw2Apa)SL4@3zk2F0(&Nl2nClQZ<g}3*l_ej}3I4k00DK>(n1u0dNl%LhYy<ZKLc}3+Kl2*6j74U?%2%Toy0qjL>_=!CrFt#5?i7-o4jGn^BZS+1V*>^w?SGW>HMC;AO@@LiqxjYc_X)N7xYPR`i!aT{H6blmpQ8XMVW)RxJp#2wG7_J@CW3MS_VLYAaqe9{M50g9lp7E&B;aZ-t2ANFq#BuZDP#)0Y?|SXgZMtPr<C2J?QaN};_RzhA4~3Gq#l3_c>&wI7NA-e+cLF3mFMFbByjLK{UxI3Kg`aT%7u$8bu(NzrPx|D<3BuHn0O<1$nHrkG->F9T}5{bpe_wH>_wwzfR#Z<Ua=EEr$JDhu*t`%b#x(v)by%hjtzXbAbL$ctB0R2Ff(6lOradi9&DjXSG)1oBeB^Z6FkZ>|<r@7Bsjkq;|ZOo~rn;vlTLZXrY*Yt)2A!+BuFPy@X2!2z6ha0QU8E7T~Vby+%5Yx^S<v)j$p+1&Ufy3ClN#q+XRST~+$v@?OXif&G{PH;P!-4c()lEg0>!^ZwGEZR8<XXgazuTg>!Kh-EY=iTliU~C3LGtLdHYjdv^12@-z*4#;L(K|b9uqwoDRMKTL49y4+>x~xbz*;-&P-A(r4-}f@XoL5nJ6e6Mopx&*v42HB+XQ*giY;Wgz#tpHszdjwhbo<#@pN+sR^o@vt^6-q`wG=B`?Un{&wbFx97fNs>bBvmAWq*t`GNC>=kC~itfwF>|C`>n$PO*!=IM-`$`yRQcX&YpVy4i@I6{ix9@Deqp*QK_NhQI<?KXz6z%ZQDL*W5MuMnN-Lqg#Q703xe^v$#QUp{?MN;`AnUjenE5>Y9`w?O;=LJv46^nfLy11?T2-~a_1aE`D6{0`!Q4ST0aB^QOQ$8w%3=m1AFXBM@o7Lb7-u*2{H9N3Vz?yAU3+e(uuwb&HLmm)IA725GAvm~eEgh8rAJ;e(+CCTAjh)pu4PiUj$>()wrgmmTK@Nh=_<or7(_SM=GSD>w19i!3qQBkDq6Gd~ilaVhdV%q}NBzjb(dG6G^Ytqwz5rHu>=@{LYtZ>&yD^EI&5M}gkz>s1!7lP6xit6&Dg*TyiTB(;{Qhd{EpG>FoA#6h#wB{i;4=2)jn)Duf=@9O%j}hX2KE)L7UIL=y*b`H<MlOT6!L02xIGadA_S1X>FGcBA9{n#*SDs+J+bs~B?@*-fE%V^`sD)=>EY4NL=tGaer&QElxM`$rffZ;yJg?y_56)m_CSkpL8&S=KwA|imfAGi;52t67ej0TuwJH<{pnk{E%0PR-M*ctQ-o3evE4vr{Dn0fsH(G2$044bWx?Kk%Aqp``a0pVC0;9S>qCr9g0So|=pcSQT&ogJ{#+k{al1fcxGAHNW``4+`c&u3V*qW-7|8J3BVLr)SkG*&A?%v%1WqFdDb4Dh*ckk!gYp=Z?zlFI#ZHk;M*}?bVw1-B3x)Woh9y*Iq;bV6&WN;;nWS*jUBY;*q6VxXcgT=QVn<A*O@G(?Va5XxG5$30icsGVd_V+;`RuRa-Inb)pki^t<jZ8@BO`@UMf~bL8iooW#y+Vn)5wA4f)3fX1i$p89&VDouGI#32dOPo=yaMW0?KlfKGluJiu~$gNgnvj^1UxqcZK4y+IOt<23=nDB_Quhly#w2c-7;XWi9rDmejEFt?P@`e^(smk_x;=3wRbC<DM&15S;w@Gt}U=TeYAs;`$?Rwjk|4A9QLx&L2Q}fx#Hd(?AkfQuXBcXXw2<Xd3$24NaKZkK((W6G6G4=oZnXJ9^0ppB4MIh?SSJjsG48jC}Y@FdUL_-y69<YWzs41&)~eOdL*D-SIWC}5FTsWtCj5(%8T~yKiywd6aR%z-W=Yl?b9oGtEv{(Dz&xk^449IP14j7x>$L}IHhGgC43pmgwJL=Vl6A}g8Wv0AKS|+rIwB=EK6%J!t6E{gA2YgR<Kaxt5%)2mr6>mlR2K^X+C6<CmU)1PUSscx>k_vh)^Go;|M|mSuYaN4eS6!#H1lwi$j)|utq6*)^C*W1Cej8R#(xg6<!lf&Wh=-gd;EY8WfGWirj%xbIp&(0-?urY-ur8qlPXOrv@5`>5*3G95#U7hVCfsi^NTHE(`Vy42!IE^_t-ChQUzO(KvpElZb(`mB`J{U|Kl6d{%gr%{CRfP9WGB^Wa!|Si!>-`I5!T6f<=tSFrygvnwEEX3y_So<y{<p<xatz+Omtg{AHabUA~mBia8EOwZ%%j}GB~FetNoSqYufq(9qIp^5%xrdTq;xx_BH8sAz+|Erz5WjNU1tI=pB3-Mu?aqcBIBARq_8n6GLl>4AjXu2Q3)eM;YWH9tZ(MAam<|xZjg5kR&6N>D4oFz*Hp~MYWV<A@Kw6WBT@)6>joXvSKWcCV|oGHakF<&}!uECKwIyZlM=F&^7;DhckCt7u(GADr!e~Ro)LY0L5it9BNMIU0{fA6h=r>;P7L+lgMhNu-SBDrGTJ#I}7^ZP-!m&*x?UMM=(3i-F3>*9BKM>~YB!hI>;PdshCNVEy?^rpdKZ234i-U~3HJqWQNjuLjG=~&8%ZAg6BmuOFN%3!J_{*y^=$~tbat7%F%!`2f04LMMRR|v!zan0mkj#A5v6zYxC#=sN$V=f<Hp|ol6A5GHSFcVh{T;o>a4-<Qk6RnVHV#i`E9tDv?P{Y>?gjcYKYyzenwjgf6WxYgLk*>XwE#js%;x>*blj(7er?(*|vU%;4R-YfUO9HO9SOr(s>&(^*MZh{kB{h0vrR#uD*tE&iE-dZHvg~CO4p^j^FqTN#BFw9(v}bVt;X&aL#qf-jOqqmvR!&_ksP=%E3s-#Y@D(2mj75!(5t~28E!k;x16XEd8}<L@!6N92Tr3sw8X5G6Ml8!Pna79C;A>J(x=eW*6?dBisoaeg^KB2<S+qX08vk1o7Yt$*2SV_FT^qWGe3eVV57!Qkhg}L?V*DMMmr^ne9zW~fjJ|hv>-4}s>$cINw{(rFWKIl6D+x!HdUbUdhW*{$^5zbR41W0zOawefA>wXjce{#cLYTmvlIisjK>00GLxefjQ$6$#P7sV~5(L7zPrr)<=u^?^S>wZQhp;v<5kz-_=-mR+2Ku&4N-eI|BA2s8G%xUPH4%@K3PT_~@8z%DY!R8)QjlgCj$*1#>^QrGS{1n)cu#BF8I!DuLkdccP_3Ce4q2W1repZT(3)cLbKQBQAYddXqQV(<h1<|&R%@y#c}U=V`iNkI=UnpjtV&{o6sJw0IkWXE4#|9WnFlHTlTd?wGJe4ZX$}`(Vy~i?EjMz-(5@4Gk+E7j`KpuXu~|J8kB2K6JvKe%hEFgE$ujBYQx&NuH;+8judvrsZ0url(@CTSr-h+eO3D)fYjkeX#?o8~g$&i5rZt1FW#`gGaXtss8IV#}a)%gqjUA_qXsNLGhHyF5#e31Xtxi<9p<5!xQQ<%^QtmJ#ATAaJms~{^bry+06Y*t~?Nd<bo$cCRRyNsTfKIB}QCV61Ke^10$!d`vpeF9GeAt*Q0!b-p6MZ}q-VtF+OIgx(3wjz&`{M8Pl#Z7_E6E@uj5+#{Q%3&r;yIp}iNvI6l0Xs6Zx8xGJm+f^v%(ZQjJArnq2kcf(K3|XLNi(j>qNR~LTdVwjulewGT2g_m!=a+wDxT&g+hG{MFcAOXI5z7(yc->OG%?h#gZ%%zwWjd?CbUd_nB*)*-!y)SLqRgQSw>vipe4-@{N9`)3TnbwPZbt78L2Y^=z#(_6g-ArktuzqFaR1s?z?r{K@6dFTS>&M564J-I>}>d>dMr-xzk^A1?p)^2gW)MFLV<WZaxzz$^GS&ABKV){;<HI!Py@ddeq?noTXqOHJCxkxP$rb?yb1V@!kY?F|M!>4_EVws30UWhGHRSw_zaPcn(Pb~{pYD844Ec+4p}OhccjR}X*?J)IF!Kv4zA^m_rCbtt`jW_RCz1zQMn%F7NL=0Si6Mag{GvaJPEC&DlUX;*p7fePgGBy?GDE$r^GYh_AqcGH~1G_92>Mj~}5P;I+QS>a+y?YD!ylWwmKt@Q_eLebzPzZNGd#N%?Au%=MaU}17$3IPBybxb~CTD+G;9=^U(6*+7N$9vs=(AEPo@>d9Z{WHb1&JHkx!^8it9AqZ@rvlAFJM`|(*6xlU3p~R>qtsZJvT&ypF)IdQ$1#nM#@1h1xa(|p=0=kHe%@_lOfL@7yGBTa+C6Vr&8ybR-O9?lTQy*bR<&BPWCnUpxvg??U;a!!%E-b^Z=Uj5U{Tg?O7{%~pSc(-E=~<NE<Q<S?2dC`_JwB#?1!DKC+z+ELm{gZfbkHm>tlO)f0#c$YIjEnpBSLC6S)lsQv{719GMc6hM)W0u2Hq#YwZO+mvfJT{Mg9Zt)5U&>9-+QpLI7U<aoaQG9R#A6%aLBot<Mn244o$mDSY(-?0&@o;MH2B#q8(mt)fBYb3n0tL{QSO*F2R#hz(>(hv0cTE$(hMaII_=euR-H^;j=M)cfnFYxw<*K@QQ3w9nB_<JY7Mx13@8N5}BM>#C(YGI4&Vnb>AJ{V8hCiyQy2%7x@DubI6{F^5f4;%NHh<?U~9nH?59x5+UU|<xY2^U}vmc)t@D497Jm`yi6zgqRfj^qgl(aB?~KUAblCpirv^pTC)P*P-&_;RnZT?gVb8A~)J$lV8ZvZR7g@ZH_Yj<(0_D*>a}!07_i)J5hEs%V{JIugD>ml{)=wa_bHWW>%z?%7Trv;E}kYw|YjIL*h&CYkU1t?@qWR9@LG9kkw(<5FZQW}dW9CY{AM=a#51ekGY5urm1n;c$R4l4yPTgNtVuUjo;9CN84L!+uH)VEs+_=P|bQ2!FkT5&@;Ac%{T~s_ns<=*cVHHa@%+bc*}07jB7%z#LlWwg#^rJ?2L|@9>}@*!8QgS{MHd1n?&pe}rCDt?O3xUf#O+Ek1((JtHuWXULj?<081n$8=jJrcGsjB$;*MHhuYh`WU2lVA0Ru#dEIAbZlJw2CDi{90^eDL%Mf-O&}uA6gtTBi{D-T4gCHg93+?Dr&rKCoG?#tj;U+dr5{pveowOxzg#{?eh(#|V?Em81M>C+y;p~e+Sop&B0qKc==!7R%a>T^|3zonSJ;<l)}7UD*Q(zwZ@E@^z2aJ1tMFHKr9vM*r3w0oX7+2Emgg`FVtgNqLk-*blsfkn!=>P4S(HQU{5pt<t<kwl2OAIY;&TiB^MpqBDbDxz`KZI$rimaj19HN;KBozJMzbg;p3Y3D1-=vY>EGe)zqbf=!L&jXPx3Qq5CgUasWEy2yOf{wC-An9DnN}k-`wBt6mQ%N(rJ^W;V}|tGlg2M!dn=cyHeZOsBJESY-M3cHyw<w#cp(qw=rnfx?QX7)OWUVaqiaEH?GrS(K69pt<0xsNPyuk#=z@3Dvxjr&mv2l?&Ma9+*UE>T!yr)*J|qTmD={U`k{47J%iqF*6&odlQ=DFm9U!X##R_HgGyfUHxj1$aYuU&2XA$!dJkbN<ulS;_zx;&LtE&QdpPJF`>poF6F6puad8QeP)r%P2BUVsZ$<~?Mgs5Q>B)qgl#hp#bGqr`Q)hW}(#JSauaHklNuyT($bA?E(xb|a4hNzlbW{|73-5{ocMKbBJZw7GRO^e_fa6f#;Eg9fGb$>fNGwU@;fqYnmFK0HGB2F_9ZzdG`Z#Bc`q(r*4?S$XfuA;2<v2fbuA*}!>$;A0A1xGLbdt~-1+ral(#n0{4tj03ADp?weA(M0*3(n(gGM8kmnm}R6IC>u`SxfqL{HCH>Oe$_fIH-Y0FWS5qEmAkLYJ!$I22}IQLfK<TRBH7_9C2j(OfS^k3lGL7jjx{B!WinpaGq4E*><n|KgrTy`(;JLzMa_{Dyj9C=`=JP_hs{Ms2S5dSBcF=gmGui?Q5okG)6+9af}^NpN$#aN47I1E(;RUasAMikj2|ipVKKe-e-4cc!kV`$u6#ZljPdEZuSo`M2J<)l5dz4WO6SN^8^`SnP1J$nnb7kfU@Kiq`&MG~z1-+QO}&SFqYA7_bux0I0LshZPiw$Mf-KwrR3ND$K<mY`LC={!ELVUUxXQK&T%LrWg_zSS`4#cQJAgk3e$yS+-WH0>S}`@^%QD&c`Dg4*ER*j98ir31P8_fjCXarQ}D3b#O)gQPXu4dqbc@XgP!?uC<-$+L&ThhF-3?fJz4$Xr*d#-tq7^QJo|84ZtX<QwznkxH)11Y_z3M;f~YwdxJCOh|wieGd@KRH|=rJ>?CUs{r%nvLT30lEL;xx(a>RdsROnt>NHG_xW~u90}%R$-<kbKZM9C6)d(&+Tp<~D)jOzy{(fxMiBv_hF&DNkVuAE24oA@sNZl&^Dia4b#R+G$U}?HQWG4TjSVFsedu6q9XYH=-Tw97V&QfEku(Z^?R(yTtW!erwhZzL{zReftQ`_&tt#@#$lCzZ>MSi7UKYgGyo$+(`-9){~>fE>9(qV}Wc!y(3-Gpj7<hCj@$;b2nA%!(zzZ57)sklUC=zT~LP0usrdglaHE9<)$8Shs&K?1A5t&SXlceg9HIiVFwjfQ+yb(O(NE%x=0=d|GVeRzzoGm=?#5DTJzkdn=6wnkMV@_0J5)nBK;@--qj)d{rdrels-qBJ5ehytDgFc>E5l<<Ot(IrQu&hsndY#9NzUbV{X3q^4a!I)AUV;0{1!`7IV5TVzs+m$=D?TXl^aAgq5dzZy)4Ch9|jA-j&JMVXgE$Brzk|atteJ~n&A-)yP=7Pjpjz}l@*1<t8>b&CR;`tQOGp*IP9SWC*Nhf_(&Y+h%>$U?An$8cqr-O-r<02?Dp=9z{WzHA_Uc8-|^#cPSpUE*}ENc0#2nbr`km|P0?vXOt2O)86);5VKEy|t6FCu_gbbU$1Ys4cNLs?sLUiZWwELXK$<;{tebJ$a(v{-Dq+V5hinU)!a>R<3cz_T|n>*B@@+o9Y__!m}<lTAzr->V0lip!XsucysGWT55U`?JSMXzyCrVI*?ECg3!*bj?vAAy|Y+8+PVwSPLGiwBM`l+^y~I`1Zm<^7~MzRS*5O?b_}Zk%<1SoU#-IPjMq4pg3&2l$yW>N{S82O8_6^huWhV8N^WI89CYT8+EIpgG*fx>d{nH3x`f>zp&a2MCQ@p%=?xcEHw0K1<MJ;@yYQL$7RF7_!o|FRz)Fh8GXHATYvh|FD!cjzqJY2Vnomp=XQxUTjWu5R>nAjnqs@J!2hk(^O#;U_DH3VApY{&EO96HwQ$D~S?rwGVy|Vw3GeXAHHWD8^w*WY4<!SiRqx0h1*{9HaO#-Z;+=w%0o$&0Qpy-{?HbURd+i!lX>yrCS2u6iz?;Z1ni*3#3th*3WiB1(((;?mD=9FqJ_>@NX9JO7+79DUfh2)_Ok<0rF9{AfjsXjrbpcuV14JvQ&O5hC5!Kv_xIbktdSdU&J4Xq}VE5I1MNa^Knvb5((wY5nMi#*w)^|K{4D-@R%ZnWoN#>j}sj~dDPnPVrNWYDa_s0``4rkQ<4QCkX*hF<YnP<&mOf~y0yV~J^`J#w-e*DY%Cgg^ECgw3EIZx6l1YgBx0-rVDhK~wp&J*5m^_ThS_xw(K<nJA-=P9thAfA%wiC<~dZMY_cewGdw9837sl%%WOu`doUf)pT+fXrRdoqCNfe`*n&XPd&?$?c2!hX+|Vc1mK3zObeA%zWLfgbqV5tbgj7lqTptP5OiE{4HTmO0gQnB1Me_>2;CJ8L2{BQC+QWA_;LR=CwaN3`RkYCy=#CaQwYyJbP4WF04(nOHX(d{I~3m^L8O$NWs0d<vsdd#jkE`ZP)HqHY#ZFjy1&EthWp3XSudrUga#Ju(KtstnepW!Fags4Md@EOul&pOnk4rUga8;qCsHz(jIy&^^ZiuBES6@v518BlCQ2S)n)504A#<T0qtanR@)nlnE1@@aLI1OoZ&V>nhyE|RwlG9*ZEF>v>(%d>TRsoYDB0Fnl{Pf$gv6W544-ZYg9srsZeqhxg)8IgBxo0h33N2`AxRFno@rd7mm5lMPvYHd1Dvur`0Veyz^iy?qw=ZKdtVNvB%2BIDU$k)y2PDJVpC7870YmDona@@!TR~aU|2m&p92(<qs3U%_WO0n(XTP480L9zM{Mxni>1Ymp{M!*dh<Df4}&Q%+wp@%JuDydL?i29Tc8K3E=r`UqFRwBmIy{X0y(VC5AA&{0)b?V$7)B8V+Tw7y|eM&xt~i42O9wYcLl~B9e5vy<WeSD>(Um;I=tnQo+b8q_z~aebho>W5d`<dkvn1-HF*xTD^VzLY{LxV+JKsoSEs#1mKiU*DASLUt&gA_*9_h$}bgmK!^c`_+}FC>;uPfva9WTxyX$H%^%Q{@yW475YFif1O-lx4LuFZi`iA5$=McP#xz^-3x(jE)QUX{DXbI<4RI+Dm!e_o!%%IJaymS@uw*{$T>fAJWMVpQVg*I7ug4y{EDpuHb3F9Kwt{JgJwwg9aEYC@ozw`;+sV+;!@-$n4-np%l!x6mVglO~gEGKS6!c?_;2S5pN~xtegty!2z$EqeamsS=Di3cK@xL*Wb;v$tstTzxr7nDxs0<&b*VZ>w!!fIgiu&ob=(Ld1H@!%rYW=0L6(cXkroY*88|)=Mk(2G3mdrIyzM+z=`EX-%Zni<7gLKfQscXnSHA!;V8%z#u4V*I3I#0{|-OpaVCgYX~^fI(Q71CPG>jT5S7G%VRN%*U7N;uygxu)9{kQi60>(z)jNTH-O9U@MH1E&ler9)1qoTCBygMk##Z9Lm8?3njRY`EAUd2EkUu;jRDcXodTxQ?CXNseZFK31VOt%Lpm%`ju@1cpTLTNXyxlF)!*6AT1bUbkc(HiTKk4q@Jfv6Vy1Kh&^zIM`<yNI4oid2<Ho#v~TphZ7qK$TQ}yjK}&xp{Vd0cSD|xA|Qi}2*hBBEf~Atc09VSp%|A(qaogQb87_@!*_PKEA<7m$&)f&JyLMdZlmd#kTAL~+EXL0R-I1T^>=bcP<owCqq|TnH8WK1{Q%}b*YjEYdF<&d#XDyRzy!(|5C`953gO|fcb<z#WJ%%6>?IV_Q*rgq+qVkXJ9LWTC)ijs%&+5$r`UY>6h52L`;n(thb}2OttN%*GakZB9Jl&*dS`WTl~IVxiC~e1{2Lic0OfKRs*zF>W2UFQ%u*i70ARmtmvDvD@7qw%*+2n!`E(Z);qW}`t{S*Mk}9TAM45!0uLx()=%qppD}|3#`yh$Hg@<vitwD#_1>$6A0aK>|V)7C@aLE|7-#Qs%>K01bHI8wJXz;x&BP4J;MDfT|G)R;PSJ|QBms?k}cRDF#1w2ExsyGDVi)s%EWOSL1Z|$+LfXDKsOh)c~iD&Ld!OX)?wd}RwL&ii(Kse+6kq{p2HU~qjZXm=BI>`%UQ3C+)+cyg-LnfgyAe0(%oHYBe>7FCfg-Uj$uN<WKDimve>cJf3yQGiewiI&fxt9)#qYB4Y$OOW;2j}M@p68wkiQ=3c9YXMPCndc!oIZ3yozSln{2SK}XYk}ZQLfq)NRqJykY|D!K~lmyrX)>Ni4%!@i$(Xw8}1v$EJgMQ-c-)z2My?bGtC9j$q$3ku>hgZaoKRjoy=UX*k8%p(lJ5elMENoMYSwx4M;!h7t+9sje`eVFDud=C9Q6M9|PQ#qFr8-d2|TtI&S;g6lvwglXh4vqZz)gh`cU#sSj1c(gbvymE6z=W;p1}3|5f_P4<pgtvhI*TO`M9bhJlU>A~JO7@Y!Dva}|UPnNAA9?Rt6IhG-=ZF1IxJ0to#wb*SE9f{$!(PL)T9?-RcIeV?>15;gFqJMADzc(>^8f=Z5Y1A(}4?JPSL^xB5ke`HN!P~{98Dc@uK46w6KGEG@Wqo*~?{)j*A#ko71|}4Xj*~6SP_?s#lGgtI$?*yKCW7p+JZhc!bRo#v^Xzo4e9gI~#mvp$8@PvEk4iR<N=FMz&8*bZ<D+oS_kzh;5FqtK4KD1B-pQD<H1VY_(^P>q3CQ;1tcf7@-@|sv9D$!I-akF`rakqe$K0zofam<@VK<{2{e3UGk7Eq}&{>Q<={Z?UPFt}B4{Fcw7!ix9&~%Snn8t;}u5*1Uk5lPtbx~w-IAK}jCF{*tZ`|F%C>(f;9qV$9)5Z-Av{Tm9ln|006u4<BkToNmH1W9GZc}jPzB?IAT0OVl*J~(cn|WpA%(uzir9UwOBprpaF?8JT4GzZpE#$>w7&iRyc@`0Ah(@%?zcm=#et&jzlC!hmEm*Kmz7#$&YahoBf~d~|Z-Tq0fWr`g3g`60820Ps5Iw~RnUX}b4P}w2I4@6H2jiJy=3}u;DT2P1aj~mpeoSi^oOUS~xJ2=KY=puCrghrt_NWAAi{liNE}mS6L9e}trxD)B^I16#MOs75R}y_e0a){<FEk?V#A+asb(!X!Mj={0>8nx9Iq{e(JW_ROh7T@WaOS?*gtwdT#MK^nt=2#*!`lz%3Nz&qb28X=I>@=m&NQCH?lU8GQ$j#;z8~RV`n@LQLZcdO9R=!cLv3i+8x?bhQ1A58qks8IQNnLi4n9hAh%}VPe0~Te0^JNoxe&NEr9Z+q7r&>RIWqq)cP-slU<~2}ZQLAx;G!<cDT(U%KaD~&vr#R$ji02nC)9#mhBNzko<4y*NiyXRNVqhS_)m7?Vo9tI-M)ZO2-Q|xXG&5e{r}XWy%6sY0frS8>pU6~;N2ARFRv#U2|v^%dDQ2KMJTxs;Ub%kOqwp&BfW*9JzDi!eKt}KI(i=S4CX)U03b%hB0MWsvW|cSTL;ZTpz|beqSZ({8PrlGVle0pB0WS?HOFc*W=l?{L#DHUM;7#YGJ|ofs>jO~XD`BJPiV&De7zG)>zk&~-U-vKP0j|3XRUL%?MzWi&q}6}g~t>+ceeJrz3$}Px^4+S9bEzLT73-p6dEe+bh=ZKN!=p?d`e<ojKOJ*7$_k>Z1sblJW{ssM{T_xvnZ7}f&IhdS*Kc+YN5Q7rP)<vC+9`*x=tX}hB$M^Y(OyuI*ITpN_3;`XcS<Ny_cX4g*}Lh*)IASqDU=ZZe_$70b<#u0{$e6b{EkS3z0arf~iu}<Y*;1+bchq3H@(BHm12UEQT?KEly2C*4D(FrJI%VHVFguSR&$O5$6~6`Ea?8Sah@$T8zjz2P_z@73tbew|2U;<3G2I4wIcuBzsKE?t$1p2Wa%rvdNet2|9_42b~h7m{A#J=z8Rem}`^uTBdmIl++NFN~(jwjmE_YWw?2kU=wrD_$#%0<?U*Dr&7Pl#7i%REGO)U>u+=~G#6A?<N40`&W@FvcbQt8aIj}EwJ~>^GgCsJL~6AZWelNDjN!F%v<(O8MAAqp6-}b*##xv}iKF7U)&~CvXep=Sz+n|Cn>&c5P`h162*4c<26vidt)Dts{3d@gM9+v^$QeAzGR}ZwgdBn%>%9uiP6mNMu@do_o`OpIQ1BE{$T&5QIseF+^Y+a`fivP6`t<gacxP<*?Kk<QvB9@*&PD#faCYz9sjgJfvb<8;+^nqZU`m0CPdx1^yWv%96$r@+St=!Z9PM!`RvsN0E4v_Is$&|;cH5Zd@w7GSq7+R=C2^#;0~FSvf@%IrQIc_!OjP#a=)5yQRx>wNavG;(H^FNV?l1?FMS=)d97=JMo*FaB^H{y<;#6&`Gd&9#9IgE^;>qWPk65IjHbg_wg{cs2jZ9RwO`2`zono-~X6Zh%jA?=}?=yDBT=au^mP6+0mv7f=+qZ#$Za?tvRPQ4=*_}H;QdeTDva8(>g7yon+<imGTON)zeMPV-oQ&3DYu<%v&A%`$<H8^gh{x*aR_`Ex+?tHKQy-t^^ll}fxF>_2huI>N90k99o04Ja-)CIhR6uhhB^I#5s3^(WQf`6!t$v$*)MV8Q#(59T?=wc<P&GEwCxY=@RmrR?stS0Pa)e>G+y`Qj<(j6`rV*7Ko7DV$HPPQ_L#>bx47x0%<29X&SCR#Lz?NV^&+?(yB{EBgq%m`V@zlFdV!rq`=CMitrTAOAUKrfG6T`ZnSSsGQ@YTjiZ_=&eFLJc;35gFFCib!EsA%bMGBf&p?ui~ZVsce92eZ-f*$n+4ZisdXUUQ}aOz|7ei#Y)$NTQF$2xU<dgEA}G*rz~V6LaV8)D>(;9FdHUiHBzN3B?D>6P6zlEClT#VU#yPG)_D=l3-r({lO_kbb)lpBS*JH@#okR2p<NjMY~pH1{*JM-?N7;{FdgsW7fu?(Aw#ADNa+WbfDjP`f`k?EZ*6KceEd0#IU?7!{3<O5(>XaJ3?_p4JliJE?JtZ>ibcg0{$jN?6g&vRk;*7)7)4$;*%ro6H(pBoyxNAp3?%xwcl;AhYs93Evd6IwGgHeRWmP`f|JJ3q&P0ZYDHuXGda)3ta<O1AE1!QsXx*8P@q3aw2-JPV^}-qH~{)Ha=*FJ_mg}Va|>1OuT-|kn{qy&`qU22VAK)&!ED`u?njxH7y-R0lR2D?c7u45=I_KEFI5?}SHI&G(+>Ppn*uG;_gY?>-YNM-sB@F2^Y>b2VHU6XWRK-@RbcYvgXx4)!QmS?fyf37OT0fAwa1YtFat<KvhjH`LF)RO-OX^Gb>%PB9A%ebW6^S})$-bA4YOO-6CW6rwb5_lI@{>mxmC7`d)%p5ibq?acXO=T$F9{nX?M}_q&1n~iiG99TaLOmziL&&h&Z+^wm5Xk%H6$@UvgZbx80YHD+DDxzE1I{mhN{foC^1Qt?qGNZ((!LUu=!`x-bxEW9{`42z|#bt`i0&^}^^U(>e&Qcl+oVgICnU!O57?eR0D30VCEN<>Pu(VQq$O>9+@nM8>f}F8@df+B9)Oedc2zK|}w+w+M7tr#}oJ29eU-2J2E}d#^=GFh;cKq|~(;^n$!ENWM4kmr}|-5?>2jnry~yCKZJZ8jr&AOZm=-Z5GA-TYKF{S9e*lXIcPKZ{}=XON=Ujm5pwBsO4^Lw~kycyP_qmNRWchq(DI({D3Gz`wje$a}H3#gQ#UV)^YMKPIe<6m{)WUmb}=fZt}C)cIM#VWmuXst;XmJtLy>Nt%p5lrBT?Zv}1Q3^lPzUWV~v=oKlS|*rp6XC}5+Thg2(1OHvyX)qYMBs?VgRX66^B%%h_Qb+PG%eZl1E2gRmyE%s=siSuD9A0*2MRNj2H%QmCR^AJUo1t~U|zmTDyvWNNj|2#afr3)3Uq_W4d7qg%-ex>aF3!;ZQNDr4`nK9MwJ_<UX0N>Zs$ML5lii?BC9s3aT18Cu)y+_#6`>#mux3znl(nX;Bwq9O)|CQ$B)He4|My57t5?*>^aYSNM$UxE@9<`jOX*6uUrec@?a1QzFHvC$0ZG;D8fJm5Qips)km$tw7B}|5Vnc9p7HW}8nsC}78Hkn$LlBz2Ax7KUh<(=C0178dNr{MkaZoRyIWgEISG$E@WI`cbGX=v6lH>%hICd4olUE;!*UfrCc*Q3ODkE60ntG^E<f_}NgGOyDyXZ^=ND$ykbj2b7=6?sy9I1zS5WCZ;tyCX)HRqs?0ao^q?STfYKjqWt935I=H9H#)1AYsAcTad_tDF|f109-oV2p|EII2*0@5&troFYJGRZr!_?FBI?(l{-v8!LP0gcx{(u3u~*gy|KGPyN)2puWzkxT_>D)zCBv?o$~GKI(!jntJkwMmyI!U92mxAfX4d|g1)>NqsfBKGR=h=GtUSaG;eLG8+5AQfJ(Cw0S%8nU9YW({8q_#CZ_{cf(0y`gyh#S6Um3wl}buGE7g=2Y6er@o8Z+|2@RAz>NBCa_Thf@qdtD7E$zoIX!MPTq9Hr($&5di?!nh-b$dZuUex^nonvm9oMU1#eZX#F7*U-&cBk&%L0m*)J>9W#B3l+|#I<C(%am=2A_=GI%3$#wT)L~PYsQqMK#oIJ(h+Rp@?p?wi{JQjGL2xNLt)M2Fm)t0`eZMo7>v)D!<@6jxBA{>k8KtNX<Wdp)_QH&qde;7{M1@bz1(C<DPo;#`-A1ZZi~F!Fd-8AtBDS9UBtL?_5B6><iG6?Op+IJZZW}>IBEe7>f`QMNR1*4-lr64M{w^9UF>o6jRF~xMWMVDTzpb(pDLHRcgU20wKLBSTT9jUoVqwVa-18xc6qbB{$PjFx3GG^PA7rNeURHgcB@qC^AG~LHim>Sor@6uoYOd+zR*0vm$bVPi5NF{379Z^vVGD=wKsV`3BNQ=$Kv0qZTtEJ8fArH&+`q=pg5DRy#5V?5&U<L4hN51*8E{>a_B3F08`aT$4zW3>NuLQ268{_jRvjuehY-n$siZ64Dqee2*nkOy3jm>bDUc&qUFKm&6Gi%?3AjLD9)F^Ro=PltEln}BE7%HzQm?GBXiy9!QNmno+RJ5k3I1&zq&ydKDcAaX=gG_+ghzNeQw!gfiQ1W8?2zCqCDEiB)9t{e02BNJd!^1!i5$Zi2BbuuA<n5qT}MQ#Zie+)YaQ{|Gn)R+`q|}HcL6(I%||%!n@@^@h_|v1S`U)B66i_GA3QQZI|Yp8x`{snT(e-7wiA2bD_N1O}=WBvavNGvQSQ!gF){KBqa$l48h1@4LA*K9IM0J+@oz|fAhJNqxths=$t$@;)h6F)?VPXNQASba90lgcwzQB4{J5ALzLyk*=y($p3Y?B?A4L{Rm^fW>x@a2m`qqxq`1?#P&phLT{e%|>*AoHGLoAjD-q3#6$!@@1jlY((<xjDt6C7Wm^bspp(5?%I5!-+Lhj2Pnp|K~^yb7bLwLT_MeC??YehmB7wytd1rdh?s4(E{m*LTB?!>`@t@Ytxu6_&4tk!j5w`6ap)!mJaD##*p)kQiceG&>lR2#Lm?ef;$>Wa_l10JN-6-WHgsv$d`afR*L&fUs(I>}<sFt`l_$w^CuDhIjyF6R&9O<>A1G%FDf>?(5VAPuqb!QeVG(-SqD5pNXqc`!K&3*M`xC1IDK`xF4Pd*=>_iwGjVQ})YnvTh=<B=9(bU9=N<aQ=c43u<B7DELzN@R;-8^n@OPJ#?&~wSP!Lbz3eC<1!{P-@HeYE3qSSfEG_QhCTGHcf6%#V9%W2L_|O<jRjgKtdsu!Aw~<g4LPiQmq993R-;t>;m8EX0PSXiwPHpijmMB!&z38<O{XO$u}KA%hOeEsv*NhNhbNs*FQBA7I(PwoXmsBb7!z%D2pG#-Hy!IlGH@Ev(c1*-qmrTl_kKoM4I{z0iJ+xS#wfK364?Wpe<GJix6Nh}rRRnF9YdwF^+Y((EFbc3Ws6^qnO$P`mV}4;*o&SdaHus@Hq>Uqg_pwn&>D)sF2S_5Kaub*)SeaU#3=!kc!xUOE<;L@M8>@*?OBXeefR!-vNFVhQR0{=@h%%<39pP}vTztdGP!?0I}VJpLt%_!H21s~o#={coSAqi_hWD<X2!OR=^VWTcFH_laIdw0gj=!S_7KCBLf3MAL-%i&SKdWk;^wM82#T%Y*&{ssz%yqKiVzCOO3*bTmNDg!HPZ_Fuuz<EzpZsNK^)k%7<q)HUzkkf`4Cmjd8`A$Cbe9^w6@v|t<=_cH#Ys%>K&23$W8=G&{d0ab`m9i5R3-nTtfO+nz=m;7YhsO+AwE%aO8^td4(90ZylQvB}inXZ|0I4p&pOlYpP2D{wbeJaq=n_#QXRCWAA><QFNLb_BwXaJavlj0;H7?s9jN2EOp1lnu#)IM#YXzcQ2~nkxRQ#*|=TV_BYCK@4BCU;bL~L%=<xGKlNnKdCeBM;V=Q5xwdu}Gv`IFpklV;QaX+|{i4ZrVT!#W+*Qx9Lc|ah7VSu%7r9L*d)~Cl-Zt>Lo!WY38-JsfPoKijKEpS}%&=l98AT$Cd1W%(wI$8JSU@pRlwHz0ZO=?nz){H1u!cq<{%~!QbnH^b^=D@N*<s382~w7|2_FKzLRFKhC`=45_I61?+|FpBFE=EDXNXNjlVq28e#5Zn`E9t#dl}LNa>-JJo5B$AA-kKZDQ+%!-1lp0m@}~Pv&Z-<hS?&-t|heKmQrw!L+&0}5Aj}rPPGF(@46#P=D~pS@O?hJm6S_hQt_Ud#6s!t#dj2Qrs_p3<s4pW(LUfT&#nf2DMd|htAi@B2C~SNCSFaxNzt^siSVUso7KOJ*cI{iFFh=PrHJ)5w?U_<>QE|kHh*k1y1pQjK_1QwBNL8n3QPkrRZh_Z92tN}_$boMhCG-&!VQ~F`Avv~i25c>+hn0MvEUWw_NDkQKW7ZfJ!_s)p?YNl(vNqRuXpkFLNjfKL+l12f(h7$>vSX<DspW~OJFB0!{K6m$Kxn2U6NCJqL^4H#0rD5F<!3I(eSfEiGep#WY6oujFKVDGBs8^pVT0|J9N1riCX;aP@4T-1q~_~k0^R!B8&`*!(m=&UxqCTX{o-ukvI!8L?h<J29?L*z?h^*(9pCYY6}5%g@T7JjPKk^+1yyINcG#!*Mm_PmF%i6he3A3SMd`9L>2?=u4BA>yexq(YbAA2&aUrnRkqhF<?T(B|6Zn4mnqE|=4ggkmnzOBs|w}1&@AHXxWBNVX{{tA!D8OclJktDoL2eOg+Z0WDFi~CI!-zSc5-gJIJX$m{8onKl;(r=Eu>Kk!_x71Zs`~Zq||A~Qb`kFAQse~%Azx`0FkWKREr)xrW#C8{fuhmK8VD`Y1~D2j{x%9!Y$eKp40&1z@E`&NXllyr*&qxRy?z0Av(b3nW_+|DHOZ#Y@=^O?lKSOPO1WB^pWIn360q{tNm4^C1;BwFg<pLN1<9Mb6S1tudwF7lDUxC-H@CyG0tWjNs5dq&g00g=?0mIHguu*Rzk;(-SiTaB(m;+=>G0z?Y&LR0ujc*rfSU5T+l$juU>1?AM{b<l@l#^V<!YY4pCeAZ9HN{vmd<UDX({x999?)>qdnarpLm$pkT3FOc#ny$Ys4MsSahGT?)T?80A;kLMqE@qvHYEGjNd)K$goDTsao?hqF9{J0bQGr~(sy_E88xyGG-v8G2x0U2w6(_8Jcf_nFQC<B5Y60$nF8xiMr>)3;94`EN29`bVTkzQt2RUqYV`dV{@I&xiNkIN*bCbU=!DQKYd*-<HCRcxe-QH7SU_<DhW{_ln`dHXaI+lD(n%+G~d0Vcbg7BBtV_X1U3}+%v==p-{IGycGTe3{d8K^SgbYp82GQr>vB6PO=)HzyORB4}+P-f-NT3P%WKM!#tdYh%LB=Na!Aj>;yEu5p*uK36#*l%(T4ut~S4$wappA!wb-k(HZ)iA=b`5S6{NUNLBU*Z{5;rQh3)y@mPyn$QPDwp`du!on7>;H*U?hx{>AU`(Xw`eXa3af>mYSoLIhBtql~=7tt&KG>8O)=B@i{x5=Z6@4V=je-cnK#xqnVvCFsi!E(InmDYqRu*aOULIj0y%n$lNwJe!~XX09{FFzTzdKT%>EFefAryRyjAg16HvoLojy>lo3jZ81;#uLQYj@e|gT`%W#!3?p$(fu;%jwt|Dk-6WzC7qp`^EqjTMtfjIoZUXdx?%R-<bJW!qQ0UtJQrgojIamN0lCx^Mj+vNXYlUSLm4vdK#I}Cj(rvpW@u~a?ACzZA3oGOGmowps59svOb*2<w{$Ib?1zKLZT!$ZI%;O(p6d~xqUv`FbMzz0IMXBBUe;YI7VAbkqonpl2>sn6$nbIF;luE*<L`-wFd<iIMh}pa69nTBf4MdY(*#UO5C8~ff$mmyxGj`75dh2IDp$APtJZ}Z=2cqmE5KRG=cCruarJZ(7@KdGaDSvJCE<Ex<R@94w4s|U!%}5!V}r*md&+9O)}Fn21!Pse-|ilJ-V*r+&;u1j;N6&Gci|L?{dLqR6$^#t9K=lfgsdm{IeB`jAI0W;yQlEB_wZq8H=+K+v!kQfvxg7A0}uaywZk+7@ju@d6ADONf~A?a`B-?Wwk4H^wQz~9Z-F2gnMu@0eZ$Q;y~>`<idogg6T;EZEC}%}Ja;TB3qfnB!(>H+Li5+%8g+Djbk!n{u2Hb4vi>3Q!lP)4zx>SnRAEq}3{^xFcfxz0-%;zv8&}OaC6n3-trua$&*eDfyT{X*(-GJ8!VX7!0WhXO-mPVFI&;pe#f0hRI1$eyJZ2vqlyqd?W97?4?)?V!wc~jQP31IgCiYZMYB$R*+f8#<b{-o%w%txgb4<+yp+Ib(PA9`fwACZ^83vv*AR6vJfmWM_V+eE;N?;}$t!!1wJ4mK!$_CgrUW#`#5RV|2PREUC*RI_tE<n?1AO!5oONUSnD{FK*O{r&Fd<*~YP3Jq_@znI($+!CF(ncpT&7=qeu}P9gh-f}Nvw<j35)&@JdqJYc0UE_p3ERxA%(}40VxV8>&Mxdd2U!>0nb=R8HN*?MTi#r&P&lH91E6g4>}yWhb_J0@w+fFjj5dv`F-k_)=kdlyN6h`BoR}M-%riNdeUO4^$YudvD_xx%vp|Rf+Os)MMq5PosPyC|d>~*a#1m$OrZbTcC1Q#@{C<%?F;NPxiAS#WK7F10z)kca(!4u|t~cr&oNV;ZNqO<K6OpW*IxQi~kdqeQ-|6PXzORlls=Cv0Tpihi{&emAuHPG+;f<#?X~<tqzAm;Va~O2+o5SwGp;iU`G}oc)l(>BOAofKetwCV2!GuH{F*LPflID47=J|^nh=j>)ODGt=hBT3C>P!GHhWHS0;PeR1I+6f2ye=7(VFJnmg)$|7F({SAP>uskq}x62eh@GS8cAi*K2TV3Og@9dY5;6}%$}!0wL4qs?=(snW*CPsrOa0{d2p7GIpR_1phQ=8G56i+BpxWBHUGk*nKIv~i4jRXqg}6UnX`Gy;48igHRp{r3j@axR}N`nI?-G;1gH}p=~N){GQ@k&kcy6=j<n>5dTkQyKLOFJCq~Q{Zn^olZn?>Mxm_hpeiL^FVSOC*UofTY6ncO}R9te!&lv#~VY<50sM?<<B2A2&PQB?w<sS^q)#4mFb(YVE23M--ESr!-U-{Ne2=SD6?D(NzAMk9DLlG&Fl1A_I;XIZFnP55$zYrJSiEb<BEv|)2;7n|kw<&%6JZ`|`{V^r?U!tFRJUAq*&9^<<G!Ht{;d>+>zg^S-%(Dzvc1k!OHoDV@M*<EwG1KXWG2tl4Y6?@N+Hf==^zxit-@$7S=0SDsi{D;;eDMVS_w4c`{P*I^izn8_GwbpP7f&v}geT80KfL(*@~0P1t&30L<sabjXO};<;OP_i;-kwS!<SG4mWd_FGs|I>O@bJVvsp^+KqklV3+X#3fee%0lk5~wTdk0$v%XP#w*or_tHOU8rFf!L@=fME5=Wtd9UowAM{t15%qNbV8Vg1keR=mjaQRqc+JuwZi9G-an4P^NU0pHbN?{<b(jmj4-yZaXL~syxB{-KuNU=ScqZ89-8DZI*&Y#O~7#c%B6SaHscTq<W#|KNzYzwRK_f$iv-K%Vu*HK>)^9!Ea`xlX(GrfOdfEJPTCdrH9Tj;Eln!PR2K#EszbkU7I+;P$hi)%F2u`hmg@%-Ymi>I(_pI&@n!H?fv{`bpIE<d{X8XkOo@w{X`3O{{({gL|kalBLU7MXecI#%Jc%TFx$@%-|K$RGG&f-j&TG=n$H-x{>_F|wu0zXeW2ty%B}S<}bRh36Mvz)Sd7??Qw&V+m3d9$t<K$1Url(8(~Z$pkXAakFR*hnLWl?_#%9Q^UYqQa+Yqj1z&7ODQZ*;ncCMRI|n=oGwzoRvsW(l|<ceQtL{GcFeZ6M3Acyrf{jGJKniudcts5iflbVH0BA%!pggag_ynT6#-_W(vRRRZ?$qSf?Hj~TV5y)j2UN9Mb>H=1<qX=U`-eTsHG=-c=FeQ4w75Leqe+pr0)2uyOs4MQlD_Lldii2T#1$A!^9y`cMLG~qrn*-LCM0DV8rm4nKHs(^ZgvNeN9lMgN{Q+6|sN?I>0M0a8PK`wf9JA0ykz5(fGaJ4o-d46k$dS#JfpXJn{!wm;@(#<6z=x(?X*$od|jvd9dN&pqJ{m^O|0Y7gCntUj`8HSl!}Cl%I*8(VEGVMQdLo{?D$Rp^*q<yb<@FudgdOpr&KK{_=bYZ2t6V{lF6ihuzb`#HaMqBEwTwLOoq?8BMR3E*Z$QGQx=M3WOmW)2H-KCQ4n{IymT@^!GiRl6u=tG!>2sj2cPK(<dbbSR2d9EX^M%s}1a!==>&%WRYc38_Mmtu{EOuHlp*K!|JK4Me(|$li32A8wR6xZl67P_k}toO6DoLe(ZKBl^-d&FfF`5NsD@0q0AU=B28|L_diXcr*dMH^!+l6y_EP-sxDMP@`C6IJm0@vLGUj=7XI|3Us$mXMqc5gW6F?Vmq_=K8Q|i}GsPE}wX8yB%vv@{yVtJatUq?L4}`RnF}6fb;=nK!jk9i3nTGv6ztbN1dxv(@7}UhLHn9E1kyAyhLIb%GBoMB<t0g&^1KGw{Ju-T_iee|=uBLT(?+~t24^QA-%66Ko!l6w5LvcTyUUwMLSTb_6s_9-`C!aXvv~e9gvM<ha^a?AYz^dLVZ|_vg>js!=7J4|xhp|bxJ<M$L{a3Kj_g|5%K3*1)2d$o^yh)c)NV3E}OUVRX4dpS^l9SAJ0+;D^I3W5U4xNrI1lj_cz?>o*1r)u)93yzcxh|{?%jD=Olarv*J)bEX#SwiHVnQSJP-SB|X%S-Fx@pSQp;9}zhc3~y;~tF2qAiZla2F^B>6$i#lnKm+?hibu^(7w1@nd%$XPcLGG{bu7v#bu*j5GIYXKltB8cy6<R>s7^D513H(s4#LoZ&G>f6a9;Z4~@)Jo6q)8OO>>Z37rcwVr<Ak^{^*PP9r=j}xuTiykLh72n2jqE|oHaWdDJGtQ0lIl|)I4luv|1pPNhgVO-v{h<fuhLwFde=x}vOqU1aXKXeW-xy2CvFCeGCSSPaq#t`i8Ipe9@q50l#-ADvx>m23n|;oSNbtatk+Gw(7d_aLEyRzt^scf&8;(6Y(J-Reqg8W`)=-!E4O<g%gv8yfm1_K4XI%}R&k^^0L#b%yie>hhik1ttQ&F|z_fEP2CY{0YhRFVU(uNWTeQ?B??Iv1>Edc{CzG6LGQtC(Fkz@8ri+!ba6G(1#EA|?mz;o2kN1wwJB4TW$3I9CC<{sg%SN4Nmk5X;EQex_E56(n)Ug@^+;mywLoi}gnx5Yz@zib27gjbIq^CQN~ga-{F`@sKM7ym}an&%f^p!wzE(~IY%@BEo{@yCnj*2QOJ(0Ow475u*Z{>ASu{=jyf%OAmizi=%w;`|;9xcJ4>%MamyA6@?V^5>Ty<0l_NvCmP9O1jfe(HsPv0~4#!%jaY#g2%su%0IjO{^j40b{YQo80z^keDo|ob2|1xIuN4{roldWBd{+EyzQU#dipfox^?sQrJKb<`ZU26Xm~Oi<2*G}CeA8SzL9%3@`WY*W4p4oUVgB+0)z{PU}ba1((?OQ>*Y1;y0x`kyZ^wdgDfk+E8y9kT}te=Qr;|YKQN1@cR*-ib>1<sYanUh;nwZSof>>cy1k%37I+gN=njUIr}MNsI2lv-3fA}xxYghcw!>yi;OE3L(v1(h9lZaAX-smIR@X(L3@(BPGMZ=X4l&QH4=jH)0vVXHX~fU2_g}&Pw;k-AbbD<b1(H$=$-L+kVrVIYr*$%>tTYxbtW|hDZ<-AJr$)k~HNg)UHc-?{a-Y?jSd+!!$zBh!kSOm`>mYBH6);m+>aZ0HcH%{5jc9iaR0=~xMApIRtTk#2SW0LQ(N$pyt_@q`$u%8imECN*otR~DGPYU=f%Ud$p(RkF6CpfyWwTPRljl%959>@3h)9c83;y7MZ8V;v<7NV}b+_N^9tAuBXfPfL92Z7~MWiVgUcxwzyOfelPOLq)I#}TX_UYgRMwT}%pJ{45$}~#x`VlISh~Of#bL3r2e(JDU7O9NVH{<O|Vq}b+q)k@%@(O!25v4I7oQ&cMCg_s}mn6Q{<hbJ+I3w>gwF(SX?$>L(+bamm4db)TRbx*?ZS9jGkyp^p#V$4>EeLR2;gO@-6%DQKZV_{+`0rL8Boa>?^?|jA;Ktc4ETC&IrMMQz=CiwGnm_`sIUw*|cCpEzRij4~C;06c<XG+k-1noo@@SNePBW1*&v1F7si6LQ^jSVWto+$w0E;A>lKxPJx_Yo{J*`3|uye&~?p&Nx32IC`CTGlsja$=G?S7-maQ@B1Tf|-)UAAZnJ;5P(60oeCfY+eTJY2)Y@;XLJ<2FYDUb=Jg%Ka!wIwf21?jBE_!EI!G(wlg^P0>$G)z;~#Wro|sU<Au-Btx#D5pSsJ@r%5sI*{lQ2p7GGWGfFVbEiZFKzj!PFONIu#}pIsR^7wFnP>Nc&V)4nl+etxk6}yuK*+%HY<*!<5yiP_fhbOs6EXz!A@x_v+pFmAgl0%~2u~ndg}~7`ipbo?yPb}?kGXXRqgKx+$GCGJ!>dWzn=_x`^<$76pv_iq5x9E0CBO{fye(OK=+K3sL2yi-yH<x!F9shc4-`Beb>Sp$I#|wuA{tAD;!PL+c@zJ63;tOu;6Dl9(ZPA#M_t3oY1kNm<d^N{HJs*iig`?uFg%~BMY7kHe5?#AE#}=~A@2%D;~O__<lUP$Z{`#39JH!)^)9OZDm8GlO;>NSV5lAlRhDQj^Xhr2Qd`mOxG-5}f@BXdt|Bk$<Z!1#6G#+CYwvDW(d~{16lfFA%i8vPK&1XgwT^n)ICb*sd(fzKjpQe!bP#_gST^iK9J~__zUdW%;!?EgRtGUgGRMjdWDS=pnP;>tkNeVd+h4Dg-%WV0;0dA)6@Sw}>5s%P;g*fG9UbOUOz4u4JelLf`m;tUq5TcxT#Aizt6bj+M~PBU?RLGgeJ{y)#*<brva<Am#>;=rHDHr6P4pc$O?M1uRp!vCDJL<q=%7eU@$Z$_cUiEj++SJWT}=*&bwl*!PGw`Owv9eEbb*ulA=kbw*c%<zu7!6X#pC*?t&#A}Cnl%DAyIimJ!QdKZ#Rqgrt#|$U!nZa`rscFL^y}Yr;&E|QV`BoJ2)O3j9SCP?mmt}!0U^<j-DbyW~9{N_eI`H6X%9YvL}Lm1Ytn!?2Vt2;x-Nj&ciyNQ_ywo5DXrJFuE`(dOHk8ez8y}V5$$3NTED41?&Q>B>ZpTrb*>G^pbgdg)^SBd@NpuO{;gQ+1<n#=Mt{q@_vNZJ;QiQ!z;uYht}QX9F>kuj)=IH@V}^7(HOeN&4kou{J0V8dy{I@&bh^-^hQ}C6B(BhB3Z2B*%H~j!;97IB&fT=s|^zU+2j!A$^&Fh`>5xJsl1j8izKN9rO@W}t1p=~qtb;Xr@2_XQeSc4=9hU99+5+MwQj%TF7khyLflCgXrWFa&8pb4HEI5N0Ic-(ZgqWCO0pcjV-{zjqU=1s;a{c%^>e4EPf1NLBWc2vkfgV*r`YkNp2DQVK=pZ!wZ4M=)f-yibfO8zZ6t$W%;-L?-h?A5Owz$frEzhydGA6|De;_?HZa61r3uOm^Nm_aOAoIlhgSo2Wp`Aj_(-5uBtLv+c6g{qWocK#h$d?fAHTeu?Gn~KqwSGOP5*6lKvOcmc$^D)0U>d7*pBLFiuaSfe;Di^!8sPwzWT%7$=EN-Q%%kd2aL*Jnd#$gip6A<FaE5BIs%b4UJE~s)wZY=Ggq7lz*LZ4=ma}JYc<MslkRc1f3VEHAr#<&s25hJJB1ZA9jJ)ac|14`RJJdWB!{EHw2K+fhm<`W4>?N7j;(XB<}pHsp;5gTOoA|m^EhZB=S3Uq*pMQi1D8V$(jl@{D3(wUE<vaUpRgL}-y8JrP5SqiZ##}LUyBzq8Mu4mxClFeJ5GO_BkQO?s2~a2xlYH0iSv4`<Gpsv+AmE;jr}lek&KV-2<6XlzyKR*Fw;RB1oKv3C|%M+!eG$n45`Mw%m@op;iwI~(ww`UW*Fs|i9Mv9fbZNYkod0w8(>{c#=RC#;Ak~qTnjkTmD=`pWxb5Bv1BRoS9W)5ckbA;u!zDL!FgNW8wIVSR73_<xFi-*a)wSS5ELe05ObJSm@20#Q~HZkIi5kP(vwEi)J((eoOzT?Mj>C7hX{4x3Wo;~Kh!|3JO2r%$jf5nspONv;T1FImGDD8<_kUs!Zsh)8D3d4n0=D3%jwSK+5)dBmFv%6Nif~-oxm+$Lp1<Sm`CGTPLLo>MdFaqocsvzNCf9&G=T_(PaFz!$Gfp^LSJ?P<^=!cwS_+q`9c69un0w)nSbKVf=*#t^--N=oZM18CU?#T5%_3f(3i?#J$Ud{D_3+ZVfJDr3u5Vv2JMsmU~F9@({S(n8W}R#%WrJKqZ61=3ChX{kC9N9+>#?~=ahwWyNdo3LW>69_j*#pfk4{U#CpfG&?XoN(TsuOg3(ETtk+6A9{N!)Z&X6lG%Y*Ykdd|SAELGMAn2a}Lu7+5DHknP>saUzH__cbg>DM7)oiauqh(^+d7ed|eX@DXIT>Q<ve}?yyhRCo16ek_5wQL&d>#cm5Dx{$L8CacXQ97ejv-cD8oE2040>K6@3!#2#bB{mNO-m)QF3B6&Y{QO#}nzXnpQsKOn+QZBxU0Q^uuf`4jU?^rod*`!v;kIR{YwwAwgf0kq2|I_dD^cYwvLEiC!rL1WsHk67Y?n0C&{*KCX+L0IMb3h3<I2HyFbW`*7?+9d7VF*aWbj`*g7LEaN{s&-jmHRw6fW-|LYT(b>l9K3b*!TC^<zzC(!ZW-{>$39J&140^Z-J#>~227@+tozU|*nF|#)n3v;tJ*{b2yn!_}+5b#*51&Z4v%Hb+_b|u07zBIdIj)!kS2g%8=@O)*k`><VQYbeF+yWv7Q?*y_@7PX?eqnz=ie)&zB4Sch(GU@u9G04(5V<bM6>b_~?bPv3!mY2o7{)nG<L2ZKgH~H}0f-qy1QwrB(~2|08sV(SEzB=CYI|3@iV9RCWad#!qg0$C5!u3ae)bR&pN6xnWBZ`JV0(AdemsvxhP5$~TTy3>znPaNKbZPTC17Ur0{_i;mAuo1lIScI8ehd>BjR=N-0XJF32#wGiizWo)T((CD@-#X3iCp?s>jtPF|~5D`GQ}%M6Zr%ro`7N^TcnBd4XUuIww6*!ly*z;&0|WhGQ&T7gsiH91l`#{$z;SX5msK%{U~#c&c#|J&KaY-7)bhUu;tUA@sn*fAVcOB*r<t<E(7;TjTw1SGX5s)(zZ_Ij|V3KJcu-Rql2vJ$vb>-H@B4>5;VQM}#@{pjxeL?hwmMsfZHzmB-^?gtC-K;TLs@64h-~>l@{rmAfduYO=VRnMUbNV}gbAaEEEo!-mwT;gYAr3!h6OoK=+|nbeA;@7NHJCqCW7j=HiLAwVhl>8;q&pMd})CvkU^I5ei55oZ|9W_r$HI+W3F3G#wGJ5W#w0wcg_sMQ>MH1-d=!<=OHp_das&<sDOrD{!J3VgQfi!K<Q1LV6f`O$G}m=nD)ReanUA9=~IC}K)}_L|nZM8q_??`ZQQPZ3I;OqrITom9X~vE+tt-zg}spS$HcY#e#G${k>^lSEW$xtS)=jHFkL_bMhw#s<sZXimpbxgC=WMx+xV=fA?&sZJbMJ#Adkrjyo&`Ox8eO8U1`L5?Yh4!tz7QQ-+N@uyPol-ynY2G(podBolqO&hrocWM?B;_p*=P}s?Hk0j<OHxk_tjm>05Gmpo<&u$Vr<@=J;7<Pq`w@LXew73QOVIBJOhJ#Z@^+hq&9lqC~GPJMEaoDowGf(k29n5=@s|MmnxkgCF5BYTV!oVhN0_X~@4NXKW(J?NV&uV3>vbmaeM0BM}Hmp+EOto4|S2^|L7Yeo#*Zn($_l8joE9Lr5<oJ(GsYwKwTfatX>X=yLDe1qIBJxNj&h92=YOvGxK*J7X2PifLzE5c8mSs_a9@X@emhNVuV7NO(6-ZJ6%(XbF#pAF*(OL}O<=(4pzkA(2?iFrGb|s?7@<`oqF+3X~QBH#CB&X4Pv|<CfW4I*h4whU2<A?)xpx{2-TKWf`eKP4RzG;+CNEfPHw74Z(>OA91wO|9y#3$Fpu8=IFkLR4?t4?iN|05xYs#4Z>sh=}~4x^ybiIBXy(0=kgaY4GSNl&$9bX*&=%@Meah!ZrmofeL$bz4LRGT>ME{Bc^meI3DwI^78xMY(y`8tgq}2x2mbj?vbFugS?6Rm(UvI)2gO7*T9F$U`@Z-bP?US`CIKTCP|BvXaCp`ZFnj(iA=9X`$3H%jZ5Or}FSu8X9_x+l^G#J^pqvDkBKb0{%3qq_O3+qTtvawEdI*KJaK%W#EXg9|q1jZgm3hukmF*^?NB%lc*pa^ik+AdC0-Hoh}FkR3-V7LJKK?K=6hMP|Mv&9@VUmBKnXOMv_h{?Au7vM?oYJb4N!0NBqACWxeEdeL}qj`<qo%_)BC(6B8}R#!xijbj+7(E%tf#L^;_R)dTJ(t8sx4!#u+HPFAgYu9c+q)$)8c=K+yb3Oa=K12#$?W_POh)lHWmw|6%(z)a!Nl<L-*eerNi;MG`NG?FP)1UBI1tbnGl7WG#{{l($#Y3>J;L*R{vgI=5L1qz*6eeE`i*KfE>*KafxpH}ajci@J|yc%Qm5=<ei8x35@+$f}vpyAZ3Mxpt(_N!>xEW-0OIz`0WjFXei=_nNnDji_FZ4mw^!1krDXxNdIOEU#yg`U4R(r(8c^jy}h7~FvKT8^6bsGjsJ7x=kkPQ_ap-WuvG`W7-q<9X<d<N!H)jDb4@Rp0KO9HSd8R%km7DlrSbOYkFR7UAf4i|ZR2-_5g&br@vqXSi~7eX>iMJQ=<VGrrCoyvY7ZO)OF>ClDUk({(Z2^cJUGS=KNTHE{h1-!L5!QK>I-9_Y4pT;y~EyCi{xm}qJ-E-RYK%p(wXIGh34@uEOsk&u%-BaE?Nto`w_1z!&KPtg4gzG@Ew<__KCA;M7-%8#8G4UO||Wv8-TTdQnV3}cDU(|qID5i2|bQPC(7YU4??5Rph<weH@?-$K^~w&L#vmPpo#69fM_I}G}k=F5omWR*8pHJ`pk^o2RMLbcAABDaT#EYzQ9Vkx>oQsDlWQ%dT+dWRjyv`2Yzz9Gp!%a>Bb7CRAC?&<l9H8h8xPqIB;Yn;c(>Z-v~dwQqlyS(5;Yk`BXnyN-1$HeqUit4S!*1NvnMWt{D)3-*GUciADW@009ScXyTA6bKrbnc<Bd_EXmYwsMR`h#A_wOUpO-t?>k=ui^u3vLAE5CK(RjFKrN38#=jP`0)!GIxlGNJ_9kM0p|L1M+tgFe%s`0<ADCPg~%>z*CUS`*__JcRB$pcCX@BH@3EG_bMBe&7BCAnv&@!eEaGL2b^YCR5VtemHI4`yWspoi={@|97sG|eQ2n0>h@c0)I?6Aco#&!<5V0N4cSfBsO{B~0AIm5bM3#f%Ig(!7n0A{D+a`uztY<mCG~a1iukg<sKUZ%6ryZ+KQ`WuS1Rl4#R8!mtpoQYoojr*i85Q&t;%|Jvl7Ov)x(~-mW($O&RK5sE=|mmcsIrLa;#VyL|zob7~xJNU`^NxilU3Xpml_>5<=~OG8np3Np~*3^6S`o5udJmBA#cge79IN%}N5{1xCG(Pp}@5he|6L6Kc2HCj8=<I%a;|Ksw0%pWRVJsL#<J`jS0GP;UJ5IEr9Pv;0I}#7RW>hhX(|g_bTLk-Syf1Uhqhe4yn)<`L3DZm5}9bjm0tiBO~1ypkUVRXxKnrjO(0`eE=#6T_IL&HBX$t-K`!D36=!pMlfvims&FT2*}--rH=a0?~MyL%|h|;{+<HnZ}vCL(4e*o?#fzvR_mE&TCh<m-C0tEL*oUTPxExTC3G`#y9t7ZML<#F*{&)L8ITRSd&Hm&X=%jvh2yCl@yCIzny81PVOK1<u}+seCtxaj{INK{t;Uz)qs6oh~=tMn{D~L7=M*k$>50X%G@g!7IhPI!96b~?8+&IWNAfygf4tZV==2FH2(|NHn?L6%<5E(Zh86ll?~G_4USN|ff9!PR7`XldUk8-YZ;u;nN?~>q;|frcwThK)zwL462xP0aHN@JjiF>>V`Bf3639#&?%a0K$lS*h-6*T(9Jy|8nRDG_%3qqrE!7N|R)KC0)a<YGn*Ls(UumDDz|@*5P6sMYjLu3yER7Y6A(zJ9elugDv=yEO-Gf8CcLPlVuj)w&{kpI&>KD<pZl+iTbtWxjWX{SGH?}q+xc*@-Har30#wUMM*iIXz*1<@muoWd@j~$g#RlMh4(VL$gD!_bQBZJ#TeF$@bC~Pq>Ccfj|t87zN(0IK%7NSpA_C6)|ly#=PN8FQlH*4>0#w$ck39Q!1@lc5du#ab)hmji8-e0Y)?4nj6l%`h8I~ALOrboFMqqZGH;UkrO&Jn;U>Z9~w;b(d-wA!6J)s-rxN@G%m9{WJRyBk|HfZFEzgQ(9oC6Ph-Lyi{}9cW~rQ_N<g{@8Y!N!aKNB1}f!w0h(VnDLFi=~5=soOfHO)fAd`WXbTmx~^`cu6VQhKLe|nH+o3s-XpA5*z~jNeY~8wjgz`7s=~9{1G)#(TC4DQ%yApE#)NqpjO|&826eqS<?vb&II_>I%O78U|Kd}^vHg@%2tK{|!{yH~DVfBw{rutyyn1^1;pI=XJYp||v-@V@)*G$2-nuf*?*D);5Pa_Ck1l`eBC_r?gwCbpW?x-=MbG|-*~Sof7dra!#S@tsk<fG{d@dg5y_XD%d*kMf)?V?=ZwiWAl%Tlu2XOrL2jmxSp>xsgTJ2qH!ID5z)}87a!en5{`c8SRVxigt;h*c8K9Qv#25lba9OfS}Et&16`PIpiltxsPE?GPs0VARo3nDzqIvb3RK#)CJmehLO>Yt$6kUVDuer&tjI_M9^lkR?mVBx0u-QDUoZ0};xi_vPDX(w)<u$6l-0hLF6QQ!e9hwvz6^^m_&*(tALns~h<0-+82p{J$PJ4op%rCv%+Z8I_JSVF7M<=HIy)y=Klow|Gzxm0V#ousja)wyUQkro^Usw;Ga5isMReGrT!EEVHwkz|fqOHH*$bLg+!t*pFT_Ym||o708}WY*Fa3$GM(LL;RZUa!ZLQ(OPg)uj$Tz(iFzX%QQWNX0}97MpjBcArdB4y}+jj1-g3JXV_Ys#fj-^N;E}hrLODw7*3$@S*tM=^milUwhcKOU0#IaBpdy2JlckdA;D;ok4$+M<udddMmc=?A8>PXFTlo12o|dWv++F^ASpt;*EY?z;IJ;-tVAmL*EefI&4i2J@!_Fn=gn8_H`<^VCRSZ1KU}~QHd_+u&}$gH^6Hh1zOq@YTAXdcrcW>uLyj~*H&kl=S$uJu%cK;(VqpDaWKgdPMX{4$GeKk#7;jq-=&OFfo`qWc3?wMg{jFuBca{55I*v~^7f_}Id|Vi2Y^&$Ihr>gHWHDeNpnp$BUaCHv}J6bJS@;)(~`_;;d%CUZG9cLKe@>|ukm|U*1}uO|AuuIgi+-*S(RLTJ*g%hX7vFE%e0b%{~$4XIp9*a9A|mTsp=W<4J4SM!*jFO8s(<$Fic6|o}({aLwm9~6qzSn`r{C&Wl?+GboX0BYU_r9bXwEH9?*BLfSGFsBhTJLGotPG&b<5^+=VF|9Y<s6MWgfnD~FFpGN;(%(jyFgma3b_@4pg?vcpJ9!=?-KO|$6TyyYIYfGrS78~Srjbd<CI-(OlA<qFCddHDZU4kZkbM4n&#dsJGwcxrhTh*E#J{65GWKjFlv7f&5QT9fVs_N0CB8xTr3k@FV<;-8TGKSIQRObzg3N&$ef*pm_`iv1g`#v)9APB!o<XMz5jWunW!rOdJ-3+po!d!AnW9=>@_C8&4&!87>dW6Cn{DZTh<0*X}Q9(JLqh+O!>`72fZi*O6uvY_oocZXxzI=#Wbv}@LDwaiTe?;ANRG;2$|Mx|0WmfSH*-+afs<=zZ8$6+q9Z}i<_qA(`s<Y$eJtr53%@%y|5zX!dx-x?i1wo{4Q(8g3y@x?D>&TtDejLS^Dmp=z0LE;5s1Zf(`_m4o7Kea$w)`&hz!s`K?4~v6-ut@j)MUZYwA|VCP9ai?ar0-9|Mzz|r$!^K-VFL(A$WB>4G>}4fKuVJ2;7=p9b<xPmB8q|Ns;FVvL3K+e?P<kyxOj5ggtHMn=HUk38t(^W-<N3ROlY^9c_`_YPD#VOP{uS$rNv^?#fbl~C4QK-*#CC<LrMvvxDc+UU^jZE5y}^j90k_tX^c_*0^a?*ehvH#NamBsLUX2iW?7ux*}D7zH2n3&mz3h<nTuTNspNB*BIRpA8khekl>7m7@5{@Nv+3c6uO(;KmNSJUv;uCN0*nL#qOtK1U(Y59c;CP>Vxdq|q5qU9yUk_Oa=(d;BMG&7=(c!QODKG0qcsAqLd7)ZmTc8O?zUkg9E|s2P;M&^|N8rsmzYo;Mn?f0tFU8;OgvyRhsazoHF*v*lV5)9m=3~TaDYM_Q-?eiFaB>*1$-<IFgjAcri|a8;M@sfQWW_cJ(&%%7dk?JPlw?%AcU{tbo3KE4Y60IE=d~wHz@xHYMdLUeB^&1GJAUQg-*Qio=;OaoT2F7id;GP2sso}vf)gPW-L!q=S{vk_%vkm+1OK**y@={@GyWS8o(?EaCb8iqq@FQp89$wVaCcfa`BHdM9z3oL4KZBjN_UNJ&j6PJU|A>DFG$_YaJZqfN^=mD9NRkK-&^LR60Zy-+T#DY$47{xp&M%GngiD$QTFwAEfj8ki@*_M886u3TX|t4pG^M>MucWF;Om?h;I2Z`z9w`9H)?yEG6kFLN`Xrnsko?qjwE0ut=YiMNL|{Mo#}oI+u8HVAwdNXlod$E}fo?>Hh?V@th<!r|f1(_8EFEO{2Wn(5t3XQcQT7#~!*<q_QNh<xk<pS-@C3g+Z3iZBa=;9epFlNdF!w&4S|<8QoJ%IffkEcpI>zESm^CotOTWq=DZ7mB-Y-&w&>|gBM@2>;P?jN_UK}s2x1czp^M78DEM-;}>6L)BNiw*7<`Dk*JTxVlru@P(ButR8KT*r1pKxoYXonp6Bx271_xf?hWn!t12O)aMe%Lf$q1)VGoS=$?h>+Zi}&V^p|*>jLB6jKwtjQcph1oAE7M9%Yza%tpHy9et>7I;8my=f0nh}|Dx6WgjQ26`0r>XMantUr{f{})Re>Sxwt(L1&U;L*~|Wh02K0dJ66h~%gK}rZ<yS0?KSPEY58@nV-~;D%c=8?m6t}|qwaUWfkIrxEAXK=bppfViW-}7b8JmdQ*|Y8#u&zLX|uxNbomL88Ii{4v_*deA`xB-a^i6NJ;_aQ#94p($uIEk_C%a{U#AF$N~IwLK)h9aiE<Vij6S2E|1P*jf^wdrssd?@PFND!bTN{t|Bpn5*xx6tX+fDSo4(%E2_c29^b&4@dW_T+dSIcmx{1Ixs%zV9=KF58@-I@5#2&F?M#@OeI*|!$2--n^aNO;qMgFmC{l`DTBMKTL6Fpr5N(UM00&%%a2@o0reUAcyPWEph<UL(!`06PrOfa;Ji>Tyoz(SBqkfa@7Wx5oguq5T}h!FO7c#%aB4C_EPMf6CN1O)UI(k&VK66`2JJkX8RLNS|s-qKcj92}!(_I{c)rb<d8iV}%g8=+-!HR>a@EUwua3BBNpQBkhJLI__ynLdPM)11kF4rlS7P<=|Ohfj$qJi}v#j<PS5!TFO=%aqDu)`~#g*4k2ulvjKn;TcFwO6Xwe>i8r)Ni}}-8C~<9P=3}QOLf-_J;sk%x{)t4$);F9-OK>wn8U#dnmYA$PSaCpl%O;f+Ore@0F|DXJ0Mchpu>TT5s(f><Li0`Ujo~0qXP#=qF^V+`IHc!)9(1B)$4vBW4ADoO(ge?Z~~MO5mC6;v6W-9BVJ5KP+fJ^YV|PLscv|@SFMu~)_cUxEfLTfr^FAV4o6PN$#%BOoAo=j?G0q2I#V+;GrWSxyoT|X9vk|k%z504c{>&Hn$bIzu+j;24Q~-ueLQa&HYZ;$^F1Q$A{z5MW*}E8%E2BHhaigMHmZ`~3lEWEPxf*n+wbMxf4{xp{2#VEvaKA;b=Ldu7f})I*lwqBtJHK<1{50+@GaYzE)u(JU;Go>E$AlwNr^3VSL*m$aU1(B>-$%uf#VAjC)mURVlXs`O?A>667_Xvj6?vYauyl*0G+Hx-C@pYl-{`6jG_G!5J0zc?jtTNudkM)H8%6`8e>qR#NcQdLB)5;3pE-F@N4HgoJl5_@K2f}tkZr3e>}Dtk2;T=EC9f7muf{9dn^?u9BCQ{jAdrnK-;H2yFeHfsCpLgY*ngWvJp)mo)@719zhN(?_JxmUW>IL6%yn#Hn&Q`)T8k@qKuG!k<!~Ssf0jb--qcU^kfm!9aD1YB`|OwE<Z_iSBh2kP5pvkCb98YaVZ!6jx?7gpQ+0M{0lccxDt|lhPI)Pg;5hH7mnzs<a6~kdte##57F!Elc*G@$E^vg=6{Fl$Vb#J8z`s(XoQb(k{0;7ORC$b_0xDWhWdV(eM%R7c{b9#2-;0k+lqrWQq+wTzB7A=Xx59&Z4#!vpKr5-<+UxNRU)v>1h|>=nz_u&+A|S-Fz=JZHc=9#af^hvXXnb6SJ8GvYF(avQO?*H|0-<qY&TOn5OI6?N>HN|C5;BPXQ3t~wjD11k%UpS!#>j$FR?vDS~bmb8rATib|yEEdCke*6-)m903@0T>z~=;s?qGM?UDSzHPZeZJ4Fk=$=WFBj9A{T)@NOjOdn`lMXx5+B%+@(`Ufv><Dlr(0FsE@WG_D<_hjvgi6{3|xIPGfWV$wgB=|5fQQKMQ*cQdDPvuq4A^!oE0j35ez`T2Hn&}rb{pJ@&>D)6#m3io<voyDx4(YNs3Z%W#4;*LNx}wicY`6XkJpVkhW6jr0)J%Ltn%Cb6l`b5NF?XD7t$6W<KgOQ-xaa=qSPPw-c~EIz>YFg!R`Z-z?P2ET+ghDdmV<(g$X73K%ZvtRv#do@W`lbQ5^?OTNjPfIG1Vv}ZJZ4_TBN@bG>aDf*K)(qE{e~J{7f{-&z73=?5sD{DMm}F>fnC*>RKJNT^1`f*j3HOlzxsy?-rtC3wWtxX$u80RtQH8_{#eJ#EBV^gtAwhQI^u@{2|lY0P8=e3Et>2!u#!W)1jHYoo<E_ADii+BeS*<?10D-)*jBWZT1j@j?)KZ--Y04S+Dr3yOs3}0&jb!5A>UnR!zp)HBS;%W1W$`s$nuV(;x}j*(V&rWa1(BtJ#O#Y3z{ue`v-3pljoB)$ySF-=FhYcN#nE(i7XA64Jv}4?y8k5I+Fh5y#ke=J`iv{X{|GL90DkY)cQ?w(_8D>mIa*pz-60XZJCEuNy=USLtng^P3&WY{DLEM{;4lBbj99=i=TNYF_btrdRRv5?yQY=J}DuCQz~pJVC!Ci+o;*cHs8+lf<V^wqg52;vC^!6I+l+{xh;(>z!nSHj4Nk(~IZ$6OZ!M$~~kxSvu~L`-Qxef^?qEmTo0Zd5wR};!m;Cu8|3u14ZE?h<=Q*s8Fq^kEDO{6COVdq{RBWh{|0^NiJSwblc5SD7~@1Sh!Jw3kYlDaQ4{j=duIx4@6kj$8@Xt5^dP{gv^1Tu^n2lh#zPwc{-H7FDCm7Iu|}w%;2#*a|pj6SDufE>6p4rQJ(iI+jYcNYCd+&wV>kh-;rX{>%hn(tYEH5{aEY@m>;V?LG+($dd=IFtr{XV|2JDgKa3Asvm2`4EiW$Jdfhr~_1l9^$5{ViYYg0A50isI(cGwY#us;V%bK>k2LU1p&O6KG&i9NMeB7Wb#>n^BFW7U>*ddJUY&GoGcpOX+z@~|Od%cG7oIt~sojP%Ia+c)Z^Yrdl<e5r6!&pHwIe#p6`KM$WG<5|Jn7!6mOVy#ly?&Ec|KlIo_0rfXg^m)u%p6VplC1Q96ms0x7W;CVCw4N*9e4XV^1yOH6#UbNf6vZHh_fJ`&eh~|ai{uPzfp0V!O!9KuP%OVZM8-)O_S6C-d^3P`IY-SmF-QnuyZu7OknyY&kCjps7#4|Q`a#80#P>v_eZh1lb<Ni?s>M}mep#vV~A@rdq<jXBZ_%S-^2&=_w3?-YKkQ3Y@vJTX6{T%DB*+}g=p&bytrh|MMwW5uGWpJz|wj_+e6^+)i&qb{qmy1bc$^~7e?Oa7hk5fvQgRDuCCPQ<9?dM4DpnRzD_jy%y^i7k=lX!&hBcpGUpJJ&Jc8P{)opdX7}=kEWw~y713c_e&5&&0;oAffH{AJehqW<<TuIN+cVbFlcD_wp(9Yrd5lVzf3TqHf4%rMOzb}ZNBIYM^mo>uKKv!!5YZn5Z?I2Ult2N3)v`a)Jp$0hz@D?e(cqP*@eT1T;dE#0c*hp^SE@-GRy6P`kbdXLX*@@t|CRVX`fEPTlE+4a9tipei|93JUAN%oGwNdY-9<ufy>9J~25`!@m!)UeXA$ifio7s%go~f2Prr?M=scOd$!wAaKcaHaBF<qER&e>#Ws7MduIzpqQFg<!Jk~>F?mpP6?cA-@EA>pN_wVU+eomBw)~gaCvywpY&%yzmBpcYLAmJAO^rK&F+*r2Q9eZ)FH`qV2R;sRbchEatraY=)y8Ad>gmAPax%VTe4X3uU8e-Ha)Kd!o40U&`HY-}twzY9X0?L{|XyWT+9NUt4Y-`2;_{X&+OLIh7wzQPZvjzh^dA+((r9gayRYGM19Ghj#a2Mk6=)OxYK7t!5Be-EvoD@m|t)4Z&tXs#&2w;6O#5kZA8N%I4d41=uzfyx6LV2x{wpbCZ387Mc^L4tJGRm9R*KS$553S~igX-VxLFx$sB&^BYgSz>R2X)Ils5oHsLMC4ERmP}qva2C1&o!Rc&Eq)?TD{33^cl7=i~-nfx6LkA61^d4pX>{x+5Ce6`dp*F;cx_LDqZ>ukKqmT7<8C$1%r!7=wq0%puc}EV8dT@HLilpKG!(joMRjs+_^>mdYo87OV;oWhBFs9d*qBle#799*@x7x1>nustQgT{rKjpP_MEV9q&J07V?~@|IsEqt&eLyM5S6}|T-!g8eEN)R-IqTUEfVezOJ34o@H_EJh)Z~9_zZz~>MK=DX?mx+;^Vc275CI1()fR!SjW_HCGO7Q9W(smM=XgMhw|%-e}*dt-bGm|OQ;obU?}6MSQZq~wYuRnAHoKOgM{sHm|^LACtY;BJQa}3Z5E-<TD_zBNA_#zB`FH|LZB&vRMbV%-}!0kIDf%W)}b@ar{d#h)~zK=Ttc4FZR0Brb2Lo`3%p;6S~R}|s_pWJ%V-y3t<IMclR$%ILU)I!XcN*e6;H$X@~8)qaI8f;(8uuCClnL<L^^xJu;!l2e@(?Zu;}9TdvVA5T)!BJ60x~|mNvtPhCYR*#v=nkt$Kr28*^-(Ovs-Q@p~}sfygN>Gx=Bt$X2eu0Db(HXW1AzkxuP2li#O*;W2#qlu;#~Q$+TM%nC?T9b(S4Cd-kPzx;&6d0Y|X@{E*=4#UyI`3jqO#;r7MIIAKR3EfjDp+r}Mq!-C?5Iy~*-JPi2IJPM6ffp(akC?x^|5HT4d<~sRWi0ChY<cgbjfSf6kd4VQ_s?V$5L;-0guy>h7PSxv@}Q4@&fl4TN>`Ytb!`Pv$^=kUw*#^u0r;b^(c0e+#^YtHbr8S-eh9L#K0^Z23O>LDXmj`Qm-GYG;_}v<h}lUc*iCA?C!ER`@pf#rj;a6Tu71)7;g2Az!fYkb<xY?K+dUqhOs>+^-v}E(oKLX+fUl1t6e_jLTVX;Rh8)2~4?71(1Y89<;V#<f#iX`DU~QieXx{_{K#=$5?)5*Xhe#Qe6si3V9kE!nM@bRG1JEHT^eLeJazcZXQLA@dB=9&6aIoF+@%Rd}En#i=wdl>$EBCyPz24`W4c_0bkAumiN7uXX;3dv`81&kNu*u|f3YS<kH=Ohpv%3n@`kI4T^?n=Oe$x7lx&Ts<!t|n|V9Y2C*6q92oz-o6qg||}G68c=^iHrW2-|}R%BaKnd;M=_16kBdBXyn!u*e9`2N#y0&8&<d51Eze%$JlKnnc|V#!-p3ez#KDse3x!EbEIDKBHZd)<`!d$OG3cg<2aoVJU+ll)8P``~Goj=&~;*yGVYHFa29fc4@Wt-sXC(yy}0q>_+;aqRISN=onP02j;mr%U|M>`^we%3;*5?yR=i@UaRc*E3kv`*5Yq%SMI=t?Cwsqp??i^lRoJ?Kc|inSYy%Oy8EC`=%nS{)hf=wuf%fxfIspR=2PfjWu>-R+jt;BP&dk()jOERF+RqW@CbjcxNVnGfe>AavIr$GLs9N*q^*C@WjbyFtJ&SGCj0eEVTBh|>BqfUX?9>%EA=|Bz`qD{84~kTeI1Zke72sEOPyR49bnas7&(xr^za;)8I;Kb3RTi2J-#S9ihrha?Fqw|_%!&>gmk8jWCdn)y|#wU$Qze`@%KfaMQIr6N_kT)eq>|#`W+3w!5M`<*Gc>IAljLK;;W00v!UlcU&FW)tJMv>hStKC{!Q>MDC?Up9(F03ag8Yr=woZAvdy(PZu+JM5>KgC)C3!>s(MrZ?z*po3y4|4#|%FripcBZbCSryLV7jwx*KQCK*T`6SVZm9#EM49B&02IJaI%3QQ4jH3S9Vw{zVd*y2)jIS+A^DNZ;w-e&FAA4T^G~*x&Csb{{`rr73TOUlJQ&=w0A#8#swl85w=;M_7G{aZr<A{4*&F{y^OX-skfcpgH_p5O8f1rfY>}%l>XTdsz)T9Y^jPZ!e_`NMz{R|BCyK`0RoshmV_q$3w+`#n(!z`)g_v<xg@G`j}5Pp&`>h`pxAh^!5)zIr0p7ipKfw-U|8qT6s$xLJ~7A+-s)W<rlF%RNGzyM*WvOT&DlZ{gw6II%&USn<-mn@`>GvA29h_WmMey;UeHk`jjV2ATxd9uiXkuu2pJi&wQYljfr8A@{(gp_7Mr>15(R!sIW^)uP2zfZ0)I_&xl8#uWMbjZ{MqK!eK6!Exxo85W=<pq6YJ-FCmcZyOjs^II}`E$=ALBD0p)DV>~+Q@A`1??Ci3%@%OC8@{{H;D2vA}I!UM*k)=Z)$&&h$OA0Dgo6=vc+=ESAx#tVhG7~Yc7p(vp+e-3nRD5A~H{MAI$5=ugy{n=7`RlcsU4nmH76i15j!&MmpAiqORFbmMZ$rN6ha`0ICG1#bY8Rd6QRy-cNgPUs3njQ}UBNIbwiiZ!gLI|&(4hdZy9x|4{Ps?H19$^1jCK){Gk|Prd%LpEx>K5l)F^Yiq??`Qk(i#wPNx_+mW$pC&k)aW=1$lEd1$4&1)uIb*s4SuN8qp${Ns|l2uuO`a)fw;u6?KC<0;F~W|u!hIR?am^4eytzC+}52lqZb{w|Y31h)^ytyM-V@HIKYu|kmkiv$s(H{+S6hMknPGzh22Kp0WpmqEC65L15@FgNZOU2W@=lG4x{lSROl7H^Vwm~NurT)=tbx_4Is&mq~7JP=EW0p|^!(SNi{NA9(2r;i!++a*U~<_a=HT{Fh4g_jc_Pk+RDBIW|UkGM6h0v)eyy2<jNNAE51?`XQzH>h|2J_%Y+FMpUUyj?L#hkqAwlKmNT9_2IyzuaUQW<E);Cqq#Ko^1FbZMJ*m^?G)ffJP(ExwubF0avExLl0g!y@gM%_O<5E&U?#`dJW?&Mm(q@J>?HX^j^pkVxRcrlNeV2nb2N8(=R&lZ*XPl4?Q#p&yZ@o<VrQ!r&KT4*tbBuQby(8eVWA^qu-`-O1a0Q?$>(Tv79wYe&mPojM`^mOsZj}F=(Tt#}XOJR97lG^#QMo(49eLg2UD<Z;#W0JkCbI0Q_lhJY_(1&iok<@_5y%^@Bys_JjJfpmj`1e?(S<ytO&72&BcaCZm)7eyE7+53J*p-lV&D&^>7Foy*J$`>p*$Jtr>tkfRcJFbw*`^Jy<n#TeSudJnr;>-WxA;9>Nmvpw%U(Tg091oi2hiEo+IJ6>UVd>EjgEye0Kn#(6+Ob{1E<ChT>2zlYQ2}pn^A|-*Rj0?dW0WkVLW-cM~XuIQhULhrKC1-G%V!AR9-Bi&nlANDGWd8q}d-v`}j`L3V|9py$bd*L9n;<}PZlGJc7!otX6>lQJk!1?o27v|%4{kz(<Z#A3$Cl$HXSFA?75l8?dz@tRhjZRlWXT@Ol6=nl2{>P2Kgs)Cs;aBHt3k|YcF*fX64>2!uX^ga{vHryFlr1_K;cFSNyp>@RW6s7Z(YB6-2y<-IzAaZgqoU<3T3|!iwCpT?G;2f7DP*4P9{DojRWP753bD1$x%P-bYR8;`skXd*#2cZXm=P(iOW>@KXamx46#wD6bFM*u{-Ga^Zeh25C0uB2X>?rhn5l-DIi>bOe)rx>$u?)>e*bmQd(?Zxa3001A4v8_G$il$(AxtOQoi&mnkrCulVlQH}<{`ly-;Yx*9J{xK-V(h3=dx;?c$`Hrwr_nw(L<@}C_RTVzNH>0_<P%2{0OjHBpr6poq$OiNH*bnzi?<I`?*20w>q_r~4!24+m)C_j5kln1ugjVFb+%tqnm#}9oAqh)Z|k-*u4ZGo8(%X!;^mZaGq^`fAC0H12PG>9JDE_wL{&${i&aU2{F#K&Y@IQ6_FiivA&OexL#e#w``p?S}nTlDcz!5$h#W5fo=h!k$fzg5EbxHb3zZ;bNNv@XVC*1|UWUdIUkb8r+PRv$g>MUzQ1E_};t6sb-+)yud1vz$7Q+hZfiMtT7-{Wz`Jl=()=7#Jk6a2EUIl}wO2Ee|LkJf*ar!A=u<IJgVH3vN8@b=$7rIa=kQsPfIF8`qayZ+c@z1nUg$!oLb4nCdS5&-D+7y*ANWp8~PUrK@Uc_JoZvsLEdjkZ~ty=?q-Pa(4Z*adQ+-qDK=!$6?lFstv3q_fW}?;QJ|?9H>wxOisgb@QJ!0NuRdqVY=hol{CWfpt;KQKt(gr%&A|%^Xj9`%BSvmUgb3S7WzoGPx_-6zF=w<Hse;eD|!uAc@r*L+yUP7=AvG@L882q=qMUvOB8bWB6U!DOW+Hz5>9pb{M8gdfT<$@>IBZej}~uq>@CV4G!;xMW~liTFK4K5tLpG-@L$b3RDuiqyq%*#d4jK*U?$UU-02QrA3=X+sxw_9t`^mIssln5+`#oIbJBlyX~gfm;=c3B-0AeXaZAb*a5|l)$*-Edc*<cqnG7+a4^9bO)}5f^R?C$qS>lKEm$FQI7f(2Dbgq2iPPx9B6O)~IBGR<F72>^I`oyiM`_OkeyDL}i&l2-15_~b^;R*)CcRMG&o|!RLa7q_m-sK7W<6KZqrRZT+fnL6TT`&jtt9xO6dpo>adn;s}Jedg7q?PGR`tB!32ohR8d-mOsv@23&WGgRc?jU13MC0erxHQ>8g_`0sHi~{OJ=CN({fDHW_#OFTsZSdgu9)E8w6&|4{TLs}HKb(4yn^(TrkQ?;#h~Arg=nN3g&D#uP3g=^oFNHi0_5ON`UTa5wVRXzG3lEG*|Lbeo^^>;Q$aQyi6moBtbXz&tncn`A(Ob4hqIP+Pe&(|mPSUmL0$aE!hwx^%l-pn5g48453D6ef3jyEkg)eh^qq4`i$w=kR2u32=uF;i_YLPnOopP#T-bAwD{FIODP-#psx$ZYH|l#ulnC7w7EGAoUEf-(eLc|RMXbT+jqcgfGE!lYkn!@TwC{c=@pJ!7Hb9hE5j^PKlP=84XpDD{OGykXu8-9hHKG8VurK4YPQBMl$sNmmP?1@Vr2R;x=_a2}Hryj%IF7M;teSIsc`uv;@<yrCxm_+TJy#A$M=I)^^#{%Z{Hy5HPSKsZ75O#*B%V1oarut&j#VBUPhqh?=LnUj&b{vB{>h<D0l=QbP~tsnSTezJYhg$d3hHC}Mn-%|pAj^(e9Vag=#3r*oAi;oFURH-L=dHKIC1znaA^dlnmN4$;pX{{(C_EO=aSo~J%oyaw?AVaVoI*bom|_UUt3?pX!@L-Q=hYk&O9QMqCNzsLV2|~G^P0EXA(Y+;pJgW?}?uNUb(bB&GAmXd<utJy6kC4kU$MkPl`UJ?37Awk>4Q??)Nj9L4C0UoL!MrmA#)$8uMoLCFc!2yuP$_v)w-Yig`m95tCrIwzIpvw!ezb<xcIvIwdss9lU9so$BtJ`?Wd@z;}>j(2uQcXJ@yzN%lSG-g^E1{vGG7-Sv7c+55EzYl8emVmE3`OC5s^!rIr-w2*T2Ic)cbAqXcRRW;Q$hDfw|s&TV`bgvEHZCZ5VdP52F5n&!6GJ=A888y4;$7ojvXL=h)Es^JpOp52Z`gIpRK$poDbhD3uGOb>-+mBObX}x|90tt_Y@u)k2;VDpii(y(%b7|yxCLh=7mUE-c>mfS9Vyp(@+{Rr2v!69=>!pU{qvk+>{)Xha42y!^XarIG!dC5A;R=ToDHo07@N~Jfn6<@-*Jz4SskBcI<j)2$x-gdEfF6pUjVT5jy5*AMQe!28sr^*uc6YV9zXwZ=Px1KFMbhz<*$4Aa_Ck$GNev^z4*&~<nas|*=qnt`yFcUvt*OFrhVSj7`_o2k?Otuyv+wqHbQ*S={ci6(jPOS10HTy($BGITqT`;`x?^pKq5x0vNNS0DZ6=<i2sWSrj>S3F9Q!HOfHqT>nbb!;bicNnV)AQF0`cW>%9>@&5+dtit9ylA#w;}%jvmUQNm`R!>r8${2159(78+GW0(=#rkpL+V8E>Z8YOM7T%o@UCX$X&YyN!XMm9p0)dkO{FTsw{ODOJ}-a}f2=S9A-lcq+{bZR*o*jE(UZ<i^3D8N>%&fXm_N_&i1x8r>;5KA23|&%-_t!44tmC_m_#eJ_(VPU@OaNa<63FS7-)r!#n9W(&d;PN;OABPQc(X|yA`EJov~)8q8Q`F^f$t2rK@hp-rD&2c+~36|5EIBy#5;bbOG=CSTi0<<F@!2t>Tuvy_|IPQ+n{RmEmxvfRg7$Bdg(SVZsrQA8i2MRJTp5ntPzKvj{;${pF&%;SGerWgclPG#IaEHt%z~>TcFyq@Nrq{T?`_0L?`^fI6s`m`I2(j_o-B_|BhWjUe0((D1P7_mn3`}2ArG=+=v@U5cP|NWUZStK@$_GIqZEl<KyCO@O`Y7eKX*7TzD^56FK-7rY3qsiyD4^}atKB!7y`(G;1GwlBP|o<domr31N5jc+6nEp0J@#Rj8{YdlZcQ8_t(B%7PggX0#!vBTGNvlRyB%8U4qB4|@(6UQgr}CP`QO7~f$hX3!s?o(3-fZf%WNjeE>3xwiti*}tJu_x_o(=g(<o#<x4WDg0~O0e;7-8pHhaS(!=+yy0bLNt^XQjQSTa|O++(IQzxawK)atS-Ag@GT2hk~k8HomS&7(UxVnA_WpSC8k=`L$@kE2IJy#radP5ISPG{mJnr~ENFZkzI3NtEQ7vb$_17pjO~WvNQ{%#=ahz@iRd%FPcn_e;8HLyUB!PizjBjI<C6qy=qLB#xR$WrZ-9^1PH`@_c$Q7%aAUTmn@a4(KXZpHX+RUYB~ukY(oms_lNm2NR>dF(Dlvt=R5=sq$fyq!K0(I>HUpmbjRFF)g=`gf_q%kvCe}n^AnICA8ID`Aiy!t;$-+nUGo0aTJAm7MQGip>a{44a2kMIqV^0=Ue;@6M=vxU$`DBUPQrAWPB#A%pTs5s6mOdo|3K>Bb8!^b6H?Sc+h6oNC;*+I*O~BHjzlYnd{!zIpGYuG7+)lrZw+Ib&`x0?aqv74-N%%LdI?JLRZ49uZ4HlK}JK*9I4q6s%G83s+@B=c-v8q99>0|<E8^5viLsn(xC?e&(-dx(-{S|2Wt!DVKV0yM*`=~Tj8RUVJIFIIwM~<9iwE3;xD3zx{u>YJ8cA@$m#Q7^lnk;J@2A_LT6;RKk(N@sX4>sXJjv7djxgH^E<9kfT8P9xVgT!S*@?$cb;J9FVgKN(=ClRnYPU}NUU^-?a4MeZA$1ME?&RkRs@V9RF~DYy93SMCc5nkZxncl-YGn@(nAII+W7>}OJ_tVafok?(4UVZ-Bib^Q{v!>3lkL+%`<T>{S*TTeRV6zZ}pn4YSBFb(`OFCO=99vJ`H66yej$pfR3pKbk!IhaB)A-cyfTW+!R`_g^vfm6zy&XKQAPJ)&2VR-Mh@w&sGifiNp@%e3)Gzmo+3NsX7GZiAPb3G-Gs`@dX?!F@Y{WzJd2r<+EZRp48~k1S2`cMe5Kc<bW9Hi|En?3bBqJ0S$FPBxxNZ?$_3KU5j9r^+B-|28!EnK2lr<Yz){8$fW8EfrFKck%)stOEZh11RS`aj+-Yjf()RrA_N_vTlcH`aN|6vr3=DI599vuVFX+{Av#r1h7R{Ox8JCRh?2TnPZi-U9mD0`M7q%qd(kN>4cxU-S&w5qpVua6`;GeMw(p=K*ok595fp&~;s%k~wv#wKQLwrI1Dg!z4XBw`57?E#k(;yrifu(+vWrOXZY6=*lQ)!{U)~oumlEn;zPZ1>yZ#Nh2s6`+yckDI<HZEK6uJi6hPB>2Ef!^WZGESn-f~Kz#AF3$ulB&<LlB|E$6$UC9Whvhj?C#hoi0LW013d^!As^WLg^0X4R^T1$>PvAkchzK#5Q5X&#0CM+)N$#8v@x}a1hKI;RE1IoRhcIijKUA#|x^5PC2Y-AfFQMhL$of-Kd@zMecl^9dnTzo>1_4cQ7w@0Bs@0gwu3p@(5~S-DwIZG$g=5O2um|CK;|w#t(S+q)8rozSEB4*0?*OX&{7*W(R}DiEo?ZGM(;(b|xj}>p4d!h+RI3GNWW(Rb~ToD<+($pL%c5O7l}2v1B_;$RaXyx_dNWfjA}p!l$}~%Ss`^l;izDeg=j6wT-oqdCJTUk>T&%I!?y_{+kbRIy&Rw<7nWp-h2TGWFM2K%RLC}PofW_3C(OkTt}e-M`Ux~+1=cOe~%JG2(_Q!bqXC&t@;Xq1Ulll6JJZ10Qn$$E`u<Hgv}B0zO&jYR-9xDIoMq~_OW#|N3dm24$)Dmd5G4pF%TmpYl57xK@i0Ejp`j>Ojmc&(>hf}l9(jaFJ}T_0olEEf~_MxL!X5ymMb0Rc5Qoge-p-oYHk*ffwb2$z_(LslNh)*^1-$nc3Ke(SB{TN#)|VCo-$Xg9vM0Z%#A6T*BOqO%%G7fLT7!3;uxPG>J1$J$dS5lw4>8$N-wE2(n_<G$){czNkmVFT_DK7q`_4~_-90Mu>Bry9;n|L9)1gsQtV8I&VSk4-db@UM$w3g4KndSeh!hWGd@Q<B1b>G4vQHM^uS668o|L6SHcUzQXII*7@`<S2wTGR5YwCVvxX`ElD-#e<1LJg=D3el$r|8^EO?P{n(+abY&23@jO1)m<HiH1Rc74at|fnox??s)5{c6UG`uA!e4~J}lPjhsgo`LNm#&DetROj@xjsA`Ql1+3WYU?x<$8$3(Mid0%Ja1#>hwD{2iPuA)?hKm6E-XP%+yV-1~?+X`>xiCjHZ~8HRIzGxVtF4VX;78`rJgXv;+)r>zMR9Mbrj*D|y|qd}|#8owBy{RhssRxLOMA>Oi+h3={_qkp=f!@moWnZHKO>M~Qr6eMfT)lLxK%xI1!F>kW@kE-ZedJ0eLIO47pWg0T8#eJ5P2-Q7UvIQ0Q>s=*B(_|D-O)N*5)eZFxcJg#It5h*e!8KlLtIzBlJTuK@d6J}utg;m%Y<VhAJuNbvZ%!lnm`HDZZ_=w0B9~a3duF&zJ?KyYBo$C}+!5>1Phbc(OvtLt!g`#Ko?zoL>i`T9#m3&t{B&ji^crNrrsMXd?55v6Guui1t&Eq2SjhHXhlll>@#ns)_@GjgvSwBWWs&I%Xe9M_xuv<4ZU4i;C*U*`KJl)S%YxIf(d|@5BIf!C6hLyvJF#P8+Egoql;8bWcvRiv|e|@*cTMsoY=yQ{v9$ttd5yF|L#!C0J?xyrb-C^+#>PObM3z*#(<L*2!Pff&W7PMi|gOGG`w)K70Sx*?%o<Q|?CdbW*LrNJ6jXRE<=D2kXmuXw3L12lSUZLwQNFCD0k5?mmmQiIVk8^U9TV?^83X-{W=dj=bE2J8^a|VSd9k8~V+9Xg@at(Pw&3Y{^;m1KLk%;9RefgmWw6Gj7S;>z`8FeF?CY%w>!rVV>H!GqTukZ4;a%qwO<@txMJ1sQjSf=-;YQ5YEI@YtF1OX3TsPh_`sY11vR4z`RD#^rZ`3v%QcCxH#GrBq)*;Ul4CK;i+yLx~9LDuCu|2*b4mU|Lk)@jscML(0f<ick;l4k+g2xX`(`&?ryNuaAwDx)OsH7Pt9pI0*8^Q6xRo}>ZG>^L2AHD#U%nYPRVHFA;XLr)EwH1@2h;yYrZu_TbJ?ylF##>M{Jc_g;knw+3&5H<Lj7%bvYQL1qCHY^KD5nN{n@t{G0V6;Y<I|hFQ7*6|$el2ojU2L6Sb>v_@qH!<)mWzUFd~!Np!F5SLw1yFLdEH2XMfJZ?T0&5A!>wJ*`39eR6@6kPTNw}RwgswC`(Ca=ZF2_%O<DlVFYo$}TSPTbLI3!ma;u>;j2*<e?44K6iml!-j&#C^F7oV>7;f*^clPV(n#dlC9rpAOo{*!z$rn}p3+K*?gPiD-EY~F7GAkE6(Y*FARvdAZMf^6uHHn#L*hRDnp{o&OjS-3a5r){Pv`EnH;i15b9dfP<OutD;@i-#9RFN3%Z>_(%U(0DAapRiON|H=N%lVl!MRiPVsh~9jf;sBPG(G;;w6~gt5bE#su>QN-yS3HoUOn92U90WFj6JAstgl_sOw{Q_tlUeWcd^zW&_2>eHb336p*kbX6zf~7Krz?%>b0%9YmYS%D=foJejq*MeZ`=lmwgJoMM9*0_u2O@KfnC^+57GbhkZngi_1&RQt7$EK3;W1ZWb-gE{1P$3L6D$&Y*W6`HsMZ_)>&@ePE=f`lA_RK|u=~nehoD>msH19f3<tQ18y3@?Lh{tWsRSdj!GqMJhR67V7NT2QN8ZptQVvv)o#|{uSc|mM~u6!E&)gKhP_8qgJmutZ#GnwyHaO_qXf5b9a5Ky1`)X@cbS~rF*`!yT9c~=SSb!tKOBt14gWt+FKZ7!Z6ef@%^jIUtazU^Y~o;FA<HxIr%U>&>x(|GA*uJqz^<oz(4ANTu&<wP5gV$-p2*R<S*}W_A?b}lZphATl#y?zN0bOTX#Y^*6_})>rUXfVYs<bS_<#(Zz%7a+U6Z#JVG}Q93&jWT6O!8@W*3=N=Ea6Y=#7oXRrYzD&5gJh-L$X0ua7A{pO%~6pf4Q3x!X^aq}z_-`9#p6Q@T1f&G%yoW1H#2EfQ%SQr52{Y61LbO*W1C$YFtgradP&J^g5U>p@m%qfiB|N725@wHd+Uxm^^dH#0e+vS7O{8HoFtN7>5#<v&C9=`p5|KR#ix&7krS9lZ$ea`n>D0__nzP5%A(wBKF1>ItO=i?{@(iBQXs}yxZR@91&(5WeD4_m?DkS_{iVdKJMlylmV)YDLoz#GfO!&}$!VLK|MHdv8#c@gox;tQ$N^P6!fEcBUNtuXg+!v*g=?|ywHrF-L@8RYEOl{kt9=wO0=`#DS_1I`R^E(m-kwr?APF;GPAF9R1_ax4D_HVw7JL&z86`Z_;|JRF*b6F2VNH|#gKtmR>J9wb|jrb#l6@XAs^%x@s3wg?A3=S#FG@W%$riiSxjkVLDLG11Xt*~OEWj5rWwt@UEB11vaQPUF2rh#E=7zPnf3xEsoel3PwpRgT1AU|5I_YQGhh4Y%mKmd!i|-;C3T#`DWISRXiZC|DZ3{u-Ty2g~p;cMc|C&V3y<YH+LCI?MY!ao)kI+zJ%)U3o&JA_`77V)j=oi<PT{Q(iwQg@`^Uf*wdBVCGu2jT*~}8y=4;{hi_(EC>rMUj_#q_eqz*e^|Ga02={6xa>9N;EUD#Pj2BCxD6XtX^H4Zz-_5hwpb2Vx7TVRh)o-N`P{OuG!<1GRLV<@IUyS#l$R=tr3M`sgI2SweUR3#^g)tFo5=wlBZ>}#|Kd<Y8{&k>6DCe-YFw?(QP_+pES<K*nk5i4iU<<POXyu7=p7wnSxkqna9?#waF)GnYXc|;3;QU^s|oF+-UZ}E(HpOLIBcJ%Kr$Zr_+6E<OE6Xj%0b!(=k&oAD%|Pyd9mAueu99(>r9jLNQgSIXPF3chzS_TLiW@Pf-<)TmG@9XfWj}x@yRC#r3Qzb!?)0ym$_3DCD3R%D!3dY7rQn^fVzEUqK^WU&m`IIManKi#301$5&2lYBzJ3jHPnTL%OM2;?G=Fs8yvbhO|jJi+VQdGtQN1n{YGtz<^X4g{_<WgFIL2v;MZc8k==pRQs||XT>aA2P9KLGwdxxo#Zuw{%II~_)q9#ZA6@FSE!kHOPo<|BRGC_!O<q2-Q!rN?p@duDI>lq3Euu#!V-}m)l`NKD!(}h`OiuMm$B#r<ISaZKoGZ}=AE4Rl5xwK;d{=UIVcPW)7cs>Q3#*^mA$BVh_D$XeylNW85Us^DGpm<1-rAW~bmyVkY@F-*&f8mee8H|@_+<TU;o=tLGPD`xl35%jIjLg8U2LoTAa98pLcb}K2BsHA?wo*=O55wU@2-Z2$~4&0?0+}4hUxpyP`zpTwXN-X?auc08=8U*L{2<3bh*a9v+R(^EBum~PloLibhnZ;VJ<!&Oq!2WhNl(h2`y3D3ztnn`jVT{2=1gXZgvP5DMMHc2qaX&dlj);p5UB3^eRdDrY9nx$)({``gvbWue3nGSv~=2#nHr54IE@@=6V~`>(ddkL`^oBEK4!WE*z>toNM^%QddK1Js&oeXg<q6O-0F?R?3|23{aP^<9H?059=2Xd8-xd^y1XErCM-G2_o*mMP{p93U}6bY8&fYc+#!$u6PD^*EcaPL3kUlmCBMfV8Gql{^}cRcQ}9)!%s2P0$wGqw(ti>=W+E$kRH}yBRV$sUi1-$SEzW#e5<vsx{PE))P=3p^^J8>e2P;G*W=DUHc;JAZOh;=`3ZJX{^np(`4>57qX3g~o}!n;nWoyR*LGo4b@?<CI<G$yVdVOwREV{#*VOdugeg<N>}L|zCRQ!UuInppF}iLie=O>MEa`tNXR=&UZ)nNYCc!5vghf8BzqQSARw_IghTYJ0fDZ(lM)vNr<6#f2G;Iu`bW{kJ!;WC=>eh5R(HZKNPNPAWk~<|R0-D-R@iIvu82Xk0<LqvhI5{$oUM5P$!EY|gq{;AmiGMHaz10)nzFr~{p*g1crvDfIDLdv~Bvw0-u9@RAQY0Nq^r8tlabRE7_lN?U&9a*$6Y!kBx)RS#G@Nv?2pl<xA_b(Sp3z+meRMJu4jh`*0&ez-t*hEM$*k)w(pPya<0W&Hs$8oVp{@Gpp{@!CnTS4`t?W}_0K%Codw-REFa7gqYInCcHqfD=zP?#=by0_+o@E?7i#pjpd2FLLeevl|F;^*DQb<?D!O5We?nzYm3N>l&vJ$-{lv;dO*o9`QNU~u`yd@!`!uD|Yx8B$SnhMonGqP4~y#~XugF&+4yUjqEUP;*{@m4(6UXi&T$cSdZvN_e*&_Q!ST^{~rz;uaaCp(v=Wue7nz3Zz$)b4C-uf74|lA+&x)wy@Q;!x&_PfTNxN+%-0r+z1`P7FQeb0Sr;mVJ8pA86>f{M_O6tr`j#SupOG7w#{qoc4r8{yBnK>HRnkup@s?+q<;PzL+<a+>HqZ>l1>x`hdFerA+Gj?7P(B#}XM&v`BycL5?hyq{twJ<gQKTNEOdW0g}y02vp>uaK(PTl%jFd=3DC7Ty>G$U*DtL$@^Qt4G5$iqZeY;X-6g4N^+kw&+z2JGq;@B7w(I64H|h1RRU<>-`IW|z0k?3x>+MzvfvG-_LB!i#7E!nSnOr@R<Hj7a4FDp?&j6Asd$qIaIs^A86#u8<PJ*y2wM@*TiCyZP$hM008ywQIS-gDYR&A{Hh|LALSXr}--5w4=#FO3(CxJv`QxL4$4#Cqtw<5NBUEqm<c8r2B^gjOZCc{&@B=aY#Hm}d*#_|<MTPZRq*unpBE%0xJC!&xZBCVjN%|NG9wqHA^?J!%bjszKn%mS;vT)1>BsTOW>H)ie4@@H2jw=pNlLPlU2)mAoUlaONN2K2bQCgA_l|mak&-9R7h@Cg9E0}ZAk?>${nkkvqO%fjN!X!IUf7IoazzGxwMSl4dwWrj-&G96{*!q|{lFX3IA>1|H+LK?>@GW$V<CB2`3u0xFEpdeWqM!xISkbfLeV`{<hE4ZG!JrXX86En)`)~kK(*3A;j!tmsYy4;wjk}a&xWY<M0b2@_l&d!ElZ8$@PL@TNXtdF2C_Y79422L2CET#z?@lry#JM#c!{lSoY>uTw0CGf+47-S>y2%ia62}<}AQ&@H*;T>FWLA$=_r_5x>W(I{rBV%4CW&lzE2N!3!ts2e%(iZeuvrWnA0s9k;^I@FEOfZnq$kZs;>m?IVmy_9mBOsh2whaZQVF*Sl@8dYS_?E6j?cEbHMeshZEeo1&7~4|C9YJ*b;>w!TKi3LdP%gMC?|;<_ml;R;b1a`xsZ|PD4$uYw1eGcn#khbT~(a|o71Mv3Uh4PLJEj2!QTkwakk#aQhhMBwwck~5QPC(rr;u|<+vSBuQ!c2mpfn`c4KOdr!gTF9ukxq0d(5YI4vEOPPMkrJ!$o`wEfki8l5!O0Pg%5;5TQR?JE#Ar-Eu)syn`}qkF1o_%z7pY<*M6lme-e?8$DWjB61pV9Gny-6DTrI4#&>E;$TEz-6j)Bq789Z^d6asUUa}`YvhzY_ab-m5}4P*c)k}+Sw*u#wR{Gf}3EV#b6T<NXPR<RCDs8)8Qb)BfmqTUS<Tg(*E1}FIkC%k79;XS%T8cqOkQO;6K8Oy1!)>&*c<Vpq?UpiP<T|e?QFUzjHJNq#{TFU6o!qqXABIzSEVjo_B?D%Q3)gvqQ?6zuBr&%x~r3VKO!;aX6+B_b^{rYdCY(2`e0s2Stvg!ZbC}JcgU4aC2FSYd|gE+$X-9g}AM|>${sMh<VdnGld_~h_X9ml*lA+vwV0}&AJdJ>b;AQE4VvRxd%rO%@KR(4~=zAm9Q);gUr29SLzox?tRfmB+|Sj(-)Su0#Lft6JJJ#t9@h#AkIC>aJx>L-Jaqm^0C!sqeT+QBNTC+IzHNLHArN$7Xa09UIHkNoSWGIUuPRq1{7A1`+Pju+3HpJJp#4|t@jB>A0B}WLy=1}l)MsQc9u%Su<U@d&p9L;0PTz$D1$EaZp#Fgovof1#TH5*T?~o~dtH!gr5JHwqe<?)>!GImllnMKw<-OpQ@1s1H?O<$;;~2dx_6<YOeqb>a>*s9`90QeWw=0LHyA`Yb^{EancfdM#xOzYQwrVH-R-@-x2hX&s3>6S->k~y=?&j2OI0ke+;HL)t|&a-+rJAm9cjDkd{w>i^j=7dc3QQooMN4qqYuvSg-uvV;IM2sSd75iK50E{AM(Lup}K{Tp^24W8u*i9jLCZy7_Xo5<x=pfhQE%b*Czu;>1}I759EqS0mkibF;MTCw%Oi-YthC;mpSN;yh0<wOXeUwyijZQqX)uZSvoV>J$Y?7&Q<{DnMo1>`-TYf8!KGj*$UG|m_{a@QNWjwr64_=^l)a$UOYwSMlH(}8C%KsSEBsSBG_x6&(vrJ<SDDunWfo80W?daS7#2GNr%CoPHRzkcY{3002v?@rhOPS=*NuDnZo}lu-hPoLL<h-#dIZyTY*1(H|jM<sPT)D>1hj%A~`MffeD(R*ii81fxzw7Z99#WM?fuPjsSNjL8F3GJ#2Loa4kxRjQR60@Un%Wu?_(wp?W^SA{-5&haj6!=wvai^p-qVj)P|6$0oGnp3Pg6U{%6FbMU<0o5IJJ8pPV;g6dD=<(sL)J5XBq2KhuAdU(Y}TAbvqf=A%uqkWOVI+50-z3d(tEVWp9(;{Ita=a>YA>qda4HXkQ)PMo1V5Lq5Fv8X`hOjz^5ki$Xs29*JDa{rJr{U=-nl=%U)(@lv#$X7K;7URl*upsK4=0gNS|a~&jFBQfai;?^Dp*Y>Xwrd!1W(*PFpGpV-FfE~G<!Ht%*nl|I8Sz(lVj10N5|{pomVpZ*cENM6&cWVV_8IMP<iT~$7@Unv5>@b;n}2)Lt*-F)(KM^QAPBgFeP9+Xs^s-!^3X{%~2PPzNp_=5Ff@1?%c_^hc*)m5-7fV0wa_2h&w-U7vN67EgFcnd<Ab%SS*$N#Zno1k00bX8$#0uE?aiohUlm-9`ZEFr{*#<FmEXIvHJRom8HCem7H<H4eR=_kjY4&%rMZ4|Aa|6ZC{Eu`AhilomcFGKI{z-gJjvos+z+kx(<(+b(4<;QO}gMK((j=x(hHp3)m*EB(*{_Df^~3YNb6Xiy7ABnUE#o7frT`GypDc?M`*|4Fj<Rp$szT!?P~xm~*5tr&f?=FtxMNs|xE8RCm_r+yzlEUSJ+?fgQYM1?deK>;bP1XO|`#D)AWu0s)n(L|x#9X^uwn4qF&P<CA&DgmEu?{=x7cc!$B&8WfgFei`<?XP?ip5RoW0G6?X1xRR2pNka(?-ndzBW0y?86lhcwedSHE%F8%kKOaS~>>z8wofC}<Hq;jkhzZ7G2ds=GT*S&rfN!|IL)4Z><0L-<%nsFS33CgK;{bnfJkd#hz~rbHw~iwM_&T0UMlt(TjEcv@m=nK~#+6TY5l92H412n$J@ArgV^psF-HmB6mZ_zFuBE#Qa%f_WZq8kB8mm}H5x6t!Vx|YJ5>}wQn;m)w`L#oCgAi84c{jj}aw0fkza3T-kkMf#5Q^bO##y+Le0(E$fYSr9(u@4a2I4E@V-+%z+2%50Fs&@V(xpeZKd=!(=(>S9p?$(ejW?@__8~Q9^0N;zZurH)@GNo8$A3KrKBIUtX(3-d?%<Px``X*D^<Qh(U%UU>=4*T3NLS6*OF-C!^Fqu5;+Aqlk|D5e5w^{|>&?Nx=BQVx9osxgF@||xGEcC*r|uHC|6-T#`zIiXbbvzQqXpc+*duWdx+x!x3xg{t`{^?_!25&yv{$>gNg8!R@#DxBBnm5!^1e|yYEzC!dEYB8I$I{6ccdQrEcEaoGwjT17rAaf4DxHCF|g2Nt>jgw>a@qr&cqQnnP78chc7hA42u&$L2S63MQ4YfkS{zu&;Xm$E#McDKc}6qn!+ZW<K`*y+z|$|&L^HJoFQEtxMF(fR6$_u7I_t`-2%N5mU8%|5@Rsu2R)YzZI)f;LgZ=$ECdy%ITHG7m+~i3V{U~-tV|Uc(~7Y5sJVC6*Bz{DMVBV<tgj?+c|eEefz(5Df_Slro8H7$7B2x|B_z)AVLqVQ$%uTa<lQtHqWJyJD|9JV78dw*tX+HyXn+S^Rr8LzNSK}{;uoXbUI-@0B7typdb5w54$m&)JCVFXKxRQlV}*?C$Yh8RjnudEN~eRi==U6BK+eh{@u1V!W;wYs_IK#cyI-&Ggv{S%i=^wOnFz#pwnei@6MW*urlf=Ktyn&Lq<@Ad+g`JUr4?Yr+79zF?t03dPxI5${nFFoAfBruK4F<dAiIFPTt#h_<4M0qj~WR$XsA5TDtb>dHLCBaO8WE){?T%}WhgDxq3wQc*PHttrI3XuLi}BLLhEFev)MD`MeBW(TQ8wdlVvOMM^xk=)!OaDW{YBjwZLPmt<CTQ8XXe}W8x;=4V7mhQ~nUCsIFH_$^ZEdp=ksIX>H^|Oue<}CiN92llA7ZoQ%GHGGuAn7LSjCr1IrKjGuOU2no(0Y!=O+xMIY$z?VSo4ReuFQRL8ZDXhpI^3LWa2&`8$lgMvq2ignQcB^;m*+}e8ke18g*#azQj*<nBuv915JBVeEL6nO+2QCAtTT~?y2FCmdgSUwPC2C<&Aq?PHA}(I0)1t%Nw(Qr6=!z<KMTf-Hic&#(FRCf6SMPz4Q)7hDo$d9ldRX690SJjne06(sbG`1qJj$y@bSpwD{J>@Yc3d9^W(0k`yX5Qbn{1V0CPvt18+i&x(gru5(jT~~=?GbOmwIZ|rD@gJWGCg^9n9kO4ZG%fWhDzHS<+@!YMTx^hD=lR$fm%sElxW4%cuB6^z9Cb>kPv=7d22wP2ee_zm#kM!`3;<*6wn8QaVSKOj8#Y3|x~it7hzR2<G6-C4#?)Tph8Rz*13Wk8TV0B1WOKj?+O0q&i2h-`s$mREThXoC?;g<fj}E{+D7$pi`lcB?(|f!b<)!i9aLcNCNPqxH?4VLoi&1{rfwNV0-rdij0-}lp!6R%g<66e(?13XWw%OTk*@w&z^lR6CE<s0HqUN2Bg#UZ5;%`t?MNp=7UpM-K!<<HSaV}x0lpGW;+?su~D-EhaU5y#t0#c1~5dX=t)osEuA{8*`4kBeF+y1Pe6pU$a3$VbSZy48s+rvsXqBBbCTog7Fs{?pm%l0cwhIu_i~!sK%<8vtdkYjzjlBp^^uoA6{ReW2}jZJ<7gDP@4PcWw8_*iSLfzHZstZ{hIUr;erTWT>~iCUAxT@MQoDI;>GtzNlKzsm=%->|V<_fR-0CUp$!FikK+C5DAdN7je_;4f4i=>&@)4E%BW?Om=r24ppTA_>(Cdd$)VbNY_0@1gS4(AF!Sx3V>ksh9<xgQ6esTFPmp{4uhs%Gy{PpDzF8?R|{$B|4`D^&`Z}87A;QxLJuYc)W{v3Y%2)_Fnd{LnA*FV^%FNsurD#BoCjtC*tsn&H4Rk%ySt`DB1U<@8CmllPaZ@9L;>sD~PV$fEaJ9Y7)@x_90%gjha2^Kl*Z@6|`-yV+tDnY?A{;U&VRPINYf5d2E3=GE5IlX78CXjZ?lat>>ehU~446_OpGgWbYw^prFNGoTpuI*AtgivYoL|VQfp)lL}9|&)igOZ=ZhW_u|A^Pzd!|Do;d_lZd_a3BINV18_fh@mc*-!L-y=pXWL{EPz50t!Qo-#}@KHv~+?El}SF)+4)op?K3Uvn$u8I-pMi2eH~URG-BQ(O^(q<y_CTo-%?afNBGe?&cm<)OgrRFkkF(7SEVtspqkk$^SAH1Fk7sRYAKl8l9|V?^W&N65ctw<s<j+}UeT)sip&uHK_RQy*`xZ}FPL)n36|Dx2gew^iTW-hl6Ki6<i48a%L7a0l1}gqaaZ9xIFKkM{6psmnE7$1z%jZFdK_j*WE;r+Uk+l;Du~>e2o6d-w6#?OQ(OfsxOFW#6sgi*mX$;FIdmZgmGcbo(|UTH+YXHy3ZGKdtVA%p#OlFr2$^(ck9)A-lR#8c{iVDBP-U)-ciB?s|1?es_O<6(f(}*uIK@#-EGO{7(fcH9|*Yv<hs?q|ytUERgV4;s`LfO^G<HpHj^JUn3j=LK;i3X+l5#SOQ}+YWH6wWq_0T6;QjIx39OO#pk~we`?~0lYo^V84zlBP0C={a}(4Xu0)_4d_3s}P3Q@Mc;m?>mhvTEmCYMBZZBS6ZhzIQa*=MxpIrXa<!|8L`!~1-e{%U#rvA>HygGja_awq$|2y7}KcH{^<?_c=<_Az%rG0S<z%CH`fb;h;!;~l3=U*c71i@p|MX7UGl&aDh#-!!<$C2MhrH{yh<O|EG&w06gj4Y(*3Ze@AL?e`lHJFs~u<xMi2nsr)e)Jp2G&(rxkIqp_9E|)CFu6^%k~pJw=EKo>6Qs2>nP7yzdN<Wz{E%Wgiz)9v*@1f)MfJwrVXvK24!%19iA;RH*Tt}ob$T|=DKSDc+F>hVEJ9guf&&@1_b{R*uOufTWk*F>aASBxsUvrzBa!sW-^C!hyvDnu2&<9B`^|?y@WsP)nO1jnPTKWYem)qE`>C&B-ABFQL|e4cIX<A-?rHi8xQPPt14+261H~jYycKv~>_2RG#|6#>7PFb2gw1gHFrEKn7btPIzBGFV^@0S9n6~08G&#S`9(gOk3axLV$8<1i7uVo0-o@0N1#e}2>uznA7&0m2r6OlcyEJ)j7LG8hF3TF2BnCB;z-C>$_VBDZKFZ)Wy;)5Vg0ZAqq4G-iAS!aXCT1e}w|7vvUMe+I%@-+O8sV*AbA!6dr6Dn4kqURJ4U0SsC;@3Ic~U1YZBpV>fW_=yVTR*?Ij&B3(8g{RS|{;j*!Pp@aSO~Le<Pvqc9LTDW(kV7pyA^tm1Av>l*+?P1IS`#y`5_PexSOiZ%2c%Z}<neX)T%keClBKxu<KYl-4u9IfKbp&1JzPM0?p6MuRYLIl6Q@9Qiy?6~re(OAO+2JREAV(xW7NB+LRr|9%f{A<F8oLi`O~L40{XMK%Yy;Xl9!aAk8Q4RQ04j?(PAF<<Ec7%#6Fg_==)6q&jEKQav2Lh$5bh11;da;FIvaHLy%cei&p_i{ea((`qOM&1T}l~H4qSz1H)mwY$V466QHoSC`cG>_)2AKWR-QgKN+P$5xNi$HRnbb<ECdXn!9Cze*KwDgq)#_9L4N{E2<_aeDScD<gx&v{OK!Feh+=}Vz&ws*_R7{9YxMh#irtVFu-D>bcJD%3t}%3Jp9x4FYb&0Zi#zbtG|AHW@8dj|dd%K!`KxQvO}6ahNGj~f$;{5iCy<Q0^m|IDe&@Pxo4ROPb7{MRh&qHu2;K+bgK^+A|!lB6->n59K|7a3Op7enLBfso6~^vM%xOZTb&kCS$0(p;`M-S(qk_N<!u4Pa1Q`;j~xoO~@si5c?1-0#0Y4H;>ANNx6oLs~}s`57#Y6s4xY>P=Y|)i<O&8WniVTmtmh%TT9BGP_|#yOHq~@9$JLYX6|&T}*Er*#}jM+9e<4%9Q|^PCx1&Mq?ubCKjD@Ro23*>%d$yRE@bZ<7PLG9Na=%!^vF?VXUETsleZQSJyj^T7fZhG(D;Ac{BBo5`sr9(WhMoyXNk=l%SLN->oo}F4W=Kn4t}}$p^rhFT6kx^+b|ldMLT)xaHF`uHARf1i#t_nmNPbrZPw4o*N#1i_#Hc4QRhOnj8l@Cp*(@3JS};@{n@2lI)(=Iac$UcATGadPl+NU2&CS?2K$0W5-xRF3i?*wAhSfj|EWA6+PbFek)wv-bZNV=kK?)Ur+zi?rEL=pTQrQr`@&Z0G?&*WjU)<)dHDdGHwoHMAwdIsZ~&A>aqqI|3qPDiN?GOi?#|C<9Kn-qlE)GOxQFrUaaaq6I!_`ov<6-Tb1YTy1GY%F}!6rythDVU0gIQ@2@(SzmZByhqRlHsqAzF80tsQekl2{_eAQz&rmxhZu5D<F9o)nU{ayF6|{K$3DuS<2%l%)mxe}xX!->d_&v5?G`jU;seb>0xRft5+A*E!6IO~Z1&ZraDwH6-3M@L{6F*WlJSAQd?JA!Nh9uGDE4u>p0WV-QUlTjzNF%p2(9kY$;G^GNep!4DRWBRUKQz?2y1tavsV0N;Rn)x8tx~zvI=ue8YTgfduM;EAYG}h2@)7HxKcktD=@Or!Vf14JzkT+1$fh&sF3V#|7c3URETkE2Ub5nMxpnhK^G4|_D}I+W#V^}Ge|!0l&gHK!e=Ib+zs5QH<>mjl{1H6(M|$)jeEE-X9)5B8V+S6f;q*6`KY)M!ou0wBKcEW!LDY-ZRn#1RNJ?XD0m}UtD*OrQkpJi9Z_^s)A7B1l2S&bM+pGp=PF1#7@ox;jne-_Xk@Z}@+iVX<Nmf^d`N)o=<c~Ol+o&s~I>h**NLyg($Q_fO#-nbpho~Hc{GfZCDFdIiYm_`mHXH%JEq*FMiI}caObpxUzy*@rAKX3mr4VkMG4K#ZQglQui81(wk_TB8HWtHZ!#_YSNkaHg=<_Lu$i~H1eCqnF)yaLY(^xf-{4Pu}m&&drzNoS~L;?jFMeon_k5YOiV``*s#;$#_+Kvsd5*A@3l(u1^6^4uy;mD&9E9eFwu7;1{y7*g48Tuzo7Rnb;O2cH#?(|XE49z)%_!13pblWGEDcc}&NtxBWC45E_DrZE1*Y2eiMt5Leq@GP*K_knc@iy0Aj9VqP74}HMDCD;(=h+kOyjI}6j=O^+gQlp57yiR28X;Y1p7ao#QKZ|<CU7GWoauo#kH4WKD!oER=Y%CH)Aph%Nz*nYTjZy3`(qMsk;6MCY{CLoXVC&dL)Z%f8=1AWUnl!Pxox<!>6f5Fof0+zYFvT4!j$4a-H}afpt8vVa9Q>D^s@9>7quWf8)i|JK|*26S`>HW^cLt8Eb*#nSan&#PflupCBC)n3y3>qIpw0Nz1l`?wVqK7{!pwdV&Ny&WL|CL#xOtg(-q0g{a$crs(TfEJIK}#D&^}9?>SXnJO^T2!?BvygHpp@0*@NbHKw^R=RA71!1Z_y5`yytMY%!M9?^2B3xucVIVInv)X9>S7ewt#c2*<S-N?b6D+0;P%;pm02|rh;sc2(oI27LB>Blk=3Z^ZlEq^i^p{jZM_Lu)7G;mRqy?RXXR1o|D`~KE|AS<UDnWX)S!I->N1F7v~7!FSQa5J~)@EKqD<lG!%p2=uDUE_DyYzGBv&oQfq?56iunKoc#c*H&D+fIIYbFj8r;I^S^uAE~|7>gAooLgq`REl7V75Yz=g$4J@3Fi70Z)zN`>64OSYa4R8(bLJr9H)qvg;-uQEHrEH3i_5((EqSX0(_04IJFtk9;kj?r4OC`cncOxxC&`G@1MQzlRGM>8+l4nHavyr@bfbvay#T__!*q}&q?-tZ@M~T2Icd#o=e@!<*y7P@{#j|=_10r;B^5-P#g0pned1h`OXt`9nM?ucU}=Ij+h_}Mt7kr4g~Xwe8N?HrNLVnZ}WI7$-XJ($|yCBxtfiSLM<n7PATz9%g2Nr$je!!(tZzn4EbP}!@LSsasXeYDVKUa1I0QT4%-3s#v;SbL<#<iN~dJP+zUWa?tNtHF=5!-wLQ0y(=+F<0$QtLG6j;hakFwZ%%|Gdy{KS(XlaNx1AT-93Y*Qp>!-*o3XB+!16C(79o6B7Q}iKw!%7)O^R#9WgsH*hf8gOy<PzP77QPopj@r1evb0mWndQgP`2GPq$|hNuxLL`l3tB`jv?v*V!Fo0R<P#JQsfWL33mIyo>@XD>of~vas@x2IeVUR=lhU%L$`{tdMK^AjORZAtdG&C=67dMAZ2nt{FA(83pYhVt!VyI$+Rh5l&jk_HoG-r=qr+cHhlw#fG$sF?L~aaTpxjh1+xxOy>fF4seDnHORl+Tk5)LC0P@DGM%io|L?FS$k{Ot1ox%?6Q`tjw@FMoUaudJy1SBgyd<>iMyhc2Kd4OM3U2LJq!)NB9B`m_&85BC#P&G`=f^)vYEM-<~g`ab9KAE;H*!X?`cQ909+J9}pDY*e?>ZwMwQWO+ooz<qsZdk-@<zFAFsg=B0IN^-)hP-O!jAF%%#hSyjwSXKvQf^ggfkpInU`hHjo^meVJN+orFVkq>6aQBQ)o4P)cv4a_bLicuuKP>+67nTTGZizGnYf8xB9`d!9HK)Ut|Co-p^l)Iy1<;|tp>rUkn~kbx^bJ`vCHFN=CKGp(yvhw}WM%CjJ3Z-(b*-5GWQH4d4f=o=OSZ7QwB&V@(+5jIF%safavDShLP#0l<HEu8Jpep3Y8FBzv0hz*cpxj|BUV7$N7*?{Ud}LkqY{jUk~JKj7qo79S~bP^xZ9Z&%HGO(?$?<qlx42yzY+Bj3`WJX<7gZeEL&mzyyS^gn4?nhXi_M7zA@T`QE9&HU4w&!d6WmuLC)I2JRL<yMGnk$n8iAf()2fcbhFheIUb$`Zg((<;A@d|D+(~Q`aFa+0v_76Pl#ggl6>U6{#KQe;!bh|55)7)k|>=c6VHe~QJW!pP%kxbqDPZx%+QzZVe16Pj%m(_!*p1+-|RUAI35##kG3Wbt+Ks+rHp?Voo5oQrZZiPhi6EKi9tk1UdrDc7p94WSnQnO#c=Q7F)g5@eZrP{zI>+m(z&u%G5Q4}eY;Q~LP+0<HRPh_UY>BDvsedIQ>02$YH__e>U88&w;uyV)Ci}a-Rz!Sm#FI<v++lboGF7%57aT!>wGSJEvvg(OKCtvKGoasKko{1Y1L}2P1R{xYC%?G&FZSi7CF9XV|5u^r3BG0=0=iF%q}N|R9k>+J8S*gC8=dA%c-8A%ZI(I8%U|9W^V@bz9mzn4Sw3t4ff0(gi)`1)CJm*$<VfDOU;J)&S(&bLzB#+9fBS@6P*gAK$r<pK>UR@iOw^2vo;#N8S|s`*6{LLCfxocE7+Rn6AROtNMj<6@rtu&@c}d#-a;}1uN(FQJz9~|f9gZeeyai*8`)gQ7Tr@aQ`xFy1Y{#$5?5PbZ+{07?{z_3A75G)M{1~TAJ6c-Ssk9jq*8c;%+2CZG!JuB;+mnW2$umN<tEP2e3`(t5w2`r9S%oi)*#2mlWvU2%xM1;(MWoN_0{bMl#*v_jq=@^xst`Liex5}zH;0hj)Mb}c|YuqC&$Plw4r_Yk)qc!Va%|nj7=?8I$=k#6=v=ejnBP<^89imJSffIZp^uUi$xsZ?GS?$2_tIdC6^bH290`q(8}0vvIEuHrqRr4{iApl)hldNR4dtM)*ikdVA{s(CA#2B#qu>ZU0(XK%Cw(;a8M~PHmru~o!wd@#Zj?!&M9H4f6^5~?-tWL(0bUV{VP*JHvFDLwJ0&BvC*X%YVCu3NfTqFt(~Hz5zcI^rumf&0p;Lx6EB2xLZ482YxI}?LMGAvgz{d$M;Z~v9r)<QH8qE&cB|AbUw@InSW3SQ9bxaYwcUA`9G`VHDrOeda-30|6*<47KmQDaeV=e7zC+3{)W(^<Y_0Jyj^{hw34*u3WChLQ(yem2y?FDhDrl}N0EvHqe|~-W8{sFlw!MF6<85bm`_BGe-GNDQ3R~Ot`L}EJ1-jB5N!2}{^LflOZFVU8%>3Q;dVPKC-b)T1{l6X1`IRG3XNq>kq}qG?8}&U3y9|>Ye<r~v%zn5aCgT&h=t-s(E<=c#+w@z*V`#)DB?PQ}YuLAd^mmA3<4?W$az=YY!T_@K0pg{}c)&cr7IHQ|8zPo(^8o*qm8x024|Vyf>PC*~9GZv9gipr|EhEHxfF+Rq#Q$a*2L=>c_Iy)pHZ@!+iCN5CV=hfC?R};`P^#Z?gcTNq2`6|WDyq2WA-;DnvRV~gACbb>JsD$2@0Fut(a{*596)1Qj|RiD=j+g89wcibMDJ!(B@72qjwa!8@R$boxX4qaOg8ucdYIORrAi;Tz<xp%rkDqlqV4c0l3vpK8Y^kOov!L=Qa}zut1j8Yp6V4UFGv!Y7qOD_=aFFdqIRas?Q`lgjc;BK+!|v0I1kmkJr3JlIP)DJ^B4JW-%o%hG)Hl;2%>H3#J?!n73UFJk<-IW)z|U`0T{jfx|H{|CJGG47JjGQu}gYi%NGPBr67P94U0}sN2X+fl+5!eXm<{>2OaTw&kKf@@(RV|s~8#3J84G_@011HBou;$Sv_Gq71Tx{b~G^l?I`XZ4IGjvKsJqivR7jcP!!$}-ag&SN3>Hb{v*ZVq9vjEs2c|*FAHM%2*cG!(LuTKwZ**Ysc&zGcWZAYs4*Jvnwlj+H(;)c?Q=0Fu=6{;VRPcGh@BqL+N+m%-d1=igVdtxhG6x*mqD~UMBOSTEiE8>+6F2vbfhc&le|}{y$?G&4wBtn7=$s9x=9>hhW8~uZ?k!sddALV>`zE!qe_cZPvMa-2Jdt7;DD>MDFTHU0R0rLh>{1BF-SvSTg20Y#6^D|1o9lg2c;ZUS<dnj7Y0n2k)|+I2PeF{y|Lz6GytPfU~ohQgJ;1-gWesT2W~%VcbgzrHrs8CE5r)~WKOmK_$F^uu<J2W|I6}mh1H-`yzT=H4pGe-pux!>V|+-TG)z%|1kk2C>i3^-6qX(n4_RpTM#oKPxLC3^GlE@++o0H`J9^yk^;KQz_fxMA%*AcUyV~~7)*Kv)gGtl~pxqHC^a}A=Q#BDaJfIR~iz>E%LM5R~qNH9XLqL!o!B*|Y6f+PNuodB0B>@5S*ZmXmGLP77Rky;e{q;S}zJ?hK*VfrvG_?U+z3ylPqy*~jN6kTj+BHbOj!o6@zD*yzZu@|csv4G%c+rMYC<|p}Q%vWKZD)ILy}tecQ!FCn>HaRdv4~bMM0$*wu#i+1#KUy(Ezk`ZFutVW__^ctrwlYjvD-lSM)5+SJU@yvmf*PCK56y>vH9lvC*}u)&~gNa5Y5y;$lF$bq;?=TEPl~gISpjR;=x&fz2HBx#RsgLZVURhGB^u(k_yo0ws-AJQoh`E;LlGF+Cflm@P6^;VIr{@JuNk^o$-IYg~1smiyCl*{@_ezIpupJwH`z0S&+;CeA8sJjks;nof;30a0Qt%6{RB>JS`A?S^>rY-Sp#t0@(5J6_G7N8%sgl7}@Mst%ty|w<>ti^_r8xaPT-955?y}(DD-fn5_OHz0R<)4jS33tZUk#68_P%uJgEg7Qk4Ft>a<01-!p#&Br{40nUhg;`t)4N*Z}c-GOc1>B9j^8v~r%10Lx?tC5TcTLkKPsx1MOw8JjwROl4R&nT8KL8sZXkG;iDyssIW!CTj^n1j4Kc4Hr1HbRAQr!p&{hMy!L*3VO$spaVn{o^i9#|ZgJgrtq=JO5x2?!4PKi`N_3v^nVh>;ezHwqEDl(AB-Y?bUUW%N80&PP7@r5g;*!9U=S8)-l|H^U{iDl{TKZ)p|W#+jT3n{egmY!yX8Uf^_wU9z8nl9v#bP$6a_f3f0_&Af$#s0?>HEUu0);@fxyTlD^}~ikstc^IR+oFg$+V%h%zhXX7MvqYb6W<Ua)cz@-qD31MZCA872Ay;fzLF?Tt<QDfHD7v>^{j&%EWqHk-0@@x`TL`5hi996S3>}dMKsGZmf#05_4TTo9KkpRB+Y>MO*OxA_(r1NBGS|^w1Oj<nLN+M6>FmgC_SXen`IE1*&9?uD!HT?ygJbXcuMDx{8HjL$lG;!87MR)3)vfQO?7vv#StykGJi9fC~f2QomHX%UVm`J4bt5`vXHgf!SkBCfj2P^r01rdfNhKL!EQ0RHwu_i)Rf-Bl!g@olb!i+Y^gJ7y7e9~eN%$F;8PP&7W2&u@h&)W|66z@@ob&dJ;osa2CKl0|1YV*I2o5+y|ISNf&G5^RvPc)fR$HA2e&7mB{6c6}#;=kJrgd&9t8_X4fls4vgIM>QIwGsKV!DKkb4$)9h8&gJN=zbHpJ23@rMf?CtOnb?#9QiKo)(T9NoAlAGX#Kr*?cL_Z^yx$N52b{DdF?{A7uic#HB`x>21<ZIOh%nta(uVpUAMov!?5@$)$#;yyp5>vu<JEonH-!#)v_nS4Ruv}>Qn8NHU?qz6U9|KX=|T);az3p6BrgrfeVKY!aR`yn89uzY4D2kBt2UfGdffg5L?+_6mYe=y$j>=p8XA3-!PVhjbATrhP&BZUhFKl%g<|u`zLZZf1li&|EQd$$xRwvoImy1toW(GYyaZ%6LOtqm^g}hBQqWU628$-36zO~-o8Wr^nZHx{!6yNMV(Iju+%yHsus8#WP$r9!Xo$)tkDlJs#-Zf|Azkk3wZNeADx=<$&U$3;2-F@^mzVvCvkd4Ao!ocp9O-v{n_QeqkYdI)PtYlS07&f7HY>g=pOy^%in4{i7$}7`u^J6X?JKgb#JYqb4}`%*lYKI#ND^LDI^=blGdEA5>c5yi-5Pp8wa_+6^H+foq#+OK&*5Wn};`%xK3GhN}2k|CWr!;LkpHkz1Ok|Lj_6O63XTIbYH6;EBDa@MG+06*RWl(XTecXbvUCl?F(TSd0b%2n{d$|6`29|#lV9zyX@PVMxS+3Ob$ibpoqbjlk8Udd8{06c;Zx;w>>i4mH1T%Tr)Rtmd`*_Wm1IWEc9I3LCJB&*{jjojfTS>W=$ks*Fne49t;retdiq))0ar@7KPc#=#eAqvdbCcl}_qQT!rQ2+7`EmNjzbJDLSZ_Oftq6%`e(KA)z1B)@^`6g&|i&@|T!=YADAG4Ar8|!<b8odqRn8Sgxoqco9A=c!K+u*A!Q;YeMWzC6a=m0DIx~oxR%bgX$dtbtbN0v*N_X#jwRdCE-qOcYPZ;SbKZq$P43|$c<BltumNLs|mq}3}D-!qTz7pw1*K(xe+-5V`fjx*x2g<F@Q}wXlJ)iE*XOsy_daJj<5;VcoYtfe51XCtB&w|IeltN7Y(oB!#rS$JA(6LWsBk7uh(HyBljl7PWK2C?!bRYjZ?VoFMHk!zCkGJg1)K!BV!}^c=aQ~-!QN4=A%>6xz87``}iZCoCBlp&d<AkYuFn?(IXJ*U2d-V=$Jk($#;i6O#4Lz&${i&aZtRK)6Sz_^DycaT)6Zw<m`O{lw}@(!`&qs2nESv0O{qQl>3n$q4w;Dy0g;9l=1|_*8a#LX6fKHy$|QI5C2RMSTfhr9|;(xpa;K_tB{KA`pC<xanhXt6-n`G(ESgxv0|9SXAVC7IprTv*$3zpN}F9^3)&H<67K##aL90y_ywE#ZjuWIN1oeJ*biZ^c@9kh2hd@xLb3Pa$w~V>bmv4j5WNuNYiP-1xbk4*6*_)<)D6lvOZ-@-S2tJ*Ni@+pv*SGn!@Ecf@2}Eqqr)N7xs2nBGrzY>H(J+kwr{;iwBVN_8^xEGf6DQo`-nZu*pH3|@tK*_A&DKwDDcl;I<tGd-HJ-fo#?AFyElp1{S`5=$eg0%+0PvEJ^KyC^{EIxpP{<`2PG={cLBo%JTSqn{eV2xehi%M51<MvS#*e-{Wn7A`lW#I`W2M^A;kvDs6pbCKXmSOC-+Ydv7!&D_|NGT7yl<@o;CgvZvl(~C)D5$l5V^T4hj-d3RTo0NrsH?WDt4uB-U@aq!j5pkiv$|i71(H$-eqdYWPAeRT+Ps6=DhVIs4ef{vP#)hs_@1Q`6};8F+gawX-bxmt22Z5Amj^tCl~~?pH6Z5i1~(kaOu3IoThfS<}2Cqm04#e6tB-at7yG6D|n;R-r|6?(2|%rbC&r2qo{Z>?WB4(t@-;bU7r10XTOn<V&k79Z{o+NSGod9$K^`Wh$^%qw?+mPDp}LED%mBX@F_xYKU=KLb69>>sJXdQ}r7-;?&5jAmzEuB~WVV#tdvK&&dE_<%3@3G?u^#m<<^?4b+f{29b{dk<*ZrfB0xB4Gc5n+5vUnZMu~=s|m!OThYP?j4FHGes?00{8bJxEv0miYpZY|F&pD^ka!0}m7b7Xh0I;Fr~J3z|4#-g^gweUVyQTiXncC<9g{o8#0`Xef_4Rvz6Q?WImtIvJWjPdYGOWTM+Fte3w!(Z2UVxvJO?qo+cQ5W&zt$f^Lb(0AQ7+AWg9H?woPz^P+!LO(H=^dD46~UdC<bRkQ2ZF3Nbf$iY5w9&}@2Bo=PVPh@oaBriGpY);Wcp3^I7^^g-gpslX)F(OHzr<*fkaD^DH~Ai3OwdiF)k%7Wf1n3u(4XRL0_d7ixs&8|U|P||9VVw6)#=v0vaz}qMNQCvV;=X0XWuo<_yU8awImnJ1x^o8L$K*y?SMAJwoN&JVFC=pm!9e<|c`WbTRt|V$h;94r2w=y{y&(W03BuXgv&%Vzv4#<uJ8xHlt6iuy6WGEEqt_stSvN#I%=zEAAK>&qMC9{k5&#F89I!6#;phL&K{HJ7MX5rH+c}m+C%l@_4E-#jDEj}-U`4eI;e@B44H0QRoe1F6{##d>VC3_@cZhc39ko<{78DC&-Q0g;+A<A;3$f&(!CiLdgQuB7Z^%ap}7b$rrF$wsCQsz3lwY^$(clACcUYK(@wOo?+YmRdR^R^Hffv?WQg&@)4PA1J3Wk11JlnP8_7@`OGTjm6#2ah`{1sMI^p7?qSbb1C4^iZ|kXdW-DF%UTr?MXL^u9D7};m8>qUZ_+BCB+qqeN<B6Rf4)VxFVco^f&;jsByykTMT{58D?qbfK%9AufH8=ZHBOQL!jRg5cG!<v?1iS(!L?GMo|l={ZyRP7+uT95zbmVmxJ(9)BA9cDwqU2S8Q3!xv0p0!8MQ_hm#XiJfK-SLQ%rxgkmN`nl{r?tfi*TtDZ7!+|^fP94RX(-bRySxHpfE<>K^*?UP<)ra7*ngR=bK&8jmQ@;Z0%HdV>h3;1jjwT|H~gy$lwv2%?1-+IL1!d(}kWt#jV+n%=4v!vnXq|KOTIJx5JlBGw;ZWF#=h$^PnOTJr$33(gj5ttTSXH21u82EeZTld^-y>Jjy)#8(Et@YaK{Vl3gLF|)tP+Img6|1z|s&?N<1U02kQ!Z_-GUL6iHhn3}MQ3{uVZaAD5<77p&z_c)d<66uj$sA4!g2G=A^F1RGmSi&;SYk6D!%AZmS~{BJ4GF}mT6o_q3L>}Ubj>Zv91tE>KNopw<6L|`)+9w-Df&o^kW|a3ymq%hhzS&C$+9%LTiIt0vRk7a1gH59=H|wSC~}49Kzg2eT>n@9zZ>75713?Ev#+r)i&>J)Jy@bbUi6>8i$7@=)kFjBa#%w3E!xNYx}hgv+#|iutbZ9wQ}oQb<}EaV8?ETBIHy|F31mV<t}V0Wbn~})vWoJ{5(DxjmSv8xxH2cF?@ToN`Vg#YP;2YHK_P@$f7v9n(pYGhChuu%O!dy{E))c_0-4k#COYO{*AnT7ojuKAHg=fv%Ue;$E<si-_k5JTS7K5OVVsq$x|05q`bsWn{DLVPm#dBSxpy72|ndzR&zvcR6h1kln|s!QRpC=12&<B>*3CtJ|plKJiQUFuh#TMTCJIl+zhK5_`wFuZuwTIarPl)q@#Wk1zTlLx$1-J`i7X5OqXxNIX;az#2%f&QF?GI_pgUQVrm#XAKt6(z<ey0!bvlJ7>;_)fm^9p_uddEKDA*N%R2bLxe=kS;Mh=(TwIhY1k7XvMNra!BT(G}nwjqAqC8xzMk2%$pb~Jp()(?3IqaNZz&VVDH}NVE;@!-^7qx?xDxjvmUfTqQ7`<Ak3bXtT+5Mw&vpZmg7z}_fz<ayctLpyRdL6oS6BR6Q0N&i7wE<GSL4WM**6wbs-@9MWbnaG&&@UkBNI=dIZ%r69bnnjE<~FeEuvxcAPeWr*b?!Dy(RdsQ2%i{6DmF1r@s{-9_T8ltT`b{o1m|LS6b&N&X8jw$!fPMul`oaU<MYvw0(@Y}?!UdWUB6%3Ti+9?Ub|TDqBzE=R-;fN9QHkSaH;L?Ztt4AW@#xzZVc~i2|-dx8;>{*=0pG5wbKg{Kp*<2D1wTM2`=z{T<{v6S@AMzP(~r$5Fz_MrBCt3HthA{G3A>=|1wIIC@&9sDM(9ob7uoFS5r+~4>Ow4RPW!cri$DMvue?5z3x_&T$$%vlTqL--3+t(OQ<N(U8<@yb!J{=x5BiR6KF}I;gnVBN>6K5n&`i~C-_{o!ks4Ed@H+qBA;ts3fDa{paR3Z8m{g=z-VE$DwJQ!Fm>aGFg4c3{V%siHWjG&-pv1qVZ5*pK^A09t)b-ha(wPkBno!-R0b<l52r1xy4shLsA{_3v=##`!08IiIg$1vO6HnI&)26avBpyvaD`^dMMxFoaL9+0HGb5=tO=e!oeU?<9)D*!K^$_>BrBZt*zrc0)=G=Kmem&uAl8@Zu__|>D-Bq;bSJtD9Q68wS|}&L^_To*nLNpyJ_OuF3n0Sn3iwe??$2XkSCI{DadM=db2c15M8U5@LL`vf=oAh1@kt-#qH|}Wg*^5!>lX8>^T=1ibwd(Rd>lm+UG5|S8#9-U&L643|F=X#Bm}>8nDcd!%+D$z^WEWqGA)Pr1Jk-9p4i<k^pqaL55L;$9t{Yo1s~C0`6;!-ftHI4?)M1yJDVZYhTGfU+=Sb*_^1~@(j?fk0D4_uK?@<y(x?6Z2M3s}+W~gspt4kMtenNcS+PaFKp`zsfxfT>!=f`5bHXtqxb(Y(y&uC;H^%}sXT|w1d)r&{vC>tbuRzpx6vN$((xI@FuesTwU=M3R$f-&zq6KYOztkq=)Gve>Ndmm^Q8e~F6~~6R->ElaadN!@Vuj^Tg@e~Dh1n-eHhGHgLo===xlm^43eKL7kw!k@()n>TNd^>Z3guIqHd$99&(IibqB>mF@Ss2P!}g@864uK<lD1D_NJ9>Ywn2C0C#{24p&i1(KlP@sYdTF^Wz>x0tH*((WrTNz<cm>BRw%fE>r)DfdHCl)yU9@AdBuI_mATX0NFmf6oqBL)7H{6T*^m*H=G4JA=)I5e`#n!A@u?+vUlB4?H!0eORAU>6#%d%;7yd{kFK>^nzHb6W+_+FIGmp&$RBhob_UXjpu0S{m&s)K_#SRdSfG8>5@PS($O@^_{p<!j^(sP&o=fX&P?Y!79uCI$tA`vdQ@esqO{LaxTS5m&Yw79(Hik6#_Mh8rwfsA4R637U74Zv*3J0ELECvsO|WAL5D2~0pf@|<ayk2<qLQ$X3yD@k<+f?826#GOM`dJMoB`NI_O>hP|(@Guc<c&6Wa_#`Pr`p)x!o>GNAq1mgu1y#vS>s6CIdqUxbECN?sBzjMoSiOMa{th}cs804~5`Ji2>YR1VI|l?47bNgni0VRMn`*T>C&qLw3xJoM(GVw8wVc3!`+I;T4w$=Z(CyWrg(KL!LcdGNI)gGrm-8>~wRL_JdEr?ZplNQ*neFv-1NwOG*Q#r^U8(&G-8sEK`bu{R&a@(CS$IL&8Lh>lvva~ujv$<!p0iJ$^UZXI_;2FofJ^rj=&v;8>;fa!B#QxD2u0Ii&>v1v3q2ZwP&@=4Es6O-7}m+8**b=6nM})U6Ifj6_khD;jaR%dJ{iDe-Uq4QIUK`MA43iJR}5t)!xJoRs3AAEVU~nSIncwzw^16a?s5i~q(g2+h@d;$DT{2PCk>5oHu8bfUb42E-QJXjU$021$wQ$gXLz6r>YQ0N$bNBnPJRJdwe%G=r&xC=@0qFbU=XmSLb{4T4!Ouff?-uxd>qnlP>v-wo)EVvOd=<QX6sg%XwJo2xIXRs{0P`5*RGkXmw45<YwLUH4xB-i5NC!1?YGxz1Zj@eG+1K2GNGH>t-ZOw4yQcCPeLl-St}&poci_~wJn%EIU@X%)c-=ow^7R?wXFA15s84oBDKqyfwd?^6k)wUCRxW31OEv$VFgc++}Cg(5U>X^i6ur8?fVp)imH>KIqJex*ARem;ix-VfMyo(MZAC|T#qaV1rHIvzZD^vH03-_<1!L!73i#0aWFh9-~;^QF_4|&$)rWl<sE!da9?}-wf<}E`fK-J+k9>B8|ew)kpwQD;CYY`f!m80PM6^i1X+j1R$vCiYkDlck*`zTi;n0T7GZPu5C+k02n=<1!S&``A`c7J(pe!D_axfeufv)W;i`~t0D0toB%M^zx-MY(l582AT<l}Ya%IVBMXRy1y{9h+uqwC)K!V^8ioxx~Vu{>2o+lEt-3mUu@Igi;=+B_ASStHz$5<g34aechh!Pe-D{K=IlLS-Y&KV|DCM~8_mD_!Rx?!4?!H%s`zBFmQc%_C#Rn%(QZ0&X9ojt+}Tg2m{TMU{#3hp)6%aPqiDSIDcIdnSRN1XG9oeN#j2X-ii|Mc5#UXbf7<74F>tjmbkVq_mq-bP*2H>#Oe*>@;o$$R8rER%Z}NhvcYbJ#u)3hw5{d}&GeC;ad4{Kz5j!S+cDag3F%R<r9$vR`yuYc$1PVoPLVT_LyeS+7ZN&RQ$5BJAS0Kz4|jkgcRbQ}xMrfEdHsBj2*NZq{t|n%#b!+vodLxFWHm+9jC*G9iv8Fiy63n4D}NX5uP`UvP_4qIa}hqDCS<b}QnXXH;)o$XE?>xMx|{#l=Dg(e}CTw#UuR#H|pIovCLujzp*23f{u$>CTcBaWSOttvw9F0;yo!Jt8~5u;I^7SZe|^v?3kn6r>+3*Lvru1;UH##<D{?Rp0}iqb};@$R#o*455bz3LiM@z=s{W9x{|MwQpq(;?ZybE3sDtW_Nq5h=9ihmoh?C78al{LkvfP(|3Gw2oh(1fmk~@bQ)VXm#p1_%=STl9=He<j`s+$uCX;{2xETwaquKrHSz-v;olbj0I`U3E;7sR1<6GL56@OP5`TnmdE%}DmjH`5Uq2sVD~J;(ShfYM$#<)ivmo{AbkG)qpCjUR;p06$JedTArIKI1Rmy-%2^L3AY?04xhzl14xJh&t4g;e>blTs+bmfFW1hiRBVYY&FTQsYI`qH-dM0Z$pAP$n?w_J}YGcPbOsw@$y&T+qx8{@v2-WXcN1dy&(4lK0m&zC(AG%d`vS^b^2>osz+*sRupBF{*-mkb62qlCXt;Vy;`;YZKD1CO6#8YV<9X4K!`tE5u8Ls0}9Nay#|R^~k`b~ODV_Omx_rg1{B&KH;ubW^`mU40`YlV#q#FjWNtgrXm*b&}@zol8hx+N$V}T{j!oZ&azP(S)g@V!fxufX%{inkVENT!5409V|AAz2O<6vIT)5GZ_PiD<c_v-nf~g2DbfZ0xU4!GU46b?ah$ua%ajY#IkxQ7CEFz(ihjyzqG6!KwZX84gN)mpf+mNy_y<|o4y8iqt<YY&hsR-v+O|Z5(dVwjIN3Dg3}?I4ipCtY#fbIOYJOPFKymI4fl`)oa1hc1yWZ5>$j$feLVRxDLUfJ(|xM$RlI=t?zJ*$uAD?n5MNOw|9H~x(W8d%(!^5AMSY6xweh!9m$r2eQRfqlL-eU7B^rK^k-6ov<W!x$w?%YF`Z1d3m}e%zA^Ck_k?SbSJQ|GndPqV;Xx`H<k47=wOFT5(#G&LgKk=!);j^;MQ~K+xF4|dXad<=x&rz?~P^$VFN&T`x>c6As6nu)Qr#~TZ2o3?IFiYJ(`ZEDbueZZGmq-ZsJ49yuE4UZQjS5bYR>WldO8)dKK{YF=F;zs`M_yh@3?=>8tFLctgloIiyBU2dO{$h*&Q;!>?uvM)pUOs{+SU)IwNBhBVl_os#Rh8ZUz+iQgBK}@o1|n>$80Nn-4Me53~NdTYg{~zh~@j6R`l85X~_n%FlaOko4PHlAy26>b9Hex*JGV~+pBM*qm~XKV^(eE0#r*S#59V&R}XEp%dCyJkG{UXTUWBA9u-4T^#fFtwT&R)w2cjAgXq!p*_n2O7>p}IYoL1+Tp0)w96j=iA<1uHIOkSG9zb|{P+7WOYFq)v<R(61+8Bk_4@pQY6xD?2HS9GH9dyiDL04N%RK<0{gN7KgT~i->)sm*hmR?vMYQvjVD^}L_`IoSjpTbUuUGCdi*iW5je@$68{=~uYoJsai6o!uJLwU$~UL8wCXO(UOSllM1v_WqId4(!(auODW_p;%(Ogf7RM_o<ECzIpOiMhW73|rDBvoz$`H*V9lC?-|c59?mnIYslX#?%gc8L?C}V2sU03DWSWRxznW>VQ8&$r#xg<-Rx?nsyEOhIJEZ_94&gaRo1e-r%9^B|8j?86-MDN_O%ALLW^`pzc4pFkSP~nX?Ep6w&%E3ACU=!`Bs^Xode0rHnCaP8`?CC>Nqlj@MCAPf=6>24mPIyS=ibx|Rb=c$BAFXkxlh(@&k8weD<GDHWOUcV!h;%8^U?pPO}(8<+6lPMSB|zt}xXI(uE=B$ln5i@|Kj!{$~HPBFKF7xB^54k9|DnxvVmBg}i@OWopX3zk+emSJv)wH_)TQ`oE#ayz7_hNhC-#V~3U?XfF4;SQ9%NQF9xWKp`+Q*I=wEG+88TOJFkG$)cH4}*#lNl}_~p*&)vQ-+s9YVbt&Sz18ZaI+RtCgp#={5Q(j^W)G4mw@W=bQOA!-B+Eoy7}^R&ausDs`>8t1pf1Z<{D@XcYNoa0sOx%ueb#5_KCn*d5WpUF-Y+9%RfV@&j^I>d(ZwJAAUg)Cf{SnF<&Mc7f^6djR?k1Re&m8WbdV~JR?xiS>1c!b3*g~_Ur@r=nIK!!aJtGdL7E&L@BvF-!Vt<F%Aw`N;OPmDMiF1R>qv=Q^cGR>7m}NE||fgJc>V?Kp)TEFWM&TSNg!e;GsPG`y`9HFXa=8e1)E2EGEU!YMG}#<-wtRlZvaAHGj_QLOI;8UE=`Rj~SZ=rs=(F*N{D+IsJ%6fi^Zyf2HfiR#J0?`LCYxHc?f<5`93GWjPes2`a)g?WTYW^9iB1d>2c9b2=yRO|O`l$sZFU&r>6u+4-iykbTq75f{El7VPg3G^motU<s?F0#mfJSU&FI?>jO}8m!s}*sZ^%o&6C57N#_eRORP5n;(m*$&T<BiKi|s3vsDN6LefYX8V&sMZEljgYy9A>r)sR+e|5y*+(KfJ!X=}hNe-XyVbpV_>J`);l-}$+SOfwpOA80cXT;Y=)9xEK3L#NTF0m_bUkS#WW1o1RZ#1Lh$hX}hla+nvNTRJD@vu}mF7j=TMN0}@NuX3jqd0!Rw?Jgch3;ssy`CBD+2mxeJ5P2-QB=IMD?;4okqRjhTrT#`%uk|W%f2gG3s$jih9hnnw`1$T1Nc>{z>M_Jfo}W)m@R*j8q*kSY`9fs{|KJFUm`FWiehEqhMlV<p`Wp%m{@hVNA~yqOS5$2I*gxH;v=23@f%;(k)$jx1|}O)n1EQ4+YF#vf((6lXi^tI}<kKz=riK`}}Up*9${m-)&8uvw|VN_-P))=ZiaKc=xn9&Mb-08p-XMvKf5MWEZ8FiFq(K@3@U?<<jCZ{VOYAyV~Bu{ayR|YOS`mM<+hqSl?W)ODR34?U{~~^)!e`@-|s@(?`M2;oSVIgh(;DRLMn-8IIcpIuf58qOSijeNE{cQ09>><9rDJCAebXhOtKfYBfJDB1?<OwLR&#Zq~NCyGnOePI+>}e3XEX==~Cjpfo#-Jitr_A7Q3(@rJ2wLpXNg!OubTG&^^{x?A4^xnOl0WsG{wWLME=RR?EE3&I%mBz@xCirOu_n8LR%XAfBs?oF>cwUZ>xV+kx3?;@$1h_T5TxaPyh2Q@1)%_@=W2r%09xmMd)$DqX!bpg9|3lfYjTkzbzu#!K~FUe+QD%W2!#-I@dl(K}T&_nz@>uCnjC8cWr61kBKf2L`*o)S;Q2qT2(sC>OFf%ynPdJpK?UU;XvTO0(OE6P;_Q@$l+0LyMQ%V(T$1?Y?}Dt+5Y@n6X#Rb1cP%^9DNZ#y(G1W5DjJHQ?ygyRpP+!%v{^C;)S;<amwC4bSgRVesp`)X9Osz<3)_6$gkt8~R)MNEZILuTtWEC*)g%Kx;wU*Cph*r?tK*S6o<LXKPzC5xA6>JqSriIBhi(QtUwqmYKCf?;7TFZ#;UuWU(vUX-eqf_`&(l-zN9YvXMJ%V5!S3IdiQ&#WR}r#{qB$y+hdi?)Unr|I0I{tCsj00UW3idE@C-JB%b>BYf=ZbZwA<(toePB}k6pNt7a#D7UN$tc`>mk>ffQSO%{HvO5fSN}x96U(s*R27bhd?X~<3J96soF+Sfbe8pO_l+`)u|i5rReTw5_j|RyVs~^tIK%=O$4Gpbi!t*}lACi}LVk~cQmphKyva{VweWr6<0sNX3vEey3TSeee4--ZW17ED)!fkw1gQEw`4E072+OBg=p{{f_tWM|R!k6FKFH4`z>{dnt&l6N)W>{urEs2*w98Ldvbvkp)nHKk5770Is2mBppQ?-l@YTnR*J}ll%4$g{KLA}4UqY=sj&JH&4RJMvvm&h>y~Hh3C*uFjRL7qEy&N~r&1Y;9Mhy>$&Eh6{-!iMD(Gh65_+NoKrfC44eIG{jkx(AsoJ%E^)+A;L|1%Bn)0F1wbH4BxsaWg^e5gTa{e^>=R>tA|3q1WuKg1u-D|q4HiGQwBBFF+G0Aj2T0cc4g^UA|$JOHU@6tz?{?QT5kHP1uY+|K#rcsOvDL`}Ws;OL}zM6X8lO1zNY^TVZAR2#AcYLkbV(TF<K*A#vty-n5EhbP_MB*eP_%iuk|=tvMKY4#%!Ktc&q%3;wpbdF<8Cn&vG*2$u<$q*3kt>v3bo!hssFE5wc&1P%y_VreCY4Q3J{J7mIm&%8oW_dYkib@$m3kojt-+~*)nvjinh1V4a^~&IogCoVq>Tk@fG#taz>wj_a{{c}7dfW'

def _runtime_code_sources():
    try: history=list(get_ipython().history_manager.input_hist_raw)
    except Exception: history=[]
    found={}
    for src in history:
        if not isinstance(src,str): continue
        m=re.match(r'^\s*#\s*(0[1-9]|1[0-3]|C0[1-7]|21)\s*·',src)
        if m: found[m.group(1)]=src
    return found

def materialize_own_notebook(require_causal=False):
    doc=nbformat.reads(zlib.decompress(base64.b85decode(NB_SNAPSHOT_B85.encode())).decode('utf-8'),as_version=4)
    anchor="NB_SNAPSHOT_B85 = '__ML03_FULL_SOURCE_EMBED__'"
    sources=_runtime_code_sources();seen=[];parity=[]
    for cell in doc.cells:
        if cell.cell_type!='code':continue
        m=re.match(r'^\s*#\s*(0[1-9]|1[0-3]|C0[1-7]|21)\s*·',cell.source)
        ensure(m is not None,'EXPORT_SOURCE_MARKER_INVALID')
        key=m.group(1);seen.append(key)
        runtime=sources.get(key)
        if runtime is not None and key not in ('13','21'): cell.source=runtime
        if key=='13':
            ensure(anchor in cell.source,'SELF_EXPORT_SNAPSHOT_MARKER_MISSING')
            cell.source=cell.source.replace(anchor,'NB_SNAPSHOT_B85 = '+repr(NB_SNAPSHOT_B85),1)
        compile(cell.source,'<ML03_V403_'+key+'>','exec')
        parity.append({'cell':key,'runtime_source_present':runtime is not None,
              'sha256':hashlib.sha256(cell.source.encode('utf-8')).hexdigest()})
        cell.outputs=[];cell.execution_count=None
    expect=[f'{i:02d}' for i in range(1,14)]+[f'C{i:02d}' for i in range(1,8)]+['21']
    ensure(seen==expect,'EXPORT_21_CODE_CELLS_ORDER_INVALID')
    required=expect[:13]+(expect[13:20] if require_causal else [])
    missing=[key for key in required if key not in sources]
    cfg_ast=next(c.source for c in doc.cells if c.cell_type=='code' and c.source.startswith('# 01'))
    cfg_nodes=[x for x in ast.parse(cfg_ast).body if isinstance(x,ast.Assign) and any(isinstance(y,ast.Name) and y.id=='CFG' for y in x.targets)]
    cfg_exact=(len(cfg_nodes)==1 and ast.literal_eval(cfg_nodes[0].value)==CFG)
    doc.metadata.pop('widgets',None)
    nbformat.validate(doc)
    file=WORK/'ML03_V4_03_FINAL_FULL_RESEARCH_4_STAGES.ipynb'
    nbformat.write(doc,file)
    if TOKEN and len(TOKEN)>12:ensure(TOKEN not in file.read_text('utf-8'),'GITHUB_TOKEN_LEAK_IN_EXPORT')
    audit={'version':CFG['VERSION'],'code_cells':len(seen),'required_executed':required,
           'missing_runtime_cells':missing,'configuration_matches_runtime':cfg_exact,
           'source_parity_verified':not missing and cfg_exact,
           'source_cells':parity,'notebook_sha256':digest(file)}
    (WORK/'source_parity_report.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    return file,audit

IPYNB,SOURCE_PARITY_REPORT=materialize_own_notebook()
mark('SOURCE_PARITY_GATE','PASS' if SOURCE_PARITY_REPORT['source_parity_verified'] else 'PARTIAL',
     'missing='+str(SOURCE_PARITY_REPORT['missing_runtime_cells'])+', CFG exact='+str(SOURCE_PARITY_REPORT['configuration_matches_runtime']))
mark('PASS_NOTEBOOK','PASS',f'21 code cells, 4 stages, causal module, nbformat+syntax verified; {IPYNB.stat().st_size} bytes')

def json_safe(v):
    if isinstance(v,dict):return {str(k):json_safe(a) for k,a in v.items()}
    if isinstance(v,(tuple,list,set)):return [json_safe(a) for a in v]
    if isinstance(v,np.integer):return int(v)
    if isinstance(v,np.bool_):return bool(v)
    if isinstance(v,np.floating):v=float(v)
    if isinstance(v,float) and not np.isfinite(v):return None
    if v is pd.NA or v is pd.NaT:return None
    if isinstance(v,(datetime.datetime,pd.Timestamp)):return v.isoformat()
    return v

critical=['CELL10_PIPELINE','STAGE1_PHYSICAL_AUDIT','PRIMARY_TARGET_D01_GATE','SQL_DUCKDB_UNIT_CONTRACT','DATA_SMOKE','REAL_STAGE1_SQL_PREFLIGHT','SOURCE_PARITY_GATE','DISCOVERY_EXHAUSTIVENESS','INDEPENDENT_METRIC_RECONCILIATION','PASS_INPUT','ALL_STAGE1_DOWNLOAD','PASS_DATA','PASS_SPLITS','PASS_TRAINING',
   'PASS_VALIDATION','PASS_INTERPRETATION','ADVANCED_ENSEMBLE','ADVANCED_SHAP_QA','ADVANCED_TEMPORAL_CANARY','PASS_OUTPUTS','PASS_NOTEBOOK','TEMPORAL_CANARY',
   'MIGRATION_TARGET_SEMANTICS','FEATURE_G1','FEATURE_G2','FEATURE_G3','FEATURE_G4',
    'PASS_UNIT_AUDIT','PASS_BOUNDARY_AUDIT','TWO_YEAR_CALENDAR_QA']
# Validate the whole ordered canonical design, not a few convenient comparisons.
required_canonical_pairs=['M0 → M1','M1 → M2','M2 → M3','M3 → M4']
if len(ABLA)>0 and {'study_id','confirmatory','comparison'}.issubset(ABLA.columns):
    completed_pairs=set(ABLA.loc[ABLA.study_id.eq('CANONICAL') &
                       ABLA.confirmatory.astype(bool),'comparison'].dropna().tolist())
else:completed_pairs=set()
canonical_confirmatory=set(required_canonical_pairs).issubset(completed_pairs)
_primary_canonical=CANONICAL_YEAR_QA.loc[CANONICAL_YEAR_QA.horizon.eq(1)&CANONICAL_YEAR_QA.protocol.eq('ROLLING_TIME')]
full_calendar=(len(_primary_canonical)==len(CFG['TARGETS']) and
    _primary_canonical.target.nunique()==len(CFG['TARGETS']) and
    _primary_canonical.status.eq('FEASIBLE_COMPLETE').all() and
    _primary_canonical.missing_test_years.fillna('UNKNOWN').eq('').all())
no_fatal_qa=not any(v.get('status') in ('FAIL','PUBLICATION_FAIL')
                    for k,v in STATUS.items() if k!='PASS_GITHUB')

SCIENTIFIC_EXPLICIT_BLOCKERS=[]
# G5: отдельный аудит пригодности, не требуем в первичной H1/H3 программе
# h=5: проверка полноты календаря, но не основной прогнозный горизонт
if int(PANEL.boundary_year_at_risk.sum())>0:
    SCIENTIFIC_EXPLICIT_BLOCKERS.append('BOUNDARY_HISTORY_UNVERIFIED')
if STATUS.get('G1_DOMAIN_COMPLETENESS',{}).get('status')!='PASS':
    SCIENTIFIC_EXPLICIT_BLOCKERS.append('G1_WAGES_EMPLOYMENT_INCOME_NOT_VALIDATED')
if STATUS.get('HISTORICAL_PUBLICATION_DATES',{}).get('status')!='PASS':
    SCIENTIFIC_EXPLICIT_BLOCKERS.append('PUBLICATION_VINTAGE_NOT_VERIFIED')
if STATUS.get('ADVANCED_VS_BEST_SINGLE',{}).get('status')!='PASS':
    SCIENTIFIC_EXPLICIT_BLOCKERS.append('ENSEMBLE_VS_SINGLE_NOT_COMPLETE')
scientific_ready=(CFG['SCIENTIFIC_RELEASE_ALLOWED'] and CFG['MODE']=='FULL' and CFG['VERSION'].endswith('-final') and all(STATUS.get(s,{}).get('status')=='PASS' for s in critical)
  and canonical_confirmatory and full_calendar and no_fatal_qa and not SCIENTIFIC_EXPLICIT_BLOCKERS and len(METRICS)>0 and len(PREDICTIONS)>0)
if not scientific_ready:
    mark('FULL_SCIENCE_GATE','PARTIAL',
       'V4.03: status set by confirmed source vintages, boundary history and H1/H3 model QA; see blockers')
else:mark('FULL_SCIENCE_GATE','PASS','Full empirical hypothesis test, all required artefacts and scientific validation')

MANIFEST=WORK/'run_manifest.json'
def write_manifest(publication=None):
    pub=publication or {'status':'NOT_PUBLISHED'}
    payload={'experiment':'ML03_FACTORS','version':CFG['VERSION'],'mode':CFG['MODE'],
       'source_repo':CFG['REPO'],'source_commit_sha':source_sha,
       'source_handoff_sha256':hashlib.sha256(raw_index).hexdigest(),
       'total_assets_stage1':len(ASSETS),'verified_inputs':source_receipts,
       'all_stage1_downloaded':bool(all(a['asset_id'] in LOCAL_ASSETS for a in ASSETS)),
       'years':{'first':min(YEARS),'last':max(YEARS)},
       'n_municipalities':int(base.entity_id.nunique()),'n_panel_rows':len(PANEL),
       'n_supervised_candidates':len(TASK_PANEL),'n_predictions':len(PREDICTIONS),
       'n_model_evaluations':len(METRICS),'n_paired_contrasts':len(ABLA),
       'n_advanced_superlearner_evaluations':len(ADVANCED_ENSEMBLES),
       'n_advanced_shap_local_rows':len(ADVANCED_SHAP_LOCAL),
       'advanced_explanation_status':STATUS.get('ADVANCED_SHAP_QA'),
       'advanced_temporal_canary':ADVANCED_TEMPORAL_CANARY.to_dict('records'),
       'advanced_shap_interaction_rows':len(ADVANCED_SHAP_INTERACTIONS),
       'advanced_vs_dev_best_single':ADVANCED_SINGLE_DUEL.to_dict('records') if len(ADVANCED_SINGLE_DUEL) else [],
       'github_colab_secret_name':CFG['GITHUB_COLAB_SECRET'],
       'publication_date_audit':PUBLICATION_AVAILABILITY_QA.to_dict('records'),
       'organization_grain_exclusions':ORGANIZATION_GRAIN_EXCLUSIONS,
       'h5_feasibility_report':H5_CALENDAR_GAPS.to_dict('records'),
       'progress_widget_count':1 if _PROGRESS_BAR.widget is not None else 0,
       'excel_shap_local_exported_rows':sum(r['rows'] for r in EXCEL_SHEET_INDEX if r['source']=='SHAP_локальные'),
       'supplemental_bdmo_receipts':BDMO_EXTERNAL_RECEIPTS,
       'supplemental_bdmo_load_audit':BDMO_EXTERNAL_QA.to_dict('records'),
       'groups':{g:{'n_features':len(ff),'model_features':ff} for g,ff in GROUP_FEATURES.items()},
       'status':'FULL_SCIENTIFIC_PASS' if scientific_ready else 'DIAGNOSTIC_FULL_RUN_SCIENTIFIC_PARTIAL',
       'technical_status':'PASS' if all(STATUS.get(k,{}).get('status')=='PASS'
             for k in ('PASS_INPUT','PASS_DATA','PASS_TRAINING','PASS_OUTPUTS','PASS_NOTEBOOK')) else 'PARTIAL',
       'scientific_status':'PASS' if scientific_ready else 'PARTIAL',
       'scientific_blockers':[k for k in critical if STATUS.get(k,{}).get('status')!='PASS'] +
           ([] if canonical_confirmatory else ['CANONICAL_M0_M4_COMPLETE_SEQUENCE_NOT_CONFIRMED']) +
           ([] if full_calendar else ['PRIMARY_TWO_YEAR_CALENDAR_INCOMPLETE']) +
           ([] if no_fatal_qa else ['FATAL_RUNTIME_QA_PRESENT']) + SCIENTIFIC_EXPLICIT_BLOCKERS,
       'run_completion_status':RUN_COMPLETION_STATUS,
       'scientific_ready':scientific_ready,'causal_status':'CAUSAL_NOT_IDENTIFIED',
       'status_details':STATUS,'task_plan':TASK_STATUSES.to_dict('records'),
       'selected_models':DEV_METRICS.to_dict('records'),
       'factor_cutoffs_by_horizon':FACTOR_CUTOFF_BY_H,
       'target_specific_test_windows':TARGET_TEST_WINDOWS,
       'stage1_physical_qa':STAGE1_PHYSICAL_AUDIT.to_dict('records'),
       'economy_manifest_audit':ECONOMY_SOURCE_MANIFEST_QA.to_dict('records'),
       'regional_context_not_municipal_g1':True,
       'regional_context_dictionary':REGIONAL_CONTEXT_META,
       'regional_context_audit':REGIONAL_CONTEXT_AUDIT.to_dict('records'),
       'regional_unit_qa':G1R_UNIT_QA.to_dict('records'),
       'feature_staleness_qc':STALENESS_QA.to_dict('records'),
       'discovery_exhaustive':DISCOVERY_EXHAUSTIVE,
       'scientific_open_limitations':[
          'G5_HISTORICAL_VINTAGE_UNVERIFIED' if not GROUP_FEATURES.get('G5') else '',
          'FIVE_YEAR_NOT_FEASIBLE' if len(H5_CALENDAR_GAPS) and H5_CALENDAR_GAPS.status.ne('FEASIBLE_COMPLETE').any() else '',
          'HISTORIC_MUNICIPAL_BOUNDARY_CROSSWALK_REQUIRED' if int(PANEL.boundary_year_at_risk.sum())>0 else '',
          'G1_INCOME_LABOUR_COVERAGE_INSUFFICIENT' if STATUS.get('G1_DOMAIN_COMPLETENESS',{}).get('status')!='PASS' else '',
          'SOURCE_PUBLICATION_DATES_MISSING' if STATUS.get('HISTORICAL_PUBLICATION_DATES',{}).get('status')!='PASS' else '',
       ],
       'data_smoke':DATA_SMOKE_REPORT, 'sql_duckdb_unit_contract':SQL_DUCKDB_CONTRACT_REPORT, 'real_sql_preflight':REAL_SQL_PREFLIGHT, 'source_parity':SOURCE_PARITY_REPORT,
       'unadapted_municipal_wide_assets':UNADAPTED_MUNICIPAL_WIDE,
       'g1_empty_cause':G1_EMPTY_CAUSE if not GROUP_FEATURES['G1'] else None,
       'target_availability':TARGET_AVAILABILITY.to_dict('records'),
       'distinct_seeds':SEED_QA.to_dict('records'),
       'test_metrics':METRICS.to_dict('records'),
       'ablation_results':ABLA.to_dict('records') if len(ABLA) else [],
       'advanced_comparisons':ADVANCED_COMPARISONS.to_dict('records') if len(ADVANCED_COMPARISONS) else [],
       'advanced_model_selection':ADVANCED_ENSEMBLES.to_dict('records') if len(ADVANCED_ENSEMBLES) else [],
       'advanced_additivity_audit':ADVANCED_SHAP_QA.to_dict('records') if len(ADVANCED_SHAP_QA) else [],
       'outputs':{f.name:{'sha256':digest(f),'bytes':f.stat().st_size} for f in (IPYNB,XLSX,ZIP)},
       'publication':pub,'elapsed_seconds':int(time.monotonic()-START),
       'interpretation':'Predictive association only. No automatically inferred municipal policy effect.'}
    MANIFEST.write_text(json.dumps(json_safe(payload),ensure_ascii=False,allow_nan=False,indent=2),encoding='utf-8')
    ensure(json.loads(MANIFEST.read_text(encoding='utf-8'))['version']==CFG['VERSION'],'MANIFEST_INVALID')
RUN_COMPLETION_STATUS='COMPUTATIONS_COMPLETE_RELEASE_GATED'
write_manifest()

# GitHub writing is intentionally unreachable before FULL scientific acceptance.
def create_or_update_git_file(remote,local,branch,content_sha=None):
    request={'message':f"ML03 {CFG['VERSION']}: {Path(remote).name}",
      'content':base64.b64encode(Path(local).read_bytes()).decode(),'branch':branch}
    if content_sha:request['sha']=content_sha
    obj=api('PUT','/contents/'+urllib.parse.quote(remote,safe='/'),json=request,expected=(200,201)).json()
    return obj['commit']['sha'],obj['content']['sha']
def verify_github_file(remote,branch,path):
    obj=api('GET',f"/contents/{urllib.parse.quote(remote,safe='/')}?ref={urllib.parse.quote(branch,safe='')}").json()
    blob=base64.b64decode(obj['content']) if obj.get('content') else base64.b64decode(
        api('GET','/git/blobs/'+obj['sha']).json()['content'])
    ensure(hashlib.sha256(blob).hexdigest()==digest(path),'GITHUB_READBACK_SHA_MISMATCH: '+Path(remote).name)
def verify_release_asset(asset_id,expected):
    url=API+'/releases/assets/'+str(asset_id)
    r=SESSION.get(url,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,1200))
    ensure(r.status_code==200,'RELEASE_READBACK_NOT_200')
    if 'json' in r.headers.get('Content-Type','').lower():
        meta=r.json();r.close();u=meta.get('browser_download_url','')
        p=urllib.parse.urlparse(u)
        ensure(p.scheme=='https' and p.hostname in ('github.com','api.github.com'),'RELEASE_ASSET_URL_INVALID')
        r=SESSION.get(u,stream=True,timeout=(30,1200))
    h=hashlib.sha256()
    try:
        ensure(r.status_code==200 and 'json' not in r.headers.get('Content-Type','').lower(),'RELEASE_ASSET_DOWNLOAD_FAILED')
        for b in r.iter_content(4*1024*1024):
            if b:h.update(b)
    finally:r.close()
    ensure(h.hexdigest()==expected,'RELEASE_ASSET_READBACK_SHA_MISMATCH')

def publish_verified():
    ensure(scientific_ready and CFG['MODE']=='FULL' and CFG['PUBLISH_FULL_PASS'] is True,'FULL_QA_REQUIRED')
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    branch='ml03-results/'+CFG['VERSION'].replace('.','-')+'-'+stamp
    tag=CFG['VERSION']+'-'+stamp
    api('POST','/git/refs',json={'ref':'refs/heads/'+branch,'sha':source_sha},expected=(201,))
    npath=CFG['SEGMENT']+'/'+IPYNB.name;xpath=CFG['SEGMENT']+'/'+XLSX.name
    mpath=CFG['SEGMENT']+'/'+MANIFEST.name
    create_or_update_git_file(npath,IPYNB,branch)
    revision,_=create_or_update_git_file(xpath,XLSX,branch)
    # A draft release accepts assets before publishing a tag. Publish the tag only after
    # the manifest and release asset have been re-downloaded and verified.
    release=api('POST','/releases',json={'tag_name':tag,'target_commitish':branch,
      'name':'ML03 '+tag,'draft':True,'prerelease':False,
      'body':'Stage II ML03; scientific PASS; evidence SHA-256 '+digest(ZIP)},expected=(201,)).json()
    with open(ZIP,'rb') as f:
        upload=SESSION.post(f"https://uploads.github.com/repos/{CFG['REPO']}/releases/{release['id']}/assets",
         params={'name':ZIP.name},data=f,headers={'Content-Type':'application/zip','Accept':'application/vnd.github+json'},timeout=(30,1200))
    ensure(upload.status_code==201,'RELEASE_UPLOAD_HTTP_'+str(upload.status_code))
    asset=upload.json()
    verify_github_file(npath,branch,IPYNB);verify_github_file(xpath,branch,XLSX)
    verify_release_asset(asset['id'],digest(ZIP))
    publication={'status':'PASS_GITHUB','branch':branch,'tag':tag,
      'release_url':release['html_url'],'evidence_asset_id':asset['id'],
      'evidence_url':asset.get('browser_download_url',''),
      'notebook_url':f"https://github.com/{CFG['REPO']}/blob/{branch}/{npath}",
      'excel_url':f"https://github.com/{CFG['REPO']}/blob/{branch}/{xpath}",
      'initial_commit_sha':revision,'readback_sha_verified':True,
      'publication_phase':'draft_assets_verified_before_tag_publication'}
    write_manifest(publication)
    manifest_commit,_=create_or_update_git_file(mpath,MANIFEST,branch)
    verify_github_file(mpath,branch,MANIFEST)
    published=api('PATCH','/releases/'+str(release['id']),
       json={'draft':False,'target_commitish':branch},expected=(200,)).json()
    ensure(published.get('draft') is False,'RELEASE_REMAINS_DRAFT')
    ref=api('GET','/git/ref/tags/'+urllib.parse.quote(tag,safe='')).json()
    tagsha=ref['object']['sha']
    if ref['object']['type']=='tag':
        annotated=api('GET','/git/tags/'+tagsha).json();tagsha=annotated['object']['sha']
    ensure(tagsha==manifest_commit,'RELEASE_TAG_DOES_NOT_POINT_TO_VERIFIED_MANIFEST_COMMIT')
    verify_release_asset(asset['id'],digest(ZIP))
    mark('PASS_GITHUB','PASS','Release tag, git artifacts, manifest and evidence passed SHA readback')
    publication['manifest_commit_sha']=manifest_commit
    return publication

PUBLISHED={'status':'NOT_PUBLISHED'}
if CFG['MODE']=='FULL' and CFG['PUBLISH_FULL_PASS']:
    try:PUBLISHED=publish_verified()
    except Exception as ex:
        mark('PASS_GITHUB','PUBLICATION_FAIL',type(ex).__name__+': '+str(ex)[:130])
        PUBLISHED={'status':'PUBLICATION_FAIL','error_type':type(ex).__name__}
        write_manifest(PUBLISHED)
else:mark('PASS_GITHUB','NOT_PUBLISHED','No write without explicit switch and scientific FULL QA')
if STATUS.get('PASS_NOTEBOOK',{}).get('status')=='PASS' and MANIFEST.exists():
    RUN_COMPLETION_STATUS='COMPUTATIONS_COMPLETE_RELEASE_GATED'
    progress('FINAL',1,1,'Основные модели завершены; переход к причинному блоку')
else:
    RUN_COMPLETION_STATUS='PARTIAL'
ensure(_PROGRESS_BAR.n==850,'CORE_STAGE_PROGRESS_INCOMPLETE')
# The unified progress remains active until causal diagnostics and BOTH archives are verified.
quiet_print('CORE_STAGE_COMPLETE',CFG['VERSION'], 'comparisons',len(ABLA),'evaluations',len(METRICS))
quiet_print('Local outputs:',*[str(x) for x in (IPYNB,XLSX,ZIP,MANIFEST)],sep='\n · ')
if PUBLISHED['status']=='PASS_GITHUB':quiet_print(PUBLISHED['release_url'])


#### Определение причинных гипотез и требований к данным.

In [ ]:
# C01 · IV/IV · КОНФИГУРАЦИЯ ПРИЧИННОГО МОДУЛЯ (разделение прогнозирования и ATT)
CAUSAL_CFG={
    'V402_OUTPUT_DIR':str(WORK),
    'WORKDIR':str(WORK/'causal'),
    'EVIDENCE_ZIP':'',                      # пусто = найти в V4.02_OUTPUT_DIR
    'RUN_MANIFEST':'',                     # пусто = найти рядом
    'TREATMENT_REGISTER':CFG.get('CAUSAL_TREATMENT_REGISTER',''),               # заполнить путь к реальному CSV
    'VERIFIED_BOUNDARY_REGISTER':CFG.get('CAUSAL_BOUNDARY_REGISTER',''),       # путь к проверенным периодам границ
    'POLICY_ID':None,                     # если в реестре >1 policy, указать явно
    'OUTCOMES':['migration_per_1000','population_change_pct'],
    'EVENT_LEADS':3,'EVENT_LAGS':3,
    'MIN_TREATED':30,'MIN_CONTROLS':80,'MIN_REGIONS':8,
    'MIN_BASELINE_YEARS':2,
    'STRICT_BOUNDARIES':True,
    'PS_CLIP_LOW':0.02,'PS_CLIP_HIGH':0.98,'MAX_PS_CLIP_SHARE':0.1,
    'BOOTSTRAPS':299,'SEED':1729,
    'AUTO_UPLOAD_IN_FRESH_COLAB':False,
    'GITHUB_SECRET_NAME':'TRIAD-RU-Colab', # имя, не сам токен
}

progress('CAUSAL_LOAD',0,1,'Конфигурация и входные файлы')

#### Контроль SHA полного Evidence и исходной муниципальной панели.

In [ ]:
# C02 · ИСТОЧНИК V4.02 + КОНТРОЛЬ МАНИФЕСТА И EVIDENCE (без повторного обучения)
import os,json,re,math,zipfile,hashlib,shutil,tempfile,datetime,traceback
from pathlib import Path
import numpy as np,pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression,Ridge
from sklearn.pipeline import make_pipeline
from scipy.stats import norm
from matplotlib import pyplot as plt
OUT=Path(CAUSAL_CFG['WORKDIR']);OUT.mkdir(parents=True,exist_ok=True)
ROOT=Path(CAUSAL_CFG['V402_OUTPUT_DIR']); QA=[];CAUSAL_READY=False;ESTIMATES=pd.DataFrame();INFERENCE_STATUS='NOT_IDENTIFIED'
def audit(gate,status,reason='',**kwargs):
    QA.append({'gate':gate,'status':status,'reason':str(reason)[:500],**kwargs})
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def findpath(custom,candidates):
    if custom:return Path(custom)
    return next((x for x in candidates if x.exists()),None)
ZIP_PATH=findpath(CAUSAL_CFG['EVIDENCE_ZIP'],[ROOT/'ML3_EVIDENCE.zip',Path('ML3_EVIDENCE.zip')])
MANIFEST_PATH=findpath(CAUSAL_CFG['RUN_MANIFEST'],[ROOT/'run_manifest.json',Path('run_manifest.json')])
if ZIP_PATH is None or not ZIP_PATH.exists():
    audit('EVIDENCE','BLOCKED','ML3_EVIDENCE.zip отсутствует; необходим результат V4.02 FULL')
if MANIFEST_PATH is None or not MANIFEST_PATH.exists():
    audit('MANIFEST','BLOCKED','run_manifest.json отсутствует; необходим результат V4.02 FULL')
manifest={};PANEL_CAUSAL=pd.DataFrame();TRANSFORMS_CAUSAL=pd.DataFrame();evidence_info={}
if ZIP_PATH and ZIP_PATH.exists() and MANIFEST_PATH and MANIFEST_PATH.exists():
    manifest=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    if 'v4.0.3' not in str(manifest.get('version','')).lower():
        audit('VERSION','BLOCKED','Ожидаются результаты V4.03, получено '+str(manifest.get('version')))
    else:
        expect=manifest.get('outputs',{}).get(ZIP_PATH.name,{}).get('sha256')
        real=sha256(ZIP_PATH)
        if expect and expect!=real:
            audit('SOURCE_SHA','BLOCKED','SHA ZIP не совпадает с манифестом')
        else:
            audit('SOURCE_SHA','PASS','Архив V4.02 проверен',zip_sha256=real)
            with zipfile.ZipFile(ZIP_PATH) as z:
                names=set(z.namelist()); idx='evidence/methods_and_evidence_index.json'
                if idx not in names:audit('EVIDENCE_INDEX','BLOCKED','Не найден индекс Evidence')
                else:
                    info=json.loads(z.read(idx));evidence_info={x['file']:x for x in info.get('evidence_index',[])}
                    def evidence_read(name):
                        member='evidence/'+name+'.parquet'
                        if member not in names:raise FileNotFoundError(member)
                        rec=evidence_info.get(name+'.parquet')
                        if not rec:raise RuntimeError('FILE_NOT_IN_EVIDENCE_INDEX: '+name)
                        target=OUT/(name+'.parquet')
                        with z.open(member) as src,open(target,'wb') as dest:
                            shutil.copyfileobj(src,dest,length=4*1024*1024)
                        if sha256(target)!=rec['sha256']:raise RuntimeError('EVIDENCE_SHA_MISMATCH: '+name)
                        p=pq.read_table(target).to_pandas()
                        if len(p)!=rec['n_rows']:raise RuntimeError('EVIDENCE_ROW_COUNT_MISMATCH: '+name)
                        return p
                    try:
                        PANEL_CAUSAL=evidence_read('municipal_source_panel')
                        TRANSFORMS_CAUSAL=evidence_read('factor_transforms')
                        audit('PANEL','PASS',f'{len(PANEL_CAUSAL)} муниципалитет-лет; SHAs сверены')
                    except Exception as exc:
                        audit('PANEL','BLOCKED',type(exc).__name__+': '+str(exc)[:220])
# Токен Colab TRIAD-RU-Colab остаётся только в V4.02-ноутбуке; для локальных outputs не нужен.
# Если надо повторно получить оригинальный handoff, авторизуйте V4.02 указанным секретом и запустите его.

progress('CAUSAL_LOAD',1,1,'Входные Evidence и SHA проверены')

#### Валидация реестра фактических вмешательств и исторических границ.

In [ ]:
# C03 · ИСТИННЫЙ РЕЕСТР ВОЗДЕЙСТВИЙ; НЕ СЧИТАТЬ НЕЗАРЕГИСТРИРОВАННЫЕ ЕДИНИЦЫ КОНТРОЛЕМ
SCHEMA=['entity_id','policy_id','treatment_status','adoption_year','source_url','date_precision','exposure_definition','spillover_risk']
BOUNDARY_SCHEMA=['entity_id','start_year','end_year','boundary_stable','official_source_url','verification_scope']
register_file=findpath(CAUSAL_CFG['TREATMENT_REGISTER'],[OUT/'causal_interventions.csv',ROOT/'causal_interventions.csv'])
boundary_file=findpath(CAUSAL_CFG['VERIFIED_BOUNDARY_REGISTER'],[OUT/'verified_boundary_periods.csv',ROOT/'verified_boundary_periods.csv'])
if register_file is None:
    audit('TREATMENT_REGISTER','NOT_FEASIBLE','Нет проверенного реестра вмешательств; ATT не вычисляется')
if boundary_file is None:
    audit('BOUNDARY_REGISTER','NOT_FEASIBLE','Нет официального реестра исторических муниципальных границ')
REG=pd.DataFrame();BVER=pd.DataFrame()
if register_file is not None and register_file.exists():
    REG=pd.read_csv(register_file,dtype={'entity_id':'string','policy_id':'string'},keep_default_na=False)
    miss=set(SCHEMA)-set(REG.columns)
    if miss:audit('TREATMENT_REGISTER','BLOCKED','Нет полей '+','.join(sorted(miss)))
    elif REG.empty:audit('TREATMENT_REGISTER','BLOCKED','Реестр пуст')
    else:
        REG['entity_id']=REG.entity_id.str.strip()
        REG['policy_id']=REG.policy_id.str.strip()
        policies=sorted(REG.policy_id.dropna().unique())
        active=CAUSAL_CFG['POLICY_ID']
        if active is None and len(policies)!=1:audit('POLICY_SELECTION','BLOCKED','Более одной политики; укажите POLICY_ID: '+','.join(policies[:15]))
        else:
            active=active or policies[0]
            REG=REG.loc[REG.policy_id.eq(active)].copy()
            if REG.entity_id.duplicated().any():audit('TREATMENT_REGISTER','BLOCKED','Несколько строк для одного муниципалитета и политики')
            else:
                REG['treatment_status']=REG.treatment_status.astype(str).str.upper().str.strip()
                valid_status=REG.treatment_status.isin(['TREATED','NEVER_TREATED']).all()
                REG['adoption_year']=pd.to_numeric(REG.adoption_year,errors='coerce')
                bad=(REG.treatment_status.eq('TREATED')&REG.adoption_year.isna()) | (REG.treatment_status.eq('NEVER_TREATED')&REG.adoption_year.notna())
                valid_source=REG.source_url.astype(str).str.match(r'^https?://').all()
                valid_precision=REG.loc[REG.treatment_status.eq('TREATED'),'date_precision'].isin(['YEAR','EXACT_DATE']).all()
                if not(valid_status and not bad.any() and valid_source and valid_precision):
                    audit('TREATMENT_REGISTER','BLOCKED','Неверные статусы, даты или ссылки на документы')
                else:
                    audit('TREATMENT_REGISTER','PASS',f'Политика {active}: {len(REG)} единиц, {int(REG.treatment_status.eq("TREATED").sum())} получателей')
if boundary_file is not None and boundary_file.exists():
    BVER=pd.read_csv(boundary_file,dtype={'entity_id':'string'},keep_default_na=False)
    if set(BOUNDARY_SCHEMA).issubset(BVER.columns):
        for col in ('start_year','end_year'):BVER[col]=pd.to_numeric(BVER[col],errors='coerce')
        good=BVER.official_source_url.astype(str).str.match(r'^https?://')&BVER.boundary_stable.astype(str).str.upper().isin(['TRUE','1','YES'])
        good &= BVER.start_year.notna()&BVER.end_year.notna()&BVER.start_year.le(BVER.end_year)
        BVER=BVER.loc[good].copy()
        audit('BOUNDARY_REGISTER','PASS' if len(BVER) else 'BLOCKED',f'Верифицированных стабильных периодов: {len(BVER)}')
    else:audit('BOUNDARY_REGISTER','BLOCKED','Нет обязательных полей или внешнего подтверждения')

progress('CAUSAL_REG',1,1,'Реестр вмешательств и границ проверен')

#### Предлечебные ковариаты, динамика исходов и допустимость выборки.

In [ ]:
# C04 · РАСЧЁТ НАБЛЮДАЕМЫХ ИСХОДОВ, ПРЕДЛЕЧЕБНЫХ КОВАРИАТ, ВАЛИДНОСТИ ГРАНИЦ
DATA=pd.DataFrame();EVENT_PLAN=pd.DataFrame();VALID_POLICY=False;EXPOSURE_QA={}
if len(PANEL_CAUSAL) and len(REG) and any(r['gate']=='TREATMENT_REGISTER' and r['status']=='PASS' for r in QA):
    D=PANEL_CAUSAL.copy()
    required={'entity_id','observation_year','region_name','population','migration_per_1000'}
    if required-set(D):audit('PANEL_SCHEMA','BLOCKED','Не хватает столбцов '+str(required-set(D)))
    else:
        D['entity_id']=D.entity_id.astype('string').str.strip();D['observation_year']=pd.to_numeric(D.observation_year,errors='coerce')
        D=D.loc[D.observation_year.notna()].copy();D['observation_year']=D.observation_year.astype(int)
        D=D.sort_values(['entity_id','observation_year']).drop_duplicates(['entity_id','observation_year'],keep=False)
        D['population']=pd.to_numeric(D.population,errors='coerce')
        p0=D.groupby('entity_id').population.shift(1);y0=D.groupby('entity_id').observation_year.shift(1)
        D['population_change_pct']=np.where((D.observation_year-y0).eq(1)&p0.gt(0),(D.population/p0-1)*100,np.nan)
        D=D.merge(REG[['entity_id','treatment_status','adoption_year','spillover_risk']],on='entity_id',how='inner',validate='many_to_one')
        D['boundary_verified']=False
        if len(BVER):
            # A municipality is accepted only if an externally documented stable interval covers the observation year.
            verified_keys=set()
            for row in BVER.itertuples():
                for y in range(int(row.start_year),int(row.end_year)+1):verified_keys.add((str(row.entity_id),int(y)))
            D['boundary_verified']=[(str(e),int(y)) in verified_keys for e,y in zip(D.entity_id,D.observation_year)]
        if CAUSAL_CFG['STRICT_BOUNDARIES']:
            D=D[D.boundary_verified].copy()
        D['adoption_year']=D.adoption_year.fillna(9999).astype(int)
        D['treatment_status']=D.treatment_status.astype(str)
        DATA=D
        exposure_regions=D.loc[D.treatment_status.eq('TREATED'),'region_name'].nunique()
        good_treat=int(D.loc[D.treatment_status.eq('TREATED'),'entity_id'].nunique())
        good_control=int(D.loc[D.treatment_status.eq('NEVER_TREATED'),'entity_id'].nunique())
        all_regions=D.region_name.nunique()
        EXPOSURE_QA={'n_eligible_municipalities':D.entity_id.nunique(),'n_treated':good_treat,
          'n_never_control':good_control,'n_regions':int(all_regions),'n_treated_regions':int(exposure_regions),
          'n_year_rows':len(D),'strict_boundaries':CAUSAL_CFG['STRICT_BOUNDARIES']}
        VALID_POLICY=(good_treat>=CAUSAL_CFG['MIN_TREATED'] and good_control>=CAUSAL_CFG['MIN_CONTROLS'] and
                      all_regions>=CAUSAL_CFG['MIN_REGIONS'])
        audit('EXPOSURE_SUPPORT','PASS' if VALID_POLICY else 'BLOCKED',str(EXPOSURE_QA))
# Covariates selected from values observed strictly at g-1. No post-treatment feature in nuisance regressions.
COVAR_CANDIDATES=[]
if len(DATA):
    prior=['population','births','deaths']
    factor_cols=sorted(c for c in DATA if re.match(r'^G[1-4]_[0-9]+',c) and pd.api.types.is_numeric_dtype(DATA[c]))
    COVAR_CANDIDATES=[c for c in prior+factor_cols if c in DATA and DATA[c].notna().sum()>=max(50,int(0.1*len(DATA)))]
    COVAR_CANDIDATES=COVAR_CANDIDATES[:12]
    audit('PRETREATMENT_COVARIATES','PASS' if len(COVAR_CANDIDATES)>=2 else 'BLOCKED',
          ','.join(COVAR_CANDIDATES))
CAUSAL_READY=bool(VALID_POLICY and len(COVAR_CANDIDATES)>=2 and len(DATA)>0)

progress('CAUSAL_PANEL',1,1,'Панель причинного дизайна подготовлена')

#### Когортный DR-DID, контролируемые группы и региональный cross-fitting.

In [ ]:
# C05 · КОГОРТНЫЙ DOUBLY ROBUST DID (NOT-YET/NEVER CONTROLS), GROUP-REGION CROSS-FITTING
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression,Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
C_RESULTS=[];C_DIAGNOSTICS=[]
def construct_group_period(data,cohort,year,outcome):
    ref=int(cohort)-1
    if int(year)==ref:return pd.DataFrame()
    two=data[data.observation_year.isin([ref,int(year)])].copy()
    if two.empty:return pd.DataFrame()
    base=two.loc[two.observation_year.eq(ref),['entity_id','region_name','adoption_year',outcome]+COVAR_CANDIDATES].copy()
    base=base.rename(columns={outcome:'y_base'}).drop_duplicates('entity_id')
    current=two.loc[two.observation_year.eq(year),['entity_id',outcome]].rename(columns={outcome:'y_now'}).drop_duplicates('entity_id')
    z=base.merge(current,on='entity_id',how='inner',validate='one_to_one')
    z=z.loc[z.y_base.notna()&z.y_now.notna()].copy()
    z['D']=z.adoption_year.eq(int(cohort)).astype(int)
    z=z.loc[z.D.eq(1)|z.adoption_year.gt(max(int(cohort),int(year)))].copy()  # never- or not-yet-treated
    z['dy']=z.y_now-z.y_base
    return z

def dr_did(df,seed,bootstraps=299):
    from sklearn.model_selection import GroupKFold
    if (df.D.eq(1).sum()<CAUSAL_CFG['MIN_TREATED'] or df.D.eq(0).sum()<CAUSAL_CFG['MIN_CONTROLS'] or
         df.region_name.nunique()<CAUSAL_CFG['MIN_REGIONS']):return None,{'status':'INSUFFICIENT_SUPPORT'}
    x=df[COVAR_CANDIDATES].apply(pd.to_numeric,errors='coerce')
    # Exclude cohort/year information: these are design variables, not confounders.
    x=x.loc[:,x.notna().any(axis=0)]
    if x.shape[1]<2:return None,{'status':'TOO_FEW_BASELINE_COVARIATES'}
    outcome=df.dy.to_numpy(float);treated=df.D.to_numpy(int);region=df.region_name.astype(str).to_numpy()
    prob=np.full(len(df),np.nan);mu=np.full(len(df),np.nan)
    folds=GroupKFold(n_splits=min(3,df.region_name.nunique()))
    for tr,te in folds.split(x,treated,groups=region):
        if len(np.unique(treated[tr]))<2 or int((treated[tr]==0).sum())<20:
            return None,{'status':'CROSSFIT_NOT_FEASIBLE_IN_REGION_FOLD'}
        pp=make_pipeline(SimpleImputer(strategy='median',add_indicator=False),StandardScaler(),
                         LogisticRegression(C=0.5,max_iter=1000,random_state=seed))
        mm=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=10.0))
        pp.fit(x.iloc[tr],treated[tr]);mm.fit(x.iloc[tr][treated[tr]==0],outcome[tr][treated[tr]==0])
        prob[te]=pp.predict_proba(x.iloc[te])[:,1]
        mu[te]=mm.predict(x.iloc[te])
    if not np.isfinite(prob).all() or not np.isfinite(mu).all():return None,{'status':'NAN_NUISANCE_PREDICTION'}
    clipped=np.mean((prob<CAUSAL_CFG['PS_CLIP_LOW'])|(prob>CAUSAL_CFG['PS_CLIP_HIGH']))
    if clipped>CAUSAL_CFG['MAX_PS_CLIP_SHARE']:
        return None,{'status':'POSITIVITY_FAILURE','clipped_share':float(clipped)}
    ps=np.clip(prob,CAUSAL_CFG['PS_CLIP_LOW'],CAUSAL_CFG['PS_CLIP_HIGH'])
    odds=ps/(1-ps)
    residual=outcome-mu
    def aggregate(inds):
        d=treated[inds];v=residual[inds];w=odds[inds]
        nt=int(d.sum());nw=float(((1-d)*w).sum())
        return float(v[d==1].mean()-np.sum(v[d==0]*w[d==0])/nw) if nt and nw>0 else np.nan
    point=aggregate(np.arange(len(df)))
    rng=np.random.default_rng(seed);clusters=list(dict.fromkeys(region))
    blocks={c:np.flatnonzero(region==c) for c in clusters}
    reps=[]
    for i in range(int(bootstraps)):
        draw=rng.choice(clusters,size=len(clusters),replace=True)
        ids=np.concatenate([blocks[c] for c in draw]);v=aggregate(ids)
        if np.isfinite(v):reps.append(v)
    if len(reps)<max(50,int(0.8*bootstraps)):
        return None,{'status':'CLUSTER_BOOTSTRAP_INVALID','n_valid':len(reps)}
    lo,hi=np.percentile(reps,[2.5,97.5])
    status='CONDITIONAL_ASSOCIATION_DID'  # assumptions not machine-verified
    return {'ATT_DR':point,'ci_low':float(lo),'ci_high':float(hi),'p_bootstrap_two_sided':float(
        2*min(np.mean(np.asarray(reps)<=0),np.mean(np.asarray(reps)>=0))),
        'n_treated':int(treated.sum()),'n_controls':int((treated==0).sum()),
        'n_regions':len(clusters),'n_bootstrap':len(reps),'ps_trim_share':float(clipped),
        'inference':'region bootstrap of cross-fitted nuisance predictions (nuisance not refit)',
        'status':status}, {'status':'PASS_DIAGNOSTICS','n_covariates':x.shape[1],
            'ps_min':float(ps.min()),'ps_max':float(ps.max()),'share_trimmed':float(clipped)}

if CAUSAL_READY:
    cohorts=sorted(set(DATA.loc[DATA.treatment_status.eq('TREATED'),'adoption_year']))
    for outcome in CAUSAL_CFG['OUTCOMES']:
        for g in cohorts:
            for event_time in range(-int(CAUSAL_CFG['EVENT_LEADS']),int(CAUSAL_CFG['EVENT_LAGS'])+1):
                if event_time==-1:continue  # omitted reference period
                yr=int(g)+event_time
                sample=construct_group_period(DATA,g,yr,outcome)
                if sample.empty:continue
                est,qa=dr_did(sample,CAUSAL_CFG['SEED']+int(g)*17+event_time,
                              CAUSAL_CFG['BOOTSTRAPS'])
                C_DIAGNOSTICS.append({'outcome':outcome,'cohort':g,'year':yr,'event_time':event_time,**qa})
                if est:C_RESULTS.append({'outcome':outcome,'cohort':g,'year':yr,'event_time':event_time,**est})
    if C_RESULTS:INFERENCE_STATUS='ESTIMATED_CONDITIONAL_ON_DID_ASSUMPTIONS_NOT_CAUSALLY_PROVEN'
ESTIMATES=pd.DataFrame(C_RESULTS);CAUSAL_DIAGNOSTICS=pd.DataFrame(C_DIAGNOSTICS)
audit('CAUSAL_ESTIMATES','PASS' if len(ESTIMATES) else 'BLOCKED',
      f'{len(ESTIMATES)} group-time estimates; {INFERENCE_STATUS}')

progress('CAUSAL_EST',1,2,'Когортные оценки DR-DID')

#### Событийные кривые, претренды, неоднородность и интервалы.

In [ ]:
# C06 · СВОДКА ATT, ПРЕТРЕНДЫ, ГЕТЕРОГЕННОСТЬ И ГРАФИКИ (ТОЛЬКО ЕСЛИ ВЫПОЛНИМЫ)
EVENT_STUDY=pd.DataFrame();PRETREND_QA=pd.DataFrame();CAUSAL_GRAPHS=[]
if len(ESTIMATES):
    # Cohort-weighted mean; weights = factual n_treated per cohort, not the model seed count.
    rec=[]
    for (outcome,event_time),g in ESTIMATES.groupby(['outcome','event_time']):
        w=g.n_treated.to_numpy(float);m=float(np.average(g.ATT_DR,weights=w))
        rec.append({'outcome':outcome,'event_time':int(event_time),'ATT_weighted':m,
            'n_cohorts':len(g),'n_treated_cohort_sum':int(w.sum()),
            'aggregation':'cohort size; SE not pooled across overlapping clusters'})
    EVENT_STUDY=pd.DataFrame(rec)
    pre=[]
    for outcome,g in EVENT_STUDY.groupby('outcome'):
        leads=g.loc[g.event_time.le(-2)]
        pre.append({'outcome':outcome,'n_pre_event_points':len(leads),
          'max_abs_pre_estimate':float(leads.ATT_weighted.abs().max()) if len(leads) else np.nan,
          'status':'PRETRENDS_OBSERVABLE_NOT_PROVEN' if len(leads)>=2 else 'INSUFFICIENT_PRE_PERIODS',
          'interpretation':'A non-significant placebo does not establish parallel trends'})
    PRETREND_QA=pd.DataFrame(pre)
    for outcome,g in EVENT_STUDY.groupby('outcome'):
        g=pd.concat([g,pd.DataFrame([{'outcome':outcome,'event_time':-1,'ATT_weighted':0}])],ignore_index=True).sort_values('event_time')
        plt.figure(figsize=(9,4));plt.plot(g.event_time,g.ATT_weighted,marker='o')
        plt.axvline(-.5,linestyle='--',color='gray');plt.axhline(0,color='black',linewidth=.8)
        plt.xlabel('Годы относительно начала подтверждённого воздействия')
        plt.ylabel('Условный ATT (единицы исхода)')
        plt.title('Когортная динамика · '+('Миграция' if outcome=='migration_per_1000' else 'Изменение населения'))
        plt.tight_layout();f=OUT/('event_study_'+outcome+'.png');plt.savefig(f,dpi=170);plt.close();CAUSAL_GRAPHS.append(str(f))

progress('CAUSAL_EST',2,2,'Претренды и неоднородность')

#### Протокол причинного анализа и контрольные суммы.

In [ ]:
# C07 · ПРОТОКОЛ ГОТОВНОСТИ, РЕЗУЛЬТАТЫ, SHA АРТЕФАКТОВ. НИЧЕГО НЕ ПУБЛИКОВАТЬ В GitHub АВТОМАТИЧЕСКИ.
for filename,frame in [('causal_gate_audit.csv',pd.DataFrame(QA)),
 ('causal_group_time_att.csv',ESTIMATES),('causal_nuisance_diagnostics.csv',CAUSAL_DIAGNOSTICS if 'CAUSAL_DIAGNOSTICS' in globals() else pd.DataFrame()),
 ('causal_event_study.csv',EVENT_STUDY),('causal_pretrends.csv',PRETREND_QA)]:
    (OUT/filename).write_text(frame.to_csv(index=False),encoding='utf-8')
info={'version':'ML03_V4.03_CAUSAL_BLOCK_1','status':INFERENCE_STATUS,
 'causal_ready':bool(CAUSAL_READY),'panel_source':'municipal_source_panel.parquet',
 'input_evidence_sha256':sha256(ZIP_PATH) if ZIP_PATH and ZIP_PATH.exists() else None,
 'input_manifest_sha256':sha256(MANIFEST_PATH) if MANIFEST_PATH and MANIFEST_PATH.exists() else None,
 'intervention_sha256':sha256(register_file) if register_file and register_file.exists() else None,
 'boundary_sha256':sha256(boundary_file) if boundary_file and boundary_file.exists() else None,
 'n_cohort_time_estimates':len(ESTIMATES),'qa':QA,'exposure':EXPOSURE_QA,
 'limitations':['No validated exposure registry = no causal design',
 'Selection on unobserved confounders not testable',
 'Parallel trends cannot be proven by lead tests',
 'Spatial spillovers/SUTVA may fail',
 'Group-by-region crossfit and conditional region bootstrap do not constitute independent causal identification',
 'Municipal boundaries and official publication vintages must be reviewed'],
 'outputs':{x.name:sha256(x) for x in OUT.iterdir() if x.is_file()}}
(OUT/'CAUSAL_STATUS.json').write_text(json.dumps(info,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
quiet_print('Причинный статус:',INFERENCE_STATUS,'; оценок:',len(ESTIMATES))
quiet_print('Причинный блок сохранён в:',OUT)
quiet_print('Прогностический контур не изменён')

progress('CAUSAL_EXPORT',1,1,f'Статус причинного анализа: {INFERENCE_STATUS}')

#### Итоговый проверенный ZIP, readback GitHub Evidence, таблицы, графики и контроль SHA.

In [ ]:
# 21 · V4.03 · FINAL RESEARCH ZIP + GITHUB EVIDENCE + FULL CAUSAL STATUS
from pathlib import Path as _P
import json as _json,hashlib as _hash,zipfile as _zf,datetime as _dt,urllib.parse as _url
# Re-materialize the full runtime with actual causal cells, preserving all 4 stages.
UNIFIED_IPYNB,FINAL_PARITY=materialize_own_notebook(require_causal=True)
UNIFIED_PARITY=WORK/'source_parity_report.json'
if not FINAL_PARITY['source_parity_verified']:
    mark('FINAL_SOURCE_PARITY','PARTIAL',str(FINAL_PARITY['missing_runtime_cells']))
else:mark('FINAL_SOURCE_PARITY','PASS','Full runtime reconstruction through causal module')
# Apply causal QA to the final manifest (the technical archive has already been computed).
write_manifest(PUBLISHED)
ensure(XLSX.is_file() and XLSX.stat().st_size>0,'ANALYTICAL_WORKBOOK_MISSING')
ensure(ZIP.is_file() and ZIP.stat().st_size>0,'TECHNICAL_EVIDENCE_NOT_CREATED')
ensure(MANIFEST.is_file(),'RUN_MANIFEST_MISSING')
ensure((OUT/'CAUSAL_STATUS.json').is_file(),'CAUSAL_STATUS_MISSING')

# 1. One curated, small scientific package: no raw panel, full test predictions or full local SHAP.
RESEARCH_DATA={
 '01_test_metrics':METRICS,'02_confirmatory_exploratory_comparisons':ABLA,
 '03_model_selection_DEV':DEV_METRICS,'04_ensemble_test':ADVANCED_ENSEMBLES,
 '05_ensemble_vs_best_single':ADVANCED_SINGLE_DUEL,
 '06_ensemble_intervals':ADVANCED_INTERVALS,'07_factor_dictionary':FEATURE_DICT,
 '08_factor_support':MODEL_DOMAIN_COVERAGE,'09_global_SHAP':ADVANCED_SHAP,
 '10_SHAP_region_CI':ADVANCED_SHAP_CI,'11_SHAP_stability':ADVANCED_SHAP_STABILITY,
 '12_SHAP_interactions':ADVANCED_SHAP_INTERACTIONS,'13_SHAP_additivity_QA':ADVANCED_SHAP_QA,
 '14_grouped_permutations':ADVANCED_GROUP_PERM,'15_PDP':ADVANCED_PDP,'16_ICE':ADVANCED_ICE,
 '17_ALE':ALE,'18_publication_dates_QA':PUBLICATION_AVAILABILITY_QA,
 '19_five_year_calendar':H5_CALENDAR_GAPS,'20_task_plan':TASK_STATUSES,
 '21_municipality_boundaries':BOUNDARY_QA,'22_source_and_units':SEMANTIC_QA,
 '23_model_group_coverage':DOMAIN_COVERAGE,'24_future_canary':CANARY_QA,
 '25_ensemble_future_canary':ADVANCED_TEMPORAL_CANARY,
 '26_source_grain_audit':SOURCE_GRAIN_AUDIT,'27_real_SQL_QA':REAL_SQL_PREFLIGHT_QA,
 '28_extended_source_recovery':BDMO_EXTERNAL_RECOVERY,
 '29_correlation_sensitivity':SENSITIVITY,'30_size_heterogeneity':SIZE_SENSITIVITY,
 '31_hypotheses':HYPOTHESIS_REGISTRY,'32_regional_errors':REGIONAL_ERRORS,
 '33_research_status':pd.DataFrame([{'gate':k,**v} for k,v in STATUS.items()]),
 '34_shap_local_sample':ADVANCED_SHAP_LOCAL.head(int(CFG['RESEARCH_LOCAL_SHAP_SAMPLE'])),
 '35_causal_gate_audit':pd.DataFrame(QA),
 '36_causal_group_time_ATT':ESTIMATES if len(ESTIMATES) else None,
 '37_causal_event_study':EVENT_STUDY if len(EVENT_STUDY) else None,
 '38_causal_pretrends':PRETREND_QA if len(PRETREND_QA) else None,
}
RS=WORK/'research_fast';RS.mkdir(exist_ok=True)
SCIENTIFIC_CSV=[];RESEARCH_INDEX=[]
for name,frame in RESEARCH_DATA.items():
    if frame is None or not isinstance(frame,pd.DataFrame):continue
    path=RS/(name+'.csv')
    frame.to_csv(path,index=False,encoding='utf-8-sig')
    SCIENTIFIC_CSV.append(path)
    RESEARCH_INDEX.append({'name':name,'rows_exported':len(frame),
      'rows_total':len(ADVANCED_SHAP_LOCAL) if name=='34_shap_local_sample' else len(frame),
      'cols':len(frame.columns),'sha256':digest(path)})
progress('ARCHIVE_RESEARCH',3,4,f'{len(SCIENTIFIC_CSV)} научных таблиц')

# Compact Excel workbook: all relevant summary tables, not large local-explanation sheets.
from openpyxl import Workbook as _Wb
from openpyxl.styles import Font as _Font,PatternFill as _Fill,Alignment as _Align
import numpy as _np
lite=RS/'ML03_RESEARCH_SUMMARY.xlsx'
w=_Wb(write_only=True)
for name,frame in RESEARCH_DATA.items():
    if frame is None or not isinstance(frame,pd.DataFrame):continue
    label=name[:31];ws=w.create_sheet(label)
    # Excel supports primitive scalar values; JSON-serialize nested dictionaries.
    def safe(v):
        if v is None:return None
        if isinstance(v,(_np.integer,)):return int(v)
        if isinstance(v,(_np.floating,)):v=float(v)
        if isinstance(v,float):return v if _np.isfinite(v) else None
        if isinstance(v,(dict,list,tuple,set)):return _json.dumps(v,ensure_ascii=False,default=str)
        if isinstance(v,(pd.Timestamp,_dt.datetime,_dt.date)):return v.isoformat()
        try:
            if pd.isna(v):return None
        except Exception:pass
        if isinstance(v,str) and v.startswith(('=','+','-','@')):return "'"+v
        return str(v)[:32767] if not isinstance(v,(str,int,bool)) else v
    from openpyxl.cell import WriteOnlyCell as _Cell
    hdr=[]
    for col in frame.columns:
        cell=_Cell(ws,value=str(col));cell.font=_Font(name='Aptos',size=10,color='FFFFFF',bold=True)
        cell.fill=_Fill('solid',fgColor='17324D');cell.alignment=_Align(wrap_text=True)
        hdr.append(cell)
    ws.append(hdr)
    for row in frame.itertuples(index=False,name=None):ws.append([safe(v) for v in row])
w.save(lite)
from openpyxl import load_workbook as _load
verify=_load(lite,read_only=True,data_only=True)
try:
    ensure(len(verify.sheetnames)==len([f for f in RESEARCH_DATA.values() if isinstance(f,pd.DataFrame)]),'LITE_XLSX_SHEET_COUNT')
    for item in RESEARCH_INDEX:
        r=verify[item['name'][:31]].iter_rows(min_row=1,max_row=1,values_only=True)
        ensure(next(r,None) is not None,'LITE_XLSX_HEADER_MISSING_'+item['name'])
finally:verify.close()

# 2. Publish heavyweight `ML3_EVIDENCE.zip` as private GitHub Release asset.
# Remote repository reference is attached under stage2/ml03_factors/runs; main branch remains untouched.
REMOTE_PUBLISH={'status':'NOT_STARTED','repository':CFG['REPO'],
 'scientific_status':'PASS' if scientific_ready else 'PARTIAL',
 'technical_archive_name':ZIP.name,'technical_archive_bytes':ZIP.stat().st_size,
 'technical_archive_sha256':digest(ZIP),'verification':'not_started'}

def _publish_heavy_evidence():
    if not CFG.get('GITHUB_TECHNICAL_PUBLISH',True):
        return {**REMOTE_PUBLISH,'status':'DISABLED'}
    ensure(CFG['MODE']=='FULL','REMOTE_PUBLICATION_REQUIRES_FULL_MODE')
    ensure(bool(TOKEN),'GITHUB_TOKEN_NOT_AVAILABLE')
    # Immutable source ref controls Stage I retrieval; Git branch is based on current main.
    head=api('GET','/git/ref/heads/main').json()['object']['sha']
    stamp=_dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    branch='ml03-results/v403-'+stamp;tag='ml03-v403-results-'+stamp
    segment=CFG['SEGMENT'].strip('/')+'/runs/'+stamp
    progress('ARCHIVE_TECH',1,5,'GitHub: создание изолированной ветки')
    api('POST','/git/refs',json={'ref':'refs/heads/'+branch,'sha':head},expected=(201,))
    create_or_update_git_file(segment+'/'+UNIFIED_IPYNB.name,UNIFIED_IPYNB,branch)
    create_or_update_git_file(segment+'/run_manifest.json',MANIFEST,branch)
    create_or_update_git_file(segment+'/source_parity.json',UNIFIED_PARITY,branch)
    for p in sorted(OUT.glob('*')):
        if p.is_file() and p.suffix in ('.csv','.json','.png','.md'):
            create_or_update_git_file(segment+'/causal/'+p.name,p,branch)
    progress('ARCHIVE_TECH',2,5,'GitHub: ноутбук, методика и реестр')
    body=('ML-03 V4.03 — reproducible computed technical evidence. '
          'Scientific status: '+('PASS' if scientific_ready else 'PARTIAL')+
          '. Parquet/models are research evidence, NOT approved scientific claims. '
          'SHA256: '+digest(ZIP))
    release=api('POST','/releases',json={'tag_name':tag,'target_commitish':branch,
      'name':'ML-03 V4.03 · '+stamp,'body':body,'draft':True,
      'prerelease':not scientific_ready},expected=(201,)).json()
    # Stream original already-tested ZIP; do not duplicate locally or base64 encode a giant file.
    with open(ZIP,'rb') as stream:
        response=SESSION.post('https://uploads.github.com/repos/'+CFG['REPO']+'/releases/'+str(release['id'])+'/assets',
         params={'name':'ML03_V4_03_TECHNICAL_EVIDENCE_'+stamp+'.zip'},data=stream,
         headers={'Content-Type':'application/zip','Accept':'application/vnd.github+json'},timeout=(30,1800))
    ensure(response.status_code==201,'GITHUB_TECH_ASSET_UPLOAD_FAILED_'+str(response.status_code))
    asset=response.json()
    ensure(asset.get('state') in ('uploaded',None),'GITHUB_ASSET_NOT_UPLOADED')
    ensure(int(asset.get('size',-1))==ZIP.stat().st_size,'GITHUB_ASSET_BYTES_MISMATCH')
    progress('ARCHIVE_TECH',3,5,'GitHub: технический архив загружен')
    meta=api('GET','/releases/assets/'+str(asset['id'])).json()
    ensure(int(meta.get('size',-1))==ZIP.stat().st_size,'GITHUB_ASSET_READBACK_SIZE_MISMATCH')
    digest_remote=str(meta.get('digest') or asset.get('digest') or '')
    if digest_remote.startswith('sha256:'):
        ensure(digest_remote.partition(':')[2].lower()==digest(ZIP),'GITHUB_ASSET_SHA_READBACK_MISMATCH')
        strength='SHA256_FROM_GITHUB'
    else:
        verify_release_asset(asset['id'],digest(ZIP))
        strength='SHA256_INDEPENDENT_RELEASE_READBACK'
    # Record remote object and SHA under repository segment without inserting 250MB into Git history.
    index={**REMOTE_PUBLISH,'status':'ASSET_VERIFIED','branch':branch,'tag':tag,
           'release_url':release['html_url'],'asset_id':asset['id'],
           'asset_url':asset.get('browser_download_url'),'verification':strength,
           'scientific_ready':bool(scientific_ready),'source_sha':source_sha}
    stampfile=RS/'github_evidence_index.json'
    stampfile.write_text(_json.dumps(index,ensure_ascii=False,indent=2),encoding='utf-8')
    create_or_update_git_file(segment+'/github_evidence_index.json',stampfile,branch)
    verify_github_file(segment+'/github_evidence_index.json',branch,stampfile)
    progress('ARCHIVE_TECH',4,5,'GitHub: реестр и удалённая проверка')
    published=api('PATCH','/releases/'+str(release['id']),json={'draft':False},expected=(200,)).json()
    ensure(published.get('draft') is False,'GITHUB_RELEASE_STILL_DRAFT')
    index['status']='PUBLISHED';index['release_url']=published['html_url']
    index['git_segment_url']='https://github.com/'+CFG['REPO']+'/tree/'+branch+'/'+segment
    progress('ARCHIVE_TECH',5,5,'GitHub: публикация завершена')
    return index

try:
    REMOTE_PUBLISH=_publish_heavy_evidence()
    mark('GITHUB_TECH_STORAGE','PASS' if REMOTE_PUBLISH['status']=='PUBLISHED' else 'NOT_PUBLISHED',
         REMOTE_PUBLISH.get('release_url','Technical publication not enabled'))
except Exception as ex:
    REMOTE_PUBLISH={**REMOTE_PUBLISH,'status':'UPLOAD_FAILED','error':type(ex).__name__+': '+str(ex)[:350],
                    'recovery':'Technical source files remain in Colab WORK; no scientific PASS is implied'}
    mark('GITHUB_TECH_STORAGE','UPLOAD_FAILED',REMOTE_PUBLISH['error'])
    progress('ARCHIVE_TECH',1,1,'GitHub: ошибка, технические файлы остались в Colab')

# 3. Create exactly one download archive with curated scientific products and truthful publication status.
remote_record=RS/'GITHUB_PUBLICATION_STATUS.json'
remote_record.write_text(_json.dumps(REMOTE_PUBLISH,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
science_data={
 'version':CFG['VERSION'],'source_repo':CFG['REPO'],'source_commit_sha':source_sha,
 'scientific_ready':bool(scientific_ready),'full_science_gate':STATUS.get('FULL_SCIENCE_GATE',{}),
 'municipalities':int(base.entity_id.nunique()),'municipality_years':len(PANEL),
 'prediction_rows_in_remote_evidence':len(PREDICTIONS),'model_metric_rows':len(METRICS),
 'paired_comparisons':len(ABLA),'ensemble_models':len(ADVANCED_ENSEMBLES),
 'shap_global_rows':len(ADVANCED_SHAP),'shap_local_total':len(ADVANCED_SHAP_LOCAL),
 'shap_local_in_download':min(len(ADVANCED_SHAP_LOCAL),int(CFG['RESEARCH_LOCAL_SHAP_SAMPLE'])),
 'publication':REMOTE_PUBLISH,'table_index':RESEARCH_INDEX,
 'causal_inference_status':INFERENCE_STATUS,'causal_gates':QA,
 'notes':'Scientific summaries only; full test predictions and local SHAP in private GitHub evidence release.'}
science_manifest=RS/'SCIENCE_MANIFEST.json'
science_manifest.write_text(_json.dumps(science_data,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
readme=RS/'README_НАУЧНЫЕ_РЕЗУЛЬТАТЫ.md'
readme.write_text('# ML-03 V4.03 — материалы для научного анализа\n\n'
 'Содержит компактную книгу Excel, полные итоговые статистические таблицы CSV, объяснения SHAP (агрегаты), '
 'визуализации и независимые QA/причинные протоколы.\n\n'
 'Полные наблюдения, прогнозы, локальные SHAP и сохранённые модели **не входят** в скачиваемый архив; '
 'они находятся в техническом Evidence GitHub Release (адрес в `GITHUB_PUBLICATION_STATUS.json`).\n'
 'Если статус `UPLOAD_FAILED`, публикация не прошла: технические данные остались в Colab, их нельзя считать загруженными в GitHub.\n'
 'Научный PASS нельзя выводить из успешной упаковки.\n',encoding='utf-8')
FAST_ZIP=WORK/'ML03_V4_03_RESEARCH_FAST.zip'
figs=sorted(p for p in (WORK/'charts').glob('*.png') if p.is_file())
causal=sorted(p for p in OUT.rglob('*') if p.is_file() and p.suffix.lower() in ('.csv','.json','.png','.md'))
with _zf.ZipFile(FAST_ZIP,'w',compression=_zf.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    z.write(lite,'tables/'+lite.name)
    for p in SCIENTIFIC_CSV:z.write(p,'tables/csv/'+p.name)
    for p in figs:z.write(p,'charts/'+p.name)
    for p in causal:z.write(p,'causal/'+p.relative_to(OUT).as_posix())
    z.write(UNIFIED_IPYNB,'notebook/'+UNIFIED_IPYNB.name)
    for qc in sorted((WORK/'checkpoints').glob('*.json')) if (WORK/'checkpoints').exists() else []:
        z.write(qc,'checkpoints/'+qc.name)
    for p in (remote_record,science_manifest,readme,UNIFIED_PARITY):z.write(p,p.name)
ensure(FAST_ZIP.stat().st_size<CFG['RESEARCH_ARCHIVE_MAX_MB']*1024*1024,
       'SCIENTIFIC_ZIP_EXCEEDS_FAST_LIMIT: '+str(FAST_ZIP.stat().st_size))
with _zf.ZipFile(FAST_ZIP) as z:
    required={'tables/'+lite.name,'SCIENCE_MANIFEST.json','GITHUB_PUBLICATION_STATUS.json',readme.name}
    ensure(required.issubset(z.namelist()),'SCIENCE_ZIP_REQUIRED_FILES_MISSING')
    ensure(z.testzip() is None,'SCIENCE_ZIP_CRC_FAIL')
    ensure(len([x for x in z.namelist() if x.startswith('charts/') and x.endswith('.png')])==len(figs),
           'SCIENCE_ZIP_CHARTS_INCOMPLETE')
report={'version':CFG['VERSION'],'fast_archive':FAST_ZIP.name,'bytes':FAST_ZIP.stat().st_size,
        'sha256':digest(FAST_ZIP),'tables':len(SCIENTIFIC_CSV),'charts':len(figs),
        'technical_remote':REMOTE_PUBLISH,'scientific_ready':bool(scientific_ready)}
(WORK/'ML03_DELIVERY_REPORT.json').write_text(_json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
progress('DELIVERY',1,1,'Один научный архив проверен, технический сохранён удалённо или статус ошибки зафиксирован')
ensure(_PROGRESS_BAR.n==1000,'UNIFIED_PROGRESS_NOT_COMPLETE')
_PROGRESS_BAR.close()
print('ML-03 V4.03 | Научный архив:',FAST_ZIP.name,
      '| размер, МиБ:',round(FAST_ZIP.stat().st_size/2**20,2),
      '| GitHub технический:',REMOTE_PUBLISH['status'])
if REMOTE_PUBLISH.get('release_url'):print('GitHub:',REMOTE_PUBLISH['release_url'])
if CFG.get('AUTO_COLAB_DOWNLOAD') and CFG['MODE']=='FULL':
    try:
        from google.colab import files as _files
        _files.download(str(FAST_ZIP))  # ONLY small curated scientific archive
    except ImportError:quiet_print('Not a Google Colab runtime; archive remains at',FAST_ZIP)


---
**Файлы исследования после полного выполнения:** `ML03_V4_03_RESEARCH_FAST.zip`, `ML3_EVIDENCE.zip`, `ML03_V4_03_FINAL_FULL_RESEARCH_4_STAGES.ipynb`, `run_manifest.json`, `source_parity_report.json` и технический журнал. Необходимые для полного воспроизведения промежуточные таблицы, модели, проверки и предсказания содержатся в техническом архиве; компактный архив сохраняет расчетные своды, графики и протоколы. Состояния `NOT_FEASIBLE`, `PARTIAL` и `NOT_IDENTIFIED` сохраняются как результаты проверки применимости данных, а не заполняются гипотетическими значениями.